# ShortVid-Bench · chạy lại video lỗi với bản sửa `lm_head` + audio SDPA, đo tách thời gian

Lần chạy trước (`svb_1_full`, `svb_2_istm`, `svb_3_av_aligner`, 500 câu) có **18 video lỗi CUDA OOM**:

| nhóm | số video | nơi lỗi | sửa trong notebook này |
|---|---|---|---|
| chỉ `full` | 12 | `lm_head` tính logits cho cả chuỗi khi prefill (≈ 11k token × 152k từ vựng × fp16 ≈ 3.2 GB) | `fix_lm_head`: chỉ chiếu token cuối; greedy `generate()` chỉ đọc `logits[:, -1]` nên đáp án không đổi |
| cả 3 method | 6 | SDPA của audio encoder dựng mask đặc `[1, S, S]` | `fix_audio_sdpa`: attention theo từng đoạn `cu_seqlens`, cùng kết quả toán học (giống bản sửa vision); điểm audio của OmniZip tính y như cũ |

Notebook chạy cả 3 method (`full`, `istm_r45` = ISTM theo paper, `omnizip_r45` = AV_Aligner của fork) trên
18 video lỗi + 24 video **đối chứng** đã chạy được (trải đều theo độ dài chuỗi). Nhóm đối chứng kiểm tra bản sửa
không đổi đáp án, và cho số đo thời gian ở video bình thường.

**Đo thêm:** thời gian từng giai đoạn (đọc video, processor, chép lên GPU, vision encoder, audio encoder, OmniZip,
LLM prefill, `lm_head`, decode), TTFT, VRAM đỉnh từng giai đoạn, số token thật vào LLM; ảnh token video được giữ.
Cuối notebook: accuracy 500 câu **sau khi thay** kết quả của các video chạy lại, và so ghép cặp.

Code commit `dadc007` nhúng sẵn; kết quả lần trước nhúng sẵn (mục 1b). Settings: **GPU T4 x2**, **Internet On**,
nên có secret `HF_TOKEN`.

In [ ]:
# ===== Cấu hình =====
import glob, json, os

MODEL_ID = "Qwen/Qwen2.5-Omni-3B"
METHODS = {"full": {"wrapper": "none"}, "istm_r45": {"wrapper": "omnizip", "video_compressor": "istm", "rho_audio": 0.3, "rho_video": 0.6, "g": 3, "contextual_ratio": 0.05}, "omnizip_r45": {"wrapper": "omnizip", "video_compressor": "av_aligner", "rho_audio": 0.3, "rho_video": 0.6, "g": 3, "contextual_ratio": 0.05}}
BASELINE = "full"
FIX_LM_HEAD = True                # chỉ tính logits của token cuối khi prefill
FIX_AUDIO_SDPA = True             # attention của audio encoder theo từng đoạn cu_seqlens
PROFILE = True                    # đo thời gian + VRAM từng giai đoạn (có synchronize, chậm hơn vài %)

# 18 video lỗi ở lần trước (lỗi ở ít nhất một method)
FAILED_VIDEOS = ["0avoXj7I3yA.mp4", "2Yp6UqQPQNE.mp4", "4tyX4O4it5U.mp4", "8OqmJB8oGtM.mp4", "BojPeImI2QE.mp4", "FkKh29UxzEc.mp4", "GSLBOuwOdRE.mp4", "H3iTVprK7wU.mp4", "KwhBtGE-mDc.mp4", "O61cCbK3hMY.mp4", "OY-uPQ4oLKM.mp4", "WgWdcSg5_to.mp4", "_GTQj7_xyME.mp4", "_uDlP4vW190.mp4", "hVG1xDJwMbY.mp4", "iIDhWvE6jR0.mp4", "olx5oFJbivo.mp4", "uGoKOpZOSUg.mp4"]
# video đối chứng: chạy được ở cả 3 method lần trước, trải đều theo số token đầu vào; [] để bỏ
CONTROL_VIDEOS = ["pAEYLYBhnSk.mp4", "81SdVr4Cqb8.mp4", "wskyXFw6C4I.mp4", "pRcxjOCF3iM.mp4", "Z2GuiatpQUw.mp4", "FPBSzdA1oKU.mp4", "qHakVESEOec.mp4", "7lh_sO9EWkE.mp4", "zCwz5h0TPmE.mp4", "iIZiJR-MHnM.mp4", "Syz-bdm5oEE.mp4", "hIWZDSR_uns.mp4", "kS6aRrdqI7M.mp4", "XOUUsvMm1PM.mp4", "toQnAXHr5uo.mp4", "OYIX_zxxS9Q.mp4", "i2fOL059y1A.mp4", "2Rl9dPOJk2w.mp4", "B7g9hmHAUzo.mp4", "6X1IEKFL33Y.mp4", "ySiKjxAnE4I.mp4", "5yssh2Aqtps.mp4", "_Y7Q2syNYxY.mp4", "UaBfM-tONmI.mp4"]
VIZ_VIDEOS = ["kS6aRrdqI7M.mp4", "2Yp6UqQPQNE.mp4"]  # lưu 4 frame + mặt nạ token được giữ để vẽ

# Phải giống lần trước để so được
MAX_FRAMES = 128
FPS = 2.0
VIDEO_MAX_PIXELS = 128 * 28 * 28
MAX_NEW_TOKENS = 100
DTYPE = "float16"
ATTN_IMPLEMENTATION = "sdpa"
SEED = 0
PREV_TAG = "svb_omni3B_f128_px128_mini500"
RUN_TAG = "svbfix_omni3B_f128_px128"
CODE_COMMIT = "dadc007"

WORK = "/kaggle/working"
TEMP = "/kaggle/temp"
CODE_ROOT = "/kaggle/temp/code"
RESULTS_DIR = f"{WORK}/results"
LOG_DIR = f"{WORK}/logs"
DATA_DIR = f"{TEMP}/shortvid"
PREV_JSON = f"{WORK}/prev_records.json"
os.environ["HF_HOME"] = f"{TEMP}/hf"
for d in (TEMP, RESULTS_DIR, LOG_DIR, DATA_DIR, *(f"{CODE_ROOT}/{os.path.dirname(p)}" for p in ['omnizip/omnizip_core.py', 'omnizip/av_malign.py', 'omnizip/modeling_qwen2_5_omni.py', 'qwen-omni-utils/pyproject.toml', 'qwen-omni-utils/src/qwen_omni_utils/__init__.py', 'qwen-omni-utils/src/qwen_omni_utils/v2_5/__init__.py', 'qwen-omni-utils/src/qwen_omni_utils/v2_5/audio_process.py', 'qwen-omni-utils/src/qwen_omni_utils/v2_5/vision_process.py'])):
    os.makedirs(d, exist_ok=True)
RUN_VIDEOS = FAILED_VIDEOS + [v for v in CONTROL_VIDEOS if v not in FAILED_VIDEOS]
print(len(FAILED_VIDEOS), "video lỗi +", len(CONTROL_VIDEOS), "đối chứng ×", len(METHODS), "method")

## 1. Code OmniZip (commit `dadc007`, nhúng sẵn) + cài thư viện

In [ ]:
%%writefile /kaggle/temp/code/omnizip/omnizip_core.py
import torch
from typing import Dict, List, Tuple, Optional

from .av_malign import AV_Aligner

def omnizip_audio_attn(
    audio_feature: torch.Tensor,
    video_feature: Optional[torch.Tensor],
    attn_logits: torch.Tensor,
    merging_ratio: float = 0.5,
    contextual_ratio: float = 0.03,
    g: int = 3,
) -> Tuple[torch.Tensor, Dict[int, List[int]]]:
    device = attn_logits.device
    def to_seq(x):
        if x is None:
            return None
        return x.mean(0) if x.dim() == 3 else x

    a = to_seq(audio_feature).to(device)
    v = to_seq(video_feature).to(device) if video_feature is not None else None
    N = a.size(0)
    if N == 0:
        return torch.zeros(0, dtype=torch.bool, device=device), {}

    keep_mask = torch.zeros(N, dtype=torch.bool, device=device)
    dominant_num = int(max(0, min(N, round((1.0 - merging_ratio) * N))))
    if dominant_num > 0:
        _, topk = torch.topk(attn_logits, dominant_num)
        keep_mask[topk] = True

    all_idx = torch.arange(N, device=device)
    remaining = all_idx[~keep_mask]
    contextual_num = int(max(0, round(contextual_ratio * N)))
    merge_plan: Dict[int, List[int]] = {}

    if remaining.numel() > 0 and contextual_num > 0 and g > 0:
        contextual_num = min(contextual_num, remaining.numel())
        step = max(1, remaining.numel() // contextual_num)
        init_pos = torch.arange(0, remaining.numel(), step, device=device)[:contextual_num]
        anchors = remaining[init_pos]
        keep_mask[anchors] = True

        rem_local_mask = torch.ones(remaining.numel(), dtype=torch.bool, device=device)
        rem_local_mask[init_pos] = False
        pool_global = remaining[torch.arange(remaining.numel(), device=device)[rem_local_mask]]

        if pool_global.numel() > 0 and g > 0:
            a_norm = a / (a.norm(dim=-1, keepdim=True) + 1e-6)
            sim_aa = a_norm[pool_global] @ a_norm[anchors].T
            assign = sim_aa.argmax(dim=1)
            
            if v is not None and v.numel() > 0:
                v_norm = v / (v.norm(dim=-1, keepdim=True) + 1e-6)
                sim_av = a_norm[pool_global] @ v_norm.T
                scores = sim_av.max(dim=1).values
            else:
                scores = sim_aa.max(dim=1).values
                
            C = anchors.numel()
            for c in range(C):
                mask_c = (assign == c)
                cand = pool_global[mask_c]
                if cand.numel() == 0:
                    merge_plan[int(anchors[c].item())] = []
                    continue
                scores_c = scores[mask_c]
                topg = min(g, cand.numel())
                _, sel = torch.topk(scores_c, topg, largest=True)
                chosen = cand[sel].tolist()
                merge_plan[int(anchors[c].item())] = chosen

    return keep_mask, merge_plan


_AV_ALIGNERS: Dict[Tuple, AV_Aligner] = {}


def get_av_aligner(
    dim: int, device, dtype, num_heads: int = 1, num_clusters: int = 5, attn_layer: Optional[torch.nn.Module] = None
) -> AV_Aligner:
    key = (dim, str(device), dtype, num_heads, num_clusters, id(attn_layer))
    if key not in _AV_ALIGNERS:
        aligner = AV_Aligner(dim, num_heads=num_heads, num_clusters=num_clusters).to(device=device, dtype=dtype).eval()
        if attn_layer is not None:
            # Share the LLM layer's frozen q/k/v instead of the randomly initialised ones
            aligner.load_attention(attn_layer)
        _AV_ALIGNERS[key] = aligner
    return _AV_ALIGNERS[key]


def omnizip_av_align(
    video_feature: torch.Tensor,
    audio_feature: torch.Tensor,
    num_tokens_per_frame: int,
    merging_ratio: float,
    aligner: AV_Aligner,
):
    num_frames = video_feature.shape[0] // num_tokens_per_frame
    mask = torch.zeros(video_feature.shape[0], dtype=torch.bool, device=video_feature.device)
    num_keep = int(num_tokens_per_frame * (1.0 - merging_ratio))

    if num_frames == 0 or num_keep >= num_tokens_per_frame:
        mask[: num_frames * num_tokens_per_frame] = True
        return mask

    frames = video_feature[: num_frames * num_tokens_per_frame].reshape(num_frames, num_tokens_per_frame, -1)
    aud = audio_feature.to(frames.dtype).unsqueeze(0).expand(num_frames, -1, -1)

    with torch.no_grad():
        scores = aligner.token_scores(aud, frames)  # [T, tokens_per_frame]

    keep_idx = scores.topk(num_keep, dim=1).indices
    offsets = torch.arange(num_frames, device=mask.device).unsqueeze(1) * num_tokens_per_frame
    mask[(keep_idx + offsets).reshape(-1)] = True
    return mask


def omnizip_istm(video_feature: torch.Tensor, num_tokens_per_frame: int, merging_ratio: float):
    num_frames = video_feature.shape[0] // num_tokens_per_frame
    mask = torch.zeros(video_feature.shape[0], dtype=torch.bool, device=video_feature.device)

    def dpcknn(tokens, keep_rate=0.5, k=5):
        N = tokens.shape[0]
        num_keep = int(N * keep_rate)
        if num_keep >= N:
            return torch.arange(N, device=tokens.device)
        k = min(k, N - 1)
        with torch.no_grad():
            normed = torch.nn.functional.normalize(tokens, dim=1)
            dist = 1 - torch.mm(normed, normed.T)
            dist.fill_diagonal_(float('inf'))

            # ρ: local density via KNN
            knn_dist, _ = torch.topk(dist, k, dim=1, largest=False)
            rho = torch.exp(-knn_dist.mean(dim=1))

            # δ: distance to nearest higher-density point
            higher_mask = rho.unsqueeze(0) > rho.unsqueeze(1)  # [i,j]: rho[j] > rho[i]
            delta_dist = dist.clone()
            delta_dist[~higher_mask] = float('inf')
            delta = delta_dist.min(dim=1).values

            # Highest density point: δ = max distance to any other point
            is_peak = delta.isinf()
            if is_peak.any():
                finite_dist = dist.clone()
                finite_dist.fill_diagonal_(0)
                delta[is_peak] = finite_dist[is_peak].max(dim=1).values

            score = delta * rho
            selected = torch.topk(score, num_keep).indices
        return selected

    keep_ratio = 1.0 - merging_ratio

    for t in range(num_frames):
        start_idx = t * num_tokens_per_frame
        end_idx = (t + 1) * num_tokens_per_frame
        tokens = video_feature[start_idx:end_idx]

        if t % 2 == 0:
            spatial_keep_rate = keep_ratio
            num_keep = int(num_tokens_per_frame * spatial_keep_rate)
            if num_keep < num_tokens_per_frame:
                keep_idx = dpcknn(tokens, keep_rate=spatial_keep_rate, k=5)
            else:
                keep_idx = torch.arange(num_tokens_per_frame, device=tokens.device)
            mask[start_idx + keep_idx] = True
        else:
            prev_tokens = video_feature[(t - 1) * num_tokens_per_frame : t * num_tokens_per_frame]
            prev_norm = torch.nn.functional.normalize(prev_tokens, p=2, dim=1)
            curr_norm = torch.nn.functional.normalize(tokens, p=2, dim=1)
            
            similarity = torch.nn.functional.cosine_similarity(curr_norm, prev_norm, dim=1)
            num_keep = int(num_tokens_per_frame * keep_ratio)
            
            if num_keep < num_tokens_per_frame:
                keep_idx = similarity.topk(num_keep, largest=False).indices
            else:
                keep_idx = torch.arange(num_tokens_per_frame, device=tokens.device)
            mask[start_idx + keep_idx] = True

    return mask

def omnizip(
    input_embeds: torch.Tensor,
    attn_logits: torch.Tensor,
    input_ids: torch.Tensor,
    audio_token_id: int,
    video_token_id: int,
    video_grid_thw: Tuple[int, int, int],
    audio_tokens_per_sec: int = 25,
    merging_ratio_audio: float = 0.5,
    merging_ratio_v: float = 0.5,
    contextual_ratio: float = 0.05,
    g: int = 3,
    attn_layer: Optional[torch.nn.Module] = None,
):
    device = input_embeds.device
    is_batched = input_embeds.dim() == 3
    if is_batched:
        B, L, D = input_embeds.shape
        flat_embeds = input_embeds.reshape(-1, D)
        flat_ids = input_ids.reshape(-1)
    else:
        L, D = input_embeds.shape
        flat_embeds = input_embeds
        flat_ids = input_ids

    video_token_mask = (flat_ids == video_token_id).to(device)
    audio_token_mask = (flat_ids == audio_token_id).to(device)

    video_indices = torch.nonzero(video_token_mask, as_tuple=True)[0]
    audio_indices = torch.nonzero(audio_token_mask, as_tuple=True)[0]

    video_feature = flat_embeds[video_indices]
    audio_feature = flat_embeds[audio_indices]

    T = video_grid_thw[0, 0].item()
    W = video_grid_thw[0, 1].item() // 2
    H = video_grid_thw[0, 2].item() // 2
    video_token_per_frame = H * W
    num_video_tokens = T * video_token_per_frame
    
    audio_mask, merge_plan = omnizip_audio_attn(
        audio_feature=audio_feature,
        video_feature=video_feature,
        attn_logits=attn_logits,
        merging_ratio=merging_ratio_audio,
        contextual_ratio=contextual_ratio,
        g=g,
    )

    if g > 0 and len(merge_plan) > 0:
        a_norm = audio_feature / (audio_feature.norm(dim=-1, keepdim=True) + 1e-6)
        v_norm = (
            video_feature / (video_feature.norm(dim=-1, keepdim=True) + 1e-6)
            if video_feature.numel() > 0 else None
        )
        for anchor_rel_idx, merge_rel_list in merge_plan.items():
            if not merge_rel_list:
                continue
            merge_rel = torch.tensor(merge_rel_list, device=device, dtype=torch.long)
            if v_norm is not None:
                scores = (a_norm[merge_rel] @ v_norm.T).max(dim=1).values
            else:
                scores = (a_norm[merge_rel] @ a_norm[anchor_rel_idx].unsqueeze(-1)).squeeze(-1)
            w = torch.softmax(scores, dim=0)
            anchor_vec = audio_feature[anchor_rel_idx]
            merged_vec = (audio_feature[merge_rel] * w.unsqueeze(-1)).sum(dim=0)
            new_anchor = (anchor_vec + merged_vec) / (1.0 + w.sum())
            anchor_global_idx = audio_indices[anchor_rel_idx]
            flat_embeds[anchor_global_idx] = new_anchor

    # Detect time chunks from input_ids interleaving pattern
    def detect_token_chunks(indices):
        """Split sorted token indices into consecutive runs (time chunks)."""
        if indices.numel() == 0:
            return []
        diffs = indices[1:] - indices[:-1]
        boundaries = (diffs > 1).nonzero(as_tuple=True)[0] + 1
        chunks = []
        prev = 0
        for b in boundaries.cpu().tolist():
            chunks.append((prev, b))
            prev = b
        chunks.append((prev, indices.numel()))
        return chunks

    video_chunks = detect_token_chunks(video_indices)
    audio_chunks = detect_token_chunks(audio_indices)

    num_video_chunks = len(video_chunks)
    num_audio_chunks = len(audio_chunks)
    num_guided = min(num_video_chunks, num_audio_chunks)

    # Audio retention for guided groups (where audio exists)
    audio_group_retention = []
    for i in range(num_guided):
        a_start, a_end = audio_chunks[i]
        group_mask = audio_mask[a_start:a_end]
        ratio = group_mask.float().mean().item() if group_mask.numel() > 0 else 0.0
        audio_group_retention.append(ratio)

    min_ratio, max_ratio = 0.35, 0.75
    base_vs_guided = [
        max(min_ratio, min(max_ratio, max_ratio + (min_ratio - max_ratio) * ret))
        for ret in audio_group_retention
    ]

    # Budget normalization: guided groups share the guided portion of the budget
    guided_video_tokens = sum(video_chunks[i][1] - video_chunks[i][0] for i in range(num_guided))
    unguided_video_tokens = sum(
        video_chunks[i][1] - video_chunks[i][0] for i in range(num_guided, num_video_chunks)
    )
    target_discard_tokens = merging_ratio_v * num_video_tokens
    unguided_discard = merging_ratio_v * unguided_video_tokens
    guided_target_discard = target_discard_tokens - unguided_discard

    adjusted_vs_guided = base_vs_guided.copy()
    if num_guided > 2 and guided_video_tokens > 0:
        guided_group_sizes = [video_chunks[i][1] - video_chunks[i][0] for i in range(num_guided)]
        base_guided_discard = sum(
            base_vs_guided[i] * guided_group_sizes[i] for i in range(num_guided)
        )

        if abs(base_guided_discard - guided_target_discard) > 1e-3 and base_guided_discard > 0:
            min_idx = base_vs_guided.index(min(base_vs_guided))
            max_idx = base_vs_guided.index(max(base_vs_guided))
            min_val, max_val = base_vs_guided[min_idx], base_vs_guided[max_idx]
            other_indices = [i for i in range(num_guided) if i != min_idx and i != max_idx]
            other_discard = sum(base_vs_guided[i] * guided_group_sizes[i] for i in other_indices)

            if other_indices and other_discard > 0:
                fixed_discard = min_val * guided_group_sizes[min_idx] + max_val * guided_group_sizes[max_idx]
                remain_target = guided_target_discard - fixed_discard
                scaling = remain_target / other_discard
                for i in other_indices:
                    v_new = base_vs_guided[i] * scaling
                    adjusted_vs_guided[i] = max(min_ratio, min(max_ratio, v_new))

                actual_discard = sum(
                    adjusted_vs_guided[i] * guided_group_sizes[i] for i in range(num_guided)
                )
                diff = guided_target_discard - actual_discard
                if abs(diff) > 1e-3:
                    total_other_size = sum(guided_group_sizes[i] for i in other_indices)
                    if total_other_size > 0:
                        adj_per_token = diff / total_other_size
                        for i in other_indices:
                            adjusted_vs_guided[i] = max(
                                min_ratio, min(max_ratio, adjusted_vs_guided[i] + adj_per_token)
                            )

    # Apply audio-guided AV_Aligner compression per chunk
    aligner = get_av_aligner(D, device, flat_embeds.dtype, attn_layer=attn_layer)
    video_group_masks = []
    for i in range(num_video_chunks):
        v_start, v_end = video_chunks[i]
        group_feat = video_feature[v_start:v_end]

        num_frames_in_group = group_feat.shape[0] // video_token_per_frame
        if num_frames_in_group == 0:
            video_group_masks.append(torch.ones(group_feat.shape[0], dtype=torch.bool, device=device))
            continue

        if i < num_guided:
            ratio = adjusted_vs_guided[i]
            # Guide with the audio tokens kept in the aligned chunk (all of them if none were kept)
            a_start, a_end = audio_chunks[i]
            group_audio = audio_feature[a_start:a_end]
            kept_audio = group_audio[audio_mask[a_start:a_end]]
            group_audio = kept_audio if kept_audio.shape[0] > 0 else group_audio
        else:
            ratio = merging_ratio_v
            group_audio = audio_feature

        if group_audio.shape[0] == 0:
            group_mask = omnizip_istm(
                group_feat,
                num_tokens_per_frame=video_token_per_frame,
                merging_ratio=ratio,
            )
        else:
            group_mask = omnizip_av_align(
                group_feat,
                group_audio,
                num_tokens_per_frame=video_token_per_frame,
                merging_ratio=ratio,
                aligner=aligner,
            )
        video_group_masks.append(group_mask)

    video_mask = torch.cat(video_group_masks, dim=0) if video_group_masks else torch.ones(0, dtype=torch.bool, device=device)
    global_mask = torch.ones(flat_embeds.size(0), dtype=torch.bool, device=device)

    assert video_mask.shape[0] == video_indices.shape[0]
    assert audio_mask.shape[0] == audio_indices.shape[0]

    global_mask[video_indices] = video_mask
    global_mask[audio_indices] = audio_mask

    if is_batched:
        input_embeds_out = flat_embeds.reshape(B, L, D)
    else:
        input_embeds_out = flat_embeds

    return input_embeds_out, global_mask

In [ ]:
%%writefile /kaggle/temp/code/omnizip/av_malign.py
# ---------------------------------------------------------------
# Copyright (c) 2024. All rights reserved.
#
# Written by Chen Liu
# ---------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.init import trunc_normal_
from einops import rearrange


def norm_fea(fea, norm_flag):
    if norm_flag:
        return F.normalize(fea, p=2, dim=-1)
    return fea

class Cross_modal_interaction(nn.Module):
    def __init__(self,
                 dim,
                 num_heads=1,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop=0.,
                 proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim**-0.5

        self.norm = nn.Identity()
        self.q_proj = nn.Linear(dim, dim, bias=qkv_bias)
        self.k_proj = nn.Linear(dim, dim, bias=qkv_bias)
        self.v_proj = nn.Linear(dim, dim, bias=qkv_bias)
        self.q_norm = nn.Identity()
        self.k_norm = nn.Identity()
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def load_attention(self, layer):
        """
        Share the frozen input norm and q/k/v (plus per-head q/k norms, if any) of an LLM decoder layer.
        RoPE is skipped, so scores ignore token positions.
        """
        attn = layer.self_attn
        self.norm = layer.input_layernorm
        self.q_proj, self.k_proj, self.v_proj = attn.q_proj, attn.k_proj, attn.v_proj
        self.q_norm = getattr(attn, 'q_norm', nn.Identity())
        self.k_norm = getattr(attn, 'k_norm', nn.Identity())
        head_dim = attn.head_dim
        self.num_heads = attn.q_proj.out_features // head_dim
        self.num_kv_heads = attn.k_proj.out_features // head_dim
        self.scale = head_dim**-0.5

    def _split_heads(self, x, proj, head_norm, num_heads):
        x = rearrange(proj(self.norm(x)), 'b n (h c) -> b n h c', h=num_heads)
        x = head_norm(x).transpose(1, 2) # [B, num_heads, n, head_dim]
        # GQA: repeat kv heads to match query heads
        return x.repeat_interleave(self.num_heads // num_heads, dim=1)

    def attn_logits(self, query, key):
        """
        Return:
            scaled attention logits: [B, nheads, n_query, n_key]
        """
        q = self._split_heads(query, self.q_proj, self.q_norm, self.num_heads)
        k = self._split_heads(key, self.k_proj, self.k_norm, self.num_kv_heads)
        return (q @ k.transpose(-2, -1)) * self.scale

    def forward(self, img_fea, aud_fea, aq=True, sigmoid=False, softmax=False):
        """
        Args:
            img_fea: [b, num_token, dim]
            aud_fea: [b, k, dim]
            aq: True: a is query; False: i is query
        Return
            if aq:
                fuse_fea: [b, k, dim]
            else:
                fuse_fea: [b, num_token, dim]
        """
        if len(aud_fea.shape) == 2:
            aud_fea = aud_fea.unsqueeze(1)
        if aq:
            B, N, _ = aud_fea.shape
            query = aud_fea
            key = img_fea
            value = img_fea
        else:
            B, N, _ = img_fea.shape # [B, h*w, dim]
            query = img_fea
            key = aud_fea
            value = aud_fea

        attn = self.attn_logits(query, key) # [B, nheads, h*w, 1]
        v = self._split_heads(value, self.v_proj, nn.Identity(), self.num_kv_heads) # [B, nheads, 1, head_dim]

        if sigmoid:
            attn = torch.sigmoid(attn) # [B, nheads, h*w, 1]
        if softmax:
            attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)

        return x
    
class GumbelSoftmaxClustering(nn.Module):
    def __init__(self, num_clusters, dim, temperature=1.0):
        super(GumbelSoftmaxClustering, self).__init__()
        self.num_clusters = num_clusters
        self.temperature = temperature
        self.cluster_centers = nn.Parameter(torch.randn(num_clusters, dim))  # Learnable cluster centers

    def forward(self, img_fea):
        """
        Args:
            img_fea: [B, H*W, dim] -> image features
        Returns:
            cluster_assignments: [B, H*W, num_clusters] -> soft cluster assignments
            cluster_centers: [B, H*W, dim] -> recomputed cluster centers
        """
        # Compute similarity (dot product) between img_fea and cluster centers
        logits = torch.einsum('bnd,kd->bnk', img_fea, self.cluster_centers)  # [B, H*W, num_clusters]

        # Apply Gumbel Softmax for cluster assignment
        gumbel_samples = F.gumbel_softmax(logits, tau=self.temperature, hard=False, dim=-1)  # [B, H*W, num_clusters]

        cluster_centers = torch.einsum('bnk,kd->bnd', gumbel_samples, self.cluster_centers)  # [B, H*W, dim]
        # cluster_centers = torch.einsum('bnk,bnd->bkd', gumbel_samples, img_fea) # 

        return cluster_centers, gumbel_samples

class SoftKMeansClustering(nn.Module):
    """Parameter-free vMF-mixture EM (spherical soft k-means), optional Sinkhorn balancing."""
    def __init__(self, num_clusters, temperature=0.1, iters=5, sinkhorn_iters=0):
        super(SoftKMeansClustering, self).__init__()
        self.num_clusters = num_clusters
        self.temperature = temperature
        self.iters = iters
        self.sinkhorn_iters = sinkhorn_iters

    def _assign(self, x, mu):
        logits = x @ mu.transpose(1, 2) / self.temperature  # [B, H*W, num_clusters]
        if self.sinkhorn_iters == 0:
            return logits.softmax(dim=-1)
        log_r = logits
        for _ in range(self.sinkhorn_iters):
            log_r = log_r - log_r.logsumexp(dim=1, keepdim=True)  # equal mass per cluster
            log_r = log_r - log_r.logsumexp(dim=2, keepdim=True)  # each token sums to 1
        return log_r.exp()

    def forward(self, img_fea):
        """
        Args:
            img_fea: [B, H*W, dim] -> image features
        Returns:
            cluster_centers: [B, H*W, dim] -> per-token soft-assigned center
            cluster_assignments: [B, H*W, num_clusters] -> soft cluster assignments
        """
        fea = img_fea.float()
        x = F.normalize(fea, dim=-1)
        N = x.shape[1]
        K = min(self.num_clusters, N)
        mu = x[:, torch.linspace(0, N - 1, K, device=x.device).long()]  # deterministic init, [B, K, dim]

        for _ in range(self.iters):
            r = self._assign(x, mu)
            mu = F.normalize(r.transpose(1, 2) @ x, dim=-1)  # M-step on the sphere

        r = self._assign(x, mu)
        # Centers in the raw feature scale so q/k/v projections see the usual magnitudes
        raw_mu = (r.transpose(1, 2) @ fea) / r.sum(dim=1).unsqueeze(-1).clamp_min(1e-6)  # [B, K, dim]
        cluster_centers = r @ raw_mu  # [B, H*W, dim]

        return cluster_centers.to(img_fea.dtype), r.to(img_fea.dtype)

class AV_Aligner(nn.Module):
    def __init__(self, dim, num_heads=1, num_clusters=5):
        super(AV_Aligner, self).__init__()

        self.num_clusters = num_clusters

        self.fc = nn.Linear(dim, 1)
        self.sigmoid = nn.Sigmoid()
        self.layer_norm = nn.LayerNorm(dim)

        self.gumbel_clustering = SoftKMeansClustering(num_clusters)

        self.av_cros_interaction = Cross_modal_interaction(dim=dim, num_heads=num_heads)
        self.va_cros_interaction = Cross_modal_interaction(dim=dim, num_heads=num_heads)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def load_attention(self, layer):
        """
        Reuse the frozen attention of an LLM decoder layer (e.g. thinker.model.layers[0]) for both
        cross-modal interactions. fc has no frozen counterpart, so it is dropped and audio tokens
        are weighted by their attention logits instead (see _pool_audio).
        """
        self.av_cros_interaction.load_attention(layer)
        self.va_cros_interaction.load_attention(layer)
        self.fc = None

    def _pool_audio(self, aud_fea, cluster_centers):
        """
        Args:
            aud_fea: [B, K, dim] -> audio features
            cluster_centers: [B, H*W, dim] -> clustered image features
        Returns:
            weighted_aud_fea: [B, 1, dim] -> audio features pooled by visual relevance
        """
        if self.fc is None:
            # Weight each audio token by its strongest match to the clustered image tokens
            logits = self.av_cros_interaction.attn_logits(aud_fea, cluster_centers) # [B, nheads, k, h*w]
            attention_scores = logits.amax(dim=-1).mean(dim=1).softmax(dim=1).unsqueeze(-1) # [B, k, 1]
        else:
            aud_enhance_fea = self.av_cros_interaction(cluster_centers, aud_fea, aq=True, softmax=True) # [B, k, dim]
            attention_scores = self.fc(aud_enhance_fea).softmax(dim=1) # [B, k, 1]
        return (attention_scores * aud_fea).sum(dim=1, keepdim=True)

    def forward(self, aud_fea, img_fea):
        """
        Args:
            aud_fea: [B, K, dim] -> audio features
            img_fea: [B, H*W, dim] -> image features
        Returns:
            enhanced_img_fea: [B, H*W, dim] -> enhanced image features
        """
        # Step 1: Cluster image features into K clusters
        cluster_centers, _ = self.gumbel_clustering(img_fea) # [B, H*W, dim]: per-token soft-assigned center

        weighted_aud_fea = self._pool_audio(aud_fea, cluster_centers) # [B, 1, dim]

        # Step 6: Enhance image features based on the audio-visual similarity map
        enhanced_img_fea = self.va_cros_interaction(img_fea, weighted_aud_fea, aq=False, sigmoid=True)  

        # enhanced_img_fea = self.layer_norm(enhanced_img_fea + img_fea)
        enhanced_img_fea = enhanced_img_fea + img_fea

        return enhanced_img_fea

    def token_scores(self, aud_fea, img_fea):
        """
        Args:
            aud_fea: [B, K, dim] -> audio features
            img_fea: [B, H*W, dim] -> image features
        Returns:
            scores: [B, H*W] -> audio-visual relevance of each image token
        """
        cluster_centers, _ = self.gumbel_clustering(img_fea)

        weighted_aud_fea = self._pool_audio(aud_fea, cluster_centers) # [B, 1, dim]

        # Same sigmoid attention that va_cros_interaction uses to enhance img_fea
        attn = torch.sigmoid(self.va_cros_interaction.attn_logits(img_fea, weighted_aud_fea)) # [B, nheads, h*w, 1]

        return attn.mean(dim=1).squeeze(-1)

In [ ]:
%%writefile /kaggle/temp/code/omnizip/modeling_qwen2_5_omni.py
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
#           This file was automatically generated from src/transformers/models/qwen2_5_omni/modular_qwen2_5_omni.py.
#               Do NOT edit this file manually as any edits will be overwritten by the generation of
#             the file from the modular. If any change should be done, please apply the change to the
#                          modular_qwen2_5_omni.py file directly. One of our CI enforces this.
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
# coding=utf-8
# Copyright 2025 The Qwen team, Alibaba Group and the HuggingFace Inc. team. All rights reserved.
#
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import math
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import Parameter

from transformers.activations import ACT2FN
from transformers.cache_utils import Cache, DynamicCache, SlidingWindowCache, StaticCache
from transformers.generation import GenerationMixin
from transformers.modeling_attn_mask_utils import AttentionMaskConverter
from transformers.modeling_flash_attention_utils import flash_attn_supports_top_left_mask, is_flash_attn_available
from transformers.modeling_outputs import BaseModelOutput, BaseModelOutputWithPast, ModelOutput
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS, dynamic_rope_update
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS, PreTrainedModel
from transformers.utils import (
    auto_docstring,
    check_torch_load_is_safe,
    is_flash_attn_2_available,
    is_flash_attn_greater_or_equal_2_10,
    is_torch_flex_attn_available,
    logging,
)
from transformers.utils.hub import cached_file
from transformers.models.qwen2_5_omni.configuration_qwen2_5_omni import (
    Qwen2_5OmniAudioEncoderConfig,
    Qwen2_5OmniBigVGANConfig,
    Qwen2_5OmniConfig,
    Qwen2_5OmniDiTConfig,
    Qwen2_5OmniTalkerConfig,
    Qwen2_5OmniTextConfig,
    Qwen2_5OmniThinkerConfig,
    Qwen2_5OmniToken2WavConfig,
    Qwen2_5OmniVisionEncoderConfig,
)


if is_flash_attn_2_available():
    from flash_attn.flash_attn_interface import flash_attn_varlen_func as flash_attn_varlen_func
    from flash_attn.layers.rotary import apply_rotary_emb
else:
    flash_attn_varlen_func = None
    apply_rotary_emb = None


if is_torch_flex_attn_available():
    from torch.nn.attention.flex_attention import BlockMask

    from transformers.integrations.flex_attention import make_flex_block_causal_mask


if is_flash_attn_available():
    from transformers.modeling_flash_attention_utils import _flash_attention_forward


logger = logging.get_logger(__name__)


class Qwen2RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        """
        Qwen2RMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

    def extra_repr(self):
        return f"{tuple(self.weight.shape)}, eps={self.variance_epsilon}"


@auto_docstring
class Qwen2_5OmniPreTrainedModel(PreTrainedModel):
    config_class = Qwen2_5OmniConfig
    base_model_prefix = "model"
    supports_gradient_checkpointing = True
    _no_split_modules = ["Qwen2_5OmniDecoderLayer", "Qwen2_5OmniVisionBlock"]
    _skip_keys_device_placement = "past_key_values"
    _supports_flash_attn_2 = True
    _supports_sdpa = True
    _supports_cache_class = True
    _supports_static_cache = True

    def _init_weights(self, module):
        # important: this ported version of Qwen2.5OmniThinker isn't meant for training from scratch - only
        # inference and fine-tuning - so the proper init weights code has been removed
        std = self.config.initializer_range if hasattr(self.config, "initializer_range") else 0.02

        if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv3d, nn.ConvTranspose1d)):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            if module.weight is not None:
                module.weight.data.fill_(1.0)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, Qwen2RMSNorm):
            module.weight.data.fill_(1.0)


class Qwen2_5OmniPreTrainedModelForConditionalGeneration(Qwen2_5OmniPreTrainedModel):
    def _prepare_4d_causal_attention_mask_with_cache_position(
        self,
        attention_mask: torch.Tensor,
        sequence_length: int,
        target_length: int,
        dtype: torch.dtype,
        device: torch.device,
        min_dtype: float,
        cache_position: torch.Tensor,
        batch_size: int,
    ):
        """
        Creates a causal 4D mask of shape `(batch_size, 1, query_length, key_value_length)` from a 2D mask of shape
        `(batch_size, key_value_length)`, or if the input `attention_mask` is already 4D, do nothing.

        Args:
            attention_mask (`torch.Tensor`):
                A 2D attention mask of shape `(batch_size, key_value_length)` or a 4D attention mask of shape `(batch_size, 1, query_length, key_value_length)`.
            sequence_length (`int`):
                The sequence length being processed.
            target_length (`int`):
                The target length: when generating with static cache, the mask should be as long as the static cache, to account for the 0 padding, the part of the cache that is not filled yet.
            dtype (`torch.dtype`):
                The dtype to use for the 4D attention mask.
            device (`torch.device`):
                The device to place the 4D attention mask on.
            min_dtype (`float`):
                The minimum value representable with the dtype `dtype`.
            cache_position (`torch.Tensor`):
                Indices depicting the position of the input sequence tokens in the sequence.
            batch_size (`torch.Tensor`):
                Batch size.
        """
        if attention_mask is not None and attention_mask.dim() == 4:
            # In this case we assume that the mask comes already in inverted form and requires no inversion or slicing.
            causal_mask = attention_mask
        else:
            causal_mask = torch.full(
                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=device
            )
            if sequence_length != 1:
                causal_mask = torch.triu(causal_mask, diagonal=1)
            causal_mask *= torch.arange(target_length, device=device) > cache_position.reshape(-1, 1)
            causal_mask = causal_mask[None, None, :, :].expand(batch_size, 1, -1, -1)
            if attention_mask is not None:
                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit
                mask_length = attention_mask.shape[-1]
                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :]
                padding_mask = padding_mask == 0
                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(
                    padding_mask, min_dtype
                )

        return causal_mask

    def get_llm_pos_ids_for_vision(
        self,
        start_idx: int,
        vision_idx: int,
        spatial_merge_size: int,
        t_index: List[int],
        grid_hs: List[int],
        grid_ws: List[int],
    ):
        llm_pos_ids_list = []
        llm_grid_h = grid_hs[vision_idx] // spatial_merge_size
        llm_grid_w = grid_ws[vision_idx] // spatial_merge_size
        h_index = torch.arange(llm_grid_h).view(1, -1, 1).expand(len(t_index), -1, llm_grid_w).flatten()
        w_index = torch.arange(llm_grid_w).view(1, 1, -1).expand(len(t_index), llm_grid_h, -1).flatten()
        t_index = torch.Tensor(t_index).view(-1, 1).expand(-1, llm_grid_h * llm_grid_w).flatten().long()
        _llm_pos_ids = torch.stack([t_index, h_index, w_index])
        llm_pos_ids_list.append(_llm_pos_ids + start_idx)  # + 1 ) # 12.09 by malinhan
        llm_pos_ids = torch.cat(llm_pos_ids_list, dim=1)
        return llm_pos_ids

    def get_chunked_index(
        self, token_indices: torch.Tensor, tokens_per_chunk: int, remove_index: int
    ) -> list[tuple[int, int]]:
        """
        Splits token index list into chunks based on token value ranges.

        Given a list of token indices, returns a list of (start, end) index tuples representing
        slices of the list where the token values fall within successive ranges of `t_ntoken_per_chunk`.

        For example, if `t_ntoken_per_chunk` is 1000, the function will create chunks such that:
        - the first chunk contains token values < 1000,
        - the second chunk contains values >= 1000 and < 2000, and so on.

        Parameters:
            token_indices (`torch.Tensor` of shape `(seq_len, )`): A monotonically increasing list of
                                token index values.
            t_ntoken_per_chunk (`int`): Number of tokens per chunk (used as the chunk size threshold).
            remove_index (`int`) An index id to subtract from `token_indices` before chunking

        Returns:
            `List[Tuple[int, int]]`: A list of tuples, each representing the start (inclusive)
                                and end (exclusive) indices of a chunk in `token_indices`.
        """

        def _iter():
            i, start_idx = 0, 0  # skip bos token
            current_chunk = 1
            while i < len(token_indices):  # skip eos token
                if token_indices[i] - remove_index >= current_chunk * tokens_per_chunk:
                    yield (start_idx, i)
                    start_idx = i
                    current_chunk += 1
                i += 1
            yield (start_idx, len(token_indices))

        return list(_iter())

    def get_rope_index(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        image_grid_thw: Optional[torch.LongTensor] = None,
        video_grid_thw: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        use_audio_in_video: bool = False,
        audio_seqlens: Optional[torch.LongTensor] = None,
        second_per_grids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Calculate the 3D rope index based on image and video's temporal, height and width in LLM.

        Explanation:
            Each embedding sequence contains vision embedding and text embedding or just contains text embedding.

            For pure text embedding sequence, the rotary position embedding has no difference with modern LLMs.
            Examples:
                input_ids: [T T T T T], here T is for text.
                temporal position_ids: [0, 1, 2, 3, 4]
                height position_ids: [0, 1, 2, 3, 4]
                width position_ids: [0, 1, 2, 3, 4]

            For vision and text embedding sequence, we calculate 3D rotary position embedding for vision part
            and 1D rotary position embedding for text part.
            Examples:
                Temporal (Time): 3 patches, representing different segments of the video in time.
                Height: 2 patches, dividing each frame vertically.
                Width: 2 patches, dividing each frame horizontally.
                We also have some important parameters:
                fps (Frames Per Second): The video's frame rate, set to 1. This means one frame is processed each second.
                tokens_per_second: This is a crucial parameter. It dictates how many "time-steps" or "temporal tokens" are conceptually packed into a one-second interval of the video. In this case, we have 25 tokens per second. So each second of the video will be represented with 25 separate time points. It essentially defines the temporal granularity.
                temporal_patch_size: The number of frames that compose one temporal patch. Here, it's 2 frames.
                interval: The step size for the temporal position IDs, calculated as tokens_per_second * temporal_patch_size / fps. In this case, 25 * 2 / 1 = 50. This means that each temporal patch will be have a difference of 50 in the temporal position IDs.
                input_ids: [V V V V V V V V V V V V T T T T T], here V is for vision.
                vision temporal position_ids: [0, 0, 0, 0, 50, 50, 50, 50, 100, 100, 100, 100]
                vision height position_ids: [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1]
                vision width position_ids: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
                text temporal position_ids: [101, 102, 103, 104, 105]
                text height position_ids: [101, 102, 103, 104, 105]
                text width position_ids: [101, 102, 103, 104, 105]
                Here we calculate the text start position_ids as the max vision position_ids plus 1.

        Args:
            input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
                Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide
                it.
            image_grid_thw (`torch.LongTensor` of shape `(num_images, 3)`, *optional*):
                The temporal, height and width of feature shape of each image in LLM.
            video_grid_thw (`torch.LongTensor` of shape `(num_videos, 3)`, *optional*):
                The temporal, height and width of feature shape of each video in LLM.
            attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):
                Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:

                - 1 for tokens that are **not masked**,
                - 0 for tokens that are **masked**.
            use_audio_in_video (`bool`, *optional*):
                 If set to `True`, use the audio in video.
            audio_seqlens (`torch.LongTensor` of shape `(num_audios)`, *optional*):
                The length of feature shape of each audio in LLM.
            second_per_grids (`torch.LongTensor` of shape `(num_videos)`, *optional*):
                The time interval (in seconds) for each grid along the temporal dimension in the 3D position IDs.

        Returns:
            position_ids (`torch.LongTensor` of shape `(3, batch_size, sequence_length)`)
            mrope_position_deltas (`torch.Tensor` of shape `(batch_size)`)
        """
        spatial_merge_size = self.spatial_merge_size
        image_token_id = self.config.image_token_id
        video_token_id = self.config.video_token_id
        audio_token_id = self.config.audio_token_id
        vision_start_token_id = self.config.vision_start_token_id
        audio_start_token_id = self.config.audio_start_token_id
        position_id_per_seconds = self.config.position_id_per_seconds
        seconds_per_chunk = self.config.seconds_per_chunk

        mrope_position_deltas = []
        if input_ids is not None and (image_grid_thw is not None or video_grid_thw is not None):
            total_input_ids = input_ids
            if attention_mask is None:
                attention_mask = torch.ones_like(total_input_ids)
            position_ids = torch.ones(
                3,
                input_ids.shape[0],
                input_ids.shape[1],
                dtype=input_ids.dtype,
                device=input_ids.device,
            )
            image_idx, video_idx, audio_idx = 0, 0, 0
            attention_mask = attention_mask.to(total_input_ids.device)
            for i, input_ids in enumerate(total_input_ids):
                input_ids = input_ids[attention_mask[i] == 1]
                image_nums, video_nums, audio_nums = 0, 0, 0
                vision_start_indices = torch.argwhere(input_ids == vision_start_token_id).squeeze(1)
                vision_tokens = input_ids[vision_start_indices + 1]
                audio_nums = torch.sum(input_ids == audio_start_token_id)
                image_nums = (vision_tokens == image_token_id).sum()
                video_nums = (
                    (vision_tokens == audio_start_token_id).sum()
                    if use_audio_in_video
                    else (vision_tokens == video_token_id).sum()
                )
                input_tokens = input_ids.tolist()
                llm_pos_ids_list: list = []
                st = 0
                remain_images, remain_videos, remain_audios = image_nums, video_nums, audio_nums
                multimodal_nums = (
                    image_nums + audio_nums if use_audio_in_video else image_nums + video_nums + audio_nums
                )
                for _ in range(multimodal_nums):
                    st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                    if image_token_id in input_tokens and remain_images > 0:
                        ed_image = input_tokens.index(image_token_id, st)
                    else:
                        ed_image = len(input_tokens) + 1
                    if video_token_id in input_tokens and remain_videos > 0:
                        ed_video = input_tokens.index(video_token_id, st)
                    else:
                        ed_video = len(input_tokens) + 1
                    if audio_token_id in input_tokens and remain_audios > 0:
                        ed_audio = input_tokens.index(audio_token_id, st)
                    else:
                        ed_audio = len(input_tokens) + 1
                    min_ed = min(ed_image, ed_video, ed_audio)
                    if min_ed == ed_audio:
                        text_len = min_ed - st - 1
                        if text_len != 0:
                            st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                            llm_pos_ids_list.append(torch.arange(text_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        bos_len = 1
                        llm_pos_ids_list.append(torch.arange(bos_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        audio_len = ((audio_seqlens[audio_idx] - 1) // 2 + 1 - 2) // 2 + 1
                        llm_pos_ids = torch.arange(audio_len).view(1, -1).expand(3, -1) + st_idx
                        llm_pos_ids_list.append(llm_pos_ids)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        eos_len = 1
                        llm_pos_ids_list.append(torch.arange(eos_len).view(1, -1).expand(3, -1) + st_idx)

                        st += text_len + bos_len + audio_len + eos_len
                        audio_idx += 1
                        remain_audios -= 1

                    elif min_ed == ed_image:
                        text_len = min_ed - st - 1
                        if text_len != 0:
                            st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                            llm_pos_ids_list.append(torch.arange(text_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        bos_len = 1
                        llm_pos_ids_list.append(torch.arange(bos_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        grid_t = image_grid_thw[image_idx][0]
                        grid_hs = image_grid_thw[:, 1]
                        grid_ws = image_grid_thw[:, 2]
                        t_index = (torch.arange(grid_t) * 1 * position_id_per_seconds).long()
                        llm_pos_ids = self.get_llm_pos_ids_for_vision(
                            st_idx, image_idx, spatial_merge_size, t_index, grid_hs, grid_ws
                        )
                        image_len = image_grid_thw[image_idx].prod() // (spatial_merge_size**2)
                        llm_pos_ids_list.append(llm_pos_ids)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        eos_len = 1
                        llm_pos_ids_list.append(torch.arange(eos_len).view(1, -1).expand(3, -1) + st_idx)

                        st += text_len + bos_len + image_len + eos_len
                        image_idx += 1
                        remain_images -= 1

                    elif min_ed == ed_video and not use_audio_in_video:
                        text_len = min_ed - st - 1
                        if text_len != 0:
                            st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                            llm_pos_ids_list.append(torch.arange(text_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        bos_len = 1
                        llm_pos_ids_list.append(torch.arange(bos_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        grid_t = video_grid_thw[video_idx][0]
                        grid_hs = video_grid_thw[:, 1]
                        grid_ws = video_grid_thw[:, 2]
                        t_index = (
                            torch.arange(grid_t) * second_per_grids[video_idx].cpu().float() * position_id_per_seconds
                        ).long()
                        llm_pos_ids = self.get_llm_pos_ids_for_vision(
                            st_idx, video_idx, spatial_merge_size, t_index, grid_hs, grid_ws
                        )
                        video_len = video_grid_thw[video_idx].prod() // (spatial_merge_size**2)
                        llm_pos_ids_list.append(llm_pos_ids)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        eos_len = 1
                        llm_pos_ids_list.append(torch.arange(eos_len).view(1, -1).expand(3, -1) + st_idx)

                        st += text_len + bos_len + video_len + eos_len
                        video_idx += 1
                        remain_videos -= 1

                    elif min_ed == ed_video and use_audio_in_video:
                        text_len = min_ed - st - 2
                        if text_len != 0:
                            st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                            llm_pos_ids_list.append(torch.arange(text_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        bos_len = 1
                        llm_pos_ids_list.append(torch.arange(bos_len).view(1, -1).expand(3, -1) + st_idx)
                        llm_pos_ids_list.append(torch.arange(bos_len).view(1, -1).expand(3, -1) + st_idx)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        audio_len = ((audio_seqlens[audio_idx] - 1) // 2 + 1 - 2) // 2 + 1
                        audio_llm_pos_ids = torch.arange(audio_len).view(1, -1).expand(3, -1) + st_idx
                        grid_t = video_grid_thw[video_idx][0]
                        grid_hs = video_grid_thw[:, 1]
                        grid_ws = video_grid_thw[:, 2]

                        t_index = (
                            torch.arange(grid_t) * second_per_grids[video_idx].cpu().float() * position_id_per_seconds
                        ).long()
                        video_llm_pos_ids = self.get_llm_pos_ids_for_vision(
                            st_idx, video_idx, spatial_merge_size, t_index, grid_hs, grid_ws
                        )

                        t_ntoken_per_chunk = int(position_id_per_seconds * seconds_per_chunk)
                        video_chunk_indexes = self.get_chunked_index(video_llm_pos_ids[0], t_ntoken_per_chunk, st_idx)
                        audio_chunk_indexes = self.get_chunked_index(audio_llm_pos_ids[0], t_ntoken_per_chunk, st_idx)
                        sub_len = 0
                        for j in range(max(len(video_chunk_indexes), len(audio_chunk_indexes))):
                            video_chunk_index = video_chunk_indexes[j] if j < len(video_chunk_indexes) else None
                            audio_chunk_index = audio_chunk_indexes[j] if j < len(audio_chunk_indexes) else None
                            if video_chunk_index is not None:
                                sub_len += video_chunk_index[1] - video_chunk_index[0]

                                llm_pos_ids_list.append(
                                    video_llm_pos_ids[:, video_chunk_index[0] : video_chunk_index[1]]
                                )
                            if audio_chunk_index is not None:
                                sub_len += audio_chunk_index[1] - audio_chunk_index[0]

                                llm_pos_ids_list.append(
                                    audio_llm_pos_ids[:, audio_chunk_index[0] : audio_chunk_index[1]]
                                )
                        video_len = video_grid_thw[video_idx].prod() // (spatial_merge_size**2)

                        st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                        eos_len = 1
                        llm_pos_ids_list.append(torch.arange(eos_len).view(1, -1).expand(3, -1) + st_idx)
                        llm_pos_ids_list.append(torch.arange(eos_len).view(1, -1).expand(3, -1) + st_idx)

                        st += text_len + bos_len * 2 + audio_len + video_len + eos_len * 2

                        audio_idx += 1
                        video_idx += 1
                        remain_videos -= 1
                        remain_audios -= 1

                if st < len(input_tokens):
                    st_idx = llm_pos_ids_list[-1].max() + 1 if len(llm_pos_ids_list) > 0 else 0
                    text_len = len(input_tokens) - st
                    llm_pos_ids_list.append(torch.arange(text_len).view(1, -1).expand(3, -1) + st_idx)

                llm_positions = torch.cat(llm_pos_ids_list, dim=1).reshape(3, -1)

                position_ids[..., i, attention_mask[i] == 1] = llm_positions.to(position_ids.device)
                mrope_position_deltas.append(llm_positions.max() + 1 - len(input_ids))
            mrope_position_deltas = torch.tensor(mrope_position_deltas, device=input_ids.device).unsqueeze(1)

            return position_ids, mrope_position_deltas
        else:
            position_ids = attention_mask.long().cumsum(-1) - 1
            position_ids.masked_fill_(attention_mask == 0, 1)
            position_ids = position_ids.unsqueeze(0).expand(3, -1, -1).to(attention_mask.device)
            max_position_ids = position_ids.max(0, keepdim=False)[0].max(-1, keepdim=True)[0]
            mrope_position_deltas = max_position_ids + 1 - torch.sum(attention_mask, dim=-1, keepdim=True)

            return position_ids, mrope_position_deltas


############################
#      Start Thinker       #
############################


@dataclass
class Qwen2_5OmniThinkerCausalLMOutputWithPast(ModelOutput):
    """
    Base class for Qwen2.5OmniThinker causal language model (or autoregressive) outputs.

    Args:
        loss (`torch.FloatTensor` of shape `(1,)`, *optional*, returned when `labels` is provided):
            Language modeling loss (for next-token prediction).
        logits (`torch.FloatTensor` of shape `(batch_size, sequence_length, config.vocab_size)`, *optional*):
            Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
        past_key_values (`tuple(tuple(torch.FloatTensor))`, *optional*, returned when `use_cache=True` is passed or when `config.use_cache=True`):
            Tuple of `tuple(torch.FloatTensor)` of length `config.n_layers`, with each tuple having 2 tensors of shape
            `(batch_size, num_heads, sequence_length, embed_size_per_head)`)

            Contains pre-computed hidden-states (key and values in the self-attention blocks) that can be used (see
            `past_key_values` input) to speed up sequential decoding.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.

            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):
            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,
            sequence_length)`.

            Attentions weights after the attention softmax, used to compute the weighted average in the self-attention
            heads.
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
    """

    loss: Optional[torch.FloatTensor] = None
    logits: Optional[torch.FloatTensor] = None
    past_key_values: Optional[List[torch.FloatTensor]] = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None
    rope_deltas: Optional[torch.LongTensor] = None


@torch.no_grad()
def _audio_token_importance(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """OmniZip audio saliency: mean softmax attention each key token receives, over heads and queries.

    q, k: [H, T, D]. Returns [T]. Chunked over heads and queries to bound memory.
    """
    scale = q.shape[-1] ** -0.5
    num_heads, T_len = q.shape[0], q.shape[1]
    token_importance = torch.zeros(T_len, device=q.device)
    head_chunk = 4
    seq_chunk = 512
    for h in range(0, num_heads, head_chunk):
        q_h = q[h:h + head_chunk]
        k_h = k[h:h + head_chunk]
        for i in range(0, T_len, seq_chunk):
            attn_chunk = torch.matmul(q_h[:, i:i + seq_chunk, :], k_h.transpose(-1, -2)) * scale
            attn_chunk = F.softmax(attn_chunk, dim=-1)
            token_importance += attn_chunk.sum(dim=(0, 1))
            del attn_chunk
    torch.cuda.empty_cache()  # release the chunk buffers once, not after every encoder layer
    return token_importance / (num_heads * T_len)


class Qwen2_5OmniAudioAttention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(
        self,
        config: Qwen2_5OmniAudioEncoderConfig,
    ):
        super().__init__()
        self.embed_dim = config.d_model
        self.num_heads = config.encoder_attention_heads
        self.dropout = config.attention_dropout
        self.head_dim = self.embed_dim // self.num_heads
        self.config = config

        if (self.head_dim * self.num_heads) != self.embed_dim:
            raise ValueError(
                f"embed_dim must be divisible by num_heads (got `embed_dim`: {self.embed_dim}"
                f" and `num_heads`: {self.num_heads})."
            )
        self.scaling = self.head_dim**-0.5
        self.is_decoder = False
        self.is_causal = False

        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=True)
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=True)
        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=True)

    def forward(
        self,
        hidden_states: torch.Tensor,
        cu_seqlens: Optional[torch.Tensor] = None,
        return_logits: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        """Input shape: Batch x Time x Channel"""

        seq_length, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        key_states = self.k_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        value_states = self.v_proj(hidden_states).reshape(seq_length, self.num_heads, -1)

        query_states = query_states.transpose(0, 1)
        key_states = key_states.transpose(0, 1)
        value_states = value_states.transpose(0, 1)
        attn_weights = torch.matmul(query_states, key_states.transpose(1, 2)) / math.sqrt(self.head_dim)

        attention_mask = torch.full(
            [1, seq_length, key_states.shape[1]],
            torch.finfo(query_states.dtype).min,
            device=query_states.device,
            dtype=query_states.dtype,
        )
        for i in range(1, len(cu_seqlens)):
            attention_mask[..., cu_seqlens[i - 1] : cu_seqlens[i], cu_seqlens[i - 1] : cu_seqlens[i]] = 0

        attn_weights = attn_weights + attention_mask

        attn_weights = nn.functional.softmax(attn_weights, dim=-1).to(query_states.dtype)

        attn_output = torch.matmul(attn_weights, value_states).transpose(0, 1).reshape(seq_length, self.embed_dim)

        # Use the `embed_dim` from the config (stored in the class) rather than `hidden_state` because `attn_output` can be
        # partitioned across GPUs when using tensor-parallelism.

        attn_output = self.out_proj(attn_output)

        if return_logits:
            # Same scores as the flash/SDPA paths: unmasked, over the whole concatenated sequence
            return attn_output, _audio_token_importance(query_states, key_states), key_states
        return attn_output, None, None


class Qwen2_5OmniAudioFlashAttention2(Qwen2_5OmniAudioAttention):
    """
    Qwen2.5OmniThinker flash attention module. This module inherits from `Qwen2_5OmniAudioAttention` as the weights of the module stays
    untouched. The only required change would be on the forward pass where it needs to correctly call the public API of
    flash attention and deal with padding tokens in case the input contains any of them.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # TODO: Should be removed once Flash Attention for RoCm is bumped to 2.1.
        # flash_attn<2.1 generates top-left aligned causal mask, while what is needed here is bottom-right alignment, that was made default for flash_attn>=2.1. This attribute is used to handle this difference. Reference: https://github.com/Dao-AILab/flash-attention/releases/tag/v2.1.0.
        # Beware that with flash_attn<2.1, using q_seqlen != k_seqlen (except for the case q_seqlen == 1) produces a wrong mask (top-left).
        self._flash_attn_uses_top_left_mask = not is_flash_attn_greater_or_equal_2_10()

    def forward(
        self,
        hidden_states: torch.Tensor,
        cu_seqlens: Optional[torch.Tensor] = None,
        return_logits: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        seq_length, all_dim = hidden_states.size()
        query_states = self.q_proj(hidden_states)
        query_states = query_states.reshape(seq_length, self.num_heads, -1)

        key_states = self.k_proj(hidden_states)
        key_states = key_states.reshape(seq_length, self.num_heads, -1)
        value_states = self.v_proj(hidden_states)
        value_states = value_states.reshape(seq_length, self.num_heads, -1)

        max_seqlen = (cu_seqlens[1:] - cu_seqlens[:-1]).max().item()
        attn_output = flash_attn_varlen_func(
            query_states, key_states, value_states, cu_seqlens, cu_seqlens, max_seqlen, max_seqlen, dropout_p=0.0
        )
        attn_output = attn_output.reshape(seq_length, all_dim)
        attn_output = self.out_proj(attn_output)

        if return_logits:
            return_k = key_states.permute(1, 0, 2)  # [H, T, D]
            attn_logits = _audio_token_importance(query_states.permute(1, 0, 2), return_k)
        else:
            attn_logits = None
            return_k = None
        return attn_output, attn_logits, return_k


class Qwen2_5OmniAudioSdpaAttention(Qwen2_5OmniAudioAttention):
    def forward(
        self,
        hidden_states: torch.Tensor,
        cu_seqlens: Optional[torch.Tensor] = None,
        return_logits: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        """Input shape: Batch x Time x Channel"""

        seq_length, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        key_states = self.k_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        value_states = self.v_proj(hidden_states).reshape(seq_length, self.num_heads, -1)

        attention_mask = torch.zeros(
            [1, seq_length, key_states.shape[0]], device=query_states.device, dtype=torch.bool
        )
        for i in range(1, len(cu_seqlens)):
            attention_mask[..., cu_seqlens[i - 1] : cu_seqlens[i], cu_seqlens[i - 1] : cu_seqlens[i]] = True

        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment
        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.
        # The tgt_len > 1 is necessary to match with AttentionMaskConverter.to_causal_4d that does not create a causal mask in case tgt_len == 1.
        query_states = query_states.transpose(0, 1)
        key_states = key_states.transpose(0, 1)
        value_states = value_states.transpose(0, 1)

        # NOTE: SDPA with memory-efficient backend is currently (torch==2.1.2) bugged when using non-contiguous inputs and a custom attn_mask,
        # but we are fine here as `_shape` do call `.contiguous()`. Reference: https://github.com/pytorch/pytorch/issues/112577
        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_states,
            key_states,
            value_states,
            attn_mask=attention_mask,
            dropout_p=self.dropout if self.training else 0.0,
        )
        attn_output = attn_output.transpose(0, 1)
        # Use the `embed_dim` from the config (stored in the class) rather than `hidden_state` because `attn_output` can be
        # partitioned across GPUs when using tensor-parallelism.
        attn_output = attn_output.reshape(seq_length, self.embed_dim)
        attn_output = self.out_proj(attn_output)

        if return_logits:
            # already [H, T, D] after the transpose above
            attn_logits = _audio_token_importance(query_states, key_states)
            return_k = key_states
        else:
            attn_logits = None
            return_k = None
        return attn_output, attn_logits, return_k


QWEN2_5_OMNI_AUDIO_ATTENTION_CLASSES = {
    "eager": Qwen2_5OmniAudioAttention,
    "flash_attention_2": Qwen2_5OmniAudioFlashAttention2,
    "sdpa": Qwen2_5OmniAudioSdpaAttention,
}


class Qwen2_5OmniAudioEncoderLayer(nn.Module):
    def __init__(self, config: Qwen2_5OmniAudioEncoderConfig):
        super().__init__()
        self.embed_dim = config.d_model
        self.self_attn = QWEN2_5_OMNI_AUDIO_ATTENTION_CLASSES[config._attn_implementation](config)
        self.self_attn_layer_norm = nn.LayerNorm(self.embed_dim)
        self.dropout = config.dropout
        self.activation_fn = ACT2FN[config.activation_function]
        self.activation_dropout = config.activation_dropout
        self.fc1 = nn.Linear(self.embed_dim, config.encoder_ffn_dim)
        self.fc2 = nn.Linear(config.encoder_ffn_dim, self.embed_dim)
        self.final_layer_norm = nn.LayerNorm(self.embed_dim)

    def forward(
        self,
        hidden_states: torch.Tensor,
        cu_seqlens: torch.Tensor,
        return_logits: bool = False,
    ) -> torch.Tensor:
        """
        Args:
            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`
            attention_mask (`torch.FloatTensor`): attention mask of size
                `(batch, 1, tgt_len, src_len)` where padding elements are indicated by very large negative values.
            layer_head_mask (`torch.FloatTensor`): mask for attention heads in a given layer of size
                `(encoder_attention_heads,)`.
            output_attentions (`bool`, *optional*):
                Whether or not to return the attentions tensors of all attention layers. See `attentions` under
                returned tensors for more detail.
        """
        residual = hidden_states
        hidden_states = self.self_attn_layer_norm(hidden_states)
        hidden_states, logits, k = self.self_attn(
            hidden_states=hidden_states,
            cu_seqlens=cu_seqlens,
            return_logits=return_logits,
        )
        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.fc1(hidden_states)
        hidden_states = self.activation_fn(hidden_states)
        hidden_states = self.fc2(hidden_states)
        hidden_states = residual + hidden_states

        if hidden_states.dtype == torch.float16:
            clamp_value = torch.finfo(hidden_states.dtype).max - 1000
            hidden_states = torch.clamp(hidden_states, min=-clamp_value, max=clamp_value)

        outputs = (hidden_states,)

        return outputs, logits, k


class SinusoidsPositionEmbedding(nn.Module):
    def __init__(self, length, channels, max_timescale=10000):
        super().__init__()
        if channels % 2 != 0:
            raise ValueError("SinusoidsPositionEmbedding needs even channels input")
        log_timescale_increment = np.log(max_timescale) / (channels // 2 - 1)
        inv_timescales = torch.exp(-log_timescale_increment * torch.arange(channels // 2).float())
        scaled_time = torch.arange(length)[:, np.newaxis] * inv_timescales[np.newaxis, :]
        self.register_buffer(
            "positional_embedding",
            torch.cat([torch.sin(scaled_time), torch.cos(scaled_time)], dim=1),
            persistent=False,
        )

    def forward(self, seqlen: int):
        return self.positional_embedding[:seqlen, :]


@auto_docstring(
    custom_intro="""
    Transformer encoder consisting of *config.encoder_layers* self attention layers. Each layer is a
    [`Qwen2_5OmniAudioEncoderLayer`].
    """
)
class Qwen2_5OmniAudioEncoder(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniAudioEncoderConfig
    main_input_name = "input_features"
    _no_split_modules = ["Qwen2_5OmniAudioEncoderLayer"]
    _supports_sdpa = True

    def __init__(self, config: Qwen2_5OmniAudioEncoderConfig):
        super().__init__(config)
        self.dropout = config.dropout

        embed_dim = config.d_model
        self.num_mel_bins = config.num_mel_bins
        self.max_source_positions = config.max_source_positions
        self.embed_scale = math.sqrt(embed_dim) if config.scale_embedding else 1.0
        self.n_window = config.n_window
        self.conv1 = nn.Conv1d(self.num_mel_bins, embed_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, stride=2, padding=1)
        self.positional_embedding = SinusoidsPositionEmbedding(self.max_source_positions, embed_dim)
        self.audio_bos_eos_token = nn.Embedding(2, config.output_dim)
        self.layers = nn.ModuleList([Qwen2_5OmniAudioEncoderLayer(config) for _ in range(config.encoder_layers)])
        self.ln_post = nn.LayerNorm(config.d_model)
        self.avg_pooler = nn.AvgPool1d(2, stride=2)
        self.proj = nn.Linear(config.d_model, config.output_dim)
        self.gradient_checkpointing = False
        # Initialize weights and apply final processing
        self.post_init()

    def _freeze_parameters(self):
        for param in self.parameters():
            param.requires_grad = False
        self._requires_grad = False

    def get_input_embeddings(self) -> nn.Module:
        return self.conv1

    def set_input_embeddings(self, value: nn.Module):
        self.conv1 = value

    @auto_docstring
    def forward(
        self,
        input_features,
        feature_lens=None,
        aftercnn_lens=None,
    ):
        r"""
        input_features (`torch.LongTensor` of shape `(batch_size, feature_size, sequence_length)`):
            Float values of mel features extracted from the raw speech waveform. Raw speech waveform can be
            obtained by loading a `.flac` or `.wav` audio file into an array of type `List[float]` or a
            `numpy.ndarray`, *e.g.* via the soundfile library (`pip install soundfile`). To prepare the array into
            `input_features`, the [`AutoFeatureExtractor`] should be used for extracting the mel features, padding
            and conversion into a tensor of type `torch.FloatTensor`. See [`~WhisperFeatureExtractor.__call__`]
        feature_lens (`torch.LongTensor` of shape `(batch_size,)`):
            mel length
        aftercnn_lens (`torch.LongTensor` of shape `(batch_size,)`):
            mel length after cnn
        """
        chunk_num = torch.ceil(feature_lens / (self.n_window * 2)).long()

        chunk_lengths = torch.tensor(
            [self.n_window * 2] * chunk_num.sum(),
            dtype=torch.long,
            device=feature_lens.device,
        )
        tail_chunk_index = F.pad(chunk_num, (1, 0), value=-1).cumsum(0)[1:]
        chunk_lengths[tail_chunk_index] = feature_lens % (self.n_window * 2)
        chunk_lengths = torch.where(chunk_lengths == 0, self.n_window * 2, chunk_lengths)

        chunk_list = input_features.split(chunk_lengths.tolist(), dim=1)
        padded_feature, padded_mask, padded_mask_after_cnn = self.padded_and_mask_function(
            chunk_list, chunk_lengths, padding_value=0, padding_side="right"
        )
        padded_embed = nn.functional.gelu(self.conv1(padded_feature)) * padded_mask
        padded_embed = nn.functional.gelu(self.conv2(padded_embed)).transpose(1, 2)

        padded_embed = padded_embed + self.positional_embedding.positional_embedding[
            : padded_embed.shape[1], :
        ].unsqueeze(0).to(padded_embed.dtype)
        hidden_states = padded_embed[padded_mask_after_cnn]
        cu_seqlens = torch.cat(
            (
                torch.zeros(1, device=padded_mask_after_cnn.device, dtype=torch.int32),
                padded_mask_after_cnn.sum(1).cumsum(0),
            )
        ).to(torch.int32)

        for idx, encoder_layer in enumerate(self.layers):
            if self.gradient_checkpointing and self.training:
                if idx == len(self.layers) - 1:
                    layer_outputs, logits, attn_key = self._gradient_checkpointing_func(
                        encoder_layer.__call__,
                        hidden_states,
                        cu_seqlens,
                        True
                    )
                else:
                    layer_outputs, _, _ = self._gradient_checkpointing_func(
                        encoder_layer.__call__,
                        hidden_states,
                        cu_seqlens,
                        False
                    )
            else:
                if idx == len(self.layers) - 1:
                    layer_outputs, logits, attn_key = encoder_layer(
                        hidden_states,
                        cu_seqlens,
                        True
                    )
                else:
                    layer_outputs, _, _ = encoder_layer(
                        hidden_states,
                        cu_seqlens,
                        False
                    )

            hidden_states = layer_outputs[0]

        hidden_states_list = hidden_states.split(aftercnn_lens.tolist(), dim=0)
        token_audio_list = []
        for each_audio_states in hidden_states_list:
            each_audio_states = self.avg_pooler(each_audio_states.transpose(0, 1)).transpose_(0, 1)
            each_audio_states = self.ln_post(each_audio_states)
            each_audio_states = self.proj(each_audio_states)
            token_audio_list.append(each_audio_states)
        token_audio = torch.cat(token_audio_list, dim=0)

        
        with torch.no_grad():
            if logits is not None:
                attn_mean = logits 
                
                T_len = attn_mean.shape[0]
                if T_len % 2 == 1:
                    attn_mean = attn_mean[:-1]
                    T_len -= 1
                if T_len >= 2:
                    attn_mean = attn_mean.view(-1, 2).mean(dim=-1)

                H, Tk, D = attn_key.shape
                if Tk % 2 == 1:
                    attn_key = attn_key[:, :-1, :]
                    Tk -= 1
                if Tk >= 2:
                    attn_key = attn_key.view(H, -1, 2, D).mean(dim=2)
                
                attn_key = attn_key.mean(dim=0, keepdim=True)
            else:
                attn_mean = None
                attn_key = None


        return BaseModelOutput(last_hidden_state=token_audio), attn_mean, attn_key

    def padded_and_mask_function(self, tensor_list, tensor_len, padding_value=0, padding_side="right"):
        """
        Pads a sequence of tensors to their maximum length on indicated `padding_side`.
        Then prepares a mask so that pad tokens are not attended to.
        """
        max_len = tensor_len.max()
        dim = tensor_list[0].shape[0]
        padded_tensor = torch.full(
            size=(len(tensor_list), dim, max_len),
            fill_value=padding_value,
            dtype=self.dtype,
            device=tensor_list[0].device,
        )

        batch_mask = torch.zeros(
            (len(tensor_len), max_len),
            dtype=torch.long,
            device=padded_tensor.device,
        )
        for i, length in enumerate(tensor_len):
            batch_mask[i, :length] = 1
            padded_tensor[i, :, :length] = tensor_list[i]

        feature_lens_after_cnn = (tensor_len - 1) // 2 + 1
        max_len_after_cnn = feature_lens_after_cnn.max()
        batch_mask_after_cnn = torch.zeros(
            (len(tensor_len), max_len_after_cnn),
            dtype=torch.long,
            device=padded_tensor.device,
        )
        for i, length in enumerate(feature_lens_after_cnn):
            batch_mask_after_cnn[i, :length] = 1
        return (
            padded_tensor,
            batch_mask.unsqueeze(1),
            batch_mask_after_cnn.bool(),
        )

    # Ignore copy
    def _get_feat_extract_output_lengths(self, input_lengths: torch.LongTensor):
        """
        Computes the output length of the convolutional layers and the output length of the audio encoder
        """
        input_lengths = (input_lengths - 1) // 2 + 1
        output_lengths = (input_lengths - 2) // 2 + 1
        return input_lengths, output_lengths


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb_vision(tensor: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    orig_dtype = tensor.dtype
    tensor = tensor.float()
    cos = freqs.cos()
    sin = freqs.sin()
    cos = cos.unsqueeze(1).repeat(1, 1, 2).unsqueeze(0).float()
    sin = sin.unsqueeze(1).repeat(1, 1, 2).unsqueeze(0).float()
    output = (tensor * cos) + (rotate_half(tensor) * sin)
    output = output.to(orig_dtype)
    return output


class Qwen2_5OmniVisionAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 16) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q = nn.Linear(dim, dim, bias=True)
        self.k = nn.Linear(dim, dim, bias=True)
        self.v = nn.Linear(dim, dim, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(
        self, hidden_states: torch.Tensor, cu_seqlens: torch.Tensor, rotary_pos_emb: torch.Tensor = None
    ) -> torch.Tensor:
        seq_length = hidden_states.shape[0]
        q = self.q(hidden_states).reshape(seq_length, self.num_heads, -1)
        k = self.k(hidden_states).reshape(seq_length, self.num_heads, -1)
        v = self.v(hidden_states).reshape(seq_length, self.num_heads, -1)
        q = apply_rotary_pos_emb_vision(q.unsqueeze(0), rotary_pos_emb).squeeze(0)
        k = apply_rotary_pos_emb_vision(k.unsqueeze(0), rotary_pos_emb).squeeze(0)

        attention_mask = torch.full(
            [1, seq_length, seq_length], torch.finfo(q.dtype).min, device=q.device, dtype=q.dtype
        )
        for i in range(1, len(cu_seqlens)):
            attention_mask[..., cu_seqlens[i - 1] : cu_seqlens[i], cu_seqlens[i - 1] : cu_seqlens[i]] = 0

        q = q.transpose(0, 1)
        k = k.transpose(0, 1)
        v = v.transpose(0, 1)
        attn_weights = torch.matmul(q, k.transpose(1, 2)) / math.sqrt(self.head_dim)
        attn_weights = attn_weights + attention_mask
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(q.dtype)
        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(0, 1)
        attn_output = attn_output.reshape(seq_length, -1)
        attn_output = self.proj(attn_output)
        return attn_output


class Qwen2_5OmniVisionFlashAttention2(nn.Module):
    def __init__(self, dim: int, num_heads: int = 16) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.q = nn.Linear(dim, dim, bias=True)
        self.k = nn.Linear(dim, dim, bias=True)
        self.v = nn.Linear(dim, dim, bias=True)
        self.proj = nn.Linear(dim, dim)

    def _apply_rotary_pos_emb_flashatt(self, tensor: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
        tensor_ = tensor.float()
        cos = freqs.cos()  # .type_as(tensor_)
        sin = freqs.sin()  # .type_as(tensor_)
        output = apply_rotary_emb(tensor_, cos, sin).type_as(tensor)
        return output

    def forward(
        self, hidden_states: torch.Tensor, cu_seqlens: torch.Tensor, rotary_pos_emb: torch.Tensor = None
    ) -> torch.Tensor:
        seq_length = hidden_states.shape[0]
        q = self.q(hidden_states).reshape(seq_length, self.num_heads, -1)
        k = self.k(hidden_states).reshape(seq_length, self.num_heads, -1)
        v = self.v(hidden_states).reshape(seq_length, self.num_heads, -1)
        q = self._apply_rotary_pos_emb_flashatt(q.unsqueeze(0), rotary_pos_emb).squeeze(0)
        k = self._apply_rotary_pos_emb_flashatt(k.unsqueeze(0), rotary_pos_emb).squeeze(0)
        
        max_seqlen = (cu_seqlens[1:] - cu_seqlens[:-1]).max().item()
        attn_output = flash_attn_varlen_func(q, k, v, cu_seqlens, cu_seqlens, max_seqlen, max_seqlen).reshape(
            seq_length, -1
        )
        attn_output = self.proj(attn_output)
        return attn_output


class Qwen2_5OmniVisionSdpaAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 16) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.q = nn.Linear(dim, dim, bias=True)
        self.k = nn.Linear(dim, dim, bias=True)
        self.v = nn.Linear(dim, dim, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(
        self, hidden_states: torch.Tensor, cu_seqlens: torch.Tensor, rotary_pos_emb: torch.Tensor = None
    ) -> torch.Tensor:
        seq_length = hidden_states.shape[0]
        q = self.q(hidden_states).reshape(seq_length, self.num_heads, -1)
        k = self.k(hidden_states).reshape(seq_length, self.num_heads, -1)
        v = self.v(hidden_states).reshape(seq_length, self.num_heads, -1)
        q = apply_rotary_pos_emb_vision(q.unsqueeze(0), rotary_pos_emb).squeeze(0)
        k = apply_rotary_pos_emb_vision(k.unsqueeze(0), rotary_pos_emb).squeeze(0)

        attention_mask = torch.zeros([1, seq_length, seq_length], device=q.device, dtype=torch.bool)
        for i in range(1, len(cu_seqlens)):
            attention_mask[..., cu_seqlens[i - 1] : cu_seqlens[i], cu_seqlens[i - 1] : cu_seqlens[i]] = True
        q = q.transpose(0, 1)
        k = k.transpose(0, 1)
        v = v.transpose(0, 1)
        attn_output = F.scaled_dot_product_attention(q, k, v, attention_mask, dropout_p=0.0)
        attn_output = attn_output.transpose(0, 1)
        attn_output = attn_output.reshape(seq_length, -1)
        attn_output = self.proj(attn_output)
        return attn_output


class Qwen2_5OmniMLP(nn.Module):
    def __init__(self, config, bias: bool = False):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=bias)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, hidden_state):
        return self.down_proj(self.act_fn(self.gate_proj(hidden_state)) * self.up_proj(hidden_state))


QWEN2_5_OMNI_VISION_ATTENTION_CLASSES = {
    "eager": Qwen2_5OmniVisionAttention,
    "flash_attention_2": Qwen2_5OmniVisionFlashAttention2,
    "sdpa": Qwen2_5OmniVisionSdpaAttention,
}


class Qwen2_5OmniVisionBlock(nn.Module):
    def __init__(self, config: Qwen2_5OmniVisionEncoderConfig) -> None:
        super().__init__()
        self.norm1 = Qwen2RMSNorm(config.hidden_size, eps=1e-6)
        self.norm2 = Qwen2RMSNorm(config.hidden_size, eps=1e-6)
        self.attn = QWEN2_5_OMNI_VISION_ATTENTION_CLASSES[config._attn_implementation](
            config.hidden_size, num_heads=config.num_heads
        )
        self.mlp = Qwen2_5OmniMLP(config, bias=True)

    def forward(self, hidden_states, cu_seqlens, rotary_pos_emb) -> torch.Tensor:
        hidden_states = hidden_states + self.attn(
            self.norm1(hidden_states), cu_seqlens=cu_seqlens, rotary_pos_emb=rotary_pos_emb
        )
        hidden_states = hidden_states + self.mlp(self.norm2(hidden_states))
        return hidden_states


class Qwen2_5_VisionPatchEmbed(nn.Module):
    def __init__(
        self,
        patch_size: int = 14,
        temporal_patch_size: int = 2,
        in_channels: int = 3,
        embed_dim: int = 1152,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.temporal_patch_size = temporal_patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim

        kernel_size = [temporal_patch_size, patch_size, patch_size]
        self.proj = nn.Conv3d(in_channels, embed_dim, kernel_size=kernel_size, stride=kernel_size, bias=False)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        target_dtype = self.proj.weight.dtype
        hidden_states = hidden_states.view(
            -1, self.in_channels, self.temporal_patch_size, self.patch_size, self.patch_size
        )
        hidden_states = self.proj(hidden_states.to(dtype=target_dtype)).view(-1, self.embed_dim)
        return hidden_states


class Qwen2_5_VisionRotaryEmbedding(nn.Module):
    def __init__(self, dim: int, theta: float = 10000.0) -> None:
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seqlen: int) -> torch.Tensor:
        seq = torch.arange(seqlen, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(seq, self.inv_freq)
        return freqs


class Qwen2_5OmniPatchMerger(nn.Module):
    def __init__(self, dim: int, context_dim: int, spatial_merge_size: int = 2) -> None:
        super().__init__()
        self.hidden_size = context_dim * (spatial_merge_size**2)
        self.ln_q = Qwen2RMSNorm(context_dim, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.mlp(self.ln_q(x).view(-1, self.hidden_size))
        return x


class Qwen2_5OmniVisionEncoder(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniVisionEncoderConfig
    _no_split_modules = ["Qwen2_5OmniVisionBlock"]

    def __init__(self, config: Qwen2_5OmniVisionEncoderConfig, *inputs, **kwargs) -> None:
        super().__init__(config, *inputs, **kwargs)
        self.spatial_merge_size = config.spatial_merge_size
        self.patch_size = config.patch_size
        self.fullatt_block_indexes = config.fullatt_block_indexes
        self.window_size = config.window_size
        self.spatial_merge_unit = self.spatial_merge_size * self.spatial_merge_size

        self.patch_embed = Qwen2_5_VisionPatchEmbed(
            patch_size=config.patch_size,
            temporal_patch_size=config.temporal_patch_size,
            in_channels=config.in_channels,
            embed_dim=config.hidden_size,
        )

        head_dim = config.hidden_size // config.num_heads
        self.rotary_pos_emb = Qwen2_5_VisionRotaryEmbedding(head_dim // 2)
        self.blocks = nn.ModuleList([Qwen2_5OmniVisionBlock(config) for _ in range(config.depth)])
        self.merger = Qwen2_5OmniPatchMerger(
            dim=config.out_hidden_size,
            context_dim=config.hidden_size,
            spatial_merge_size=config.spatial_merge_size,
        )
        self.gradient_checkpointing = False

    def rot_pos_emb(self, grid_thw):
        pos_ids = []
        for t, h, w in grid_thw:
            hpos_ids = torch.arange(h).unsqueeze(1).expand(-1, w)
            hpos_ids = hpos_ids.reshape(
                h // self.spatial_merge_size,
                self.spatial_merge_size,
                w // self.spatial_merge_size,
                self.spatial_merge_size,
            )
            hpos_ids = hpos_ids.permute(0, 2, 1, 3)
            hpos_ids = hpos_ids.flatten()

            wpos_ids = torch.arange(w).unsqueeze(0).expand(h, -1)
            wpos_ids = wpos_ids.reshape(
                h // self.spatial_merge_size,
                self.spatial_merge_size,
                w // self.spatial_merge_size,
                self.spatial_merge_size,
            )
            wpos_ids = wpos_ids.permute(0, 2, 1, 3)
            wpos_ids = wpos_ids.flatten()
            pos_ids.append(torch.stack([hpos_ids, wpos_ids], dim=-1).repeat(t, 1))
        pos_ids = torch.cat(pos_ids, dim=0)
        max_grid_size = grid_thw[:, 1:].max()
        rotary_pos_emb_full = self.rotary_pos_emb(max_grid_size)
        rotary_pos_emb = rotary_pos_emb_full[pos_ids].flatten(1)
        return rotary_pos_emb

    def get_window_index(self, grid_thw):
        window_index: list = []
        cu_window_seqlens: list = [0]
        window_index_id = 0
        vit_merger_window_size = self.window_size // self.spatial_merge_size // self.patch_size

        for grid_t, grid_h, grid_w in grid_thw:
            llm_grid_h, llm_grid_w = (
                grid_h // self.spatial_merge_size,
                grid_w // self.spatial_merge_size,
            )
            index = torch.arange(grid_t * llm_grid_h * llm_grid_w).reshape(grid_t, llm_grid_h, llm_grid_w)
            pad_h = vit_merger_window_size - llm_grid_h % vit_merger_window_size
            pad_w = vit_merger_window_size - llm_grid_w % vit_merger_window_size
            num_windows_h = (llm_grid_h + pad_h) // vit_merger_window_size
            num_windows_w = (llm_grid_w + pad_w) // vit_merger_window_size
            index_padded = F.pad(index, (0, pad_w, 0, pad_h), "constant", -100)
            index_padded = index_padded.reshape(
                grid_t,
                num_windows_h,
                vit_merger_window_size,
                num_windows_w,
                vit_merger_window_size,
            )
            index_padded = index_padded.permute(0, 1, 3, 2, 4).reshape(
                grid_t,
                num_windows_h * num_windows_w,
                vit_merger_window_size,
                vit_merger_window_size,
            )
            seqlens = (index_padded != -100).sum([2, 3]).reshape(-1)
            index_padded = index_padded.reshape(-1)
            index_new = index_padded[index_padded != -100]
            window_index.append(index_new + window_index_id)
            cu_seqlens_tmp = seqlens.cumsum(0) * self.spatial_merge_unit + cu_window_seqlens[-1]
            cu_window_seqlens.extend(cu_seqlens_tmp.tolist())
            window_index_id += (grid_t * llm_grid_h * llm_grid_w).item()
        window_index = torch.cat(window_index, dim=0)

        return window_index, cu_window_seqlens

    def forward(self, hidden_states: torch.Tensor, grid_thw: torch.Tensor) -> torch.Tensor:
        """
        Args:
            hidden_states (`torch.Tensor` of shape `(seq_len, hidden_size)`):
                The final hidden states of the model.
            grid_thw (`torch.Tensor` of shape `(num_images_or_videos, 3)`):
                The temporal, height and width of feature shape of each image in LLM.

        Returns:
            `torch.Tensor`: hidden_states.
        """
        hidden_states = self.patch_embed(hidden_states)
        rotary_pos_emb = self.rot_pos_emb(grid_thw)

        window_index, cu_window_seqlens = self.get_window_index(grid_thw)
        cu_window_seqlens = torch.tensor(
            cu_window_seqlens,
            device=hidden_states.device,
            dtype=grid_thw.dtype if torch.jit.is_tracing() else torch.int32,
        )
        cu_window_seqlens = torch.unique_consecutive(cu_window_seqlens)

        seq_len, _ = hidden_states.size()
        hidden_states = hidden_states.reshape(seq_len // self.spatial_merge_unit, self.spatial_merge_unit, -1)
        hidden_states = hidden_states[window_index, :, :]
        hidden_states = hidden_states.reshape(seq_len, -1)
        rotary_pos_emb = rotary_pos_emb.reshape(seq_len // self.spatial_merge_unit, self.spatial_merge_unit, -1)
        rotary_pos_emb = rotary_pos_emb[window_index, :, :]
        rotary_pos_emb = rotary_pos_emb.reshape(seq_len, -1)

        cu_seqlens = torch.repeat_interleave(grid_thw[:, 1] * grid_thw[:, 2], grid_thw[:, 0]).cumsum(
            dim=0,
            # Select dtype based on the following factors:
            #  - FA2 requires that cu_seqlens_q must have dtype int32
            #  - torch.onnx.export requires that cu_seqlens_q must have same dtype as grid_thw
            # See https://github.com/huggingface/transformers/pull/34852 for more information
            dtype=grid_thw.dtype if torch.jit.is_tracing() else torch.int32,
        )
        cu_seqlens = F.pad(cu_seqlens, (1, 0), value=0)

        # Modification here
        for layer_num, blk in enumerate(self.blocks):
            if layer_num in self.fullatt_block_indexes:
                cu_seqlens_now = cu_seqlens
            else:
                cu_seqlens_now = cu_window_seqlens
            if self.gradient_checkpointing and self.training:
                hidden_states = self._gradient_checkpointing_func(
                    blk.__call__, hidden_states, cu_seqlens_now, rotary_pos_emb
                )
            else:
                hidden_states = blk(
                    hidden_states,
                    cu_seqlens=cu_seqlens_now,
                    rotary_pos_emb=rotary_pos_emb,
                )
        hidden_states = self.merger(hidden_states)
        reverse_indices = torch.argsort(window_index)
        hidden_states = hidden_states[reverse_indices, :]

        return hidden_states


class Qwen2_5OmniRotaryEmbedding(nn.Module):
    def __init__(self, config: Qwen2_5OmniThinkerConfig, device=None):
        super().__init__()
        # BC: "rope_type" was originally "type"
        if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
            self.rope_type = config.rope_scaling.get("rope_type", config.rope_scaling.get("type"))
        else:
            self.rope_type = "default"
        self.max_seq_len_cached = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        self.config = config
        self.rope_init_fn = ROPE_INIT_FUNCTIONS[self.rope_type]

        inv_freq, self.attention_scaling = self.rope_init_fn(self.config, device)
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.original_inv_freq = self.inv_freq

    @torch.no_grad()
    @dynamic_rope_update  # power user: used with advanced RoPE types (e.g. dynamic rope)
    def forward(self, x, position_ids):
        # In contrast to other models, Qwen2_5Omni has different position ids for the grids
        # So we expand the inv_freq to shape (3, ...)
        inv_freq_expanded = self.inv_freq[None, None, :, None].float().expand(3, position_ids.shape[1], -1, 1)
        position_ids_expanded = position_ids[:, :, None, :].float()  # shape (3, bs, 1, positions)

        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):  # Force float32
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(2, 3)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos() * self.attention_scaling
            sin = emb.sin() * self.attention_scaling

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


def apply_multimodal_rotary_pos_emb(q, k, cos, sin, mrope_section, unsqueeze_dim=1):
    """Applies Rotary Position Embedding with Multimodal Sections to the query and key tensors (https://qwenlm.github.io/blog/qwen2-vl/).

    Explanation:
        Multimodal 3D rotary position embedding is an extension to 1D rotary position embedding. The input embedding
        sequence contains vision (images / videos) embedding and text embedding or just contains text embedding. For
        vision embedding part, we apply rotary position embedding on temporal, height and width dimension separately.
        Here we split the channel dimension to 3 chunks for the temporal, height and width rotary position embedding.
        For text embedding part, we just apply 1D rotary position embedding. The three rotary position index (temporal,
        height and width) of text embedding is always the same, so the text embedding rotary position embedding has no
        difference with modern LLMs.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        position_ids (`torch.Tensor`):
            The position indices of the tokens corresponding to the query and key tensors. For example, this can be
            used to pass offsetted position ids when working with a KV-cache.
        mrope_section(`List(int)`):
            Multimodal rope section is for channel dimension of temporal, height and width in rope calculation.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    mrope_section = mrope_section * 2
    cos = torch.cat([m[i % 3] for i, m in enumerate(cos.split(mrope_section, dim=-1))], dim=-1).unsqueeze(
        unsqueeze_dim
    )
    sin = torch.cat([m[i % 3] for i, m in enumerate(sin.split(mrope_section, dim=-1))], dim=-1).unsqueeze(
        unsqueeze_dim
    )

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    This is the equivalent of torch.repeat_interleave(x, dim=1, repeats=n_rep). The hidden states go from (batch,
    num_key_value_heads, seqlen, head_dim) to (batch, num_attention_heads, seqlen, head_dim)
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)


class Qwen2_5OmniAttention(nn.Module):
    """
    Multi-headed attention from 'Attention Is All You Need' paper. Modified to use sliding window attention: Longformer
    and "Generating Long Sequences with Sparse Transformers".
    """

    def __init__(self, config: Qwen2_5OmniConfig, layer_idx: Optional[int] = None):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        if layer_idx is None:
            logger.warning_once(
                f"Instantiating {self.__class__.__name__} without passing `layer_idx` is not recommended and will "
                "to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` "
                "when creating this class."
            )

        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = getattr(config, "head_dim", self.hidden_size // self.num_heads)
        self.num_key_value_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads
        self.is_causal = True
        self.attention_dropout = config.attention_dropout
        self.rope_scaling = config.rope_scaling

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=True)
        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)
        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        self.rotary_emb = Qwen2_5OmniRotaryEmbedding(config=config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Cache] = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # necessary, but kept here for BC
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        bsz, q_len, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_multimodal_rotary_pos_emb(
            query_states, key_states, cos, sin, self.rope_scaling["mrope_section"]
        )

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)

        # repeat k/v heads if n_kv_heads < n_heads
        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups)

        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)

        if attention_mask is not None:  # no matter the length, we just slice it
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
            attn_weights = attn_weights + causal_mask

        # Fix precision issues in Qwen2-VL float16 inference
        # Replace inf values with zeros in attention weights to prevent NaN propagation
        if query_states.dtype == torch.float16:
            attn_weights = torch.where(torch.isinf(attn_weights), torch.zeros_like(attn_weights), attn_weights)

        # upcast attention to fp32
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)
        attn_output = torch.matmul(attn_weights, value_states)

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, -1)

        attn_output = self.o_proj(attn_output)

        if not output_attentions:
            attn_weights = None

        return attn_output, attn_weights, past_key_value


class Qwen2MLP(nn.Module):
    def __init__(self, config, bias: bool = False):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=bias)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, hidden_state):
        return self.down_proj(self.act_fn(self.gate_proj(hidden_state)) * self.up_proj(hidden_state))

import math

def calculate_attention_weights(query, key, attn_mask=None, scale=None):
    
    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
    
    attn_scores = query @ key.transpose(2, 3) * scale_factor

    if attn_mask is not None:
        attn_scores = attn_scores.masked_fill(attn_mask == 0, float('-inf'))
    attn_weights = torch.softmax(attn_scores, dim=-1)
    return attn_weights

class Qwen2_5OmniFlashAttention2(Qwen2_5OmniAttention):
    """
    Qwen2_5Omni flash attention module, following Qwen2_5Omni attention module. This module inherits from `Qwen2_5OmniAttention`
    as the weights of the module stays untouched. The only required change would be on the forward pass
    where it needs to correctly call the public API of flash attention and deal with padding tokens
    in case the input contains any of them. Additionally, for sliding window attention, we apply SWA only to the bottom
    config.max_window_layers layers.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # TODO: Should be removed once Flash Attention for RoCm is bumped to 2.1.
        # flash_attn<2.1 generates top-left aligned causal mask, while what is needed here is bottom-right alignment, that was made default for flash_attn>=2.1. This attribute is used to handle this difference. Reference: https://github.com/Dao-AILab/flash-attention/releases/tag/v2.1.0.
        # Beware that with flash_attn<2.1, using q_seqlen != k_seqlen (except for the case q_seqlen == 1) produces a wrong mask (top-left).
        self._flash_attn_uses_top_left_mask = flash_attn_supports_top_left_mask()

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Cache] = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # necessary, but kept here for BC
    ):
        bsz, q_len, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)

        # Because the input can be padded, the absolute sequence length depends on the max position id.
        cos, sin = position_embeddings
        query_states, key_states = apply_multimodal_rotary_pos_emb(
            query_states, key_states, cos, sin, self.rope_scaling["mrope_section"]
        )

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)

        # repeat k/v heads if n_kv_heads < n_heads
        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups)
        dropout_rate = 0.0 if not self.training else self.attention_dropout

        # In PEFT, usually we cast the layer norms in float32 for training stability reasons
        # therefore the input hidden states gets silently casted in float32. Hence, we need
        # cast them back in float16 just to be sure everything works as expected.
        input_dtype = query_states.dtype
        if input_dtype == torch.float32:
            if torch.is_autocast_enabled():
                target_dtype = torch.get_autocast_gpu_dtype()
            # Handle the case where the model is quantized
            elif hasattr(self.config, "_pre_quantization_dtype"):
                target_dtype = self.config._pre_quantization_dtype
            else:
                target_dtype = self.q_proj.weight.dtype

            logger.warning_once(
                f"The input hidden states seems to be silently casted in float32, this might be related to"
                f" the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in"
                f" {target_dtype}."
            )

            query_states = query_states.to(target_dtype)
            key_states = key_states.to(target_dtype)
            value_states = value_states.to(target_dtype)

        if output_attentions:
            attn_weights = calculate_attention_weights(query_states, key_states, attn_mask=attention_mask)
        # Reashape to the expected shape for Flash Attention
        query_states = query_states.transpose(1, 2)
        key_states = key_states.transpose(1, 2)
        value_states = value_states.transpose(1, 2)

        if (
            self.config.use_sliding_window
            and getattr(self.config, "sliding_window", None) is not None
            and self.layer_idx >= self.config.max_window_layers
        ):
            sliding_window = self.config.sliding_window
        else:
            sliding_window = None

        attn_output = _flash_attention_forward(
            query_states,
            key_states,
            value_states,
            attention_mask,
            q_len,
            dropout=dropout_rate,
            sliding_window=sliding_window,
            is_causal=self.is_causal,
            use_top_left_mask=self._flash_attn_uses_top_left_mask,
        )

        attn_output = attn_output.reshape(bsz, q_len, -1).contiguous()
        attn_output = self.o_proj(attn_output)

        if not output_attentions:
            attn_weights = None

        return attn_output, attn_weights, past_key_value


class Qwen2_5OmniSdpaAttention(Qwen2_5OmniAttention):
    """
    Qwen2 attention module using torch.nn.functional.scaled_dot_product_attention. This module inherits from
    `Qwen2Attention` as the weights of the module stays untouched. The only changes are on the forward pass to adapt to
    SDPA API.
    """

    # Adapted from Qwen2Attention.forward
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Cache] = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # necessary, but kept here for BC
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        if output_attentions:
            # TODO: Improve this warning with e.g. `model.config.attn_implementation = "manual"` once this is implemented.
            logger.warning_once(
                "Qwen2_5OmniModel is using Qwen2_5OmniSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, "
                'but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.'
            )
            return super().forward(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=past_key_value,
                output_attentions=output_attentions,
                use_cache=use_cache,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
            )

        bsz, q_len, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, -1, self.head_dim).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_multimodal_rotary_pos_emb(
            query_states, key_states, cos, sin, self.rope_scaling["mrope_section"]
        )

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)

        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups)

        causal_mask = attention_mask
        if attention_mask is not None:  # no matter the length, we just slice it
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]

        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,
        # Reference: https://github.com/pytorch/pytorch/issues/112577.
        if query_states.device.type == "cuda" and attention_mask is not None:
            query_states = query_states.contiguous()
            key_states = key_states.contiguous()
            value_states = value_states.contiguous()

        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment
        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.
        # The q_len > 1 is necessary to match with AttentionMaskConverter.to_causal_4d that does not create a causal mask in case q_len == 1.
        is_causal = True if causal_mask is None and q_len > 1 else False

        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_states,
            key_states,
            value_states,
            attn_mask=causal_mask,
            dropout_p=self.attention_dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(bsz, q_len, -1)

        attn_output = self.o_proj(attn_output)

        return attn_output, None, past_key_value


QWEN2_5_OMNI_ATTENTION_CLASSES = {
    "eager": Qwen2_5OmniAttention,
    "flash_attention_2": Qwen2_5OmniFlashAttention2,
    "sdpa": Qwen2_5OmniSdpaAttention,
}


class Qwen2_5OmniDecoderLayer(nn.Module):
    def __init__(self, config: Qwen2_5OmniTextConfig, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size

        if config.use_sliding_window and config._attn_implementation != "flash_attention_2":
            logger.warning_once(
                f"Sliding Window Attention is enabled but not implemented for `{config._attn_implementation}`; "
                "unexpected results may be encountered."
            )
        self.self_attn = QWEN2_5_OMNI_ATTENTION_CLASSES[config._attn_implementation](config, layer_idx)

        self.mlp = Qwen2MLP(config)
        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Tuple[torch.Tensor]] = None,
        output_attentions: Optional[bool] = False,
        use_cache: Optional[bool] = False,
        cache_position: Optional[torch.LongTensor] = None,
        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # necessary, but kept here for BC
        **kwargs,
    ) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]:
        """
        Args:
            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`
            attention_mask (`torch.FloatTensor`, *optional*): attention mask of size
                `(batch, sequence_length)` where padding elements are indicated by 0.
            output_attentions (`bool`, *optional*):
                Whether or not to return the attentions tensors of all attention layers. See `attentions` under
                returned tensors for more detail.
            use_cache (`bool`, *optional*):
                If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding
                (see `past_key_values`).
            past_key_value (`Tuple(torch.FloatTensor)`, *optional*): cached past key and value projection states
            cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):
                Indices depicting the position of the input sequence tokens in the sequence.
            position_embeddings (`Tuple[torch.FloatTensor, torch.FloatTensor]`, *optional*):
                Tuple containing the cosine and sine positional embeddings of shape `(batch_size, seq_len, head_dim)`,
                with `head_dim` being the embedding dimension of each attention head.
            kwargs (`dict`, *optional*):
                Arbitrary kwargs to be ignored, used for FSDP and other methods that injects code
                into the model
        """

        residual = hidden_states

        hidden_states = self.input_layernorm(hidden_states)

        # Self Attention
        hidden_states, self_attn_weights, present_key_value = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            use_cache=use_cache,
            cache_position=cache_position,
            position_embeddings=position_embeddings,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        outputs = (hidden_states,)

        if output_attentions:
            outputs += (self_attn_weights,)

        if use_cache:
            outputs += (present_key_value,)

        return outputs


@auto_docstring
class Qwen2_5OmniThinkerTextModel(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniTextConfig
    _no_split_modules = ["Qwen2_5OmniDecoderLayer"]

    def __init__(self, config: Qwen2_5OmniTextConfig):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)
        self.layers = nn.ModuleList(
            [Qwen2_5OmniDecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]
        )
        self._attn_implementation = config._attn_implementation
        self.norm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Qwen2_5OmniRotaryEmbedding(config=config)

        self.gradient_checkpointing = False
        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self):
        return self.embed_tokens

    def set_input_embeddings(self, value):
        self.embed_tokens = value

    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Tuple, BaseModelOutputWithPast]:
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        use_cache = use_cache if use_cache is not None else self.config.use_cache

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if self.gradient_checkpointing and self.training:
            if use_cache:
                logger.warning_once(
                    "`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`..."
                )
                use_cache = False

        # torch.jit.trace() doesn't support cache objects in the output
        if use_cache and past_key_values is None and not torch.jit.is_tracing():
            past_key_values = DynamicCache()

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            cache_position = torch.arange(
                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
            )

        # the hard coded `3` is for temporal, height and width.
        if position_ids is None:
            position_ids = cache_position.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
        elif position_ids.dim() == 2:
            position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

        causal_mask = self._update_causal_mask(
            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions
        )

        hidden_states = inputs_embeds

        # if hidden_states.shape[1] > 768:
        #     print(hidden_states.shape)

        # create position embeddings to be shared across the decoder layers
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        # decoder layers
        all_hidden_states = () if output_hidden_states else None
        all_self_attns = () if output_attentions else None
        next_decoder_cache = None

        for layer_idx, decoder_layer in enumerate(self.layers):
            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            if self.gradient_checkpointing and self.training:
                layer_outputs = self._gradient_checkpointing_func(
                    decoder_layer.__call__,
                    hidden_states,
                    causal_mask,
                    position_ids,
                    past_key_values,
                    output_attentions,
                    use_cache,
                    cache_position,
                    position_embeddings,
                )
            else:
                output_attentions = False
                layer_outputs = decoder_layer(
                    hidden_states,
                    attention_mask=causal_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_values,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                )

            hidden_states = layer_outputs[0]

            if use_cache:
                next_decoder_cache = layer_outputs[2 if output_attentions else 1]

            if output_attentions:
                pass
                #all_self_attns += (layer_outputs[1],)

        hidden_states = self.norm(hidden_states)

        # add hidden states from the last decoder layer
        if output_hidden_states:
            all_hidden_states += (hidden_states,)

        next_cache = next_decoder_cache if use_cache else None

        if not return_dict:
            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)
        return BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=next_cache,
            hidden_states=all_hidden_states,
            attentions=all_self_attns,
        )

    def _update_causal_mask(
        self,
        attention_mask: Union[torch.Tensor, "BlockMask"],
        input_tensor: torch.Tensor,
        cache_position: torch.Tensor,
        past_key_values: Cache,
        output_attentions: bool = False,
    ):
        if self.config._attn_implementation == "flash_attention_2":
            if attention_mask is not None and past_key_values is not None:
                is_padding_right = attention_mask[:, -1].sum().item() != input_tensor.size()[0]
                if is_padding_right:
                    raise ValueError(
                        "You are attempting to perform batched generation with padding_side='right'"
                        " this may lead to unexpected behaviour for Flash Attention version of Qwen25OmniThinker. Make sure to "
                        " call `tokenizer.padding_side  = 'left'` before tokenizing the input. "
                    )
            if attention_mask is not None and 0.0 in attention_mask:
                return attention_mask
            return None
        if self.config._attn_implementation == "flex_attention":
            if isinstance(attention_mask, torch.Tensor):
                attention_mask = make_flex_block_causal_mask(attention_mask)
            return attention_mask

        # For SDPA, when possible, we will rely on its `is_causal` argument instead of its `attn_mask` argument, in
        # order to dispatch on Flash Attention 2. This feature is not compatible with static cache, as SDPA will fail
        # to infer the attention mask.
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        using_static_cache = isinstance(past_key_values, StaticCache)
        using_sliding_window_cache = isinstance(past_key_values, SlidingWindowCache)

        # When output attentions is True, sdpa implementation's forward method calls the eager implementation's forward
        if (
            self.config._attn_implementation == "sdpa"
            and not (using_static_cache or using_sliding_window_cache)
            and not output_attentions
        ):
            if AttentionMaskConverter._ignore_causal_mask_sdpa(
                attention_mask,
                inputs_embeds=input_tensor,
                past_key_values_length=past_seen_tokens,
                sliding_window=self.config.sliding_window,
                is_training=self.training,
            ):
                return None

        dtype = input_tensor.dtype
        min_dtype = torch.finfo(dtype).min
        sequence_length = input_tensor.shape[1]
        # SlidingWindowCache or StaticCache
        if using_sliding_window_cache or using_static_cache:
            target_length = past_key_values.get_max_cache_shape()
        # DynamicCache or no cache
        else:
            target_length = (
                attention_mask.shape[-1]
                if isinstance(attention_mask, torch.Tensor)
                else past_seen_tokens + sequence_length + 1
            )

        # In case the provided `attention` mask is 2D, we generate a causal mask here (4D).
        causal_mask = self._prepare_4d_causal_attention_mask_with_cache_position(
            attention_mask,
            sequence_length=sequence_length,
            target_length=target_length,
            dtype=dtype,
            cache_position=cache_position,
            batch_size=input_tensor.shape[0],
            config=self.config,
            past_key_values=past_key_values,
        )

        if (
            self.config._attn_implementation == "sdpa"
            and attention_mask is not None
            and attention_mask.device.type in ["cuda", "xpu", "npu"]
            and not output_attentions
        ):
            # Attend to all tokens in fully masked rows in the causal_mask, for example the relevant first rows when
            # using left padding. This is required by F.scaled_dot_product_attention memory-efficient attention path.
            # Details: https://github.com/pytorch/pytorch/issues/110213
            causal_mask = AttentionMaskConverter._unmask_unattended(causal_mask, min_dtype)

        return causal_mask

    @staticmethod
    def _prepare_4d_causal_attention_mask_with_cache_position(
        attention_mask: torch.Tensor,
        sequence_length: int,
        target_length: int,
        dtype: torch.dtype,
        cache_position: torch.Tensor,
        batch_size: int,
        config: Qwen2_5OmniConfig,
        past_key_values: Cache,
    ):
        """
        Creates a causal 4D mask of shape `(batch_size, 1, query_length, key_value_length)` from a 2D mask of shape
        `(batch_size, key_value_length)`, or if the input `attention_mask` is already 4D, do nothing.

        Args:
            attention_mask (`torch.Tensor`):
                A 2D attention mask of shape `(batch_size, key_value_length)` or a 4D attention mask of shape `(batch_size, 1, query_length, key_value_length)`.
            sequence_length (`int`):
                The sequence length being processed.
            target_length (`int`):
                The target length: when generating with static cache, the mask should be as long as the static cache, to account for the 0 padding, the part of the cache that is not filled yet.
            dtype (`torch.dtype`):
                The dtype to use for the 4D attention mask.
            cache_position (`torch.Tensor`):
                Indices depicting the position of the input sequence tokens in the sequence.
            batch_size (`torch.Tensor`):
                Batch size.
            config (`Qwen25OmniThinkerConfig`):
                The model's configuration class
            past_key_values (`Cache`):
                The cache class that is being used currently to generate
        """
        if attention_mask is not None and attention_mask.dim() == 4:
            # In this case we assume that the mask comes already in inverted form and requires no inversion or slicing.
            causal_mask = attention_mask
        else:
            min_dtype = torch.finfo(dtype).min
            causal_mask = torch.full(
                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=cache_position.device
            )
            diagonal_attend_mask = torch.arange(target_length, device=cache_position.device) > cache_position.reshape(
                -1, 1
            )
            text_config = config.get_text_config()
            if getattr(text_config, "use_sliding_window", True) and text_config.sliding_window is not None:
                # if we have sliding window, we should not attend to tokens beyond sliding window length, so we mask them out also
                # the check is needed to verify is current checkpoint was trained with sliding window or not
                if not isinstance(past_key_values, SlidingWindowCache) or sequence_length > target_length:
                    sliding_attend_mask = torch.arange(target_length, device=cache_position.device) <= (
                        cache_position.reshape(-1, 1) - text_config.sliding_window
                    )
                    diagonal_attend_mask.bitwise_or_(sliding_attend_mask)
            causal_mask *= diagonal_attend_mask
            causal_mask = causal_mask[None, None, :, :].expand(batch_size, 1, -1, -1)
            if attention_mask is not None:
                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit
                if attention_mask.shape[-1] > target_length:
                    attention_mask = attention_mask[:, :target_length]
                mask_length = attention_mask.shape[-1]
                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :].to(
                    causal_mask.device
                )
                padding_mask = padding_mask == 0
                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(
                    padding_mask, min_dtype
                )
        return causal_mask


@auto_docstring(
    custom_intro="""
    The Qwen2.5OmniThinker model which consists of a audio backbone and a language model.
    """
)
class Qwen2_5OmniThinkerForConditionalGeneration(Qwen2_5OmniPreTrainedModelForConditionalGeneration, GenerationMixin):
    config_class = Qwen2_5OmniThinkerConfig
    base_model_prefix = "thinker"
    _no_split_modules = ["Qwen2_5OmniAudioEncoder", "Qwen2_5OmniVisionEncoder"]

    def __init__(self, config: Qwen2_5OmniThinkerConfig):
        super().__init__(config)
        self.audio_tower = Qwen2_5OmniAudioEncoder._from_config(
            config.audio_config, attn_implementation=config._attn_implementation
        )

        self.visual = Qwen2_5OmniVisionEncoder._from_config(
            config.vision_config, attn_implementation=config._attn_implementation
        )

        self.vocab_size = config.text_config.vocab_size
        self.model = Qwen2_5OmniThinkerTextModel._from_config(
            config.text_config, attn_implementation=config._attn_implementation
        )
        self.lm_head = nn.Linear(config.text_config.hidden_size, config.text_config.vocab_size, bias=False)
        self.pad_token_id = self.config.pad_token_id if self.config.pad_token_id is not None else -1
        self.spatial_merge_size = config.vision_config.spatial_merge_size
        self.rope_deltas = None
        self.omnizip_config = None  # None -> forward() falls back to the paper's default ratios
        self.post_init()

    def get_input_embeddings(self):
        return self.model.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.model.set_input_embeddings(value)

    def get_video_features(
        self, pixel_values_videos: torch.FloatTensor, video_grid_thw: Optional[torch.LongTensor] = None
    ):
        """
        Encodes videos into continuous embeddings that can be forwarded to the language model.

        Args:
            pixel_values_videos (`torch.FloatTensor` of shape `(batch_size, num_channels, image_size, image_size)`):
                The tensors corresponding to the input videos.
            video_grid_thw (`torch.LongTensor` of shape `(num_videos, 3)`, *optional*):
                The temporal, height and width of feature shape of each video in LLM.
        """
        pixel_values_videos = pixel_values_videos.type(self.visual.dtype)
        video_embeds = self.visual(pixel_values_videos, grid_thw=video_grid_thw)
        return video_embeds

    def get_image_features(self, pixel_values: torch.FloatTensor, image_grid_thw: Optional[torch.LongTensor] = None):
        """
        Encodes images into continuous embeddings that can be forwarded to the language model.

        Args:
            pixel_values (`torch.FloatTensor` of shape `(batch_size, num_channels, image_size, image_size)`):
                The tensors corresponding to the input images.
            image_grid_thw (`torch.LongTensor` of shape `(num_images, 3)`, *optional*):
                The temporal, height and width of feature shape of each image in LLM.
        """
        pixel_values = pixel_values.type(self.visual.dtype)
        image_embeds = self.visual(pixel_values, grid_thw=image_grid_thw)
        return image_embeds

    def get_audio_features(
        self,
        input_features: torch.FloatTensor,
        feature_attention_mask: Optional[torch.LongTensor] = None,
        audio_feature_lengths: Optional[torch.LongTensor] = None,
    ):
        """
        Encodes audios into continuous embeddings that can be forwarded to the language model.

        Args:
            input_features (`torch.FloatTensor`):
                The tensors corresponding to the input audios.
            feature_attention_mask (`torch.LongTensor`, *optional*):
                Mask to avoid performing attention on padding feature indices. Mask values selected in `[0, 1]`:
            audio_feature_lengths (`torch.LongTensor` of shape `(num_audios)`, *optional*):
                The length of feature shape of each audio in LLM.
        """
        if feature_attention_mask is not None:
            audio_feature_lengths = torch.sum(feature_attention_mask, dim=1)
            input_features = input_features.permute(0, 2, 1)[feature_attention_mask.bool()].permute(1, 0)
        else:
            audio_feature_lengths = None

        audio_feat_lengths, audio_output_lengths = self.audio_tower._get_feat_extract_output_lengths(
            audio_feature_lengths if audio_feature_lengths is not None else feature_attention_mask.sum(-1)
        )
        feature_lens = audio_feature_lengths if audio_feature_lengths is not None else feature_attention_mask.sum(-1)
        audio_outputs, attn_logits, attn_key = self.audio_tower(
            input_features,
            feature_lens=feature_lens,
            aftercnn_lens=audio_feat_lengths,
        )
        audio_features = audio_outputs.last_hidden_state

        if audio_features.shape[0] != sum(audio_output_lengths.tolist()):
            raise ValueError("length of audio_features should match audio_output_lengths")

        return audio_features, attn_logits, attn_key

    def random_pruning(self, video_embeds, ratio):
        # Baseline method random_pruning
        num_video_tokens = video_embeds.shape[0]
        num_pruned_tokens = int(num_video_tokens * ratio)
        pruned_indices = torch.randperm(num_video_tokens)[:num_pruned_tokens]
        video_embeds = video_embeds[pruned_indices]
        
        # Create mask for pruned tokens
        mask = torch.zeros(num_video_tokens, dtype=torch.bool, device=video_embeds.device)
        mask[pruned_indices] = True
        
        return video_embeds, mask

    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        input_features: Optional[torch.FloatTensor] = None,
        pixel_values: Optional[torch.FloatTensor] = None,
        pixel_values_videos: Optional[torch.FloatTensor] = None,
        image_grid_thw: Optional[torch.LongTensor] = None,
        video_grid_thw: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        feature_attention_mask: Optional[torch.Tensor] = None,
        audio_feature_lengths: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        rope_deltas: Optional[torch.LongTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        use_audio_in_video: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        video_second_per_grid: Optional[torch.LongTensor] = None,
    ) -> Union[Tuple, Qwen2_5OmniThinkerCausalLMOutputWithPast]:
        r"""
        input_features (`torch.FloatTensor` of shape `(batch_size, feature_size, feature_sequence_length)`):
            Float values mel features extracted from the raw speech waveform. Raw speech waveform can be obtained by
            loading a `.flac` or `.wav` audio file into an array of type `List[float]` or a `numpy.ndarray`, *e.g.* via
            the soundfile library (`pip install soundfile`). To prepare the array into `input_features`, the
            [`AutoFeatureExtractor`] should be used for extracting the mel features, padding and conversion into a
            tensor of type `torch.FloatTensor`. See [`~WhisperFeatureExtractor.__call__`]
        pixel_values_videos (`torch.FloatTensor` of shape `(batch_size, num_channels, image_size, image_size), *optional*):
            The tensors corresponding to the input videos. Pixel values can be obtained using
            [`AutoImageProcessor`]. See [`SiglipImageProcessor.__call__`] for details ([]`NewTaskModelProcessor`] uses
            [`SiglipImageProcessor`] for processing videos).
        image_grid_thw (`torch.LongTensor` of shape `(num_images, 3)`, *optional*):
            The temporal, height and width of feature shape of each image in LLM.
        video_grid_thw (`torch.LongTensor` of shape `(num_videos, 3)`, *optional*):
            The temporal, height and width of feature shape of each video in LLM.
        feature_attention_mask (`torch.Tensor` of shape `(batch_size, feature_sequence_length)`, *optional*):
            Mask to avoid performing attention on padding feature indices. Mask values selected in `[0, 1]`:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.
        audio_feature_lengths (`torch.LongTensor` of shape `(num_audios)`, *optional*):
            The length of feature shape of each audio in LLM.
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
            config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored
            (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.
        use_audio_in_video (`bool`, *optional*):
            Whether or not use audio track in video, should same as the parameter in `process_audio_info`.
        video_second_per_grid (`torch.LongTensor` of shape `(num_videos)`, *optional*):
            Number of seconds per grid for each video, used for temporal feature mapping.

        Example:

        ```python
        >>> from io import BytesIO
        >>> from urllib.request import urlopen
        >>> import librosa
        >>> from qwen_vl_utils import process_vision_info
        >>> from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration

        >>> thinker = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-Omni-7B")
        >>> processor = Qwen2_5OmniProcessor.from_pretrained("Qwen/Qwen2.5-Omni-7B")

        >>> conversations = [
        >>>         {'role': 'system', 'content': 'You are a helpful voice chat bot, and please respond to me in a casual conversation manner using random voice.'},
        >>>         {"role": "user", "content": [
        >>>             {"type": "image", "image_url": "https://www.ilankelman.org/stopsigns/australia.jpg"},
        >>>             {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/glass-breaking-151256.mp3"},
        >>>         ]},
        >>> ]

        >>> text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        >>> audios = [ librosa.load(BytesIO(urlopen( conversations[1]['content'][1]['audio_url'] ).read()), sr=self.processor.feature_extractor.sampling_rate) ]
        >>> images, videos = process_vision_info(conversations)
        >>> inputs = processor(text=text, audios=audios, images=images, videos=videos, return_tensors="pt", padding=True)

        >>> # Generate
        >>> inputs['use_audio_in_video'] = `True` or `False`
        >>> generation = thinker.generate(**inputs, max_new_tokens=2048)
        >>> generate_ids = generation[:, inputs.input_ids.size(1):]

        >>> response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        ```"""

        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        # Only set when audio is present; the OmniZip gate below reads them for every video input
        audio_features = attn_logits = None

        if inputs_embeds is None:
            # 1. Extract the input embeddings
            inputs_embeds = self.get_input_embeddings()(input_ids)

        # 2. Merge text , audios , image and video
        if input_ids is not None and input_ids.shape[1] != 1:  # Prefill stage
            if input_features is not None:
                #print("Audio Features Shape: ", input_features.shape)
                audio_features, attn_logits, attn_key = self.get_audio_features(
                    input_features,
                    feature_attention_mask=feature_attention_mask,
                    audio_feature_lengths=audio_feature_lengths,
                )
                audio_mask = (
                    (input_ids == self.config.audio_token_id)
                    .unsqueeze(-1)
                    .expand_as(inputs_embeds)
                    .to(inputs_embeds.device)
                )
                audio_features = audio_features.to(inputs_embeds.device, inputs_embeds.dtype)
                inputs_embeds = inputs_embeds.masked_scatter(audio_mask, audio_features)

            if pixel_values is not None:
                image_embeds = self.get_image_features(pixel_values, image_grid_thw)
                image_mask = (
                    (input_ids == self.config.image_token_id)
                    .unsqueeze(-1)
                    .expand_as(inputs_embeds)
                    .to(inputs_embeds.device)
                )
                image_embeds = image_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
                inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds)

            if pixel_values_videos is not None:
                
                video_embeds = self.get_video_features(pixel_values_videos, video_grid_thw)
                video_mask = (
                    (input_ids == self.config.video_token_id)
                    .unsqueeze(-1)
                    .expand_as(inputs_embeds)
                    .to(inputs_embeds.device)
                )
                video_embeds = video_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
                inputs_embeds = inputs_embeds.masked_scatter(video_mask, video_embeds)

            if attention_mask is not None:
                attention_mask = attention_mask.to(inputs_embeds.device)

        if feature_attention_mask is not None:
            audio_feature_lengths = torch.sum(feature_attention_mask, dim=1)
        else:
            audio_feature_lengths = None

        if attention_mask is not None and position_ids is None:
            if (
                cache_position is None
                or (cache_position is not None and cache_position[0] == 0)
                or self.rope_deltas is None
            ):
                delta0 = (1 - attention_mask).sum(dim=-1).unsqueeze(1)
                position_ids, rope_deltas = self.get_rope_index(
                    input_ids,
                    image_grid_thw,
                    video_grid_thw,
                    attention_mask,
                    use_audio_in_video,
                    audio_feature_lengths,
                    video_second_per_grid,
                )
                rope_deltas = rope_deltas - delta0
                self.rope_deltas = rope_deltas
            else:
                batch_size, seq_length = input_ids.shape
                delta = cache_position[0] + self.rope_deltas if cache_position is not None else 0
                position_ids = torch.arange(seq_length, device=input_ids.device)
                position_ids = position_ids.view(1, -1).expand(batch_size, -1)
                position_ids = position_ids.add(delta)
                position_ids = position_ids.unsqueeze(0).expand(3, -1, -1)

        if pixel_values_videos is not None and audio_features is not None and attn_logits is not None:
            from omnizip.omnizip_core import omnizip
            if self.omnizip_config is not None:
                omnizip_config = self.omnizip_config
            else:
                omnizip_config = {
                    "rho_audio": 0.3,
                    "rho_video": 0.6,
                    "g": 3,
                    "contextual_ratio": 0.05,
                }

            inputs_embeds, global_mask = omnizip(
                input_embeds=inputs_embeds,
                attn_logits=attn_logits,
                input_ids=input_ids,
                audio_token_id=self.config.audio_token_id,
                video_token_id=self.config.video_token_id,
                video_grid_thw=video_grid_thw,
                audio_tokens_per_sec=25,
                merging_ratio_audio=omnizip_config["rho_audio"],
                merging_ratio_v=omnizip_config["rho_video"],
                contextual_ratio=omnizip_config["contextual_ratio"],
                g=omnizip_config["g"],
                attn_layer=self.model.layers[0],
            )

            inputs_embeds = inputs_embeds[:, global_mask, :]
            if attention_mask is not None:
                attention_mask = attention_mask[:, global_mask]
            position_ids = position_ids.to(inputs_embeds.device)[:, :, global_mask]

        outputs = self.model(
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            cache_position=cache_position,
        )

        hidden_states = outputs[0]
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            loss = self.loss_function(
                logits=logits, labels=labels, vocab_size=self.config.get_text_config().vocab_size
            )

        if not return_dict:
            output = (logits,) + outputs
            return (loss,) + output if loss is not None else output
       
        return Qwen2_5OmniThinkerCausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
            rope_deltas=self.rope_deltas,
        )

    def prepare_inputs_for_generation(
        self,
        input_ids,
        past_key_values=None,
        attention_mask=None,
        inputs_embeds=None,
        cache_position=None,
        position_ids=None,
        use_cache=True,
        pixel_values=None,
        pixel_values_videos=None,
        image_grid_thw=None,
        video_grid_thw=None,
        input_features=None,
        feature_attention_mask=None,
        use_audio_in_video=False,
        video_second_per_grid=None,
        **kwargs,
    ):
        model_inputs = super().prepare_inputs_for_generation(
            input_ids,
            past_key_values=past_key_values,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            cache_position=cache_position,
            position_ids=position_ids,
            use_cache=use_cache,
            pixel_values=pixel_values,
            pixel_values_videos=pixel_values_videos,
            image_grid_thw=image_grid_thw,
            video_grid_thw=video_grid_thw,
            input_features=input_features,
            feature_attention_mask=feature_attention_mask,
            use_audio_in_video=use_audio_in_video,
            video_second_per_grid=video_second_per_grid,
            **kwargs,
        )

        model_inputs["position_ids"] = None

        if cache_position[0] != 0:
            model_inputs["pixel_values"] = None
            model_inputs["pixel_values_videos"] = None

        return model_inputs


############################
#       Start Talker       #
############################


@dataclass
class Qwen2_5OmniTalkerCausalLMOutputWithPast(ModelOutput):
    """
    Base class for Qwen2.5OmniTalker causal language model (or autoregressive) outputs.

    Args:
        loss (`torch.FloatTensor` of shape `(1,)`, *optional*, returned when `labels` is provided):
            Language modeling loss (for next-token prediction).
        logits (`torch.FloatTensor` of shape `(batch_size, sequence_length, config.vocab_size)`):
            Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
        past_key_values (`tuple(tuple(torch.FloatTensor))`, *optional*, returned when `use_cache=True` is passed or when `config.use_cache=True`):
            Tuple of `tuple(torch.FloatTensor)` of length `config.n_layers`, with each tuple having 2 tensors of shape
            `(batch_size, num_heads, sequence_length, embed_size_per_head)`)

            Contains pre-computed hidden-states (key and values in the self-attention blocks) that can be used (see
            `past_key_values` input) to speed up sequential decoding.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.

            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):
            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,
            sequence_length)`.

            Attentions weights after the attention softmax, used to compute the weighted average in the self-attention
            heads.
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
    """

    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    past_key_values: Optional[List[torch.FloatTensor]] = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None
    rope_deltas: Optional[torch.LongTensor] = None
    thinker_reply_part: torch.FloatTensor = None


@auto_docstring
class Qwen2_5OmniTalkerModel(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniTalkerConfig
    _no_split_modules = ["Qwen2_5OmniTalkerDecoderLayer"]

    def __init__(self, config: Qwen2_5OmniTalkerConfig):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size
        self.embed_tokens = nn.Embedding(config.vocab_size, config.embedding_size, self.padding_idx)
        self.layers = nn.ModuleList(
            [Qwen2_5OmniDecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]
        )
        self._attn_implementation = config._attn_implementation
        self.norm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Qwen2_5OmniRotaryEmbedding(config=config)

        self.gradient_checkpointing = False
        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self):
        return self.embed_tokens

    def set_input_embeddings(self, value):
        self.embed_tokens = value

    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Tuple, BaseModelOutputWithPast]:
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        use_cache = use_cache if use_cache is not None else self.config.use_cache

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if self.gradient_checkpointing and self.training:
            if use_cache:
                logger.warning_once(
                    "`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`..."
                )
                use_cache = False

        # torch.jit.trace() doesn't support cache objects in the output
        if use_cache and past_key_values is None and not torch.jit.is_tracing():
            past_key_values = DynamicCache()

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            cache_position = torch.arange(
                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
            )

        # the hard coded `3` is for temporal, height and width.
        if position_ids is None:
            position_ids = cache_position.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
        elif position_ids.dim() == 2:
            position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

        causal_mask = self._update_causal_mask(
            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions
        )

        hidden_states = inputs_embeds

        # create position embeddings to be shared across the decoder layers
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        # decoder layers
        all_hidden_states = () if output_hidden_states else None
        all_self_attns = () if output_attentions else None
        next_decoder_cache = None

        for decoder_layer in self.layers:
            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            if self.gradient_checkpointing and self.training:
                layer_outputs = self._gradient_checkpointing_func(
                    decoder_layer.__call__,
                    hidden_states,
                    causal_mask,
                    position_ids,
                    past_key_values,
                    output_attentions,
                    use_cache,
                    cache_position,
                    position_embeddings,
                )
            else:
                layer_outputs = decoder_layer(
                    hidden_states,
                    attention_mask=causal_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_values,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                )

            hidden_states = layer_outputs[0]

            if use_cache:
                next_decoder_cache = layer_outputs[2 if output_attentions else 1]

            if output_attentions:
                all_self_attns += (layer_outputs[1],)

        hidden_states = self.norm(hidden_states)

        # add hidden states from the last decoder layer
        if output_hidden_states:
            all_hidden_states += (hidden_states,)

        next_cache = next_decoder_cache if use_cache else None

        if not return_dict:
            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)
        return BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=next_cache,
            hidden_states=all_hidden_states,
            attentions=all_self_attns,
        )

    def _update_causal_mask(
        self,
        attention_mask: Union[torch.Tensor, "BlockMask"],
        input_tensor: torch.Tensor,
        cache_position: torch.Tensor,
        past_key_values: Cache,
        output_attentions: bool = False,
    ):
        if self.config._attn_implementation == "flash_attention_2":
            if attention_mask is not None and past_key_values is not None:
                is_padding_right = attention_mask[:, -1].sum().item() != input_tensor.size()[0]
                if is_padding_right:
                    raise ValueError(
                        "You are attempting to perform batched generation with padding_side='right'"
                        " this may lead to unexpected behaviour for Flash Attention version of Qwen2_5Omni. Make sure to "
                        " call `tokenizer.padding_side  = 'left'` before tokenizing the input. "
                    )
            if attention_mask is not None and 0.0 in attention_mask:
                return attention_mask
            return None
        if self.config._attn_implementation == "flex_attention":
            if isinstance(attention_mask, torch.Tensor):
                attention_mask = make_flex_block_causal_mask(attention_mask)
            return attention_mask

        # For SDPA, when possible, we will rely on its `is_causal` argument instead of its `attn_mask` argument, in
        # order to dispatch on Flash Attention 2. This feature is not compatible with static cache, as SDPA will fail
        # to infer the attention mask.
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        using_static_cache = isinstance(past_key_values, StaticCache)
        using_sliding_window_cache = isinstance(past_key_values, SlidingWindowCache)

        # When output attentions is True, sdpa implementation's forward method calls the eager implementation's forward
        if (
            self.config._attn_implementation == "sdpa"
            and not (using_static_cache or using_sliding_window_cache)
            and not output_attentions
        ):
            if AttentionMaskConverter._ignore_causal_mask_sdpa(
                attention_mask,
                inputs_embeds=input_tensor,
                past_key_values_length=past_seen_tokens,
                sliding_window=self.config.sliding_window,
                is_training=self.training,
            ):
                return None

        dtype = input_tensor.dtype
        min_dtype = torch.finfo(dtype).min
        sequence_length = input_tensor.shape[1]
        # SlidingWindowCache or StaticCache
        if using_sliding_window_cache or using_static_cache:
            target_length = past_key_values.get_max_cache_shape()
        # DynamicCache or no cache
        else:
            target_length = (
                attention_mask.shape[-1]
                if isinstance(attention_mask, torch.Tensor)
                else past_seen_tokens + sequence_length + 1
            )

        # In case the provided `attention` mask is 2D, we generate a causal mask here (4D).
        causal_mask = self._prepare_4d_causal_attention_mask_with_cache_position(
            attention_mask,
            sequence_length=sequence_length,
            target_length=target_length,
            dtype=dtype,
            cache_position=cache_position,
            batch_size=input_tensor.shape[0],
            config=self.config,
            past_key_values=past_key_values,
        )

        if (
            self.config._attn_implementation == "sdpa"
            and attention_mask is not None
            and attention_mask.device.type in ["cuda", "xpu", "npu"]
            and not output_attentions
        ):
            # Attend to all tokens in fully masked rows in the causal_mask, for example the relevant first rows when
            # using left padding. This is required by F.scaled_dot_product_attention memory-efficient attention path.
            # Details: https://github.com/pytorch/pytorch/issues/110213
            causal_mask = AttentionMaskConverter._unmask_unattended(causal_mask, min_dtype)

        return causal_mask

    @staticmethod
    def _prepare_4d_causal_attention_mask_with_cache_position(
        attention_mask: torch.Tensor,
        sequence_length: int,
        target_length: int,
        dtype: torch.dtype,
        cache_position: torch.Tensor,
        batch_size: int,
        config: Qwen2_5OmniConfig,
        past_key_values: Cache,
    ):
        """
        Creates a causal 4D mask of shape `(batch_size, 1, query_length, key_value_length)` from a 2D mask of shape
        `(batch_size, key_value_length)`, or if the input `attention_mask` is already 4D, do nothing.

        Args:
            attention_mask (`torch.Tensor`):
                A 2D attention mask of shape `(batch_size, key_value_length)` or a 4D attention mask of shape `(batch_size, 1, query_length, key_value_length)`.
            sequence_length (`int`):
                The sequence length being processed.
            target_length (`int`):
                The target length: when generating with static cache, the mask should be as long as the static cache, to account for the 0 padding, the part of the cache that is not filled yet.
            dtype (`torch.dtype`):
                The dtype to use for the 4D attention mask.
            cache_position (`torch.Tensor`):
                Indices depicting the position of the input sequence tokens in the sequence.
            batch_size (`torch.Tensor`):
                Batch size.
            config (`Qwen2_5OmniConfig`):
                The model's configuration class
            past_key_values (`Cache`):
                The cache class that is being used currently to generate
        """
        if attention_mask is not None and attention_mask.dim() == 4:
            # In this case we assume that the mask comes already in inverted form and requires no inversion or slicing.
            causal_mask = attention_mask
        else:
            min_dtype = torch.finfo(dtype).min
            causal_mask = torch.full(
                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=cache_position.device
            )
            diagonal_attend_mask = torch.arange(target_length, device=cache_position.device) > cache_position.reshape(
                -1, 1
            )
            text_config = config.get_text_config()
            if getattr(text_config, "use_sliding_window", True) and text_config.sliding_window is not None:
                # if we have sliding window, we should not attend to tokens beyond sliding window length, so we mask them out also
                # the check is needed to verify is current checkpoint was trained with sliding window or not
                if not isinstance(past_key_values, SlidingWindowCache) or sequence_length > target_length:
                    sliding_attend_mask = torch.arange(target_length, device=cache_position.device) <= (
                        cache_position.reshape(-1, 1) - text_config.sliding_window
                    )
                    diagonal_attend_mask.bitwise_or_(sliding_attend_mask)
            causal_mask *= diagonal_attend_mask
            causal_mask = causal_mask[None, None, :, :].expand(batch_size, 1, -1, -1)
            if attention_mask is not None:
                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit
                if attention_mask.shape[-1] > target_length:
                    attention_mask = attention_mask[:, :target_length]
                mask_length = attention_mask.shape[-1]
                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :].to(
                    causal_mask.device
                )
                padding_mask = padding_mask == 0
                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(
                    padding_mask, min_dtype
                )
        return causal_mask


class Qwen2_5OmniTalkerForConditionalGeneration(Qwen2_5OmniPreTrainedModelForConditionalGeneration, GenerationMixin):
    config_class = Qwen2_5OmniTalkerConfig
    base_model_prefix = "talker"

    def __init__(self, config: Qwen2_5OmniTalkerConfig):
        super().__init__(config)

        self.thinker_to_talker_proj = nn.Linear(config.embedding_size, config.hidden_size)

        self.model = Qwen2_5OmniTalkerModel(config)
        self.codebook_size = config.vocab_size
        self.codec_head = nn.Linear(config.hidden_size, self.codebook_size, bias=False)

        self.codec_bos_token = config.tts_codec_start_token_id
        self.codec_eos_token = config.tts_codec_end_token_id
        self.codec_pad_token = config.tts_codec_pad_token_id
        self.codec_mask_token = config.tts_codec_mask_token_id

        self.text_bos_token = config.tts_text_start_token_id
        self.text_eos_token = config.tts_text_end_token_id
        self.text_pad_token = config.tts_text_pad_token_id

        self.spatial_merge_size = self.config.spatial_merge_size
        self.rope_deltas = None

        self.post_init()

    def get_input_embeddings(self):
        return self.model.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.model.set_input_embeddings(value)

    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        thinker_reply_part: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        rope_deltas: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        input_text_ids: Optional[torch.LongTensor] = None,
        image_grid_thw: Optional[torch.LongTensor] = None,
        video_grid_thw: Optional[torch.LongTensor] = None,
        use_audio_in_video: Optional[bool] = None,
        audio_feature_lengths: Optional[torch.LongTensor] = None,
        video_second_per_grid: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, Qwen2_5OmniTalkerCausalLMOutputWithPast]:
        r"""
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
        image_grid_thw (`torch.LongTensor` of shape `(num_images, 3)`, *optional*):
            The temporal, height and width of feature shape of each image in LLM.
        video_grid_thw (`torch.LongTensor` of shape `(num_videos, 3)`, *optional*):
            The temporal, height and width of feature shape of each video in LLM.
        audio_feature_lengths (`torch.LongTensor` of shape `(num_audios)`, *optional*):
            The length of feature shape of each audio in LLM.
        thinker_reply_part (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`, *optional*):
            Hidden states from the thinker model's output that represent the text reply part to be processed.
        input_text_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Input token IDs for text-only content, used for position calculation in multimodal contexts.
        use_audio_in_video (`bool`, *optional*):
            Whether or not use audio track in video, should same as the parameter in `process_audio_info`.
        video_second_per_grid (`torch.LongTensor` of shape `(num_videos)`, *optional*):
            Number of seconds per grid for each video, used for temporal feature mapping.

        Example:

        ```python
        >>> from io import BytesIO
        >>> from urllib.request import urlopen
        >>> import librosa
        >>> from transformers import AutoProcessor, Qwen2_5OmniTalkerForConditionalGeneration

        >>> model = Qwen2_5OmniTalkerForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B")
        >>> processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B")

        >>> prompt = "<|audio_bos|><|AUDIO|><|audio_eos|>Generate the caption in English:"
        >>> url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/glass-breaking-151256.mp3"
        >>> audio, _ = librosa.load(BytesIO(urlopen(url).read()), sr=self.processor.feature_extractor.sampling_rate)

        >>> inputs = processor(text=prompt, audios=audio, return_tensors="pt")

        >>> # Generate
        >>> generate_ids = model.generate(**inputs, max_length=30)
        >>> processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        "Generate the caption in English: Glass is breaking."
        ```"""

        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if attention_mask is not None and position_ids is None:
            if (
                cache_position is None
                or (cache_position is not None and cache_position[0] == 0)
                or self.rope_deltas is None
            ):
                position_ids, rope_deltas = self.get_rope_index(
                    input_text_ids,
                    image_grid_thw,
                    video_grid_thw,
                    attention_mask,
                    use_audio_in_video,
                    audio_feature_lengths,
                    video_second_per_grid,
                )

                inputs_embeds[:, -1, :] += self.get_input_embeddings()(
                    torch.tensor([self.codec_bos_token], dtype=torch.long, device=inputs_embeds.device)
                )
                inputs_embeds[:, -2, :] += self.get_input_embeddings()(
                    torch.tensor([self.codec_pad_token], dtype=torch.long, device=inputs_embeds.device)
                )
                self.rope_deltas = rope_deltas

            else:
                batch_size, seq_length = input_ids.shape
                delta = cache_position[0] + self.rope_deltas if cache_position is not None else 0
                position_ids = torch.arange(seq_length, device=input_ids.device)
                position_ids = position_ids.view(1, -1).expand(batch_size, -1)
                position_ids = position_ids.add(delta)
                position_ids = position_ids.unsqueeze(0).expand(3, -1, -1)

        if inputs_embeds is None:
            # 1. Inference tokens after second token
            codec_embeds = self.get_input_embeddings()(input_ids)
            inputs_embeds = codec_embeds + thinker_reply_part[:, :1, :]
            if thinker_reply_part.shape[1] > 1:
                thinker_reply_part = thinker_reply_part[:, 1:, :]

        talker_lm_input = self.thinker_to_talker_proj(inputs_embeds)

        if attention_mask is not None:
            attention_mask = attention_mask.to(inputs_embeds.device)

        outputs = self.model(
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=talker_lm_input,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        hidden_states = outputs[0]
        logits = self.codec_head(hidden_states)
        logits = logits.float()

        loss = None

        if not return_dict:
            output = (logits,) + outputs[1:]
            return (loss,) + output if loss is not None else output

        return Qwen2_5OmniTalkerCausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=hidden_states,
            attentions=outputs.attentions,
            rope_deltas=self.rope_deltas,
            thinker_reply_part=thinker_reply_part,
        )

    def _get_initial_cache_position(self, seq_length, device, model_kwargs):
        # Talker needs to calculate cache_position with input_ids, so pop inputs_embeds temporarily
        inputs_embeds = model_kwargs.pop("inputs_embeds")
        model_kwargs = super()._get_initial_cache_position(seq_length, device, model_kwargs)
        model_kwargs["inputs_embeds"] = inputs_embeds
        return model_kwargs

    # prepare inputs for talker lm generation
    def prepare_inputs_for_generation(
        self,
        input_ids,
        input_text_ids,
        past_key_values=None,
        attention_mask=None,
        inputs_embeds=None,
        thinker_reply_part=None,
        cache_position=None,
        position_ids=None,
        use_cache=True,
        pixel_values=None,
        pixel_values_videos=None,
        image_grid_thw=None,
        video_grid_thw=None,
        input_audio_features=None,
        audio_feature_attention_mask=None,
        audio_feature_lengths=None,
        use_audio_in_video=False,
        video_second_per_grid=None,
        **kwargs,
    ):
        model_inputs = super().prepare_inputs_for_generation(
            input_ids,
            past_key_values,
            attention_mask,
            inputs_embeds,
            cache_position,
            use_cache=use_cache,
            thinker_reply_part=thinker_reply_part,
            input_text_ids=input_text_ids,
            image_grid_thw=image_grid_thw,
            video_grid_thw=video_grid_thw,
            use_audio_in_video=use_audio_in_video,
            audio_feature_lengths=audio_feature_lengths,
            video_second_per_grid=video_second_per_grid,
            **kwargs,
        )

        model_inputs["position_ids"] = None

        return model_inputs

    def _update_model_kwargs_for_generation(
        self,
        outputs: ModelOutput,
        model_kwargs: Dict[str, Any],
        is_encoder_decoder: bool = False,
        num_new_tokens: int = 1,
    ) -> Dict[str, Any]:
        model_kwargs = super()._update_model_kwargs_for_generation(
            outputs, model_kwargs, is_encoder_decoder, num_new_tokens
        )

        if getattr(outputs, "thinker_reply_part", None) is not None:
            model_kwargs["thinker_reply_part"] = outputs.thinker_reply_part

        return model_kwargs


############################
#      Start Token2Wav     #
############################


# Using custom RoPE, will use LlamaRotaryEmbedding next version
class Qwen2_5OmniDiTRotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()

        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        batch_size, seq_len = x.shape[0], x.shape[1]
        t = torch.arange(seq_len, device=x.device)
        device_type = x.device.type
        device_type = device_type if isinstance(device_type, str) and device_type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):
            freqs = t.unsqueeze(1).float() @ self.inv_freq.unsqueeze(0).float()
            freqs = torch.stack((freqs, freqs), dim=-1)
            freqs = freqs.reshape(*freqs.shape[:-2], -1)
            freqs = freqs.repeat(batch_size, *([1] * freqs.dim()))
            cos = freqs.cos()
            sin = freqs.sin()

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


class TimeDelayNetBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
    ):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding="same",
            padding_mode="reflect",
        )
        self.activation = nn.ReLU()

    def forward(self, hidden_states: torch.Tensor):
        return self.activation(self.conv(hidden_states))


class Res2NetBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, scale=8, kernel_size=3, dilation=1):
        super().__init__()

        in_channel = in_channels // scale
        hidden_channel = out_channels // scale

        self.blocks = nn.ModuleList(
            [
                TimeDelayNetBlock(
                    in_channel,
                    hidden_channel,
                    kernel_size=kernel_size,
                    dilation=dilation,
                )
                for i in range(scale - 1)
            ]
        )
        self.scale = scale

    def forward(self, hidden_states):
        outputs = []
        for i, hidden_part in enumerate(torch.chunk(hidden_states, self.scale, dim=1)):
            if i == 0:
                output_part = hidden_part
            elif i == 1:
                output_part = self.blocks[i - 1](hidden_part)
            else:
                output_part = self.blocks[i - 1](hidden_part + output_part)
            outputs.append(output_part)
        output = torch.cat(outputs, dim=1)
        return output


class SqueezeExcitationBlock(nn.Module):
    def __init__(self, in_channels, se_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=in_channels,
            out_channels=se_channels,
            kernel_size=1,
            padding="same",
            padding_mode="reflect",
        )
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(
            in_channels=se_channels,
            out_channels=out_channels,
            kernel_size=1,
            padding="same",
            padding_mode="reflect",
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, hidden_states):
        hidden_states_mean = hidden_states.mean(dim=2, keepdim=True)

        hidden_states_mean = self.relu(self.conv1(hidden_states_mean))
        hidden_states_mean = self.sigmoid(self.conv2(hidden_states_mean))

        return hidden_states * hidden_states_mean


class AttentiveStatisticsPooling(nn.Module):
    """This class implements an attentive statistic pooling layer for each channel.
    It returns the concatenated mean and std of the input tensor.
    """

    def __init__(self, channels, attention_channels=128):
        super().__init__()

        self.eps = 1e-12
        self.tdnn = TimeDelayNetBlock(channels * 3, attention_channels, 1, 1)
        self.tanh = nn.Tanh()
        self.conv = nn.Conv1d(
            in_channels=attention_channels,
            out_channels=channels,
            kernel_size=1,
            padding="same",
            padding_mode="reflect",
        )

    def _length_to_mask(self, length, max_len=None, dtype=None, device=None):
        """Creates a binary mask for each sequence.

        Reference: https://discuss.pytorch.org/t/how-to-generate-variable-length-mask/23397/3

        Arguments
        ---------
        length : torch.LongTensor
            Containing the length of each sequence in the batch. Must be 1D.
        max_len : int
            Max length for the mask, also the size of the second dimension.
        dtype : torch.dtype, default: None
            The dtype of the generated mask.
        device: torch.device, default: None
            The device to put the mask variable.

        Returns
        -------
        mask : tensor
            The binary mask.
        """

        if max_len is None:
            max_len = length.max().long().item()  # using arange to generate mask
        mask = torch.arange(max_len, device=length.device, dtype=length.dtype).expand(
            len(length), max_len
        ) < length.unsqueeze(1)

        mask = torch.as_tensor(mask, dtype=dtype, device=device)
        return mask

    def _compute_statistics(self, x, m, dim=2):
        mean = (m * x).sum(dim)
        std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(self.eps))
        return mean, std

    def forward(self, hidden_states):
        seq_length = hidden_states.shape[-1]
        lengths = torch.ones(hidden_states.shape[0], device=hidden_states.device)

        # Make binary mask of shape [N, 1, L]
        mask = self._length_to_mask(
            lengths * seq_length, max_len=seq_length, dtype=hidden_states.dtype, device=hidden_states.device
        )
        mask = mask.unsqueeze(1)

        # Expand the temporal context of the pooling layer by allowing the
        # self-attention to look at global properties of the utterance.
        total = mask.sum(dim=2, keepdim=True)

        mean, std = self._compute_statistics(hidden_states, mask / total)
        mean = mean.unsqueeze(2).repeat(1, 1, seq_length)
        std = std.unsqueeze(2).repeat(1, 1, seq_length)
        attention = torch.cat([hidden_states, mean, std], dim=1)

        # Apply layers
        attention = self.conv(self.tanh(self.tdnn(attention)))

        # Filter out zero-paddings
        attention = attention.masked_fill(mask == 0, float("-inf"))

        attention = F.softmax(attention, dim=2)
        mean, std = self._compute_statistics(hidden_states, attention)
        # Append mean and std of the batch
        pooled_stats = torch.cat((mean, std), dim=1)
        pooled_stats = pooled_stats.unsqueeze(2)

        return pooled_stats


class SqueezeExcitationRes2NetBlock(nn.Module):
    """An implementation of building block in ECAPA-TDNN, i.e.,
    TDNN-Res2Net-TDNN-SqueezeExcitationBlock.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        res2net_scale=8,
        se_channels=128,
        kernel_size=1,
        dilation=1,
    ):
        super().__init__()
        self.out_channels = out_channels
        self.tdnn1 = TimeDelayNetBlock(
            in_channels,
            out_channels,
            kernel_size=1,
            dilation=1,
        )
        self.res2net_block = Res2NetBlock(out_channels, out_channels, res2net_scale, kernel_size, dilation)
        self.tdnn2 = TimeDelayNetBlock(
            out_channels,
            out_channels,
            kernel_size=1,
            dilation=1,
        )
        self.se_block = SqueezeExcitationBlock(out_channels, se_channels, out_channels)

    def forward(self, hidden_state):
        residual = hidden_state

        hidden_state = self.tdnn1(hidden_state)
        hidden_state = self.res2net_block(hidden_state)
        hidden_state = self.tdnn2(hidden_state)
        hidden_state = self.se_block(hidden_state)

        return hidden_state + residual


class ECAPA_TimeDelayNet(torch.nn.Module):
    """An implementation of the speaker embedding model in a paper.
    "ECAPA-TDNN: Emphasized Channel Attention, Propagation and Aggregation in
    TDNN Based Speaker Verification" (https://arxiv.org/abs/2005.07143).
    """

    def __init__(self, config: Qwen2_5OmniDiTConfig):
        super().__init__()
        if len(config.enc_channels) != len(config.enc_kernel_sizes) or len(config.enc_channels) != len(
            config.enc_dilations
        ):
            raise ValueError("enc_channels, enc_kernel_sizes and enc_dilations should have same length")
        self.channels = config.enc_channels
        self.blocks = nn.ModuleList()

        # The initial TDNN layer
        self.blocks.append(
            TimeDelayNetBlock(
                config.mel_dim,
                config.enc_channels[0],
                config.enc_kernel_sizes[0],
                config.enc_dilations[0],
            )
        )

        # SE-Res2Net layers
        for i in range(1, len(config.enc_channels) - 1):
            self.blocks.append(
                SqueezeExcitationRes2NetBlock(
                    config.enc_channels[i - 1],
                    config.enc_channels[i],
                    res2net_scale=config.enc_res2net_scale,
                    se_channels=config.enc_se_channels,
                    kernel_size=config.enc_kernel_sizes[i],
                    dilation=config.enc_dilations[i],
                )
            )

        # Multi-layer feature aggregation
        self.mfa = TimeDelayNetBlock(
            config.enc_channels[-1],
            config.enc_channels[-1],
            config.enc_kernel_sizes[-1],
            config.enc_dilations[-1],
        )

        # Attentive Statistical Pooling
        self.asp = AttentiveStatisticsPooling(
            config.enc_channels[-1],
            attention_channels=config.enc_attention_channels,
        )

        # Final linear transformation
        self.fc = nn.Conv1d(
            in_channels=config.enc_channels[-1] * 2,
            out_channels=config.enc_dim,
            kernel_size=1,
            padding="same",
            padding_mode="reflect",
        )

    def forward(self, hidden_states):
        # Minimize transpose for efficiency
        hidden_states = hidden_states.transpose(1, 2)

        hidden_states_list = []
        for layer in self.blocks:
            hidden_states = layer(hidden_states)
            hidden_states_list.append(hidden_states)

        # Multi-layer feature aggregation
        hidden_states = torch.cat(hidden_states_list[1:], dim=1)
        hidden_states = self.mfa(hidden_states)

        # Attentive Statistical Pooling
        hidden_states = self.asp(hidden_states)

        # Final linear transformation
        hidden_states = self.fc(hidden_states)

        hidden_states = hidden_states.squeeze(-1)
        return hidden_states


class DiTInputEmbedding(nn.Module):
    def __init__(self, config: Qwen2_5OmniDiTConfig):
        super().__init__()
        self.proj = nn.Linear(
            config.mel_dim + config.enc_dim + config.enc_emb_dim + config.emb_dim,
            config.hidden_size,
        )
        self.spk_encoder = ECAPA_TimeDelayNet(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        speaker_embedding: torch.Tensor,
        condition_vector: torch.Tensor,
        code_embed: torch.Tensor,
        drop_audio_cond: Optional[bool] = False,
        code_embed_uncond: Optional[bool] = None,
        apply_cfg: Optional[bool] = True,
    ):
        if apply_cfg:
            hidden_states = torch.cat([hidden_states, hidden_states], dim=0)
            speaker_embedding = torch.cat([speaker_embedding, torch.zeros_like(speaker_embedding)], dim=0)
            condition_vector = torch.cat([condition_vector, torch.zeros_like(condition_vector)], dim=0)
            code_embed = torch.cat([code_embed, code_embed_uncond], dim=0)
        elif drop_audio_cond:  # cfg for cond audio
            condition_vector = torch.zeros_like(condition_vector)
            speaker_embedding = torch.zeros_like(speaker_embedding)
        condition_vector = self.spk_encoder(condition_vector).unsqueeze(1).repeat(1, hidden_states.size(1), 1)
        hidden_states = self.proj(torch.cat((hidden_states, condition_vector, code_embed, speaker_embedding), dim=-1))

        return hidden_states


# Transformer backbone using DiT blocks
class DiTCodecEmbedding(nn.Module):
    def __init__(self, codec_num_embeds, codec_dim, repeats):
        super().__init__()
        self.repeats = repeats
        self.codec_embed = nn.Embedding(codec_num_embeds + 1, codec_dim)

    def forward(self, code, drop_code=False):
        if drop_code:
            code = torch.zeros_like(code)
        code_embed = self.codec_embed(code)

        code_embed = torch.repeat_interleave(code_embed, repeats=self.repeats, dim=1)
        return code_embed


# AdaLayerNormZero
# return with modulated x for attn input, and params for later mlp modulation
class Qwen2_5_OmniAdaLayerNormZero(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.silu = nn.SiLU()
        self.linear = nn.Linear(dim, dim * 6)

        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, hidden_states, emb=None):
        emb = self.linear(self.silu(emb))
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = torch.chunk(emb, 6, dim=1)

        hidden_states = self.norm(hidden_states) * (1 + scale_msa[:, None]) + shift_msa[:, None]
        return hidden_states, gate_msa, shift_mlp, scale_mlp, gate_mlp


# AdaLayerNormZero for final layer
# return only with modulated x for attn input, cuz no more mlp modulation
class Qwen2_5_OmniAdaLayerNormZero_Final(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.silu = nn.SiLU()
        self.linear = nn.Linear(dim, dim * 2)

        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, hidden_states, emb):
        emb = self.linear(self.silu(emb))
        scale, shift = torch.chunk(emb, 2, dim=1)

        hidden_states = self.norm(hidden_states) * (1 + scale)[:, None, :] + shift[:, None, :]
        return hidden_states


# FeedForward
class DiTMLP(nn.Module):
    def __init__(self, dim, mult=4, dropout=0.0):
        super().__init__()
        inner_dim = int(dim * mult)

        self.ff = nn.ModuleList(
            [
                nn.Linear(dim, inner_dim),
                nn.GELU(approximate="tanh"),
                nn.Dropout(dropout),
                nn.Linear(inner_dim, dim),
            ]
        )

    def forward(self, hidden_states):
        for layer in self.ff:
            hidden_states = layer(hidden_states)
        return hidden_states


# Modified from Llama with a different rotate function, will fixed in next release
def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        position_ids (`torch.Tensor`, *optional*):
            Deprecated and unused.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """

    def rotate_half_codec(x):
        # x = rearrange(x, "... (d r) -> ... d r", r=2)
        x = x.reshape(*x.shape[:-1], -1, 2)
        x1, x2 = x.unbind(dim=-1)
        x = torch.stack((-x2, x1), dim=-1)
        return x.reshape(*x.shape[:-2], -1)

    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half_codec(q) * sin)
    k_embed = (k * cos) + (rotate_half_codec(k) * sin)
    return q_embed, k_embed


class DiTAttention(nn.Module):
    def __init__(self, config: Qwen2_5OmniDiTConfig):
        super().__init__()

        self.config = config
        self.dim = config.hidden_size
        self.heads = config.num_attention_heads
        self.inner_dim = config.head_dim * config.num_attention_heads
        self.dropout = config.dropout
        self._attn_implementation = config._attn_implementation
        self.is_causal = False

        self.to_q = nn.Linear(config.hidden_size, self.inner_dim)
        self.to_k = nn.Linear(config.hidden_size, self.inner_dim)
        self.to_v = nn.Linear(config.hidden_size, self.inner_dim)

        self.to_out = nn.ModuleList([nn.Linear(self.inner_dim, config.hidden_size), nn.Dropout(config.dropout)])

    def forward(
        self,
        hidden_states,  # noised input x
        position_embeddings=None,  # rotary position embedding for x
        attention_mask=None,
    ) -> torch.Tensor:
        batch_size = hidden_states.shape[0]

        # `sample` projections.
        query = self.to_q(hidden_states)
        key = self.to_k(hidden_states)
        value = self.to_v(hidden_states)

        # attention
        inner_dim = key.shape[-1]
        head_dim = inner_dim // self.heads
        query = query.view(batch_size, -1, self.heads, head_dim).transpose(1, 2)
        key = key.view(batch_size, -1, self.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, self.heads, head_dim).transpose(1, 2)

        # apply rotary position embedding
        # Due to training process, only first head is applied with RoPE, will be fixed at next release
        cos, sin = position_embeddings
        query[:, :1], key[:, :1] = apply_rotary_pos_emb(query[:, :1], key[:, :1], cos, sin)

        attention_interface = ALL_ATTENTION_FUNCTIONS[self._attn_implementation]
        attention_weights, _ = attention_interface(
            self,
            query,
            key,
            value,
            attention_mask=attention_mask,
            is_causal=False,
        )

        # mask. e.g. inference got a batch with different target durations, mask out the padding
        attention_weights = attention_weights.reshape(batch_size, -1, self.heads * head_dim)
        attention_weights = attention_weights.to(query.dtype)

        # linear proj
        attention_output = self.to_out[0](attention_weights)
        attention_output = self.to_out[1](attention_output)

        return attention_output


# time step conditioning embedding
class SinusPositionEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, hidden_states, scale=1000):
        device = hidden_states.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device).float() * -emb)
        emb = scale * hidden_states.unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb.type_as(hidden_states)


class DiTTimestepEmbedding(nn.Module):
    def __init__(self, dim, freq_embed_dim=256):
        super().__init__()
        self.time_embed = SinusPositionEmbedding(freq_embed_dim)
        self.time_mlp = nn.ModuleList([nn.Linear(freq_embed_dim, dim), nn.SiLU(), nn.Linear(dim, dim)])

    def forward(self, timestep):  # noqa: F821
        time_hidden = self.time_embed(timestep)
        time_hidden = time_hidden.to(timestep.dtype)
        for layer in self.time_mlp:
            time_hidden = layer(time_hidden)  # b d
        return time_hidden


class DiTDecoderLayer(nn.Module):
    def __init__(self, config: Qwen2_5OmniDiTConfig, look_ahead_block=0, look_backward_block=0):
        super().__init__()
        self.attn_norm = Qwen2_5_OmniAdaLayerNormZero(config.hidden_size)

        self.attn = DiTAttention(config)
        self.look_ahead_block = look_ahead_block
        self.look_backward_block = look_backward_block
        self.ff_norm = nn.LayerNorm(config.hidden_size, elementwise_affine=False, eps=1e-6)
        self.ff = DiTMLP(dim=config.hidden_size, mult=config.ff_mult, dropout=config.dropout)

    def forward(
        self, hidden_states, timestep, position_embeddings=None, block_diff=None
    ):  # x: noised input, t: time embedding
        # pre-norm & modulation for attention input
        norm, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.attn_norm(hidden_states, emb=timestep)

        # attention
        attn_output = self.attn(
            hidden_states=norm,
            position_embeddings=position_embeddings,
            attention_mask=(block_diff >= -float(self.look_backward_block))
            & (block_diff <= float(self.look_ahead_block)),
        )

        # process attention output for input x
        hidden_states = hidden_states + gate_msa.unsqueeze(1) * attn_output

        norm = self.ff_norm(hidden_states) * (1 + scale_mlp[:, None]) + shift_mlp[:, None]
        ff_output = self.ff(norm)
        hidden_states = hidden_states + gate_mlp.unsqueeze(1) * ff_output

        return hidden_states


class SnakeBeta(nn.Module):
    """
    A modified Snake function which uses separate parameters for the magnitude of the periodic components
    Shape:
        - Input: (B, C, T)
        - Output: (B, C, T), same shape as the input
    Parameters:
        - alpha - trainable parameter that controls frequency
        - beta - trainable parameter that controls magnitude
    References:
        - This activation function is a modified version based on this paper by Liu Ziyin, Tilman Hartwig, Masahito Ueda:
        https://arxiv.org/abs/2006.08195
    """

    def __init__(self, in_features, alpha=1.0):
        super().__init__()
        self.in_features = in_features

        # initialize alpha
        self.alpha = Parameter(torch.zeros(in_features) * alpha)
        self.beta = Parameter(torch.zeros(in_features) * alpha)

        self.no_div_by_zero = 0.000000001

    def forward(self, hidden_states):
        """
        Forward pass of the function.
        Applies the function to the input elementwise.
        SnakeBeta ∶= x + 1/b * sin^2 (xa)
        """
        alpha = self.alpha.unsqueeze(0).unsqueeze(-1)  # line up with x to [B, C, T]
        beta = self.beta.unsqueeze(0).unsqueeze(-1)
        alpha = torch.exp(alpha)
        beta = torch.exp(beta)
        hidden_states = hidden_states + (1.0 / (beta + self.no_div_by_zero)) * torch.pow(
            torch.sin(hidden_states * alpha), 2
        )

        return hidden_states


def kaiser_sinc_filter1d(cutoff, half_width, kernel_size):
    """Generates a 1D Kaiser-windowed sinc filter.

    Args:
        cutoff (float): Normalized cutoff frequency (0 to 0.5).
        half_width (float): Transition bandwidth.
        kernel_size (int): Number of filter taps.

    Returns:
        torch.Tensor: A tensor of shape (1, 1, kernel_size) representing the filter.
    """
    is_even = kernel_size % 2 == 0
    half_size = kernel_size // 2

    # Compute Kaiser window parameters
    delta_f = 4 * half_width
    attenuation = 2.285 * (half_size - 1) * math.pi * delta_f + 7.95

    if attenuation > 50.0:
        beta = 0.1102 * (attenuation - 8.7)
    elif attenuation >= 21.0:
        beta = 0.5842 * (attenuation - 21) ** 0.4 + 0.07886 * (attenuation - 21.0)
    else:
        beta = 0.0

    kaiser_window = torch.kaiser_window(kernel_size, beta=beta, periodic=False, dtype=torch.float32)

    # Compute time indices
    if is_even:
        time_indices = torch.arange(-half_size, half_size) + 0.5
    else:
        time_indices = torch.arange(kernel_size) - half_size

    # Compute sinc filter
    if cutoff == 0:
        return torch.zeros((1, 1, kernel_size), dtype=torch.float32)  # Ensures correct shape

    sinc_filter = torch.sinc(2 * cutoff * time_indices)
    normalized_filter = 2 * cutoff * kaiser_window * sinc_filter

    # Normalize to ensure sum = 1 (avoid leakage of constant component)
    normalized_filter /= normalized_filter.sum()

    return normalized_filter.view(1, 1, kernel_size)


class UpSample1d(nn.Module):
    def __init__(self, ratio=2, kernel_size=None):
        super().__init__()
        self.ratio = ratio
        self.kernel_size = int(6 * ratio // 2) * 2 if kernel_size is None else kernel_size
        self.stride = ratio
        self.pad = self.kernel_size // ratio - 1
        self.pad_left = self.pad * self.stride + (self.kernel_size - self.stride) // 2
        self.pad_right = self.pad * self.stride + (self.kernel_size - self.stride + 1) // 2

        filter = kaiser_sinc_filter1d(cutoff=0.5 / ratio, half_width=0.6 / ratio, kernel_size=self.kernel_size)
        self.register_buffer("filter", filter, persistent=False)

    def forward(self, hidden_states):
        channels = hidden_states.shape[1]

        hidden_states = F.pad(hidden_states, (self.pad, self.pad), mode="replicate")
        hidden_states = self.ratio * F.conv_transpose1d(
            hidden_states, self.filter.expand(channels, -1, -1), stride=self.stride, groups=channels
        )
        hidden_states = hidden_states[..., self.pad_left : -self.pad_right]

        return hidden_states


class DownSample1d(nn.Module):
    def __init__(self, ratio=2, kernel_size=None):
        super().__init__()
        cutoff = 0.5 / ratio
        half_width = 0.6 / ratio

        if cutoff < 0.0:
            raise ValueError("Minimum cutoff must be larger than zero.")
        if cutoff > 0.5:
            raise ValueError("A cutoff above 0.5 does not make sense.")

        self.even = kernel_size % 2 == 0
        self.pad_left = kernel_size // 2 - int(self.even)
        self.pad_right = kernel_size // 2
        self.stride = ratio
        filter = kaiser_sinc_filter1d(cutoff, half_width, kernel_size)
        self.register_buffer("filter", filter, persistent=False)

    def forward(self, hidden_states):
        channels = hidden_states.shape[1]
        hidden_states = F.pad(hidden_states, (self.pad_left, self.pad_right), mode="replicate")
        out = F.conv1d(hidden_states, self.filter.expand(channels, -1, -1), stride=self.stride, groups=channels)
        return out


class TorchActivation1d(nn.Module):
    def __init__(
        self,
        activation,
        up_ratio: int = 2,
        down_ratio: int = 2,
        up_kernel_size: int = 12,
        down_kernel_size: int = 12,
    ):
        super().__init__()
        if not callable(activation):
            raise ValueError("Activation function must be callable")
        self.act = activation
        self.upsample = UpSample1d(up_ratio, up_kernel_size)
        self.downsample = DownSample1d(down_ratio, down_kernel_size)

    def forward(self, hidden_states):
        hidden_states = self.upsample(hidden_states)
        hidden_states = self.act(hidden_states)
        hidden_states = self.downsample(hidden_states)

        return hidden_states


class AMPBlock(torch.nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=3,
        dilation=(1, 3, 5),
    ):
        super().__init__()

        self.convs1 = nn.ModuleList(
            [
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=dilation[0],
                    padding=self._get_padding(kernel_size, dilation[0]),
                ),
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=dilation[1],
                    padding=self._get_padding(kernel_size, dilation[1]),
                ),
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=dilation[2],
                    padding=self._get_padding(kernel_size, dilation[2]),
                ),
            ]
        )

        self.convs2 = nn.ModuleList(
            [
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=1,
                    padding=self._get_padding(kernel_size, 1),
                ),
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=1,
                    padding=self._get_padding(kernel_size, 1),
                ),
                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    1,
                    dilation=1,
                    padding=self._get_padding(kernel_size, 1),
                ),
            ]
        )

        self.num_layers = len(self.convs1) + len(self.convs2)  # total number of conv layers

        self.activations = nn.ModuleList(
            [TorchActivation1d(activation=SnakeBeta(channels)) for _ in range(self.num_layers)]
        )

    def _get_padding(self, kernel_size, dilation=1):
        return int((kernel_size * dilation - dilation) / 2)

    def forward(self, hidden_states):
        acts1, acts2 = self.activations[::2], self.activations[1::2]
        for conv1, conv2, act1, act2 in zip(self.convs1, self.convs2, acts1, acts2):
            residual = hidden_states
            hidden_states = act1(hidden_states)
            hidden_states = conv1(hidden_states)
            hidden_states = act2(hidden_states)
            hidden_states = conv2(hidden_states)
            hidden_states = residual + hidden_states

        return hidden_states


@auto_docstring(
    custom_intro="""
    The full Qwen2.5Omni Token2WavBigVGAN model. Which take mel spectrogram as input and predict waveform.
    """
)
class Qwen2_5OmniToken2WavBigVGANModel(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniBigVGANConfig

    def __init__(self, config: Qwen2_5OmniBigVGANConfig):
        super().__init__(config)
        self.num_residual_blocks = len(config.resblock_kernel_sizes)
        self.num_upsample_layers = len(config.upsample_rates)

        self.conv_pre = nn.Conv1d(config.mel_dim, config.upsample_initial_channel, 7, 1, padding=3)

        # Removing extra ModuleList breaks official state dict
        ups = [
            nn.ModuleList(
                [
                    nn.ConvTranspose1d(
                        config.upsample_initial_channel // (2**layer_idx),
                        config.upsample_initial_channel // (2 ** (layer_idx + 1)),
                        kernel_size,
                        stride,
                        padding=(kernel_size - stride) // 2,
                    )
                ]
            )
            for layer_idx, (stride, kernel_size) in enumerate(zip(config.upsample_rates, config.upsample_kernel_sizes))
        ]
        self.ups = nn.ModuleList(ups)

        self.resblocks = nn.ModuleList(
            [
                AMPBlock(config.upsample_initial_channel // (2 ** (layer_idx + 1)), kernel_size, dilation)
                for layer_idx in range(self.num_upsample_layers)
                for kernel_size, dilation in zip(config.resblock_kernel_sizes, config.resblock_dilation_sizes)
            ]
        )

        self.activation_post = TorchActivation1d(
            activation=SnakeBeta(config.upsample_initial_channel // (2**self.num_upsample_layers))
        )
        self.conv_post = nn.Conv1d(
            config.upsample_initial_channel // (2**self.num_upsample_layers), 1, 7, 1, padding=3, bias=False
        )

    def normalize_spectrogram(self, spectrogram, max_value, min_db):
        return torch.clamp((2 * max_value) * ((spectrogram - min_db) / (-min_db)) - max_value, -max_value, max_value)

    def amplitude_to_db(self, amplitude, min_db_level):
        min_level = torch.exp(
            torch.tensor(min_db_level / 20.0 * np.log(10), device=amplitude.device, dtype=amplitude.dtype)
        )
        return 20 * torch.log10(torch.clamp(amplitude, min=min_level))

    def process_mel_spectrogram(self, mel_spectrogram):
        amplitude_spectrum = torch.exp(mel_spectrogram)
        decibel_spectrum = self.amplitude_to_db(amplitude_spectrum, -115) - 20
        return self.normalize_spectrogram(decibel_spectrum, 1, -115)

    def forward(self, mel_spectrogram):
        processed_spectrogram = self.process_mel_spectrogram(mel_spectrogram)
        hidden_representation = self.conv_pre(processed_spectrogram)

        for layer_index in range(self.num_upsample_layers):
            hidden_representation = self.ups[layer_index][0](hidden_representation)
            residual_output = sum(
                self.resblocks[layer_index * self.num_residual_blocks + block_index](hidden_representation)
                for block_index in range(self.num_residual_blocks)
            )
            residual_output = residual_output / self.num_residual_blocks
            hidden_representation = residual_output

        hidden_representation = self.activation_post(hidden_representation)
        output_waveform = self.conv_post(hidden_representation)
        return torch.clamp(output_waveform, min=-1.0, max=1.0).squeeze().cpu()


class RungeKutta4ODESolver:
    def __init__(self, function, initial_value):
        self.function = function
        self.initial_value = initial_value

        self._one_third = 1 / 3
        self._two_thirds = 2 / 3

    def _rk4_step(self, function, time_start, time_step, time_end, value_start, function_value_start=None):
        k1 = function_value_start if function_value_start is not None else function(time_start, value_start)
        k2 = function(time_start + time_step * self._one_third, value_start + time_step * k1 * self._one_third)
        k3 = function(time_start + time_step * self._two_thirds, value_start + time_step * (k2 - k1 * self._one_third))
        k4 = function(time_end, value_start + time_step * (k1 - k2 + k3))
        return (k1 + 3 * (k2 + k3) + k4) * time_step / 8

    def _compute_step(self, function, time_start, time_step, time_end, value_start):
        function_value_start = function(time_start, value_start)
        return self._rk4_step(
            function, time_start, time_step, time_end, value_start, function_value_start=function_value_start
        ), function_value_start

    def _linear_interpolation(self, time_start, time_end, value_start, value_end, time_point):
        if time_point == time_start:
            return value_start
        if time_point == time_end:
            return value_end
        weight = (time_point - time_start) / (time_end - time_start)
        return value_start + weight * (value_end - value_start)

    def integrate(self, time_points):
        solution = torch.empty(
            len(time_points),
            *self.initial_value.shape,
            dtype=self.initial_value.dtype,
            device=self.initial_value.device,
        )
        solution[0] = self.initial_value

        current_index = 1
        current_value = self.initial_value
        for time_start, time_end in zip(time_points[:-1], time_points[1:]):
            time_step = time_end - time_start
            delta_value, _ = self._compute_step(self.function, time_start, time_step, time_end, current_value)
            next_value = current_value + delta_value

            while current_index < len(time_points) and time_end >= time_points[current_index]:
                solution[current_index] = self._linear_interpolation(
                    time_start, time_end, current_value, next_value, time_points[current_index]
                )
                current_index += 1

            current_value = next_value

        return solution


@auto_docstring(
    custom_intro="""
    The full Qwen2.5Omni Token2WavDiT model. Which take speech tokens as input and predict mel spectrogram.
    """
)
class Qwen2_5OmniToken2WavDiTModel(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniDiTConfig
    _no_split_modules = ["DiTDecoderLayer"]

    def __init__(self, config: Qwen2_5OmniDiTConfig):
        super().__init__(config)
        self.mel_dim = config.mel_dim
        self.repeats = config.repeats
        self.time_embed = DiTTimestepEmbedding(config.hidden_size)

        self.text_embed = DiTCodecEmbedding(config.num_embeds, config.emb_dim, config.repeats)
        self.input_embed = DiTInputEmbedding(config)

        self.rotary_embed = Qwen2_5OmniDiTRotaryEmbedding(config.head_dim)

        self.hidden_size = config.hidden_size
        self.layers = config.num_hidden_layers
        self.block_size = config.block_size
        self.num_attention_heads = config.num_attention_heads

        self.transformer_blocks = nn.ModuleList()
        for i in range(config.num_hidden_layers):
            self.transformer_blocks.append(
                DiTDecoderLayer(
                    config,
                    look_ahead_block=1 if i in config.look_ahead_layers else 0,
                    look_backward_block=1 if i in config.look_backward_layers else 0,
                )
            )

        self.norm_out = Qwen2_5_OmniAdaLayerNormZero_Final(config.hidden_size)  # final modulation
        self.proj_out = nn.Linear(config.hidden_size, config.mel_dim)

    def _create_block_diff(self, hidden_states):
        batch, seq_len = hidden_states.shape[0], hidden_states.shape[1]
        block_indices = torch.arange(seq_len, device=hidden_states.device) // self.block_size  # [seq_length]

        block_i = block_indices.unsqueeze(1)  # [seq_length, 1]
        block_j = block_indices.unsqueeze(0)  # [1, seq_length]
        block_diff = block_j - block_i  # (n, n)

        return block_diff.expand(batch, self.num_attention_heads, seq_len, seq_len)

    def forward(
        self,
        hidden_states,
        condition_vector,
        speaker_embedding,
        quantized_code,
        time_step,
        drop_audio_conditioning=False,
        drop_code=False,
        apply_cfg=True,
    ):
        batch_size = hidden_states.shape[0]
        if time_step.ndim == 0:
            time_step = time_step.repeat(batch_size)

        # Compute embeddings
        time_embedding = self.time_embed(time_step)
        text_embedding = self.text_embed(quantized_code, drop_code=False if apply_cfg else drop_code)
        text_embedding_unconditioned = self.text_embed(quantized_code, drop_code=True) if apply_cfg else None

        hidden_states = self.input_embed(
            hidden_states,
            speaker_embedding,
            condition_vector,
            text_embedding,
            drop_audio_cond=drop_audio_conditioning,
            code_embed_uncond=text_embedding_unconditioned,
            apply_cfg=apply_cfg,
        )

        # Compute positional encodings
        position_embeddings = self.rotary_embed(hidden_states)
        blockwise_difference = self._create_block_diff(hidden_states)

        # Transformer blocks
        for transformer_block in self.transformer_blocks:
            hidden_states = transformer_block(
                hidden_states,
                time_embedding,
                position_embeddings=position_embeddings,
                block_diff=blockwise_difference,
            )

        hidden_states = self.norm_out(hidden_states, time_embedding)
        output = self.proj_out(hidden_states)

        return output

    @torch.no_grad()
    def sample(
        self,
        conditioning_vector,
        reference_mel_spectrogram,
        quantized_code,
        num_steps=10,
        guidance_scale=0.5,
        sway_coefficient=-1.0,
    ):
        noise_initialization = torch.randn([1, 30000, self.mel_dim], dtype=reference_mel_spectrogram.dtype)
        maximum_duration = quantized_code.shape[1] * self.repeats
        initial_state = noise_initialization[:, :maximum_duration].to(quantized_code.device)
        batch_size = reference_mel_spectrogram.shape[0]
        conditioning_vector = conditioning_vector.unsqueeze(1).repeat(1, maximum_duration, 1)

        if batch_size != 1:
            raise ValueError("Only batch size = 1 is currently supported")

        def ode_function(time_step, hidden_states):
            if guidance_scale < 1e-5:
                prediction = self(
                    hidden_states=hidden_states,
                    speaker_embedding=conditioning_vector,
                    condition_vector=reference_mel_spectrogram,
                    quantized_code=quantized_code,
                    time_step=time_step,
                    drop_audio_conditioning=False,
                    drop_code=False,
                )
                return prediction

            model_output = self(
                hidden_states=hidden_states,
                quantized_code=quantized_code,
                speaker_embedding=conditioning_vector,
                condition_vector=reference_mel_spectrogram,
                time_step=time_step,
                apply_cfg=True,
            )
            guided_prediction, null_prediction = torch.chunk(model_output, 2, dim=0)
            return guided_prediction + (guided_prediction - null_prediction) * guidance_scale

        initial_time = 0
        time_embedding = torch.linspace(
            initial_time, 1, num_steps, device=quantized_code.device, dtype=conditioning_vector.dtype
        )

        if sway_coefficient is not None:
            time_embedding += sway_coefficient * (torch.cos(torch.pi / 2 * time_embedding) - 1 + time_embedding)

        ode_solver = RungeKutta4ODESolver(function=ode_function, initial_value=initial_state)
        solution_trajectory = ode_solver.integrate(time_embedding)

        generated_waveform = solution_trajectory[-1]
        generated_mel_spectrogram = generated_waveform.permute(0, 2, 1)
        return generated_mel_spectrogram


@auto_docstring(
    custom_intro="""
    The full Qwen2.5Omni Token2Wav model. Consists a DiT model take speech tokens as input and predict mel spectrogram and a BigVGAN vocoder take mel spectrogram as input and predict waveform.
    """
)
class Qwen2_5OmniToken2WavModel(Qwen2_5OmniPreTrainedModel):
    config_class = Qwen2_5OmniToken2WavConfig
    base_model_prefix = "model"
    _no_split_modules = ["Qwen2_5OmniToken2WavDiTModel", "Qwen2_5OmniToken2WavBigVGANModel"]

    def __init__(self, config: Qwen2_5OmniToken2WavConfig):
        super().__init__(config)
        attn_impl = config._attn_implementation
        if config._attn_implementation == "flash_attention_2":
            logger.warning_once(
                "Qwen2_5OmniToken2WavModel must inference with fp32, but flash_attention_2 only supports fp16 and bf16, "
                "attention implementation of Qwen2_5OmniToken2WavModel will fallback to sdpa."
            )
            attn_impl = "sdpa"
        elif config._attn_implementation == "eager":
            logger.warning_once(
                "Qwen2_5OmniToken2WavModel does not support eager attention implementation, fall back to sdpa"
            )
            attn_impl = "sdpa"
        self.code2wav_dit_model = Qwen2_5OmniToken2WavDiTModel._from_config(
            config.dit_config, attn_implementation=attn_impl
        )
        self.code2wav_bigvgan_model = Qwen2_5OmniToken2WavBigVGANModel._from_config(
            config.bigvgan_config, attn_implementation=attn_impl
        )

    def forward(
        self,
        code,
        conditioning,
        reference_mel,
        num_steps=10,
        guidance_scale=0.5,
        sway_coefficient=-1.0,
        **kwargs,
    ):
        """Generates a waveform from input code and conditioning parameters."""

        mel_spectrogram = self.code2wav_dit_model.sample(
            conditioning,
            reference_mel,
            code,
            num_steps=num_steps,
            guidance_scale=guidance_scale,
            sway_coefficient=sway_coefficient,
        )

        waveform = self.code2wav_bigvgan_model(mel_spectrogram)

        return waveform


############################
#    Start Qwen2.5Omni     #
############################


@auto_docstring(
    custom_intro="""
    The full Qwen2.5Omni model, a multimodal model composed of 3 sub-models:
    - [`Qwen2_5OmniThinkerForConditionalGeneration`]:
    a causal auto-regressive transformer takes text, audio, image, video as input and predict text tokens.
    - [`Qwen2_5OmniTalkerForConditionalGeneration`]:
    a causal auto-regressive transformer takes thinker hidden states and response as input and predict speech tokens.
    - [`Qwen2_5OmniToken2WavModel`]:
    a DiT model take speech tokens as input and predict mel spectrogram and a BigVGAN vocoder take mel spectrogram as input and predict waveform.
    """
)
class Qwen2_5OmniForConditionalGeneration(Qwen2_5OmniPreTrainedModel, GenerationMixin):
    config_class = Qwen2_5OmniConfig
    _no_split_modules = [
        "Qwen2_5OmniTalkerForConditionalGeneration",
        "Qwen2_5OmniToken2WavModel",
    ]

    def __init__(self, config):
        super().__init__(config)

        self.thinker = Qwen2_5OmniThinkerForConditionalGeneration(config.thinker_config)

        self.has_talker = config.enable_audio_output
        self.speaker_map = {}
        if config.enable_audio_output:
            self.enable_talker()
        self.post_init()

    def enable_talker(self):
        self.talker = Qwen2_5OmniTalkerForConditionalGeneration(self.config.talker_config)
        self.token2wav = Qwen2_5OmniToken2WavModel(self.config.token2wav_config)
        self.token2wav.float()
        self.has_talker = True

    def load_speakers(self, path):
        check_torch_load_is_safe()
        for key, value in torch.load(path, weights_only=True).items():
            self.speaker_map[key] = value
        logger.info("Speaker {} loaded".format(list(self.speaker_map.keys())))

    def disable_talker(self):
        if hasattr(self, "talker"):
            del self.talker
        if hasattr(self, "token2wav"):
            del self.token2wav
        self.has_talker = False

    @classmethod
    def from_pretrained(
        cls,
        pretrained_model_name_or_path,
        *model_args,
        config=None,
        cache_dir=None,
        ignore_mismatched_sizes=False,
        force_download=False,
        local_files_only=False,
        token=None,
        revision="main",
        use_safetensors=None,
        weights_only=True,
        **kwargs,
    ):
        model = super().from_pretrained(
            pretrained_model_name_or_path,
            *model_args,
            config=config,
            cache_dir=cache_dir,
            ignore_mismatched_sizes=ignore_mismatched_sizes,
            force_download=force_download,
            local_files_only=local_files_only,
            token=token,
            revision=revision,
            use_safetensors=use_safetensors,
            weights_only=weights_only,
            **kwargs,
        )
        spk_path = cached_file(
            pretrained_model_name_or_path,
            "spk_dict.pt",
            subfolder=kwargs.pop("subfolder", None),
            cache_dir=kwargs.pop("cache_dir", None),
            force_download=kwargs.pop("force_download", False),
            proxies=kwargs.pop("proxies", None),
            resume_download=kwargs.pop("resume_download", None),
            local_files_only=kwargs.pop("local_files_only", False),
            token=kwargs.pop("use_auth_token", None),
            revision=kwargs.pop("revision", None),
        )
        if spk_path is None:
            raise ValueError(f"""{pretrained_model_name_or_path}/{spk_path} not exists""")
        model.load_speakers(spk_path)

        return model

    @torch.no_grad()
    # TODO: raushan, defaults should be saved in generation config
    def generate(
        self,
        input_ids: Optional[torch.Tensor] = None,
        speaker: str = "Chelsie",
        use_audio_in_video: bool = False,
        return_audio: Optional[bool] = None,
        thinker_max_new_tokens: int = 1024,
        talker_max_new_tokens: int = 4096,
        talker_do_sample: bool = True,
        talker_top_k: int = 40,
        talker_top_p: float = 0.8,
        talker_temperature: float = 0.9,
        talker_eos_token_id: list[int] = [8292, 8294],
        talker_repetition_penalty: float = 1.05,
        **kwargs,
    ):
        r"""
        Generate text response and audio from input.

        Args:
            input_ids (`Optional[torch.Tensor]`, *optional*):
                Input ids, should obtain from processor.
            speaker (`str` , defaults to "Chelsie"):
                Which speaker should be used in audio response.
            use_audio_in_video (`bool`, defaults to False):
                Whether or not use audio track in video, should same as the parameter in `process_audio_info`.
            return_audio (`Optional[bool]`, *optional*):
                Whether or not return response in audio format. When `return_audio=None`, this parameter is same as `config.enable_audio_output`.
            kwargs (*optional*):
                - Without a prefix, they will be entered as `**kwargs` for the `generate` method of each sub-model.
                - With a *thinker_*, *talker_*, *token2wav_* prefix, they will be input for the `generate` method of the
                thinker, talker and token2wav respectively. It has the priority over the keywords without a prefix.
        Returns:
            When `return_audio=False`:
                - **Text** (`torch.Tensor`): Generated text token sequence.
            When `return_audio=True`:
                - **Text** (`torch.Tensor`): Generated text token sequence.
                - **Audio waveform** (`torch.Tensor`): Generated audio waveform.
        """

        if speaker not in self.speaker_map:
            raise ValueError(f"{speaker} is not available, available speakers: {self.speaker_map.keys()}")
        if return_audio and not self.has_talker:
            raise ValueError(
                "Cannot use talker when talker module not initialized. Use `enable_talker` method or set enable_talker in config to enable talker."
            )
        if return_audio is None:
            return_audio = self.has_talker
        if input_ids.shape[0] != 1 and return_audio:
            raise NotImplementedError("Qwen2.5-Omni currently does not support batched inference with audio output")

        shared_kwargs = {"use_audio_in_video": use_audio_in_video}
        thinker_kwargs = {
            "max_new_tokens": thinker_max_new_tokens,
        }
        talker_kwargs = {
            "max_new_tokens": talker_max_new_tokens,
            "do_sample": talker_do_sample,
            "top_k": talker_top_k,
            "top_p": talker_top_p,
            "temperature": talker_temperature,
            "eos_token_id": talker_eos_token_id,
            "repetition_penalty": talker_repetition_penalty,
        }
        token2wav_kwargs = {}

        for key, value in kwargs.items():
            if key.startswith("thinker_"):
                thinker_kwargs[key[len("thinker_") :]] = value
            elif key.startswith("talker_"):
                talker_kwargs[key[len("talker_") :]] = value
            elif key.startswith("token2wav_"):
                token2wav_kwargs[key[len("token2wav_") :]] = value
            # Process special input values
            elif key == "feature_attention_mask":
                thinker_kwargs[key] = value
                talker_kwargs["audio_feature_lengths"] = torch.sum(value, dim=1)
            elif key == "input_features" or key == "attention_mask":
                thinker_kwargs[key] = value
            # Put other key to shared kwargs
            else:
                shared_kwargs[key] = value

        # Merge kwargs
        for key, value in shared_kwargs.items():
            if key not in thinker_kwargs:
                thinker_kwargs[key] = value
            if key not in talker_kwargs:
                talker_kwargs[key] = value
            if key not in token2wav_kwargs:
                token2wav_kwargs[key] = value
        speaker_params = self.speaker_map[speaker]

        # 1. Generate from thinker module
        generate_audio = return_audio and self.has_talker
        if generate_audio:
            thinker_kwargs["output_hidden_states"] = True
            thinker_kwargs["return_dict_in_generate"] = True

        thinker_result = self.thinker.generate(input_ids=input_ids, **thinker_kwargs)

        if not generate_audio:
            return thinker_result

        # 2. Generate speech tokens from talker module
        embeds_to_talker = thinker_result.hidden_states[0][0].clone().to(self.talker.device)
        if thinker_kwargs.get("input_features", None) is not None:
            audio_ids_mask = input_ids == self.config.thinker_config.audio_token_index
            audio_mask = audio_ids_mask.unsqueeze(-1).expand_as(embeds_to_talker).to(embeds_to_talker.device)
            audio_mask_tensor = torch.zeros(
                [audio_ids_mask.sum(), embeds_to_talker.shape[-1]],
                dtype=embeds_to_talker.dtype,
                device=self.talker.device,
            )
            embeds_to_talker.masked_scatter_(audio_mask, audio_mask_tensor)
        if thinker_kwargs.get("pixel_values", None) is not None:
            image_ids_mask = input_ids == self.config.thinker_config.image_token_index
            image_mask = image_ids_mask.unsqueeze(-1).expand_as(embeds_to_talker).to(embeds_to_talker.device)
            image_mask_tensor = torch.zeros(
                [image_ids_mask.sum(), embeds_to_talker.shape[-1]],
                dtype=embeds_to_talker.dtype,
                device=self.talker.device,
            )
            embeds_to_talker.masked_scatter_(image_mask, image_mask_tensor)
        if thinker_kwargs.get("pixel_values_videos", None) is not None:
            video_ids_mask = input_ids == self.config.thinker_config.video_token_index
            video_mask = video_ids_mask.unsqueeze(-1).expand_as(embeds_to_talker).to(embeds_to_talker.device)
            video_mask_tensor = torch.zeros(
                [video_ids_mask.sum(), embeds_to_talker.shape[-1]],
                dtype=embeds_to_talker.dtype,
                device=self.talker.device,
            )
            embeds_to_talker.masked_scatter_(video_mask, video_mask_tensor)

        processed_thinker_hidden = (
            (embeds_to_talker,) + thinker_result.hidden_states[0][1:],
        ) + thinker_result.hidden_states[1:]
        thinker_generate_ids = thinker_result.sequences[:, input_ids.size(1) :].to(self.talker.device)
        thinker_token_embeds = [
            token_hidden_states[0].to(self.talker.device) for token_hidden_states in processed_thinker_hidden
        ]
        thinker_hidden_states = [
            token_hidden_states[-1].to(self.talker.device) for token_hidden_states in processed_thinker_hidden
        ]

        talker_text_bos_token = speaker_params["bos_token"]
        talker_input_text_ids = torch.cat(
            [
                input_ids.to(self.talker.device),
                torch.tensor([[talker_text_bos_token]], dtype=torch.long, device=self.talker.device),
                thinker_generate_ids[:, :1],
            ],
            dim=-1,
        )

        talker_input_ids = torch.cat(
            [
                torch.full_like(input_ids, fill_value=self.talker.codec_mask_token, device=self.talker.device),
                torch.tensor([[self.talker.codec_pad_token]], dtype=torch.long, device=self.talker.device),
                torch.tensor([[self.talker.codec_bos_token]], dtype=torch.long, device=self.talker.device),
            ],
            dim=1,
        )

        thinker_embed_tokens = self.thinker.get_input_embeddings()
        thinker_reply_part = torch.cat(thinker_hidden_states[1:], dim=1) + torch.cat(thinker_token_embeds[1:], dim=1)
        talker_inputs_embeds = thinker_hidden_states[0] + thinker_token_embeds[0]
        talker_text_bos_token = torch.tensor([[talker_text_bos_token]], dtype=torch.long, device=self.thinker.device)
        talker_text_bos_embed = thinker_embed_tokens(talker_text_bos_token).to(self.talker.device)
        talker_inputs_embeds = torch.cat(
            [
                talker_inputs_embeds,
                talker_text_bos_embed,
                thinker_reply_part[:, :1, :],
            ],
            dim=1,
        )

        eos_embedding = thinker_embed_tokens(
            torch.tensor([[self.talker.text_eos_token]], dtype=torch.long, device=self.thinker.device)
        ).to(self.talker.device)

        pad_embedding = thinker_embed_tokens(
            torch.tensor([[self.talker.text_pad_token]], dtype=torch.long, device=self.thinker.device)
        ).to(self.talker.device)

        thinker_reply_part = torch.cat(
            [
                thinker_reply_part[:, 1:, :],
                eos_embedding,
                pad_embedding,
            ],
            dim=1,
        )

        talker_attention_mask = None
        if "attention_mask" in kwargs:
            talker_attention_mask = torch.cat(
                [kwargs["attention_mask"], kwargs["attention_mask"].new_ones((1, 2))], dim=1
            ).to(self.talker.device)

        talker_result = self.talker.generate(
            input_ids=talker_input_ids,
            input_text_ids=talker_input_text_ids,
            thinker_reply_part=thinker_reply_part,
            inputs_embeds=talker_inputs_embeds,
            attention_mask=talker_attention_mask,
            suppress_tokens=[self.talker.codec_bos_token],
            **{k: (v.to(self.talker.device) if torch.is_tensor(v) else v) for k, v in talker_kwargs.items()},
        )
        talker_generate_codes = talker_result[:, talker_input_ids.shape[1] : -1]

        # 3. Generate wavs from code
        if self.token2wav.dtype != torch.float:
            self.token2wav.float()

        wav = self.token2wav(
            talker_generate_codes.to(self.token2wav.device),
            conditioning=speaker_params["cond"].to(self.token2wav.device).float(),
            reference_mel=speaker_params["ref_mel"].to(self.token2wav.device).float(),
            **token2wav_kwargs,
        )

        return thinker_result.sequences, wav.float()


__all__ = [
    "Qwen2_5OmniForConditionalGeneration",
    "Qwen2_5OmniThinkerTextModel",
    "Qwen2_5OmniThinkerForConditionalGeneration",
    "Qwen2_5OmniTalkerModel",
    "Qwen2_5OmniTalkerForConditionalGeneration",
    "Qwen2_5OmniToken2WavDiTModel",
    "Qwen2_5OmniToken2WavBigVGANModel",
    "Qwen2_5OmniToken2WavModel",
    "Qwen2_5OmniPreTrainedModel",
    "Qwen2_5OmniPreTrainedModelForConditionalGeneration",
]


In [ ]:
%%writefile /kaggle/temp/code/qwen-omni-utils/pyproject.toml
[project]
name = "qwen-omni-utils"
version = "0.0.8"
description = "Qwen Omni Language Model Utils - PyTorch"
authors = [
    { name = "Qwen Team", email = "lvyuanjun.lyj@alibaba-inc.com" },
]
dependencies = [
    "requests",
    "pillow",
    "av",
    "packaging",
    "librosa",
]
readme = "README.md"
requires-python = ">= 3.8"
license = {text = "Apache-2.0"}
keywords = [
    'large language model',
    'vision language model',
    'qwen-omni',
    'pytorch',
]
classifiers = [
    'Development Status :: 4 - Beta',
    'Topic :: Scientific/Engineering :: Artificial Intelligence',
    'Programming Language :: Python :: 3',
    'License :: OSI Approved :: Apache Software License',
]

[project.urls]
Homepage = "https://github.com/QwenLM/Qwen2-VL/tree/main/qwen-vl-utils"
Repository = "https://github.com/QwenLM/Qwen2-VL.git"
Issues = "https://github.com/QwenLM/Qwen2-VL/issues"

[project.optional-dependencies]
decord = [
    "decord",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.rye]
managed = true
dev-dependencies = [
    "torch",
    "torchvision",
    "torchaudio",
]

[tool.hatch.metadata]
allow-direct-references = true

[tool.hatch.build.targets.wheel]
packages = ["src/qwen_omni_utils"]

[tool.ruff]
line-length = 119

[tool.ruff.lint]
ignore = ["C408", "C901", "E501", "E731", "E741", "W605"]
select = ["C", "E", "F", "I", "W"]

[tool.ruff.lint.per-file-ignores]
"__init__.py" = ["E402", "F401", "F403", "F811"]

[tool.ruff.lint.isort]
lines-after-imports = 2
known-first-party = ["qwen_omni_utils"]

[tool.ruff.format]
quote-style = "double"
indent-style = "space"
skip-magic-trailing-comma = false
line-ending = "auto"

In [ ]:
%%writefile /kaggle/temp/code/qwen-omni-utils/src/qwen_omni_utils/__init__.py
from .v2_5 import *


In [ ]:
%%writefile /kaggle/temp/code/qwen-omni-utils/src/qwen_omni_utils/v2_5/__init__.py
from .audio_process import process_audio_info
from .vision_process import (
    extract_vision_info,
    fetch_image,
    fetch_video,
    process_vision_info,
    smart_resize,
)


def process_mm_info(conversations, use_audio_in_video, return_video_kwargs=False):
    audios = process_audio_info(conversations, use_audio_in_video)
    vision = process_vision_info(conversations, return_video_kwargs=return_video_kwargs)
    return (audios,) + vision


In [ ]:
%%writefile /kaggle/temp/code/qwen-omni-utils/src/qwen_omni_utils/v2_5/audio_process.py
import base64
from io import BytesIO

import audioread
import av
import librosa
import numpy as np


SAMPLE_RATE=16000
def _check_if_video_has_audio(video_path):
    container = av.open(video_path)
    audio_streams = [stream for stream in container.streams if stream.type == "audio"]
    if not audio_streams:
        return False
    return True


def process_audio_info(conversations: list[dict] | list[list[dict]], use_audio_in_video: bool):
    """
    Read and process audio info

    Support dict keys:

    type = audio
    - audio
    - audio_start
    - audio_end

    type = video
    - video
    - video_start
    - video_end
    """
    audios = []
    if isinstance(conversations[0], dict):
        conversations = [conversations]
    for conversation in conversations:
        for message in conversation:
            if not isinstance(message["content"], list):
                continue
            for ele in message["content"]:
                if ele["type"] == "audio":
                    if "audio" in ele or "audio_url" in ele:
                        path = ele.get("audio", ele.get("audio_url"))
                        audio_start = ele.get("audio_start", 0.0)
                        audio_end = ele.get("audio_end", None)
                        if isinstance(path, np.ndarray):
                            if path.ndim > 1:
                                raise ValueError("Support only mono audio")
                            audios.append(
                                path[int(SAMPLE_RATE * audio_start) : None if audio_end is None else int(SAMPLE_RATE * audio_end)]
                            )
                            continue
                        elif path.startswith("data:audio"):
                            _, base64_data = path.split("base64,", 1)
                            data = BytesIO(base64.b64decode(base64_data))
                        elif path.startswith("http://") or path.startswith("https://"):
                            data = audioread.ffdec.FFmpegAudioFile(path)
                        elif path.startswith("file://"):
                            data = path[len("file://") :]
                        else:
                            data = path
                    else:
                        raise ValueError("Unknown audio {}".format(ele))
                elif use_audio_in_video and ele["type"] == "video":
                    if "video" in ele or "video_url" in ele:
                        path = ele.get("video", ele.get("video_url"))
                        audio_start = ele.get("video_start", 0.0)
                        audio_end = ele.get("video_end", None)
                        assert _check_if_video_has_audio(
                            path
                        ), "Video must has audio track when use_audio_in_video=True"
                        if path.startswith("http://") or path.startswith("https://"):
                            data = audioread.ffdec.FFmpegAudioFile(path)
                        elif path.startswith("file://"):
                            data = path[len("file://") :]
                        else:
                            data = path
                    else:
                        raise ValueError("Unknown video {}".format(ele))
                else:
                    continue
                audios.append(
                    librosa.load(
                        data,
                        sr=SAMPLE_RATE,
                        offset=audio_start,
                        duration=(audio_end - audio_start) if audio_end is not None else None,
                    )[0]
                )
    if len(audios) == 0:
        audios = None
    return audios


In [ ]:
%%writefile /kaggle/temp/code/qwen-omni-utils/src/qwen_omni_utils/v2_5/vision_process.py
from __future__ import annotations

import base64
import copy
import logging
import math
import os
import sys
import time
import warnings
from functools import lru_cache
from io import BytesIO
from typing import Optional

import requests
import torch
import torchvision
from packaging import version
from PIL import Image
from torchvision import io, transforms
from torchvision.transforms import InterpolationMode


logger = logging.getLogger(__name__)

IMAGE_FACTOR = 28
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28
MAX_RATIO = 200

VIDEO_MIN_PIXELS = 128 * 28 * 28
VIDEO_MAX_PIXELS = 128 * 28 * 28
FRAME_FACTOR = 2
FPS = 2.0
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768

# Set the maximum number of video token inputs.
# Here, 128K represents the maximum number of input tokens for the VLLM model.
# Remember to adjust it according to your own configuration.
VIDEO_TOTAL_PIXELS = int(float(os.environ.get('VIDEO_MAX_PIXELS', 128000 * 28 * 28 * 0.9)))
logger.info(f"set VIDEO_TOTAL_PIXELS: {VIDEO_TOTAL_PIXELS}")


def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor


def smart_resize(
    height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS
) -> tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.

    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].

    3. The aspect ratio of the image is maintained as closely as possible.
    """
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = max(factor, floor_by_factor(height / beta, factor))
        w_bar = max(factor, floor_by_factor(width / beta, factor))
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


def to_rgb(pil_image: Image.Image) -> Image.Image:
    if pil_image.mode == "RGBA":
        white_background = Image.new("RGB", pil_image.size, (255, 255, 255))
        white_background.paste(pil_image, mask=pil_image.split()[3])  # Use alpha channel as mask
        return white_background
    else:
        return pil_image.convert("RGB")


def fetch_image(ele: dict[str, str | Image.Image], size_factor: int = IMAGE_FACTOR) -> Image.Image:
    if "image" in ele:
        image = ele["image"]
    else:
        image = ele["image_url"]
    image_obj = None
    if isinstance(image, Image.Image):
        image_obj = image
    elif image.startswith("http://") or image.startswith("https://"):
        # fix memory leak issue while using BytesIO
        with requests.get(image, stream=True) as response:
            response.raise_for_status()
            with BytesIO(response.content) as bio:
                image_obj = copy.deepcopy(Image.open(bio))
    elif image.startswith("file://"):
        image_obj = Image.open(image[7:])
    elif image.startswith("data:image"):
        if "base64," in image:
            _, base64_data = image.split("base64,", 1)
            data = base64.b64decode(base64_data)
            # fix memory leak issue while using BytesIO
            with BytesIO(data) as bio:
                image_obj = copy.deepcopy(Image.open(bio))
    else:
        image_obj = Image.open(image)
    if image_obj is None:
        raise ValueError(f"Unrecognized image input, support local path, http url, base64 and PIL.Image, got {image}")
    image = to_rgb(image_obj)
    ## resize
    if "resized_height" in ele and "resized_width" in ele:
        resized_height, resized_width = smart_resize(
            ele["resized_height"],
            ele["resized_width"],
            factor=size_factor,
        )
    else:
        width, height = image.size
        min_pixels = ele.get("min_pixels", MIN_PIXELS)
        max_pixels = ele.get("max_pixels", MAX_PIXELS)
        resized_height, resized_width = smart_resize(
            height,
            width,
            factor=size_factor,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
        )
    image = image.resize((resized_width, resized_height))

    return image


def smart_nframes(
    ele: dict,
    total_frames: int,
    video_fps: int | float,
) -> int:
    """calculate the number of frames for video used for model inputs.

    Args:
        ele (dict): a dict contains the configuration of video.
            support either `fps` or `nframes`:
                - nframes: the number of frames to extract for model inputs.
                - fps: the fps to extract frames for model inputs.
                    - min_frames: the minimum number of frames of the video, only used when fps is provided.
                    - max_frames: the maximum number of frames of the video, only used when fps is provided.
        total_frames (int): the original total number of frames of the video.
        video_fps (int | float): the original fps of the video.

    Raises:
        ValueError: nframes should in interval [FRAME_FACTOR, total_frames].

    Returns:
        int: the number of frames for video used for model inputs.
    """
    assert not ("fps" in ele and "nframes" in ele), "Only accept either `fps` or `nframes`"
    if "nframes" in ele:
        nframes = round_by_factor(ele["nframes"], FRAME_FACTOR)
    else:
        fps = ele.get("fps", FPS)
        min_frames = ceil_by_factor(ele.get("min_frames", FPS_MIN_FRAMES), FRAME_FACTOR)
        max_frames = floor_by_factor(ele.get("max_frames", min(FPS_MAX_FRAMES, total_frames)), FRAME_FACTOR)
        nframes = total_frames / video_fps * fps
        if nframes > total_frames:
            logger.warning(f"smart_nframes: nframes[{nframes}] > total_frames[{total_frames}]")
        nframes = min(min(max(nframes, min_frames), max_frames), total_frames)
        nframes = floor_by_factor(nframes, FRAME_FACTOR)
    if not (FRAME_FACTOR <= nframes and nframes <= total_frames):
        raise ValueError(f"nframes should in interval [{FRAME_FACTOR}, {total_frames}], but got {nframes}.")
    return nframes


def _read_video_torchvision(
    ele: dict,
) -> (torch.Tensor, float):
    """read video using torchvision.io.read_video

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    video_path = ele["video"]
    if version.parse(torchvision.__version__) < version.parse("0.19.0"):
        if "http://" in video_path or "https://" in video_path:
            warnings.warn("torchvision < 0.19.0 does not support http/https video path, please upgrade to 0.19.0.")
        if "file://" in video_path:
            video_path = video_path[7:]
    st = time.time()
    video, audio, info = io.read_video(
        video_path,
        start_pts=ele.get("video_start", 0.0),
        end_pts=ele.get("video_end", None),
        pts_unit="sec",
        output_format="TCHW",
    )
    total_frames, video_fps = video.size(0), info["video_fps"]
    logger.info(f"torchvision:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(0, total_frames - 1, nframes).round().long()
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps
    video = video[idx]
    return video, sample_fps


def is_decord_available() -> bool:
    import importlib.util

    return importlib.util.find_spec("decord") is not None


def calculate_video_frame_range(
    ele: dict,
    total_frames: int,
    video_fps: float,
) -> tuple[int, int, int]:
    """
    Calculate the start and end frame indices based on the given time range.

    Args:
        ele (dict): A dictionary containing optional 'video_start' and 'video_end' keys (in seconds).
        total_frames (int): Total number of frames in the video.
        video_fps (float): Frames per second of the video.

    Returns:
        tuple: A tuple containing (start_frame, end_frame, frame_count).

    Raises:
        ValueError: If input parameters are invalid or the time range is inconsistent.
    """
    # Validate essential parameters
    if video_fps <= 0:
        raise ValueError("video_fps must be a positive number")
    if total_frames <= 0:
        raise ValueError("total_frames must be a positive integer")

    # Get start and end time in seconds
    video_start = ele.get("video_start", None)
    video_end = ele.get("video_end", None)
    if video_start is None and video_end is None:
        return 0, total_frames - 1, total_frames

    max_duration = total_frames / video_fps
    # Process start frame
    if video_start is not None:
        video_start_clamped = max(0.0, min(video_start, max_duration))
        start_frame = math.ceil(video_start_clamped * video_fps)
    else:
        start_frame = 0
    # Process end frame
    if video_end is not None:
        video_end_clamped = max(0.0, min(video_end, max_duration))
        end_frame = math.floor(video_end_clamped * video_fps)
        end_frame = min(end_frame, total_frames - 1)
    else:
        end_frame = total_frames - 1

    # Validate frame order
    if start_frame >= end_frame:
        raise ValueError(
            f"Invalid time range: Start frame {start_frame} (at {video_start_clamped if video_start is not None else 0}s) "
            f"exceeds end frame {end_frame} (at {video_end_clamped if video_end is not None else max_duration}s). "
            f"Video duration: {max_duration:.2f}s ({total_frames} frames @ {video_fps}fps)"
        )

    logger.info(
        f"calculate video frame range: {start_frame=}, {end_frame=}, {total_frames=} from {video_start=}, {video_end=}, {video_fps=:.3f}"
    )
    return start_frame, end_frame, end_frame - start_frame + 1


def _read_video_decord(
    ele: dict,
) -> (torch.Tensor, float):
    """read video using decord.VideoReader

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    import decord
    video_path = ele["video"]
    st = time.time()
    vr = decord.VideoReader(video_path)
    total_frames, video_fps = len(vr), vr.get_avg_fps()
    start_frame, end_frame, total_frames = calculate_video_frame_range(
        ele,
        total_frames,
        video_fps,
    )
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(start_frame, end_frame, nframes).round().long().tolist()
    video = vr.get_batch(idx).asnumpy()
    video = torch.tensor(video).permute(0, 3, 1, 2)  # Convert to TCHW format
    logger.info(f"decord:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps
    return video, sample_fps


def is_torchcodec_available() -> bool:
    """Check if torchcodec is available and properly installed."""
    try:
        import importlib.util

        if importlib.util.find_spec("torchcodec") is None:
            return False
        from torchcodec.decoders import VideoDecoder

        return True
    except (ImportError, AttributeError, Exception):
        return False


def _read_video_torchcodec(
    ele: dict,
) -> (torch.Tensor, float):
    """read video using torchcodec.decoders.VideoDecoder

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    from torchcodec.decoders import VideoDecoder

    TORCHCODEC_NUM_THREADS = int(os.environ.get("TORCHCODEC_NUM_THREADS", 8))
    logger.info(f"set TORCHCODEC_NUM_THREADS: {TORCHCODEC_NUM_THREADS}")
    video_path = ele["video"]
    st = time.time()
    decoder = VideoDecoder(video_path, num_ffmpeg_threads=TORCHCODEC_NUM_THREADS)
    video_fps = decoder.metadata.average_fps
    total_frames = decoder.metadata.num_frames
    start_frame, end_frame, total_frames = calculate_video_frame_range(
        ele,
        total_frames,
        video_fps,
    )
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(start_frame, end_frame, nframes).round().long().tolist()
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps
    video = decoder.get_frames_at(indices=idx).data
    logger.info(f"torchcodec:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")
    return video, sample_fps


VIDEO_READER_BACKENDS = {
    "decord": _read_video_decord,
    "torchvision": _read_video_torchvision,
    "torchcodec": _read_video_torchcodec,
}

FORCE_QWENVL_VIDEO_READER = os.getenv("FORCE_QWENVL_VIDEO_READER", None)


@lru_cache(maxsize=1)
def get_video_reader_backend() -> str:
    if FORCE_QWENVL_VIDEO_READER is not None:
        video_reader_backend = FORCE_QWENVL_VIDEO_READER
    elif is_torchcodec_available():
        video_reader_backend = "torchcodec"
    elif is_decord_available():
        video_reader_backend = "decord"
    else:
        video_reader_backend = "torchvision"
    print(f"qwen-vl-utils using {video_reader_backend} to read video.", file=sys.stderr)
    return video_reader_backend


def fetch_video(ele: dict, image_factor: int = IMAGE_FACTOR, return_video_sample_fps: bool = False) -> torch.Tensor | list[Image.Image]:
    if isinstance(ele["video"], str):
        video_reader_backend = get_video_reader_backend()
        try:
            video, sample_fps = VIDEO_READER_BACKENDS[video_reader_backend](ele)
        except Exception as e:
            logger.warning(f"video_reader_backend {video_reader_backend} error, use torchvision as default, msg: {e}")
            video, sample_fps = VIDEO_READER_BACKENDS["torchvision"](ele)

        nframes, _, height, width = video.shape
        min_pixels = ele.get("min_pixels", VIDEO_MIN_PIXELS)
        total_pixels = ele.get("total_pixels", VIDEO_TOTAL_PIXELS)
        max_pixels = max(min(VIDEO_MAX_PIXELS, total_pixels / nframes * FRAME_FACTOR), int(min_pixels * 1.05))
        max_pixels_supposed = ele.get("max_pixels", max_pixels)
        if max_pixels_supposed > max_pixels:
            logger.warning(f"The given max_pixels[{max_pixels_supposed}] exceeds limit[{max_pixels}].")
        max_pixels = min(max_pixels_supposed, max_pixels)
        if "resized_height" in ele and "resized_width" in ele:
            resized_height, resized_width = smart_resize(
                ele["resized_height"],
                ele["resized_width"],
                factor=image_factor,
            )
        else:
            resized_height, resized_width = smart_resize(
                height,
                width,
                factor=image_factor,
                min_pixels=min_pixels,
                max_pixels=max_pixels,
            )
        video = transforms.functional.resize(
            video,
            [resized_height, resized_width],
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ).float()
        if return_video_sample_fps:
            return video, sample_fps
        return video
    else:
        assert isinstance(ele["video"], (list, tuple))
        process_info = ele.copy()
        process_info.pop("type", None)
        process_info.pop("video", None)
        images = [
            fetch_image({"image": video_element, **process_info}, size_factor=image_factor)
            for video_element in ele["video"]
        ]
        nframes = ceil_by_factor(len(images), FRAME_FACTOR)
        if len(images) < nframes:
            images.extend([images[-1]] * (nframes - len(images)))
        if return_video_sample_fps:
            return images, process_info.pop("fps", 2.0)
        return images


def extract_vision_info(conversations: list[dict] | list[list[dict]]) -> list[dict]:
    vision_infos = []
    if isinstance(conversations[0], dict):
        conversations = [conversations]
    for conversation in conversations:
        for message in conversation:
            if isinstance(message["content"], list):
                for ele in message["content"]:
                    if (
                        "image" in ele
                        or "image_url" in ele
                        or "video" in ele
                        or ele.get("type", "") in ("image", "image_url", "video")
                    ):
                        vision_infos.append(ele)
    return vision_infos


def process_vision_info(
    conversations: list[dict] | list[list[dict]],
    return_video_kwargs: bool = False,
) -> tuple[list[Image.Image] | None, list[torch.Tensor | list[Image.Image]] | None, Optional[dict]]:

    vision_infos = extract_vision_info(conversations)
    ## Read images or videos
    image_inputs = []
    video_inputs = []
    video_sample_fps_list = []
    for vision_info in vision_infos:
        if "image" in vision_info or "image_url" in vision_info:
            image_inputs.append(fetch_image(vision_info))
        elif "video" in vision_info:
            video_input, video_sample_fps = fetch_video(vision_info, return_video_sample_fps=True)
            video_sample_fps_list.append(video_sample_fps)
            video_inputs.append(video_input)
        else:
            raise ValueError("image, image_url or video should in content.")
    if len(image_inputs) == 0:
        image_inputs = None
    if len(video_inputs) == 0:
        video_inputs = None
    if return_video_kwargs:
        return image_inputs, video_inputs, {'fps': video_sample_fps_list}
    return image_inputs, video_inputs

In [ ]:
import shutil, subprocess, sys
open(f"{CODE_ROOT}/qwen-omni-utils/README.md", "w").write("qwen-omni-utils, vendored from the OmniZip repo\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.52.3", "accelerate", "decord", "einops",
                f"{CODE_ROOT}/qwen-omni-utils"], check=True)
import transformers; print("transformers", transformers.__version__, "| code commit", CODE_COMMIT)

## 1b. Kết quả lần trước (1500 bản ghi, 500 câu × 3 method, rút gọn)

In [ ]:
%%writefile /kaggle/working/prev_records.json
[{"method": "full", "video": "26TM44a-6a0.mp4", "idx": 0, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.579441547393799, "peak_mem_gb": 10.11962890625, "n_input_tokens": 3961, "n_video_tokens": 3168, "n_audio_tokens": 575, "retained_av_ratio": 1.0}, {"method": "full", "video": "a_wxf-SGdHU.mp4", "idx": 2, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1355857849121094, "peak_mem_gb": 9.19758415222168, "n_input_tokens": 1232, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 1.0}, {"method": "full", "video": "mecSGL3mrlk.mp4", "idx": 4, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.141866683959961, "peak_mem_gb": 10.742018222808838, "n_input_tokens": 5819, "n_video_tokens": 4760, "n_audio_tokens": 864, "retained_av_ratio": 1.0}, {"method": "full", "video": "Duj0OyTyjNQ.mp4", "idx": 6, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.879340410232544, "peak_mem_gb": 12.075822830200195, "n_input_tokens": 9792, "n_video_tokens": 8120, "n_audio_tokens": 1458, "retained_av_ratio": 1.0}, {"method": "full", "video": "Q7EqBvNNiT8.mp4", "idx": 8, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.256768703460693, "peak_mem_gb": 12.163787841796875, "n_input_tokens": 10056, "n_video_tokens": 8352, "n_audio_tokens": 1475, "retained_av_ratio": 1.0}, {"method": "full", "video": "87BvhuIeuyc.mp4", "idx": 10, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.40991735458374, "peak_mem_gb": 10.811551570892334, "n_input_tokens": 6031, "n_video_tokens": 4896, "n_audio_tokens": 863, "retained_av_ratio": 1.0}, {"method": "full", "video": "k9lg5Gd7Q-Y.mp4", "idx": 12, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2799186706542969, "peak_mem_gb": 9.26548719406128, "n_input_tokens": 1439, "n_video_tokens": 1008, "n_audio_tokens": 190, "retained_av_ratio": 1.0}, {"method": "full", "video": "Z2GuiatpQUw.mp4", "idx": 14, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0635302066802979, "peak_mem_gb": 9.20254373550415, "n_input_tokens": 1248, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 1.0}, {"method": "full", "video": "yQvt1MbF-dE.mp4", "idx": 16, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.477843284606934, "peak_mem_gb": 11.107953548431396, "n_input_tokens": 6920, "n_video_tokens": 5616, "n_audio_tokens": 999, "retained_av_ratio": 1.0}, {"method": "full", "video": "ziQsD1EQPOs.mp4", "idx": 18, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1267549991607666, "peak_mem_gb": 9.201320171356201, "n_input_tokens": 1244, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 1.0}, {"method": "full", "video": "rh1PHqq0Q50.mp4", "idx": 20, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8332805633544922, "peak_mem_gb": 9.027292728424072, "n_input_tokens": 728, "n_video_tokens": 432, "n_audio_tokens": 98, "retained_av_ratio": 1.0}, {"method": "full", "video": "nBAE01OGZYI.mp4", "idx": 22, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.223464250564575, "peak_mem_gb": 11.542811393737793, "n_input_tokens": 8211, "n_video_tokens": 6768, "n_audio_tokens": 1201, "retained_av_ratio": 1.0}, {"method": "full", "video": "pAEYLYBhnSk.mp4", "idx": 24, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7805209159851074, "peak_mem_gb": 9.012611865997314, "n_input_tokens": 682, "n_video_tokens": 432, "n_audio_tokens": 77, "retained_av_ratio": 1.0}, {"method": "full", "video": "zrcQYUHNcL8.mp4", "idx": 26, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9960784912109375, "peak_mem_gb": 9.542203426361084, "n_input_tokens": 2219, "n_video_tokens": 1728, "n_audio_tokens": 309, "retained_av_ratio": 1.0}, {"method": "full", "video": "KuDeAIF9rJo.mp4", "idx": 28, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 11.06263780593872, "peak_mem_gb": 12.261327266693115, "n_input_tokens": 10335, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 1.0}, {"method": "full", "video": "GbJSpPohjDM.mp4", "idx": 30, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.57521653175354, "peak_mem_gb": 9.36991262435913, "n_input_tokens": 1749, "n_video_tokens": 1287, "n_audio_tokens": 232, "retained_av_ratio": 1.0}, {"method": "full", "video": "deDj5V0XxfA.mp4", "idx": 32, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3043482303619385, "peak_mem_gb": 9.261552810668945, "n_input_tokens": 1427, "n_video_tokens": 1008, "n_audio_tokens": 181, "retained_av_ratio": 1.0}, {"method": "full", "video": "9ODSLaN3JFc.mp4", "idx": 34, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5454504489898682, "peak_mem_gb": 9.374284744262695, "n_input_tokens": 1762, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 1.0}, {"method": "full", "video": "ijg1smtVFIQ.mp4", "idx": 36, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6036102771759033, "peak_mem_gb": 9.794657707214355, "n_input_tokens": 2943, "n_video_tokens": 2304, "n_audio_tokens": 420, "retained_av_ratio": 1.0}, {"method": "full", "video": "-abNj3Imno8.mp4", "idx": 38, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1499617099761963, "peak_mem_gb": 9.667426109313965, "n_input_tokens": 2530, "n_video_tokens": 2016, "n_audio_tokens": 356, "retained_av_ratio": 1.0}, {"method": "full", "video": "YYoaOGBfSE4.mp4", "idx": 40, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.525355577468872, "peak_mem_gb": 9.375561714172363, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 1.0}, {"method": "full", "video": "ZQ3iu3wMCPg.mp4", "idx": 42, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8804965019226074, "peak_mem_gb": 9.137293338775635, "n_input_tokens": 1054, "n_video_tokens": 720, "n_audio_tokens": 130, "retained_av_ratio": 1.0}, {"method": "full", "video": "_SrDOLOuTWE.mp4", "idx": 44, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 16.01339626312256, "peak_mem_gb": 12.211503982543945, "n_input_tokens": 10196, "n_video_tokens": 8496, "n_audio_tokens": 1487, "retained_av_ratio": 1.0}, {"method": "full", "video": "mb5_ZoHnqKM.mp4", "idx": 46, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.0494773387908936, "peak_mem_gb": 10.360651969909668, "n_input_tokens": 4666, "n_video_tokens": 3744, "n_audio_tokens": 666, "retained_av_ratio": 1.0}, {"method": "full", "video": "m_IUCFFELcY.mp4", "idx": 48, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5038342475891113, "peak_mem_gb": 9.381782531738281, "n_input_tokens": 1785, "n_video_tokens": 1296, "n_audio_tokens": 247, "retained_av_ratio": 1.0}, {"method": "full", "video": "ESBEIJcZInw.mp4", "idx": 50, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9824404716491699, "peak_mem_gb": 9.205890655517578, "n_input_tokens": 1260, "n_video_tokens": 840, "n_audio_tokens": 164, "retained_av_ratio": 1.0}, {"method": "full", "video": "ltv6vNSstPI.mp4", "idx": 52, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.458070993423462, "peak_mem_gb": 9.796832084655762, "n_input_tokens": 2953, "n_video_tokens": 2304, "n_audio_tokens": 412, "retained_av_ratio": 1.0}, {"method": "full", "video": "R3JcXA8OmEM.mp4", "idx": 54, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.346383571624756, "peak_mem_gb": 10.07882022857666, "n_input_tokens": 3828, "n_video_tokens": 3080, "n_audio_tokens": 559, "retained_av_ratio": 1.0}, {"method": "full", "video": "-H5Yu7-Zb90.mp4", "idx": 56, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8105556964874268, "peak_mem_gb": 9.490219593048096, "n_input_tokens": 2078, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 1.0}, {"method": "full", "video": "TJQ22uyc6XQ.mp4", "idx": 58, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7566616535186768, "peak_mem_gb": 9.020779132843018, "n_input_tokens": 707, "n_video_tokens": 432, "n_audio_tokens": 80, "retained_av_ratio": 1.0}, {"method": "full", "video": "9G-6yw9kGIY.mp4", "idx": 60, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.905703783035278, "peak_mem_gb": 11.199143886566162, "n_input_tokens": 7182, "n_video_tokens": 5904, "n_audio_tokens": 1049, "retained_av_ratio": 1.0}, {"method": "full", "video": "zpeY4mrLlvY.mp4", "idx": 62, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.79058837890625, "peak_mem_gb": 9.49726390838623, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 1.0}, {"method": "full", "video": "B1U-6d5J7Vw.mp4", "idx": 64, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.774978876113892, "peak_mem_gb": 10.574050903320312, "n_input_tokens": 5310, "n_video_tokens": 4320, "n_audio_tokens": 773, "retained_av_ratio": 1.0}, {"method": "full", "video": "U453o9dA3cs.mp4", "idx": 66, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9839255809783936, "peak_mem_gb": 9.961441993713379, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 1.0}, {"method": "full", "video": "WtjJETMkuiM.mp4", "idx": 68, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.177586078643799, "peak_mem_gb": 10.059738636016846, "n_input_tokens": 3771, "n_video_tokens": 3024, "n_audio_tokens": 553, "retained_av_ratio": 1.0}, {"method": "full", "video": "LFIUeTL2ClU.mp4", "idx": 70, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9413485527038574, "peak_mem_gb": 9.951030254364014, "n_input_tokens": 3431, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 1.0}, {"method": "full", "video": "31zomD2ZORA.mp4", "idx": 72, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5908474922180176, "peak_mem_gb": 9.392497539520264, "n_input_tokens": 1818, "n_video_tokens": 1296, "n_audio_tokens": 244, "retained_av_ratio": 1.0}, {"method": "full", "video": "Hkzmq5SO0SE.mp4", "idx": 74, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.579007387161255, "peak_mem_gb": 12.20412302017212, "n_input_tokens": 10172, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 1.0}, {"method": "full", "video": "81SdVr4Cqb8.mp4", "idx": 76, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7638404369354248, "peak_mem_gb": 9.096094608306885, "n_input_tokens": 934, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 1.0}, {"method": "full", "video": "kCPt7hqgYis.mp4", "idx": 78, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0316510200500488, "peak_mem_gb": 9.203711032867432, "n_input_tokens": 1253, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "AbvUzIEW6lw.mp4", "idx": 80, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.496458053588867, "peak_mem_gb": 10.141253471374512, "n_input_tokens": 4026, "n_video_tokens": 3168, "n_audio_tokens": 554, "retained_av_ratio": 1.0}, {"method": "full", "video": "xPTy0rDrzU0.mp4", "idx": 82, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3304576873779297, "peak_mem_gb": 10.078527450561523, "n_input_tokens": 3827, "n_video_tokens": 3024, "n_audio_tokens": 542, "retained_av_ratio": 1.0}, {"method": "full", "video": "GD-P6jsAGWU.mp4", "idx": 84, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.145543813705444, "peak_mem_gb": 12.093954086303711, "n_input_tokens": 9847, "n_video_tokens": 8208, "n_audio_tokens": 1444, "retained_av_ratio": 1.0}, {"method": "full", "video": "aGqCwX-9Oh4.mp4", "idx": 86, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.452577829360962, "peak_mem_gb": 9.30179214477539, "n_input_tokens": 1543, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 1.0}, {"method": "full", "video": "iIDhWvE6jR0.mp4", "idx": 88, "problem_type": "Narrative Comprehension", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.21 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.5042405128479, "n_input_tokens": 11346, "n_video_tokens": 9216, "n_audio_tokens": 1901, "retained_av_ratio": null}, {"method": "full", "video": "b6dkq4sBDyg.mp4", "idx": 90, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.80458927154541, "peak_mem_gb": 10.23384714126587, "n_input_tokens": 4282, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 1.0}, {"method": "full", "video": "14__F3Bhv4E.mp4", "idx": 92, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.1014771461486816, "peak_mem_gb": 9.591634750366211, "n_input_tokens": 2289, "n_video_tokens": 1728, "n_audio_tokens": 316, "retained_av_ratio": 1.0}, {"method": "full", "video": "XxfYKJP7jkE.mp4", "idx": 94, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5255467891693115, "peak_mem_gb": 10.127957820892334, "n_input_tokens": 3984, "n_video_tokens": 3168, "n_audio_tokens": 567, "retained_av_ratio": 1.0}, {"method": "full", "video": "_OCNY0nkjGI.mp4", "idx": 96, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.913415431976318, "peak_mem_gb": 11.702133655548096, "n_input_tokens": 8682, "n_video_tokens": 7200, "n_audio_tokens": 1257, "retained_av_ratio": 1.0}, {"method": "full", "video": "0OCwUDKhb1c.mp4", "idx": 98, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4045941829681396, "peak_mem_gb": 9.731406211853027, "n_input_tokens": 2742, "n_video_tokens": 2160, "n_audio_tokens": 383, "retained_av_ratio": 1.0}, {"method": "full", "video": "abR9DrA5AyI.mp4", "idx": 100, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8934416770935059, "peak_mem_gb": 9.14387845993042, "n_input_tokens": 1073, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "Xkuh_lyCZ0Y.mp4", "idx": 102, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.032896041870117, "peak_mem_gb": 9.95194149017334, "n_input_tokens": 3434, "n_video_tokens": 2736, "n_audio_tokens": 494, "retained_av_ratio": 1.0}, {"method": "full", "video": "uo5ayLrm2xg.mp4", "idx": 104, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.911521434783936, "peak_mem_gb": 12.228792667388916, "n_input_tokens": 10246, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 1.0}, {"method": "full", "video": "SmdRGMI5qsM.mp4", "idx": 106, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.43928861618042, "peak_mem_gb": 9.326557636260986, "n_input_tokens": 1619, "n_video_tokens": 1152, "n_audio_tokens": 218, "retained_av_ratio": 1.0}, {"method": "full", "video": "1kFJgInMGd8.mp4", "idx": 108, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4457149505615234, "peak_mem_gb": 9.321660041809082, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 1.0}, {"method": "full", "video": "QodbVfAEx40.mp4", "idx": 110, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.7268662452697754, "peak_mem_gb": 9.8417649269104, "n_input_tokens": 3088, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 1.0}, {"method": "full", "video": "bSSkJPyZ5vk.mp4", "idx": 112, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.77566647529602, "peak_mem_gb": 12.002774715423584, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1397, "retained_av_ratio": 1.0}, {"method": "full", "video": "-XaFOYkyICI.mp4", "idx": 114, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 12.257044076919556, "peak_mem_gb": 12.536466598510742, "n_input_tokens": 11168, "n_video_tokens": 9216, "n_audio_tokens": 1670, "retained_av_ratio": 1.0}, {"method": "full", "video": "BVdjSumD5V4.mp4", "idx": 116, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1056575775146484, "peak_mem_gb": 9.130319595336914, "n_input_tokens": 1030, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 1.0}, {"method": "full", "video": "UMEiemYRr6w.mp4", "idx": 118, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.0025362968444824, "peak_mem_gb": 9.585295677185059, "n_input_tokens": 2273, "n_video_tokens": 1728, "n_audio_tokens": 322, "retained_av_ratio": 1.0}, {"method": "full", "video": "_kVH8Wh7tBo.mp4", "idx": 120, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.701001882553101, "peak_mem_gb": 11.185518741607666, "n_input_tokens": 7140, "n_video_tokens": 5904, "n_audio_tokens": 1041, "retained_av_ratio": 1.0}, {"method": "full", "video": "VTFUr1GXBqY.mp4", "idx": 122, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.692050933837891, "peak_mem_gb": 10.555829048156738, "n_input_tokens": 5257, "n_video_tokens": 4320, "n_audio_tokens": 755, "retained_av_ratio": 1.0}, {"method": "full", "video": "y4x5PY9JBzw.mp4", "idx": 124, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.136524438858032, "peak_mem_gb": 12.099546909332275, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1432, "retained_av_ratio": 1.0}, {"method": "full", "video": "HT9uHeaOBVY.mp4", "idx": 126, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1150546073913574, "peak_mem_gb": 9.189084529876709, "n_input_tokens": 1208, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "CjzBW4HOndQ.mp4", "idx": 128, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.179007291793823, "peak_mem_gb": 12.396683692932129, "n_input_tokens": 10748, "n_video_tokens": 8928, "n_audio_tokens": 1555, "retained_av_ratio": 1.0}, {"method": "full", "video": "2Rl9dPOJk2w.mp4", "idx": 130, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.714539051055908, "peak_mem_gb": 10.859333992004395, "n_input_tokens": 6176, "n_video_tokens": 5040, "n_audio_tokens": 878, "retained_av_ratio": 1.0}, {"method": "full", "video": "Cno_MIpe7Yg.mp4", "idx": 132, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.845426321029663, "peak_mem_gb": 12.284761428833008, "n_input_tokens": 10415, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 1.0}, {"method": "full", "video": "ajvnJTQvl_Y.mp4", "idx": 134, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.624016761779785, "peak_mem_gb": 12.202828407287598, "n_input_tokens": 10171, "n_video_tokens": 8496, "n_audio_tokens": 1492, "retained_av_ratio": 1.0}, {"method": "full", "video": "BcREVtMNh4w.mp4", "idx": 136, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9180505275726318, "peak_mem_gb": 9.150755405426025, "n_input_tokens": 1095, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 1.0}, {"method": "full", "video": "WgGReZb7igo.mp4", "idx": 138, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0143892765045166, "peak_mem_gb": 9.192011833190918, "n_input_tokens": 1217, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "6wMx9nKgqNE.mp4", "idx": 140, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1807730197906494, "peak_mem_gb": 9.26347303390503, "n_input_tokens": 1430, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 1.0}, {"method": "full", "video": "XslxClszbTU.mp4", "idx": 142, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.532747507095337, "peak_mem_gb": 10.13824987411499, "n_input_tokens": 4015, "n_video_tokens": 3220, "n_audio_tokens": 582, "retained_av_ratio": 1.0}, {"method": "full", "video": "MJKdCd93wDc.mp4", "idx": 144, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.367520332336426, "peak_mem_gb": 9.746126651763916, "n_input_tokens": 2788, "n_video_tokens": 2160, "n_audio_tokens": 401, "retained_av_ratio": 1.0}, {"method": "full", "video": "FPCuC9ZQlFo.mp4", "idx": 146, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.2414162158966064, "peak_mem_gb": 10.996462345123291, "n_input_tokens": 6580, "n_video_tokens": 5472, "n_audio_tokens": 967, "retained_av_ratio": 1.0}, {"method": "full", "video": "fujA0PoO_5w.mp4", "idx": 148, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.130279302597046, "peak_mem_gb": 9.255392074584961, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 1.0}, {"method": "full", "video": "EhwV8BqcpSA.mp4", "idx": 150, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7526898384094238, "peak_mem_gb": 9.468772888183594, "n_input_tokens": 2037, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 1.0}, {"method": "full", "video": "YnjbpidCSN8.mp4", "idx": 152, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8648593425750732, "peak_mem_gb": 9.093953609466553, "n_input_tokens": 927, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 1.0}, {"method": "full", "video": "mJpo1ijaQOs.mp4", "idx": 154, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4670846462249756, "peak_mem_gb": 9.788064002990723, "n_input_tokens": 2923, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 1.0}, {"method": "full", "video": "Zi4RV-TrTus.mp4", "idx": 156, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.377647399902344, "peak_mem_gb": 12.114580631256104, "n_input_tokens": 9908, "n_video_tokens": 8208, "n_audio_tokens": 1438, "retained_av_ratio": 1.0}, {"method": "full", "video": "Mmz1Aba-1P0.mp4", "idx": 158, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.862995624542236, "peak_mem_gb": 10.9248366355896, "n_input_tokens": 6373, "n_video_tokens": 5184, "n_audio_tokens": 902, "retained_av_ratio": 1.0}, {"method": "full", "video": "w9H2ibZd9zk.mp4", "idx": 160, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.598313570022583, "peak_mem_gb": 9.364527702331543, "n_input_tokens": 1732, "n_video_tokens": 1296, "n_audio_tokens": 239, "retained_av_ratio": 1.0}, {"method": "full", "video": "GiN9w2a7wTc.mp4", "idx": 162, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.389348268508911, "peak_mem_gb": 10.741981983184814, "n_input_tokens": 5816, "n_video_tokens": 4752, "n_audio_tokens": 832, "retained_av_ratio": 1.0}, {"method": "full", "video": "a0PA53cVK-I.mp4", "idx": 164, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.23546028137207, "peak_mem_gb": 10.378667831420898, "n_input_tokens": 4724, "n_video_tokens": 3744, "n_audio_tokens": 673, "retained_av_ratio": 1.0}, {"method": "full", "video": "T7lDs4GGNjU.mp4", "idx": 166, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.0643832683563232, "peak_mem_gb": 9.958991050720215, "n_input_tokens": 3457, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 1.0}, {"method": "full", "video": "zutS7gJ4Po0.mp4", "idx": 168, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.820929765701294, "peak_mem_gb": 10.225747108459473, "n_input_tokens": 4256, "n_video_tokens": 3456, "n_audio_tokens": 605, "retained_av_ratio": 1.0}, {"method": "full", "video": "_GTQj7_xyME.mp4", "idx": 170, "problem_type": "Narrative Comprehension", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.31 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 12.443460941314697, "n_input_tokens": 11704, "n_video_tokens": 9216, "n_audio_tokens": 2226, "retained_av_ratio": null}, {"method": "full", "video": "pcnVQwKXsSc.mp4", "idx": 172, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2726836204528809, "peak_mem_gb": 9.273601055145264, "n_input_tokens": 1461, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 1.0}, {"method": "full", "video": "eE2SsX4F8TM.mp4", "idx": 174, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8221607208251953, "peak_mem_gb": 9.092699527740479, "n_input_tokens": 923, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 1.0}, {"method": "full", "video": "0CCRotntwdA.mp4", "idx": 176, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7849292755126953, "peak_mem_gb": 9.467797756195068, "n_input_tokens": 2034, "n_video_tokens": 1584, "n_audio_tokens": 293, "retained_av_ratio": 1.0}, {"method": "full", "video": "wREFmTKv5DM.mp4", "idx": 178, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.227283477783203, "peak_mem_gb": 12.03716516494751, "n_input_tokens": 9675, "n_video_tokens": 8064, "n_audio_tokens": 1421, "retained_av_ratio": 1.0}, {"method": "full", "video": "h7Mjce0y7SY.mp4", "idx": 180, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0794117450714111, "peak_mem_gb": 9.206972599029541, "n_input_tokens": 1263, "n_video_tokens": 864, "n_audio_tokens": 168, "retained_av_ratio": 1.0}, {"method": "full", "video": "V2CSquBY4pk.mp4", "idx": 182, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.159997463226318, "peak_mem_gb": 10.97481393814087, "n_input_tokens": 6521, "n_video_tokens": 5328, "n_audio_tokens": 928, "retained_av_ratio": 1.0}, {"method": "full", "video": "bHsmSHIoThw.mp4", "idx": 184, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.708397388458252, "peak_mem_gb": 9.419112205505371, "n_input_tokens": 1894, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 1.0}, {"method": "full", "video": "WLZ8GAD_Mus.mp4", "idx": 186, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4876883029937744, "peak_mem_gb": 9.734521865844727, "n_input_tokens": 2753, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 1.0}, {"method": "full", "video": "UaBfM-tONmI.mp4", "idx": 188, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 12.949832439422607, "peak_mem_gb": 12.568995952606201, "n_input_tokens": 11265, "n_video_tokens": 9216, "n_audio_tokens": 1840, "retained_av_ratio": 1.0}, {"method": "full", "video": "KF19sEfFHM8.mp4", "idx": 190, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8525283336639404, "peak_mem_gb": 9.030983924865723, "n_input_tokens": 739, "n_video_tokens": 432, "n_audio_tokens": 93, "retained_av_ratio": 1.0}, {"method": "full", "video": "7Np8y8RdAxA.mp4", "idx": 192, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.1450703144073486, "peak_mem_gb": 9.61380672454834, "n_input_tokens": 2358, "n_video_tokens": 1872, "n_audio_tokens": 326, "retained_av_ratio": 1.0}, {"method": "full", "video": "x-RjxvYlzrE.mp4", "idx": 194, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8008217811584473, "peak_mem_gb": 9.048998355865479, "n_input_tokens": 795, "n_video_tokens": 432, "n_audio_tokens": 97, "retained_av_ratio": 1.0}, {"method": "full", "video": "FkKh29UxzEc.mp4", "idx": 196, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.26 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 12.106751918792725, "n_input_tokens": 11521, "n_video_tokens": 9216, "n_audio_tokens": 2115, "retained_av_ratio": null}, {"method": "full", "video": "fYYV5CoB5QM.mp4", "idx": 198, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6824922561645508, "peak_mem_gb": 9.388888359069824, "n_input_tokens": 1807, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 1.0}, {"method": "full", "video": "nsbOHGP_uGI.mp4", "idx": 200, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7278735637664795, "peak_mem_gb": 9.03481912612915, "n_input_tokens": 751, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 1.0}, {"method": "full", "video": "4tyX4O4it5U.mp4", "idx": 202, "problem_type": "Affective Intent Classification", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.80 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.414559841156006, "n_input_tokens": 9878, "n_video_tokens": 8208, "n_audio_tokens": 1443, "retained_av_ratio": null}, {"method": "full", "video": "-32uRUUd-KU.mp4", "idx": 204, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0084023475646973, "peak_mem_gb": 9.164991855621338, "n_input_tokens": 1139, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "LQsQCioPS4U.mp4", "idx": 206, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0385363101959229, "peak_mem_gb": 9.19814395904541, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 154, "retained_av_ratio": 1.0}, {"method": "full", "video": "21IQzeQBnGo.mp4", "idx": 208, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.13506817817688, "peak_mem_gb": 9.63262939453125, "n_input_tokens": 2420, "n_video_tokens": 1872, "n_audio_tokens": 334, "retained_av_ratio": 1.0}, {"method": "full", "video": "Q0GG4iuGL3g.mp4", "idx": 210, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1122348308563232, "peak_mem_gb": 9.186144828796387, "n_input_tokens": 1199, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 1.0}, {"method": "full", "video": "PBhCY56XIcI.mp4", "idx": 212, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3056442737579346, "peak_mem_gb": 9.250861644744873, "n_input_tokens": 1391, "n_video_tokens": 1008, "n_audio_tokens": 201, "retained_av_ratio": 1.0}, {"method": "full", "video": "eEV120knBoA.mp4", "idx": 214, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9832608699798584, "peak_mem_gb": 9.966373920440674, "n_input_tokens": 3478, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 1.0}, {"method": "full", "video": "2lXyTywKtkw.mp4", "idx": 216, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.717511177062988, "peak_mem_gb": 10.501514434814453, "n_input_tokens": 5093, "n_video_tokens": 4176, "n_audio_tokens": 747, "retained_av_ratio": 1.0}, {"method": "full", "video": "olx5oFJbivo.mp4", "idx": 218, "problem_type": "Narrative Comprehension", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.84 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.44101619720459, "n_input_tokens": 10026, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": null}, {"method": "full", "video": "_uDlP4vW190.mp4", "idx": 220, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.69 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.312524318695068, "n_input_tokens": 9506, "n_video_tokens": 7920, "n_audio_tokens": 1394, "retained_av_ratio": null}, {"method": "full", "video": "ZJovAHQU0o8.mp4", "idx": 222, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0904710292816162, "peak_mem_gb": 9.206311225891113, "n_input_tokens": 1261, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "YB3bxufnn7s.mp4", "idx": 224, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.608309030532837, "peak_mem_gb": 9.354443550109863, "n_input_tokens": 1701, "n_video_tokens": 1296, "n_audio_tokens": 237, "retained_av_ratio": 1.0}, {"method": "full", "video": "SexJ0OetXZA.mp4", "idx": 226, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7521820068359375, "peak_mem_gb": 9.025932788848877, "n_input_tokens": 723, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 1.0}, {"method": "full", "video": "3l4sUgCflzc.mp4", "idx": 228, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0346672534942627, "peak_mem_gb": 9.137267112731934, "n_input_tokens": 1052, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 1.0}, {"method": "full", "video": "KwhBtGE-mDc.mp4", "idx": 230, "problem_type": "Narrative Comprehension", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.92 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.541566848754883, "n_input_tokens": 10305, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": null}, {"method": "full", "video": "BojPeImI2QE.mp4", "idx": 232, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.89 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.538921356201172, "n_input_tokens": 10220, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": null}, {"method": "full", "video": "4NlfaBAaGrM.mp4", "idx": 234, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.961762428283691, "peak_mem_gb": 10.570856094360352, "n_input_tokens": 5305, "n_video_tokens": 4320, "n_audio_tokens": 760, "retained_av_ratio": 1.0}, {"method": "full", "video": "2Yp6UqQPQNE.mp4", "idx": 236, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.83 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.473042488098145, "n_input_tokens": 10012, "n_video_tokens": 8352, "n_audio_tokens": 1471, "retained_av_ratio": null}, {"method": "full", "video": "qzUdGKGEF6o.mp4", "idx": 238, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4783477783203125, "peak_mem_gb": 9.373979091644287, "n_input_tokens": 1761, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 1.0}, {"method": "full", "video": "WgWdcSg5_to.mp4", "idx": 240, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.89 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 10.523969173431396, "n_input_tokens": 10225, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": null}, {"method": "full", "video": "rD5uIy9UgxA.mp4", "idx": 242, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4400296211242676, "peak_mem_gb": 9.335355281829834, "n_input_tokens": 1646, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 1.0}, {"method": "full", "video": "y_MMvaZvGIA.mp4", "idx": 244, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.620625972747803, "peak_mem_gb": 11.12898302078247, "n_input_tokens": 6979, "n_video_tokens": 5760, "n_audio_tokens": 1010, "retained_av_ratio": 1.0}, {"method": "full", "video": "HO65gX-k31g.mp4", "idx": 246, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1684601306915283, "peak_mem_gb": 9.217312812805176, "n_input_tokens": 1295, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 1.0}, {"method": "full", "video": "DGNI8-twVys.mp4", "idx": 248, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9468541145324707, "peak_mem_gb": 9.946372509002686, "n_input_tokens": 3415, "n_video_tokens": 2736, "n_audio_tokens": 497, "retained_av_ratio": 1.0}, {"method": "full", "video": "Igf8n7hy1HI.mp4", "idx": 250, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1320507526397705, "peak_mem_gb": 9.256290435791016, "n_input_tokens": 1408, "n_video_tokens": 1008, "n_audio_tokens": 176, "retained_av_ratio": 1.0}, {"method": "full", "video": "JPqjEZXKoQc.mp4", "idx": 252, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6659560203552246, "peak_mem_gb": 9.424893379211426, "n_input_tokens": 1912, "n_video_tokens": 1440, "n_audio_tokens": 253, "retained_av_ratio": 1.0}, {"method": "full", "video": "IhiQi9X8l8o.mp4", "idx": 254, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.528316974639893, "peak_mem_gb": 11.58561372756958, "n_input_tokens": 8332, "n_video_tokens": 6912, "n_audio_tokens": 1210, "retained_av_ratio": 1.0}, {"method": "full", "video": "vR8fuThRlA4.mp4", "idx": 256, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7268881797790527, "peak_mem_gb": 9.421737670898438, "n_input_tokens": 1900, "n_video_tokens": 1440, "n_audio_tokens": 257, "retained_av_ratio": 1.0}, {"method": "full", "video": "qP6IYn3MrcA.mp4", "idx": 258, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.945160627365112, "peak_mem_gb": 11.47216272354126, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 1.0}, {"method": "full", "video": "WBtbl0O4CZA.mp4", "idx": 260, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.33581280708313, "peak_mem_gb": 12.339423179626465, "n_input_tokens": 10580, "n_video_tokens": 8784, "n_audio_tokens": 1539, "retained_av_ratio": 1.0}, {"method": "full", "video": "HAjJZ1GOXFA.mp4", "idx": 262, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.786935567855835, "peak_mem_gb": 9.030696868896484, "n_input_tokens": 738, "n_video_tokens": 432, "n_audio_tokens": 101, "retained_av_ratio": 1.0}, {"method": "full", "video": "kzZ7TfwT9Ss.mp4", "idx": 264, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.410099744796753, "peak_mem_gb": 12.099166870117188, "n_input_tokens": 9863, "n_video_tokens": 8208, "n_audio_tokens": 1441, "retained_av_ratio": 1.0}, {"method": "full", "video": "oZlh0hjcXfA.mp4", "idx": 266, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.600900411605835, "peak_mem_gb": 9.83251428604126, "n_input_tokens": 3066, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 1.0}, {"method": "full", "video": "hIWZDSR_uns.mp4", "idx": 268, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4449892044067383, "peak_mem_gb": 9.745080471038818, "n_input_tokens": 2784, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 1.0}, {"method": "full", "video": "91kNCJOoHcY.mp4", "idx": 270, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1447174549102783, "peak_mem_gb": 9.198136329650879, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 1.0}, {"method": "full", "video": "4HYLaF6fbFY.mp4", "idx": 272, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9723222255706787, "peak_mem_gb": 9.916911125183105, "n_input_tokens": 3330, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 1.0}, {"method": "full", "video": "dtGj9oet70E.mp4", "idx": 274, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.007217407226562, "peak_mem_gb": 12.244093894958496, "n_input_tokens": 10285, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "PwSOZli5_xg.mp4", "idx": 276, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.3336570262908936, "peak_mem_gb": 10.726068019866943, "n_input_tokens": 5769, "n_video_tokens": 4752, "n_audio_tokens": 843, "retained_av_ratio": 1.0}, {"method": "full", "video": "MYM3IFheI-E.mp4", "idx": 278, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2305278778076172, "peak_mem_gb": 9.254738330841064, "n_input_tokens": 1403, "n_video_tokens": 1008, "n_audio_tokens": 195, "retained_av_ratio": 1.0}, {"method": "full", "video": "EB9bA1a5TEs.mp4", "idx": 280, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5698883533477783, "peak_mem_gb": 9.380168914794922, "n_input_tokens": 1780, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 1.0}, {"method": "full", "video": "oZLqkTvWXko.mp4", "idx": 282, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6545913219451904, "peak_mem_gb": 9.426892757415771, "n_input_tokens": 1918, "n_video_tokens": 1440, "n_audio_tokens": 266, "retained_av_ratio": 1.0}, {"method": "full", "video": "kS6aRrdqI7M.mp4", "idx": 284, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7030410766601562, "peak_mem_gb": 9.851371765136719, "n_input_tokens": 3122, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 1.0}, {"method": "full", "video": "1P6XRN-amuU.mp4", "idx": 286, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.578171968460083, "peak_mem_gb": 9.4132661819458, "n_input_tokens": 1876, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 1.0}, {"method": "full", "video": "akm-KQiLsvI.mp4", "idx": 288, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6034696102142334, "peak_mem_gb": 10.181512832641602, "n_input_tokens": 4120, "n_video_tokens": 3312, "n_audio_tokens": 584, "retained_av_ratio": 1.0}, {"method": "full", "video": "JjuQ2ink05Q.mp4", "idx": 290, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.165015935897827, "peak_mem_gb": 10.350218772888184, "n_input_tokens": 4632, "n_video_tokens": 3744, "n_audio_tokens": 660, "retained_av_ratio": 1.0}, {"method": "full", "video": "TQjp3EaFCsc.mp4", "idx": 292, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.802427768707275, "peak_mem_gb": 10.908877849578857, "n_input_tokens": 6320, "n_video_tokens": 5184, "n_audio_tokens": 919, "retained_av_ratio": 1.0}, {"method": "full", "video": "pyfj75fxjj8.mp4", "idx": 294, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.605997085571289, "peak_mem_gb": 9.8034029006958, "n_input_tokens": 2972, "n_video_tokens": 2304, "n_audio_tokens": 422, "retained_av_ratio": 1.0}, {"method": "full", "video": "05FGA7Febzc.mp4", "idx": 296, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9546070098876953, "peak_mem_gb": 9.087678909301758, "n_input_tokens": 907, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 1.0}, {"method": "full", "video": "8OqmJB8oGtM.mp4", "idx": 298, "problem_type": "Narrative Comprehension", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 5.63 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.469868183135986, "n_input_tokens": 13764, "n_video_tokens": 9216, "n_audio_tokens": 4347, "retained_av_ratio": null}, {"method": "full", "video": "bajfnk7lwL4.mp4", "idx": 300, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9188613891601562, "peak_mem_gb": 9.960522651672363, "n_input_tokens": 3462, "n_video_tokens": 2736, "n_audio_tokens": 500, "retained_av_ratio": 1.0}, {"method": "full", "video": "mrYBGL9-DDE.mp4", "idx": 302, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.656783819198608, "peak_mem_gb": 10.852990627288818, "n_input_tokens": 6155, "n_video_tokens": 5040, "n_audio_tokens": 890, "retained_av_ratio": 1.0}, {"method": "full", "video": "odgiRt2sitM.mp4", "idx": 304, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9173386096954346, "peak_mem_gb": 9.107421875, "n_input_tokens": 969, "n_video_tokens": 576, "n_audio_tokens": 112, "retained_av_ratio": 1.0}, {"method": "full", "video": "iIZiJR-MHnM.mp4", "idx": 306, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9147121906280518, "peak_mem_gb": 9.49726390838623, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 1.0}, {"method": "full", "video": "vcsHoRWlD7E.mp4", "idx": 308, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8545222282409668, "peak_mem_gb": 9.518871307373047, "n_input_tokens": 2154, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 1.0}, {"method": "full", "video": "ge-zGSapyeQ.mp4", "idx": 310, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0402746200561523, "peak_mem_gb": 9.208910942077637, "n_input_tokens": 1269, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "eAOKPLl6AVY.mp4", "idx": 312, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6703412532806396, "peak_mem_gb": 10.185731887817383, "n_input_tokens": 4134, "n_video_tokens": 3312, "n_audio_tokens": 593, "retained_av_ratio": 1.0}, {"method": "full", "video": "HchGCB08bLg.mp4", "idx": 314, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8193943500518799, "peak_mem_gb": 9.088883399963379, "n_input_tokens": 911, "n_video_tokens": 576, "n_audio_tokens": 111, "retained_av_ratio": 1.0}, {"method": "full", "video": "ECvz4IPwFQ8.mp4", "idx": 316, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.598629236221313, "peak_mem_gb": 11.929829120635986, "n_input_tokens": 9358, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 1.0}, {"method": "full", "video": "Bj7N9gscARc.mp4", "idx": 318, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.9907517433166504, "peak_mem_gb": 9.977163314819336, "n_input_tokens": 3515, "n_video_tokens": 2736, "n_audio_tokens": 501, "retained_av_ratio": 1.0}, {"method": "full", "video": "3IIaJnEspcI.mp4", "idx": 320, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.3174309730529785, "peak_mem_gb": 11.032571792602539, "n_input_tokens": 6692, "n_video_tokens": 5472, "n_audio_tokens": 974, "retained_av_ratio": 1.0}, {"method": "full", "video": "rYtG6G6tYG0.mp4", "idx": 322, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9635512828826904, "peak_mem_gb": 9.136598110198975, "n_input_tokens": 1050, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 1.0}, {"method": "full", "video": "aneq_Adxc8U.mp4", "idx": 324, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2309560775756836, "peak_mem_gb": 9.24851369857788, "n_input_tokens": 1384, "n_video_tokens": 1008, "n_audio_tokens": 182, "retained_av_ratio": 1.0}, {"method": "full", "video": "4u7eslydITE.mp4", "idx": 326, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.569871664047241, "peak_mem_gb": 12.15238618850708, "n_input_tokens": 10022, "n_video_tokens": 8352, "n_audio_tokens": 1472, "retained_av_ratio": 1.0}, {"method": "full", "video": "5_tgh8PTlxk.mp4", "idx": 328, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.367424249649048, "peak_mem_gb": 9.738483905792236, "n_input_tokens": 2767, "n_video_tokens": 2160, "n_audio_tokens": 391, "retained_av_ratio": 1.0}, {"method": "full", "video": "xyt5IrIgprc.mp4", "idx": 330, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.235715389251709, "peak_mem_gb": 9.691973686218262, "n_input_tokens": 2609, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 1.0}, {"method": "full", "video": "huSqwxMjL-Q.mp4", "idx": 332, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4469735622406006, "peak_mem_gb": 9.346081733703613, "n_input_tokens": 1679, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 1.0}, {"method": "full", "video": "2IvR2LGWvcg.mp4", "idx": 334, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.097766876220703, "peak_mem_gb": 9.99790906906128, "n_input_tokens": 3574, "n_video_tokens": 2880, "n_audio_tokens": 502, "retained_av_ratio": 1.0}, {"method": "full", "video": "4WcVCJr0UwY.mp4", "idx": 336, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.784127950668335, "peak_mem_gb": 9.08643627166748, "n_input_tokens": 903, "n_video_tokens": 576, "n_audio_tokens": 127, "retained_av_ratio": 1.0}, {"method": "full", "video": "t3YK5ezf8u4.mp4", "idx": 338, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.32216739654541, "peak_mem_gb": 9.697747707366943, "n_input_tokens": 2627, "n_video_tokens": 2016, "n_audio_tokens": 358, "retained_av_ratio": 1.0}, {"method": "full", "video": "1pNEF7I8EIk.mp4", "idx": 340, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.991774559020996, "peak_mem_gb": 12.060094356536865, "n_input_tokens": 9747, "n_video_tokens": 8064, "n_audio_tokens": 1405, "retained_av_ratio": 1.0}, {"method": "full", "video": "3D5kuvs3LY8.mp4", "idx": 342, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.445746898651123, "peak_mem_gb": 9.367486953735352, "n_input_tokens": 1741, "n_video_tokens": 1296, "n_audio_tokens": 248, "retained_av_ratio": 1.0}, {"method": "full", "video": "zdvIyL1uFeY.mp4", "idx": 344, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.064728498458862, "peak_mem_gb": 10.702254295349121, "n_input_tokens": 5701, "n_video_tokens": 4608, "n_audio_tokens": 821, "retained_av_ratio": 1.0}, {"method": "full", "video": "yB0eIpT8DBM.mp4", "idx": 346, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8227128982543945, "peak_mem_gb": 9.08859634399414, "n_input_tokens": 910, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 1.0}, {"method": "full", "video": "MbiOwSUz4TY.mp4", "idx": 348, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2918219566345215, "peak_mem_gb": 9.310895442962646, "n_input_tokens": 1571, "n_video_tokens": 1152, "n_audio_tokens": 202, "retained_av_ratio": 1.0}, {"method": "full", "video": "6C33lbRzaMc.mp4", "idx": 350, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.73095965385437, "peak_mem_gb": 10.23181438446045, "n_input_tokens": 4275, "n_video_tokens": 3432, "n_audio_tokens": 602, "retained_av_ratio": 1.0}, {"method": "full", "video": "zBC5SPmMBiI.mp4", "idx": 352, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.878445863723755, "peak_mem_gb": 12.237802982330322, "n_input_tokens": 10279, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 1.0}, {"method": "full", "video": "cI7g9nPVw8s.mp4", "idx": 354, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.16515040397644, "peak_mem_gb": 10.363109111785889, "n_input_tokens": 4674, "n_video_tokens": 3744, "n_audio_tokens": 663, "retained_av_ratio": 1.0}, {"method": "full", "video": "V8ipyMRS5co.mp4", "idx": 356, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.167969226837158, "peak_mem_gb": 12.29735279083252, "n_input_tokens": 10451, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 1.0}, {"method": "full", "video": "aw0_Q6ZOx0o.mp4", "idx": 358, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.679823875427246, "peak_mem_gb": 9.436038970947266, "n_input_tokens": 1944, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 1.0}, {"method": "full", "video": "gJjosUoZiWs.mp4", "idx": 360, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.631328821182251, "peak_mem_gb": 11.13710355758667, "n_input_tokens": 7004, "n_video_tokens": 5760, "n_audio_tokens": 1009, "retained_av_ratio": 1.0}, {"method": "full", "video": "i2fOL059y1A.mp4", "idx": 362, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.785784721374512, "peak_mem_gb": 10.561777114868164, "n_input_tokens": 5276, "n_video_tokens": 4320, "n_audio_tokens": 753, "retained_av_ratio": 1.0}, {"method": "full", "video": "j5wq7pLFlU4.mp4", "idx": 364, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.6620512008667, "peak_mem_gb": 11.94097089767456, "n_input_tokens": 9390, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 1.0}, {"method": "full", "video": "nprAhUMzWag.mp4", "idx": 366, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.665408611297607, "peak_mem_gb": 12.208571434020996, "n_input_tokens": 10183, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 1.0}, {"method": "full", "video": "IiHvzFP6VQk.mp4", "idx": 368, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.738537549972534, "peak_mem_gb": 11.940295696258545, "n_input_tokens": 9385, "n_video_tokens": 7776, "n_audio_tokens": 1376, "retained_av_ratio": 1.0}, {"method": "full", "video": "oT-R-2YcqRQ.mp4", "idx": 370, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.778355121612549, "peak_mem_gb": 12.215089321136475, "n_input_tokens": 10208, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 1.0}, {"method": "full", "video": "hfS1Io1bQSU.mp4", "idx": 372, "problem_type": "Narrative Comprehension", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.561555862426758, "peak_mem_gb": 12.218141555786133, "n_input_tokens": 10218, "n_video_tokens": 8496, "n_audio_tokens": 1493, "retained_av_ratio": 1.0}, {"method": "full", "video": "rPRGQOics9A.mp4", "idx": 374, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9832570552825928, "peak_mem_gb": 9.171818256378174, "n_input_tokens": 1160, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "sYxaFHBsxKs.mp4", "idx": 376, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.205752849578857, "peak_mem_gb": 11.57617473602295, "n_input_tokens": 8302, "n_video_tokens": 6912, "n_audio_tokens": 1205, "retained_av_ratio": 1.0}, {"method": "full", "video": "Ibe_srSbN28.mp4", "idx": 378, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.876498460769653, "peak_mem_gb": 10.93724775314331, "n_input_tokens": 6408, "n_video_tokens": 5184, "n_audio_tokens": 923, "retained_av_ratio": 1.0}, {"method": "full", "video": "gRqQQYJDD5Y.mp4", "idx": 380, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.7835822105407715, "peak_mem_gb": 10.22867727279663, "n_input_tokens": 4266, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 1.0}, {"method": "full", "video": "T2VIyWfYZFI.mp4", "idx": 382, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.7597453594207764, "peak_mem_gb": 10.254193305969238, "n_input_tokens": 4346, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 1.0}, {"method": "full", "video": "3olxUYqO8cQ.mp4", "idx": 384, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.3370614051818848, "peak_mem_gb": 9.749004364013672, "n_input_tokens": 2799, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 1.0}, {"method": "full", "video": "XjZaY7vgAzg.mp4", "idx": 386, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9108502864837646, "peak_mem_gb": 9.550006866455078, "n_input_tokens": 2247, "n_video_tokens": 1728, "n_audio_tokens": 320, "retained_av_ratio": 1.0}, {"method": "full", "video": "EPzdZd7DnsQ.mp4", "idx": 388, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7414212226867676, "peak_mem_gb": 9.846568584442139, "n_input_tokens": 3105, "n_video_tokens": 2448, "n_audio_tokens": 439, "retained_av_ratio": 1.0}, {"method": "full", "video": "xwfHWw5A2HI.mp4", "idx": 390, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.885681629180908, "peak_mem_gb": 12.255019187927246, "n_input_tokens": 10320, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "otVWxaBSzL0.mp4", "idx": 392, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9956207275390625, "peak_mem_gb": 9.544283866882324, "n_input_tokens": 2231, "n_video_tokens": 1728, "n_audio_tokens": 319, "retained_av_ratio": 1.0}, {"method": "full", "video": "s6C8atMqM1Q.mp4", "idx": 394, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.866525650024414, "peak_mem_gb": 10.236463069915771, "n_input_tokens": 4291, "n_video_tokens": 3456, "n_audio_tokens": 614, "retained_av_ratio": 1.0}, {"method": "full", "video": "xlC9oqCHQ84.mp4", "idx": 396, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5068893432617188, "peak_mem_gb": 9.36484146118164, "n_input_tokens": 1733, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 1.0}, {"method": "full", "video": "wR4Za_pz2ec.mp4", "idx": 398, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7627017498016357, "peak_mem_gb": 9.450968742370605, "n_input_tokens": 1992, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 1.0}, {"method": "full", "video": "Nql8w5AYwAo.mp4", "idx": 400, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.117661714553833, "peak_mem_gb": 9.636908054351807, "n_input_tokens": 2436, "n_video_tokens": 1872, "n_audio_tokens": 347, "retained_av_ratio": 1.0}, {"method": "full", "video": "1t0JN_QdBkA.mp4", "idx": 402, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.304870367050171, "peak_mem_gb": 11.581531047821045, "n_input_tokens": 8319, "n_video_tokens": 6912, "n_audio_tokens": 1209, "retained_av_ratio": 1.0}, {"method": "full", "video": "-dtEpy4XUbQ.mp4", "idx": 404, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0052077770233154, "peak_mem_gb": 9.134081840515137, "n_input_tokens": 1042, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 1.0}, {"method": "full", "video": "i9LSUR1OSjs.mp4", "idx": 406, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.359269380569458, "peak_mem_gb": 9.27711820602417, "n_input_tokens": 1472, "n_video_tokens": 1008, "n_audio_tokens": 183, "retained_av_ratio": 1.0}, {"method": "full", "video": "F7M7aY5fofo.mp4", "idx": 408, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.031036615371704, "peak_mem_gb": 9.160751342773438, "n_input_tokens": 1126, "n_video_tokens": 720, "n_audio_tokens": 145, "retained_av_ratio": 1.0}, {"method": "full", "video": "eMPOMRBnBpc.mp4", "idx": 410, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9912264347076416, "peak_mem_gb": 9.592992305755615, "n_input_tokens": 2294, "n_video_tokens": 1728, "n_audio_tokens": 318, "retained_av_ratio": 1.0}, {"method": "full", "video": "bSppSU7Njig.mp4", "idx": 412, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 11.057981252670288, "peak_mem_gb": 12.283474445343018, "n_input_tokens": 10409, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "72rcNxQbIoI.mp4", "idx": 414, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9969298839569092, "peak_mem_gb": 9.154916763305664, "n_input_tokens": 1108, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "jl38NDYzQV0.mp4", "idx": 416, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.498847723007202, "peak_mem_gb": 10.121487140655518, "n_input_tokens": 3966, "n_video_tokens": 3168, "n_audio_tokens": 564, "retained_av_ratio": 1.0}, {"method": "full", "video": "V30jgIuBG94.mp4", "idx": 418, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6216135025024414, "peak_mem_gb": 9.415563106536865, "n_input_tokens": 1881, "n_video_tokens": 1440, "n_audio_tokens": 261, "retained_av_ratio": 1.0}, {"method": "full", "video": "9XbP_d_u_OI.mp4", "idx": 420, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.5469207763671875, "peak_mem_gb": 11.097490787506104, "n_input_tokens": 6888, "n_video_tokens": 5616, "n_audio_tokens": 983, "retained_av_ratio": 1.0}, {"method": "full", "video": "FP7u3iJxruw.mp4", "idx": 422, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.227736234664917, "peak_mem_gb": 12.290291786193848, "n_input_tokens": 10430, "n_video_tokens": 8640, "n_audio_tokens": 1505, "retained_av_ratio": 1.0}, {"method": "full", "video": "DWqcf5QvkJI.mp4", "idx": 424, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9020888805389404, "peak_mem_gb": 9.610735416412354, "n_input_tokens": 2353, "n_video_tokens": 1728, "n_audio_tokens": 312, "retained_av_ratio": 1.0}, {"method": "full", "video": "P_cYNsKKpGg.mp4", "idx": 426, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9371130466461182, "peak_mem_gb": 9.579113483428955, "n_input_tokens": 2253, "n_video_tokens": 1728, "n_audio_tokens": 301, "retained_av_ratio": 1.0}, {"method": "full", "video": "J01MxqCqkyE.mp4", "idx": 428, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.090625762939453, "peak_mem_gb": 9.934851169586182, "n_input_tokens": 3381, "n_video_tokens": 2736, "n_audio_tokens": 486, "retained_av_ratio": 1.0}, {"method": "full", "video": "wdESby14IZY.mp4", "idx": 430, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.863502740859985, "peak_mem_gb": 11.479952812194824, "n_input_tokens": 8023, "n_video_tokens": 6624, "n_audio_tokens": 1163, "retained_av_ratio": 1.0}, {"method": "full", "video": "n7vwcWG5ZY0.mp4", "idx": 432, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.839131116867065, "peak_mem_gb": 12.222540855407715, "n_input_tokens": 10229, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 1.0}, {"method": "full", "video": "18OOnHDat4E.mp4", "idx": 434, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.665382146835327, "peak_mem_gb": 10.18211555480957, "n_input_tokens": 4122, "n_video_tokens": 3312, "n_audio_tokens": 580, "retained_av_ratio": 1.0}, {"method": "full", "video": "w6C29ECeFs8.mp4", "idx": 436, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9652018547058105, "peak_mem_gb": 9.14858102798462, "n_input_tokens": 1088, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "JB8rAfaUUKY.mp4", "idx": 438, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9289004802703857, "peak_mem_gb": 9.904312133789062, "n_input_tokens": 3286, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 1.0}, {"method": "full", "video": "1YORkMT9JoY.mp4", "idx": 440, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0954835414886475, "peak_mem_gb": 9.206605434417725, "n_input_tokens": 1262, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 1.0}, {"method": "full", "video": "D980E_PsoIs.mp4", "idx": 442, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.646397113800049, "peak_mem_gb": 9.819993495941162, "n_input_tokens": 3026, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 1.0}, {"method": "full", "video": "nGHKrbtPRb4.mp4", "idx": 444, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.828559160232544, "peak_mem_gb": 9.900233268737793, "n_input_tokens": 3273, "n_video_tokens": 2592, "n_audio_tokens": 458, "retained_av_ratio": 1.0}, {"method": "full", "video": "AmWG0oYhDh4.mp4", "idx": 446, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.949612140655518, "peak_mem_gb": 12.002774715423584, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1383, "retained_av_ratio": 1.0}, {"method": "full", "video": "gmKpD8QPvSA.mp4", "idx": 448, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5164563655853271, "peak_mem_gb": 9.3690505027771, "n_input_tokens": 1746, "n_video_tokens": 1296, "n_audio_tokens": 232, "retained_av_ratio": 1.0}, {"method": "full", "video": "Wl8UjlAedeQ.mp4", "idx": 450, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8915951251983643, "peak_mem_gb": 9.509373188018799, "n_input_tokens": 2148, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 1.0}, {"method": "full", "video": "dEp4CVa18-s.mp4", "idx": 452, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7194600105285645, "peak_mem_gb": 9.835208892822266, "n_input_tokens": 3072, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 1.0}, {"method": "full", "video": "aWo90tAPcBs.mp4", "idx": 454, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.167814493179321, "peak_mem_gb": 10.966165542602539, "n_input_tokens": 6494, "n_video_tokens": 5328, "n_audio_tokens": 951, "retained_av_ratio": 1.0}, {"method": "full", "video": "XHdk-W9L9RQ.mp4", "idx": 456, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.025787353515625, "peak_mem_gb": 9.137554168701172, "n_input_tokens": 1053, "n_video_tokens": 720, "n_audio_tokens": 135, "retained_av_ratio": 1.0}, {"method": "full", "video": "1SjR4zREgq8.mp4", "idx": 458, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.028184175491333, "peak_mem_gb": 9.157127380371094, "n_input_tokens": 1115, "n_video_tokens": 720, "n_audio_tokens": 132, "retained_av_ratio": 1.0}, {"method": "full", "video": "mVJ32Ex_gqk.mp4", "idx": 460, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.863241672515869, "peak_mem_gb": 11.194681167602539, "n_input_tokens": 7168, "n_video_tokens": 5904, "n_audio_tokens": 1042, "retained_av_ratio": 1.0}, {"method": "full", "video": "RFm_F9hS0Y8.mp4", "idx": 462, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0513265132904053, "peak_mem_gb": 9.194886684417725, "n_input_tokens": 1226, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 1.0}, {"method": "full", "video": "H3Mg7I8ElNo.mp4", "idx": 464, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9391577243804932, "peak_mem_gb": 9.13941240310669, "n_input_tokens": 1059, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 1.0}, {"method": "full", "video": "RYQlChSooWM.mp4", "idx": 466, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.960892915725708, "peak_mem_gb": 11.771689891815186, "n_input_tokens": 8887, "n_video_tokens": 7344, "n_audio_tokens": 1289, "retained_av_ratio": 1.0}, {"method": "full", "video": "b0x6z9gDzHQ.mp4", "idx": 468, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.835499048233032, "peak_mem_gb": 11.47216272354126, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 1.0}, {"method": "full", "video": "KwPGlGTOBkU.mp4", "idx": 470, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3846616744995117, "peak_mem_gb": 9.342426776885986, "n_input_tokens": 1668, "n_video_tokens": 1152, "n_audio_tokens": 203, "retained_av_ratio": 1.0}, {"method": "full", "video": "6X1IEKFL33Y.mp4", "idx": 472, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.566160678863525, "peak_mem_gb": 11.747118949890137, "n_input_tokens": 8816, "n_video_tokens": 7344, "n_audio_tokens": 1278, "retained_av_ratio": 1.0}, {"method": "full", "video": "YWhL_jNOzVc.mp4", "idx": 474, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3645405769348145, "peak_mem_gb": 9.309002876281738, "n_input_tokens": 1565, "n_video_tokens": 1152, "n_audio_tokens": 217, "retained_av_ratio": 1.0}, {"method": "full", "video": "OY-uPQ4oLKM.mp4", "idx": 476, "problem_type": "Narrative Comprehension", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.20 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.41235065460205, "n_input_tokens": 11296, "n_video_tokens": 9216, "n_audio_tokens": 1866, "retained_av_ratio": null}, {"method": "full", "video": "3Y2mE1Oph6A.mp4", "idx": 478, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.098223686218262, "peak_mem_gb": 11.009311199188232, "n_input_tokens": 6620, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 1.0}, {"method": "full", "video": "ZbsjWDzPPx4.mp4", "idx": 480, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0225191116333008, "peak_mem_gb": 9.149183750152588, "n_input_tokens": 1090, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 1.0}, {"method": "full", "video": "jtFI4H55Clw.mp4", "idx": 482, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.288902521133423, "peak_mem_gb": 10.403903007507324, "n_input_tokens": 4793, "n_video_tokens": 3888, "n_audio_tokens": 693, "retained_av_ratio": 1.0}, {"method": "full", "video": "QD1l3iZ2nZs.mp4", "idx": 484, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.245067119598389, "peak_mem_gb": 11.85904312133789, "n_input_tokens": 9145, "n_video_tokens": 7632, "n_audio_tokens": 1337, "retained_av_ratio": 1.0}, {"method": "full", "video": "qPibP0Y6HUU.mp4", "idx": 486, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.85833191871643, "peak_mem_gb": 12.3126220703125, "n_input_tokens": 10494, "n_video_tokens": 8784, "n_audio_tokens": 1527, "retained_av_ratio": 1.0}, {"method": "full", "video": "AOkiHqncwF8.mp4", "idx": 488, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.86700439453125, "peak_mem_gb": 10.297929286956787, "n_input_tokens": 4475, "n_video_tokens": 3600, "n_audio_tokens": 651, "retained_av_ratio": 1.0}, {"method": "full", "video": "FPBSzdA1oKU.mp4", "idx": 490, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1172971725463867, "peak_mem_gb": 9.255331039428711, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 180, "retained_av_ratio": 1.0}, {"method": "full", "video": "6RNtg0cLgZk.mp4", "idx": 492, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.858023405075073, "peak_mem_gb": 12.495348930358887, "n_input_tokens": 11041, "n_video_tokens": 9216, "n_audio_tokens": 1616, "retained_av_ratio": 1.0}, {"method": "full", "video": "0avoXj7I3yA.mp4", "idx": 494, "problem_type": "Narrative Comprehension", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.72 GiB. GPU 0 has a total capacity of 14.56 GiB of which 68.81 MiB is free. Including non-PyTorch memo", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.970733642578125, "n_input_tokens": 12512, "n_video_tokens": 9216, "n_audio_tokens": 3020, "retained_av_ratio": null}, {"method": "full", "video": "DcyrFhN6ncE.mp4", "idx": 496, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.940730810165405, "peak_mem_gb": 11.772351264953613, "n_input_tokens": 8889, "n_video_tokens": 7344, "n_audio_tokens": 1281, "retained_av_ratio": 1.0}, {"method": "full", "video": "HAh-h5nuzhI.mp4", "idx": 498, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.151565313339233, "peak_mem_gb": 10.357276439666748, "n_input_tokens": 4655, "n_video_tokens": 3744, "n_audio_tokens": 670, "retained_av_ratio": 1.0}, {"method": "full", "video": "Hre27BhZvM8.mp4", "idx": 1, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.382569551467896, "peak_mem_gb": 11.793583869934082, "n_input_tokens": 8959, "n_video_tokens": 7344, "n_audio_tokens": 1299, "retained_av_ratio": 1.0}, {"method": "full", "video": "f7YdwVUy0ss.mp4", "idx": 3, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2267773151397705, "peak_mem_gb": 9.258447170257568, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 1.0}, {"method": "full", "video": "EgHDY0VpFS0.mp4", "idx": 5, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.3315110206604004, "peak_mem_gb": 10.143819808959961, "n_input_tokens": 4037, "n_video_tokens": 3168, "n_audio_tokens": 565, "retained_av_ratio": 1.0}, {"method": "full", "video": "egcR6-Dq0Tc.mp4", "idx": 7, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.385568857192993, "peak_mem_gb": 12.264804363250732, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "Bk60US4MacA.mp4", "idx": 9, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.838470935821533, "peak_mem_gb": 12.110877990722656, "n_input_tokens": 9895, "n_video_tokens": 8208, "n_audio_tokens": 1427, "retained_av_ratio": 1.0}, {"method": "full", "video": "ft_pDeYZA6E.mp4", "idx": 11, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7343354225158691, "peak_mem_gb": 9.491934776306152, "n_input_tokens": 2084, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 1.0}, {"method": "full", "video": "TJR8ibF26vw.mp4", "idx": 13, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.266627073287964, "peak_mem_gb": 10.427783489227295, "n_input_tokens": 4872, "n_video_tokens": 3920, "n_audio_tokens": 723, "retained_av_ratio": 1.0}, {"method": "full", "video": "Qps4jZVasg4.mp4", "idx": 15, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4209845066070557, "peak_mem_gb": 9.322324752807617, "n_input_tokens": 1605, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 1.0}, {"method": "full", "video": "lUEjk9CVcEY.mp4", "idx": 17, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.296400785446167, "peak_mem_gb": 9.730703830718994, "n_input_tokens": 2740, "n_video_tokens": 2160, "n_audio_tokens": 385, "retained_av_ratio": 1.0}, {"method": "full", "video": "cdOd3EXqagY.mp4", "idx": 19, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.750582218170166, "peak_mem_gb": 12.03483533859253, "n_input_tokens": 9664, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 1.0}, {"method": "full", "video": "Z6U4PbCGiP0.mp4", "idx": 21, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.761204242706299, "peak_mem_gb": 10.244766235351562, "n_input_tokens": 4316, "n_video_tokens": 3456, "n_audio_tokens": 624, "retained_av_ratio": 1.0}, {"method": "full", "video": "NaDxBMFiiNQ.mp4", "idx": 23, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3056120872497559, "peak_mem_gb": 9.310951709747314, "n_input_tokens": 1570, "n_video_tokens": 1152, "n_audio_tokens": 211, "retained_av_ratio": 1.0}, {"method": "full", "video": "UxlWLoM3H-8.mp4", "idx": 25, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.45523190498352, "peak_mem_gb": 12.150622367858887, "n_input_tokens": 10014, "n_video_tokens": 8352, "n_audio_tokens": 1474, "retained_av_ratio": 1.0}, {"method": "full", "video": "_mYtKUIalEo.mp4", "idx": 27, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.8792202472686768, "peak_mem_gb": 9.901248931884766, "n_input_tokens": 3275, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 1.0}, {"method": "full", "video": "P9jqFr3TvSE.mp4", "idx": 29, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.23622441291809, "peak_mem_gb": 11.721954345703125, "n_input_tokens": 8742, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 1.0}, {"method": "full", "video": "X1c3Z8eAhwI.mp4", "idx": 31, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.148024559020996, "peak_mem_gb": 10.959630012512207, "n_input_tokens": 6473, "n_video_tokens": 5328, "n_audio_tokens": 946, "retained_av_ratio": 1.0}, {"method": "full", "video": "O7EgKJ5Km7Y.mp4", "idx": 33, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 11.118978500366211, "peak_mem_gb": 12.266751289367676, "n_input_tokens": 10361, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "xlO7zTTgtHg.mp4", "idx": 35, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2568683624267578, "peak_mem_gb": 9.263609886169434, "n_input_tokens": 1428, "n_video_tokens": 1008, "n_audio_tokens": 194, "retained_av_ratio": 1.0}, {"method": "full", "video": "52smiyrFybw.mp4", "idx": 37, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8078598976135254, "peak_mem_gb": 9.076679706573486, "n_input_tokens": 874, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 1.0}, {"method": "full", "video": "_PzDR-Nf-e8.mp4", "idx": 39, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1888091564178467, "peak_mem_gb": 9.261055946350098, "n_input_tokens": 1420, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 1.0}, {"method": "full", "video": "aiVuez4WsBo.mp4", "idx": 41, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9211833477020264, "peak_mem_gb": 9.540862083435059, "n_input_tokens": 2220, "n_video_tokens": 1680, "n_audio_tokens": 324, "retained_av_ratio": 1.0}, {"method": "full", "video": "3o2KqGljhbQ.mp4", "idx": 43, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.646387338638306, "peak_mem_gb": 12.171270847320557, "n_input_tokens": 10078, "n_video_tokens": 8352, "n_audio_tokens": 1458, "retained_av_ratio": 1.0}, {"method": "full", "video": "TIxlSCV6E_A.mp4", "idx": 45, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8884327411651611, "peak_mem_gb": 9.085842609405518, "n_input_tokens": 902, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 1.0}, {"method": "full", "video": "2dnuTGnFZyo.mp4", "idx": 47, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1532676219940186, "peak_mem_gb": 9.19890546798706, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 174, "retained_av_ratio": 1.0}, {"method": "full", "video": "9ngLXMGF90I.mp4", "idx": 49, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.135866165161133, "peak_mem_gb": 9.951037883758545, "n_input_tokens": 3426, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 1.0}, {"method": "full", "video": "BPpr-5KW5BQ.mp4", "idx": 51, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.88009762763977, "peak_mem_gb": 12.210165023803711, "n_input_tokens": 10188, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 1.0}, {"method": "full", "video": "m_haCHnvhj4.mp4", "idx": 53, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0185215473175049, "peak_mem_gb": 9.156261444091797, "n_input_tokens": 1114, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "xrOu-TB6osc.mp4", "idx": 55, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9286966323852539, "peak_mem_gb": 9.114141464233398, "n_input_tokens": 989, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 1.0}, {"method": "full", "video": "9IGdrQU77pk.mp4", "idx": 57, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5709805488586426, "peak_mem_gb": 9.380497932434082, "n_input_tokens": 1781, "n_video_tokens": 1296, "n_audio_tokens": 251, "retained_av_ratio": 1.0}, {"method": "full", "video": "wPWguBt2hhs.mp4", "idx": 59, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9139318466186523, "peak_mem_gb": 9.133426189422607, "n_input_tokens": 1044, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 1.0}, {"method": "full", "video": "7moy-Foct8U.mp4", "idx": 61, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.503977298736572, "peak_mem_gb": 10.735095024108887, "n_input_tokens": 5801, "n_video_tokens": 4752, "n_audio_tokens": 845, "retained_av_ratio": 1.0}, {"method": "full", "video": "GSLBOuwOdRE.mp4", "idx": 63, "problem_type": "Narrative Comprehension", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.22 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.63064956665039, "n_input_tokens": 11393, "n_video_tokens": 9216, "n_audio_tokens": 1948, "retained_av_ratio": null}, {"method": "full", "video": "LXXmslrnPrg.mp4", "idx": 65, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8887829780578613, "peak_mem_gb": 9.1357741355896, "n_input_tokens": 1051, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 1.0}, {"method": "full", "video": "oPhbjcl7TOg.mp4", "idx": 67, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.770766496658325, "peak_mem_gb": 9.855690479278564, "n_input_tokens": 3132, "n_video_tokens": 2448, "n_audio_tokens": 451, "retained_av_ratio": 1.0}, {"method": "full", "video": "iTdOOLn-ICM.mp4", "idx": 69, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.2173104286193848, "peak_mem_gb": 9.692691802978516, "n_input_tokens": 2611, "n_video_tokens": 2016, "n_audio_tokens": 369, "retained_av_ratio": 1.0}, {"method": "full", "video": "CpeBZFxOeqQ.mp4", "idx": 71, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6642019748687744, "peak_mem_gb": 9.40907335281372, "n_input_tokens": 1862, "n_video_tokens": 1400, "n_audio_tokens": 273, "retained_av_ratio": 1.0}, {"method": "full", "video": "B7g9hmHAUzo.mp4", "idx": 73, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.169397354125977, "peak_mem_gb": 11.247618675231934, "n_input_tokens": 7326, "n_video_tokens": 6048, "n_audio_tokens": 1059, "retained_av_ratio": 1.0}, {"method": "full", "video": "tUucQrJAzsI.mp4", "idx": 75, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6304006576538086, "peak_mem_gb": 9.831465721130371, "n_input_tokens": 3063, "n_video_tokens": 2448, "n_audio_tokens": 443, "retained_av_ratio": 1.0}, {"method": "full", "video": "TA6OBcjG1Oc.mp4", "idx": 77, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.259761095046997, "peak_mem_gb": 12.250868320465088, "n_input_tokens": 10310, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 1.0}, {"method": "full", "video": "CdwdC6E8Cco.mp4", "idx": 79, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6779139041900635, "peak_mem_gb": 9.785884380340576, "n_input_tokens": 2910, "n_video_tokens": 2304, "n_audio_tokens": 409, "retained_av_ratio": 1.0}, {"method": "full", "video": "d-wkkuvKoGM.mp4", "idx": 81, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.837336778640747, "peak_mem_gb": 10.229435920715332, "n_input_tokens": 4268, "n_video_tokens": 3456, "n_audio_tokens": 609, "retained_av_ratio": 1.0}, {"method": "full", "video": "i_mCozcHi9o.mp4", "idx": 83, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.412790298461914, "peak_mem_gb": 9.322049140930176, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 1.0}, {"method": "full", "video": "g_RmJp0IgkY.mp4", "idx": 85, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0319161415100098, "peak_mem_gb": 9.15886116027832, "n_input_tokens": 1122, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 1.0}, {"method": "full", "video": "GOg9dmnHSSw.mp4", "idx": 87, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.899101972579956, "peak_mem_gb": 11.97935438156128, "n_input_tokens": 9500, "n_video_tokens": 7920, "n_audio_tokens": 1393, "retained_av_ratio": 1.0}, {"method": "full", "video": "-DxPwypTfKI.mp4", "idx": 89, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.713833332061768, "peak_mem_gb": 12.17031192779541, "n_input_tokens": 10075, "n_video_tokens": 8352, "n_audio_tokens": 1461, "retained_av_ratio": 1.0}, {"method": "full", "video": "l7QiQSivnAc.mp4", "idx": 91, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.1021769046783447, "peak_mem_gb": 9.933950424194336, "n_input_tokens": 3373, "n_video_tokens": 2736, "n_audio_tokens": 499, "retained_av_ratio": 1.0}, {"method": "full", "video": "T-PGGc1OJcM.mp4", "idx": 93, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2088336944580078, "peak_mem_gb": 9.195013523101807, "n_input_tokens": 1224, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 1.0}, {"method": "full", "video": "k4o3XkT0jbM.mp4", "idx": 95, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.617281198501587, "peak_mem_gb": 11.63711166381836, "n_input_tokens": 8485, "n_video_tokens": 7056, "n_audio_tokens": 1227, "retained_av_ratio": 1.0}, {"method": "full", "video": "baTfh9Ryxoo.mp4", "idx": 97, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.635489463806152, "peak_mem_gb": 10.508819580078125, "n_input_tokens": 5113, "n_video_tokens": 4176, "n_audio_tokens": 749, "retained_av_ratio": 1.0}, {"method": "full", "video": "CcZr-QjAvTQ.mp4", "idx": 99, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0201244354248047, "peak_mem_gb": 9.204943180084229, "n_input_tokens": 1255, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 1.0}, {"method": "full", "video": "-0J1uWKJKfk.mp4", "idx": 101, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9003922939300537, "peak_mem_gb": 9.166626453399658, "n_input_tokens": 1146, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 1.0}, {"method": "full", "video": "4WgA39tgBSo.mp4", "idx": 103, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.14797043800354, "peak_mem_gb": 10.004981994628906, "n_input_tokens": 3596, "n_video_tokens": 2880, "n_audio_tokens": 504, "retained_av_ratio": 1.0}, {"method": "full", "video": "QUC2d9FSkcU.mp4", "idx": 105, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.590672492980957, "peak_mem_gb": 9.79155445098877, "n_input_tokens": 2930, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 1.0}, {"method": "full", "video": "uGoKOpZOSUg.mp4", "idx": 107, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 6.04 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.498016357421875, "n_input_tokens": 14039, "n_video_tokens": 9216, "n_audio_tokens": 4501, "retained_av_ratio": null}, {"method": "full", "video": "MY7puORABAk.mp4", "idx": 109, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.650036334991455, "peak_mem_gb": 9.838226795196533, "n_input_tokens": 3079, "n_video_tokens": 2448, "n_audio_tokens": 436, "retained_av_ratio": 1.0}, {"method": "full", "video": "yl79pxDOjJ0.mp4", "idx": 111, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.514650106430054, "peak_mem_gb": 10.748486042022705, "n_input_tokens": 5843, "n_video_tokens": 4752, "n_audio_tokens": 831, "retained_av_ratio": 1.0}, {"method": "full", "video": "g8JpgtFpciY.mp4", "idx": 113, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.544311761856079, "peak_mem_gb": 11.365743160247803, "n_input_tokens": 7682, "n_video_tokens": 6336, "n_audio_tokens": 1120, "retained_av_ratio": 1.0}, {"method": "full", "video": "1Q3LeFo88xI.mp4", "idx": 115, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.277939319610596, "peak_mem_gb": 12.28494119644165, "n_input_tokens": 10417, "n_video_tokens": 8640, "n_audio_tokens": 1506, "retained_av_ratio": 1.0}, {"method": "full", "video": "HdwFA-Hb0pQ.mp4", "idx": 117, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.029571533203125, "peak_mem_gb": 9.147136688232422, "n_input_tokens": 1086, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 1.0}, {"method": "full", "video": "GFrTD7uV8mA.mp4", "idx": 119, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0740363597869873, "peak_mem_gb": 9.140663623809814, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 1.0}, {"method": "full", "video": "Dawm3vWn97A.mp4", "idx": 121, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6818585395812988, "peak_mem_gb": 9.44122838973999, "n_input_tokens": 1962, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 1.0}, {"method": "full", "video": "BQc182PZbeU.mp4", "idx": 123, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.099403619766235, "peak_mem_gb": 11.257997512817383, "n_input_tokens": 7363, "n_video_tokens": 6048, "n_audio_tokens": 1069, "retained_av_ratio": 1.0}, {"method": "full", "video": "tjXVd3qBMhw.mp4", "idx": 125, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1155800819396973, "peak_mem_gb": 9.185678958892822, "n_input_tokens": 1195, "n_video_tokens": 864, "n_audio_tokens": 159, "retained_av_ratio": 1.0}, {"method": "full", "video": "YGsvvmBiRHA.mp4", "idx": 127, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3782401084899902, "peak_mem_gb": 9.743360996246338, "n_input_tokens": 2781, "n_video_tokens": 2160, "n_audio_tokens": 386, "retained_av_ratio": 1.0}, {"method": "full", "video": "5yssh2Aqtps.mp4", "idx": 129, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.919168949127197, "peak_mem_gb": 12.196080684661865, "n_input_tokens": 10146, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 1.0}, {"method": "full", "video": "auFMvAywvbk.mp4", "idx": 131, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4470295906066895, "peak_mem_gb": 9.736499786376953, "n_input_tokens": 2757, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 1.0}, {"method": "full", "video": "qTX0XvwA0qU.mp4", "idx": 133, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1487088203430176, "peak_mem_gb": 9.192445755004883, "n_input_tokens": 1216, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "X2ZDXPulIy0.mp4", "idx": 135, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0357434749603271, "peak_mem_gb": 9.146425724029541, "n_input_tokens": 1084, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 1.0}, {"method": "full", "video": "qHakVESEOec.mp4", "idx": 137, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.493659496307373, "peak_mem_gb": 9.316522598266602, "n_input_tokens": 1587, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 1.0}, {"method": "full", "video": "StG5yDCruTw.mp4", "idx": 139, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.760864734649658, "peak_mem_gb": 10.191760540008545, "n_input_tokens": 4150, "n_video_tokens": 3312, "n_audio_tokens": 586, "retained_av_ratio": 1.0}, {"method": "full", "video": "6kpIdgWGRAg.mp4", "idx": 141, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.027153491973877, "peak_mem_gb": 9.140648365020752, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 1.0}, {"method": "full", "video": "Z0YlK_pGTMQ.mp4", "idx": 143, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.934647798538208, "peak_mem_gb": 9.1344895362854, "n_input_tokens": 1047, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 1.0}, {"method": "full", "video": "ZJyYchnUJL4.mp4", "idx": 145, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8814668655395508, "peak_mem_gb": 9.149425506591797, "n_input_tokens": 1093, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 1.0}, {"method": "full", "video": "7uxn4eM979g.mp4", "idx": 147, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9298720359802246, "peak_mem_gb": 9.153944492340088, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 138, "retained_av_ratio": 1.0}, {"method": "full", "video": "GO1FuCX8YHI.mp4", "idx": 149, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.36121940612793, "peak_mem_gb": 11.867830753326416, "n_input_tokens": 9176, "n_video_tokens": 7632, "n_audio_tokens": 1335, "retained_av_ratio": 1.0}, {"method": "full", "video": "tQCM5mtqkeo.mp4", "idx": 151, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.565980672836304, "peak_mem_gb": 12.088313102722168, "n_input_tokens": 9825, "n_video_tokens": 8208, "n_audio_tokens": 1445, "retained_av_ratio": 1.0}, {"method": "full", "video": "OYIX_zxxS9Q.mp4", "idx": 153, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.008307218551636, "peak_mem_gb": 10.29182481765747, "n_input_tokens": 4457, "n_video_tokens": 3600, "n_audio_tokens": 641, "retained_av_ratio": 1.0}, {"method": "full", "video": "GZ_pUlsnTGQ.mp4", "idx": 155, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0747158527374268, "peak_mem_gb": 9.18603801727295, "n_input_tokens": 1196, "n_video_tokens": 864, "n_audio_tokens": 169, "retained_av_ratio": 1.0}, {"method": "full", "video": "VyHYgZamj58.mp4", "idx": 157, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7649879455566406, "peak_mem_gb": 9.086840629577637, "n_input_tokens": 905, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 1.0}, {"method": "full", "video": "CQHAoYgQLis.mp4", "idx": 159, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.113598585128784, "peak_mem_gb": 10.339583396911621, "n_input_tokens": 4601, "n_video_tokens": 3744, "n_audio_tokens": 659, "retained_av_ratio": 1.0}, {"method": "full", "video": "ySiKjxAnE4I.mp4", "idx": 161, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.151625633239746, "peak_mem_gb": 12.050044536590576, "n_input_tokens": 9711, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 1.0}, {"method": "full", "video": "9xWVEqxUJGY.mp4", "idx": 163, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.091874599456787, "peak_mem_gb": 9.194050312042236, "n_input_tokens": 1221, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 1.0}, {"method": "full", "video": "egkxfFQ_C7g.mp4", "idx": 165, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.783249855041504, "peak_mem_gb": 9.440263748168945, "n_input_tokens": 1957, "n_video_tokens": 1440, "n_audio_tokens": 260, "retained_av_ratio": 1.0}, {"method": "full", "video": "U-R7VXme0ps.mp4", "idx": 167, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7088274955749512, "peak_mem_gb": 9.433738708496094, "n_input_tokens": 1939, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 1.0}, {"method": "full", "video": "OCRdFBuuW2M.mp4", "idx": 169, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.082283973693848, "peak_mem_gb": 10.652827262878418, "n_input_tokens": 5554, "n_video_tokens": 4464, "n_audio_tokens": 791, "retained_av_ratio": 1.0}, {"method": "full", "video": "5yUY-afqZSs.mp4", "idx": 171, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.800379753112793, "peak_mem_gb": 12.200762748718262, "n_input_tokens": 10161, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 1.0}, {"method": "full", "video": "T0p4FtF99l0.mp4", "idx": 173, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.984644889831543, "peak_mem_gb": 9.153963565826416, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 1.0}, {"method": "full", "video": "2tVZFAqM7lw.mp4", "idx": 175, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.0434372425079346, "peak_mem_gb": 9.992795467376709, "n_input_tokens": 3555, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 1.0}, {"method": "full", "video": "uBXycVLybjY.mp4", "idx": 177, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.527203798294067, "peak_mem_gb": 12.082382678985596, "n_input_tokens": 9806, "n_video_tokens": 8208, "n_audio_tokens": 1442, "retained_av_ratio": 1.0}, {"method": "full", "video": "JiRQff5ADps.mp4", "idx": 179, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.309358835220337, "peak_mem_gb": 10.689746379852295, "n_input_tokens": 5661, "n_video_tokens": 4620, "n_audio_tokens": 845, "retained_av_ratio": 1.0}, {"method": "full", "video": "tNn-m_qLsvo.mp4", "idx": 181, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3950092792510986, "peak_mem_gb": 9.320089340209961, "n_input_tokens": 1598, "n_video_tokens": 1152, "n_audio_tokens": 221, "retained_av_ratio": 1.0}, {"method": "full", "video": "0Z3NeEoE4cc.mp4", "idx": 183, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.4828178882598877, "peak_mem_gb": 10.087646007537842, "n_input_tokens": 3855, "n_video_tokens": 3080, "n_audio_tokens": 569, "retained_av_ratio": 1.0}, {"method": "full", "video": "K8S_8sdPRDI.mp4", "idx": 185, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4300529956817627, "peak_mem_gb": 9.772928237915039, "n_input_tokens": 2871, "n_video_tokens": 2240, "n_audio_tokens": 414, "retained_av_ratio": 1.0}, {"method": "full", "video": "dxSsSfynCUQ.mp4", "idx": 187, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.446425437927246, "peak_mem_gb": 11.358879566192627, "n_input_tokens": 7661, "n_video_tokens": 6336, "n_audio_tokens": 1104, "retained_av_ratio": 1.0}, {"method": "full", "video": "PW-6MC05h3A.mp4", "idx": 189, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.067044019699097, "peak_mem_gb": 11.723413944244385, "n_input_tokens": 8750, "n_video_tokens": 7128, "n_audio_tokens": 1364, "retained_av_ratio": 1.0}, {"method": "full", "video": "O61cCbK3hMY.mp4", "idx": 191, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.88 GiB. GPU 0 has a total capacity of 14.56 GiB of which 248.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 12.141539096832275, "n_input_tokens": 12527, "n_video_tokens": 9216, "n_audio_tokens": 3107, "retained_av_ratio": null}, {"method": "full", "video": "ypwjQw-1Ltg.mp4", "idx": 193, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.268225908279419, "peak_mem_gb": 11.360935688018799, "n_input_tokens": 7666, "n_video_tokens": 6336, "n_audio_tokens": 1108, "retained_av_ratio": 1.0}, {"method": "full", "video": "PKoZQwUxvnQ.mp4", "idx": 195, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3566455841064453, "peak_mem_gb": 9.728247165679932, "n_input_tokens": 2735, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 1.0}, {"method": "full", "video": "yM0pV0G3-gc.mp4", "idx": 197, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.290281534194946, "peak_mem_gb": 11.32942533493042, "n_input_tokens": 7578, "n_video_tokens": 6192, "n_audio_tokens": 1095, "retained_av_ratio": 1.0}, {"method": "full", "video": "rTcusJ3Nfqc.mp4", "idx": 199, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.45670223236084, "peak_mem_gb": 12.156249523162842, "n_input_tokens": 10028, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": 1.0}, {"method": "full", "video": "mXZjcxVwrLY.mp4", "idx": 201, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2116103172302246, "peak_mem_gb": 9.230978965759277, "n_input_tokens": 1336, "n_video_tokens": 864, "n_audio_tokens": 162, "retained_av_ratio": 1.0}, {"method": "full", "video": "oQVdUilguhc.mp4", "idx": 203, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.9772512912750244, "peak_mem_gb": 9.991386413574219, "n_input_tokens": 3556, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 1.0}, {"method": "full", "video": "E9NW_SJTrco.mp4", "idx": 205, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7484912872314453, "peak_mem_gb": 9.089425086975098, "n_input_tokens": 913, "n_video_tokens": 576, "n_audio_tokens": 120, "retained_av_ratio": 1.0}, {"method": "full", "video": "iYrFMa1TBQQ.mp4", "idx": 207, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.629311084747314, "peak_mem_gb": 11.694536685943604, "n_input_tokens": 8657, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 1.0}, {"method": "full", "video": "Sin_6xmILaE.mp4", "idx": 209, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5062167644500732, "peak_mem_gb": 9.375890731811523, "n_input_tokens": 1767, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 1.0}, {"method": "full", "video": "w7K419G-nJM.mp4", "idx": 211, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7956175804138184, "peak_mem_gb": 9.496110916137695, "n_input_tokens": 2102, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 1.0}, {"method": "full", "video": "pOMKJXCtap0.mp4", "idx": 213, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.846108198165894, "peak_mem_gb": 12.200450420379639, "n_input_tokens": 10160, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 1.0}, {"method": "full", "video": "HWpq9mZ5Sp4.mp4", "idx": 215, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0476717948913574, "peak_mem_gb": 9.193728923797607, "n_input_tokens": 1220, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 1.0}, {"method": "full", "video": "NOpfz1iawe0.mp4", "idx": 217, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7625470161437988, "peak_mem_gb": 9.031304836273193, "n_input_tokens": 740, "n_video_tokens": 432, "n_audio_tokens": 94, "retained_av_ratio": 1.0}, {"method": "full", "video": "HiujBqqlyvE.mp4", "idx": 219, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.038407802581787, "peak_mem_gb": 9.137089252471924, "n_input_tokens": 1055, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 1.0}, {"method": "full", "video": "b8RVswCLGnM.mp4", "idx": 221, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.846708297729492, "peak_mem_gb": 12.218570232391357, "n_input_tokens": 10216, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 1.0}, {"method": "full", "video": "UMdQ1VnfT54.mp4", "idx": 223, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.963216304779053, "peak_mem_gb": 11.984750270843506, "n_input_tokens": 9518, "n_video_tokens": 7920, "n_audio_tokens": 1385, "retained_av_ratio": 1.0}, {"method": "full", "video": "xcxsvhrlfc0.mp4", "idx": 225, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.813277006149292, "peak_mem_gb": 9.486788749694824, "n_input_tokens": 2066, "n_video_tokens": 1584, "n_audio_tokens": 298, "retained_av_ratio": 1.0}, {"method": "full", "video": "sAgv2JvK0iA.mp4", "idx": 227, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.288386821746826, "peak_mem_gb": 11.035873889923096, "n_input_tokens": 6703, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 1.0}, {"method": "full", "video": "u45J1YdCWvo.mp4", "idx": 229, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9301881790161133, "peak_mem_gb": 9.96367359161377, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 495, "retained_av_ratio": 1.0}, {"method": "full", "video": "VMltK9d0ewk.mp4", "idx": 231, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.478378057479858, "peak_mem_gb": 11.85973072052002, "n_input_tokens": 9151, "n_video_tokens": 7632, "n_audio_tokens": 1342, "retained_av_ratio": 1.0}, {"method": "full", "video": "bt1qugf_ej8.mp4", "idx": 233, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.63351058959961, "peak_mem_gb": 12.100847721099854, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1437, "retained_av_ratio": 1.0}, {"method": "full", "video": "eVp2MEAkC44.mp4", "idx": 235, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.8289225101470947, "peak_mem_gb": 10.266963005065918, "n_input_tokens": 4383, "n_video_tokens": 3456, "n_audio_tokens": 610, "retained_av_ratio": 1.0}, {"method": "full", "video": "3j9Eq0Ky7f8.mp4", "idx": 237, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6674609184265137, "peak_mem_gb": 9.43016004562378, "n_input_tokens": 1928, "n_video_tokens": 1440, "n_audio_tokens": 270, "retained_av_ratio": 1.0}, {"method": "full", "video": "yr8b945WDdY.mp4", "idx": 239, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.58842134475708, "peak_mem_gb": 12.157748222351074, "n_input_tokens": 10036, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 1.0}, {"method": "full", "video": "4wJs2hhedj0.mp4", "idx": 241, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2431097030639648, "peak_mem_gb": 9.251508235931396, "n_input_tokens": 1394, "n_video_tokens": 980, "n_audio_tokens": 200, "retained_av_ratio": 1.0}, {"method": "full", "video": "KABpAbGs6bQ.mp4", "idx": 243, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.464132785797119, "peak_mem_gb": 9.726462841033936, "n_input_tokens": 2729, "n_video_tokens": 2160, "n_audio_tokens": 382, "retained_av_ratio": 1.0}, {"method": "full", "video": "Bn2HxjKRi7s.mp4", "idx": 245, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.7371461391448975, "peak_mem_gb": 11.432705402374268, "n_input_tokens": 7882, "n_video_tokens": 6480, "n_audio_tokens": 1148, "retained_av_ratio": 1.0}, {"method": "full", "video": "E1FfWv-iIJM.mp4", "idx": 247, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.339426040649414, "peak_mem_gb": 11.818786144256592, "n_input_tokens": 9031, "n_video_tokens": 7488, "n_audio_tokens": 1307, "retained_av_ratio": 1.0}, {"method": "full", "video": "a7YYrZ3mdsY.mp4", "idx": 249, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9072978496551514, "peak_mem_gb": 9.537042617797852, "n_input_tokens": 2206, "n_video_tokens": 1728, "n_audio_tokens": 313, "retained_av_ratio": 1.0}, {"method": "full", "video": "4R3s-co2_lU.mp4", "idx": 251, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.004253149032593, "peak_mem_gb": 12.226788520812988, "n_input_tokens": 10242, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 1.0}, {"method": "full", "video": "e-3Km4K2G2I.mp4", "idx": 253, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.715925931930542, "peak_mem_gb": 10.515622615814209, "n_input_tokens": 5132, "n_video_tokens": 4176, "n_audio_tokens": 741, "retained_av_ratio": 1.0}, {"method": "full", "video": "Bc3vnp7VHVM.mp4", "idx": 255, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.587040901184082, "peak_mem_gb": 9.7795090675354, "n_input_tokens": 2891, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 1.0}, {"method": "full", "video": "dNqYbetriJ4.mp4", "idx": 257, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.141465663909912, "peak_mem_gb": 12.26485013961792, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1514, "retained_av_ratio": 1.0}, {"method": "full", "video": "-eUTGNm9ZFY.mp4", "idx": 259, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6309499740600586, "peak_mem_gb": 9.371688842773438, "n_input_tokens": 1754, "n_video_tokens": 1296, "n_audio_tokens": 242, "retained_av_ratio": 1.0}, {"method": "full", "video": "68_42cTzS80.mp4", "idx": 261, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9190244674682617, "peak_mem_gb": 9.542760372161865, "n_input_tokens": 2226, "n_video_tokens": 1728, "n_audio_tokens": 305, "retained_av_ratio": 1.0}, {"method": "full", "video": "OdrWu71KIdI.mp4", "idx": 263, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7594504356384277, "peak_mem_gb": 9.441871166229248, "n_input_tokens": 1964, "n_video_tokens": 1440, "n_audio_tokens": 273, "retained_av_ratio": 1.0}, {"method": "full", "video": "Q4ftOkPtOB0.mp4", "idx": 265, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.0079004764556885, "peak_mem_gb": 10.646863460540771, "n_input_tokens": 5536, "n_video_tokens": 4480, "n_audio_tokens": 816, "retained_av_ratio": 1.0}, {"method": "full", "video": "l0Ilew3e9CI.mp4", "idx": 267, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1556894779205322, "peak_mem_gb": 9.203960418701172, "n_input_tokens": 1252, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 1.0}, {"method": "full", "video": "IgGPXPR1J1A.mp4", "idx": 269, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2441065311431885, "peak_mem_gb": 9.25277853012085, "n_input_tokens": 1394, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 1.0}, {"method": "full", "video": "jNQep4ocOGQ.mp4", "idx": 271, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2628369331359863, "peak_mem_gb": 9.250809669494629, "n_input_tokens": 1388, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 1.0}, {"method": "full", "video": "qB66a2x8KXE.mp4", "idx": 273, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3171141147613525, "peak_mem_gb": 9.691875457763672, "n_input_tokens": 2608, "n_video_tokens": 2016, "n_audio_tokens": 373, "retained_av_ratio": 1.0}, {"method": "full", "video": "vVlQm8-AHt4.mp4", "idx": 275, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9347712993621826, "peak_mem_gb": 9.096914768218994, "n_input_tokens": 936, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 1.0}, {"method": "full", "video": "XOUUsvMm1PM.mp4", "idx": 277, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.0374319553375244, "peak_mem_gb": 9.957705020904541, "n_input_tokens": 3448, "n_video_tokens": 2736, "n_audio_tokens": 489, "retained_av_ratio": 1.0}, {"method": "full", "video": "_Nqw7iCl4eM.mp4", "idx": 279, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.968464851379395, "peak_mem_gb": 11.992745876312256, "n_input_tokens": 9543, "n_video_tokens": 7920, "n_audio_tokens": 1400, "retained_av_ratio": 1.0}, {"method": "full", "video": "8kMaXa0PYow.mp4", "idx": 281, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2307929992675781, "peak_mem_gb": 9.25842809677124, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 1.0}, {"method": "full", "video": "A3VEekCcv_s.mp4", "idx": 283, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.95610237121582, "peak_mem_gb": 10.624693870544434, "n_input_tokens": 5461, "n_video_tokens": 4464, "n_audio_tokens": 796, "retained_av_ratio": 1.0}, {"method": "full", "video": "Mm6HOkr_HXc.mp4", "idx": 285, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.548782825469971, "peak_mem_gb": 11.134899139404297, "n_input_tokens": 6995, "n_video_tokens": 5760, "n_audio_tokens": 1005, "retained_av_ratio": 1.0}, {"method": "full", "video": "c4x0ezi7rP8.mp4", "idx": 287, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8072586059570312, "peak_mem_gb": 9.079692840576172, "n_input_tokens": 883, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 1.0}, {"method": "full", "video": "mqFCIRSDSxE.mp4", "idx": 289, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.945698738098145, "peak_mem_gb": 11.991466999053955, "n_input_tokens": 9539, "n_video_tokens": 7920, "n_audio_tokens": 1382, "retained_av_ratio": 1.0}, {"method": "full", "video": "zCwz5h0TPmE.mp4", "idx": 291, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6412878036499023, "peak_mem_gb": 9.42463731765747, "n_input_tokens": 1911, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 1.0}, {"method": "full", "video": "SbFqCuO11JU.mp4", "idx": 293, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.00974702835083, "peak_mem_gb": 9.949219703674316, "n_input_tokens": 3420, "n_video_tokens": 2736, "n_audio_tokens": 492, "retained_av_ratio": 1.0}, {"method": "full", "video": "nl7-1jkIIgA.mp4", "idx": 295, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.270502328872681, "peak_mem_gb": 10.418438911437988, "n_input_tokens": 4843, "n_video_tokens": 3888, "n_audio_tokens": 680, "retained_av_ratio": 1.0}, {"method": "full", "video": "gAc9Kp7CHn8.mp4", "idx": 297, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9354948997497559, "peak_mem_gb": 9.500437259674072, "n_input_tokens": 2117, "n_video_tokens": 1584, "n_audio_tokens": 285, "retained_av_ratio": 1.0}, {"method": "full", "video": "0vl66E4x5OI.mp4", "idx": 299, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5069472789764404, "peak_mem_gb": 9.375603675842285, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 1.0}, {"method": "full", "video": "KLeJVuRRk84.mp4", "idx": 301, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9741475582122803, "peak_mem_gb": 9.14908504486084, "n_input_tokens": 1092, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 1.0}, {"method": "full", "video": "Lzccd-ol1Ks.mp4", "idx": 303, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.759654998779297, "peak_mem_gb": 12.222253799438477, "n_input_tokens": 10228, "n_video_tokens": 8496, "n_audio_tokens": 1499, "retained_av_ratio": 1.0}, {"method": "full", "video": "-7dHm4baH38.mp4", "idx": 305, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7518901824951172, "peak_mem_gb": 9.425642490386963, "n_input_tokens": 1914, "n_video_tokens": 1440, "n_audio_tokens": 279, "retained_av_ratio": 1.0}, {"method": "full", "video": "b53-JgOg1i8.mp4", "idx": 307, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4354839324951172, "peak_mem_gb": 9.30418062210083, "n_input_tokens": 1549, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 1.0}, {"method": "full", "video": "TQ1qkHdbyzM.mp4", "idx": 309, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4978561401367188, "peak_mem_gb": 9.314813613891602, "n_input_tokens": 1582, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 1.0}, {"method": "full", "video": "nfqtKk9tob0.mp4", "idx": 311, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.806729316711426, "peak_mem_gb": 12.214367866516113, "n_input_tokens": 10202, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 1.0}, {"method": "full", "video": "Ata-X6fKUas.mp4", "idx": 313, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.1591837406158447, "peak_mem_gb": 10.014625072479248, "n_input_tokens": 3625, "n_video_tokens": 2880, "n_audio_tokens": 513, "retained_av_ratio": 1.0}, {"method": "full", "video": "qdbBYgRMOEk.mp4", "idx": 315, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.597854375839233, "peak_mem_gb": 10.794612407684326, "n_input_tokens": 5982, "n_video_tokens": 4896, "n_audio_tokens": 869, "retained_av_ratio": 1.0}, {"method": "full", "video": "mBMrGwB0v9w.mp4", "idx": 317, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3370189666748047, "peak_mem_gb": 9.310630798339844, "n_input_tokens": 1569, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 1.0}, {"method": "full", "video": "RgygVeaQXkU.mp4", "idx": 319, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.089288234710693, "peak_mem_gb": 10.961466789245605, "n_input_tokens": 6479, "n_video_tokens": 5328, "n_audio_tokens": 934, "retained_av_ratio": 1.0}, {"method": "full", "video": "xQUom0axvAg.mp4", "idx": 321, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6746180057525635, "peak_mem_gb": 9.429158687591553, "n_input_tokens": 1925, "n_video_tokens": 1440, "n_audio_tokens": 263, "retained_av_ratio": 1.0}, {"method": "full", "video": "7WGoDmVMbU4.mp4", "idx": 323, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.520464897155762, "peak_mem_gb": 10.452305316925049, "n_input_tokens": 4945, "n_video_tokens": 4032, "n_audio_tokens": 713, "retained_av_ratio": 1.0}, {"method": "full", "video": "AZS9Vz54oWs.mp4", "idx": 325, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.8671345710754395, "peak_mem_gb": 10.532753467559814, "n_input_tokens": 5184, "n_video_tokens": 4200, "n_audio_tokens": 769, "retained_av_ratio": 1.0}, {"method": "full", "video": "ZUXFwwscme0.mp4", "idx": 327, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6453192234039307, "peak_mem_gb": 9.424002647399902, "n_input_tokens": 1909, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 1.0}, {"method": "full", "video": "y6S8EUrrpP4.mp4", "idx": 329, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0613865852355957, "peak_mem_gb": 9.643821716308594, "n_input_tokens": 2458, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 1.0}, {"method": "full", "video": "B0M-_FeUYVE.mp4", "idx": 331, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.882483720779419, "peak_mem_gb": 9.154558658599854, "n_input_tokens": 1109, "n_video_tokens": 715, "n_audio_tokens": 140, "retained_av_ratio": 1.0}, {"method": "full", "video": "DM8l9e9cEXY.mp4", "idx": 333, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8747999668121338, "peak_mem_gb": 9.140621662139893, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 139, "retained_av_ratio": 1.0}, {"method": "full", "video": "jmrtSmuXI9M.mp4", "idx": 335, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4442164897918701, "peak_mem_gb": 9.321349620819092, "n_input_tokens": 1602, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 1.0}, {"method": "full", "video": "DPnvcwAWV-Y.mp4", "idx": 337, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.650697708129883, "peak_mem_gb": 12.20294713973999, "n_input_tokens": 10168, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": 1.0}, {"method": "full", "video": "jBLELOjSYfY.mp4", "idx": 339, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7857937812805176, "peak_mem_gb": 9.840555191040039, "n_input_tokens": 3087, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 1.0}, {"method": "full", "video": "4kslnZ1gICo.mp4", "idx": 341, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0946636199951172, "peak_mem_gb": 9.19819450378418, "n_input_tokens": 1234, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 1.0}, {"method": "full", "video": "toQnAXHr5uo.mp4", "idx": 343, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.540940999984741, "peak_mem_gb": 10.131836414337158, "n_input_tokens": 3996, "n_video_tokens": 3168, "n_audio_tokens": 561, "retained_av_ratio": 1.0}, {"method": "full", "video": "__aZIudBQjw.mp4", "idx": 345, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.671801090240479, "peak_mem_gb": 11.893212795257568, "n_input_tokens": 9254, "n_video_tokens": 7632, "n_audio_tokens": 1343, "retained_av_ratio": 1.0}, {"method": "full", "video": "wRWKim49T6Y.mp4", "idx": 347, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9618430137634277, "peak_mem_gb": 9.138675689697266, "n_input_tokens": 1060, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 1.0}, {"method": "full", "video": "7lh_sO9EWkE.mp4", "idx": 349, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5627968311309814, "peak_mem_gb": 9.372931480407715, "n_input_tokens": 1758, "n_video_tokens": 1296, "n_audio_tokens": 227, "retained_av_ratio": 1.0}, {"method": "full", "video": "KMngzlwkYCU.mp4", "idx": 351, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7838294506072998, "peak_mem_gb": 9.471372604370117, "n_input_tokens": 2045, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 1.0}, {"method": "full", "video": "_J6U2xhnBX0.mp4", "idx": 353, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.148285627365112, "peak_mem_gb": 12.328513622283936, "n_input_tokens": 10545, "n_video_tokens": 8784, "n_audio_tokens": 1528, "retained_av_ratio": 1.0}, {"method": "full", "video": "8-QqR2J8qxQ.mp4", "idx": 355, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6334521770477295, "peak_mem_gb": 9.373329162597656, "n_input_tokens": 1759, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 1.0}, {"method": "full", "video": "S0ye2uVX2v0.mp4", "idx": 357, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.007070779800415, "peak_mem_gb": 9.581512451171875, "n_input_tokens": 2256, "n_video_tokens": 1728, "n_audio_tokens": 303, "retained_av_ratio": 1.0}, {"method": "full", "video": "N8q8dFk0Ark.mp4", "idx": 359, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3182528018951416, "peak_mem_gb": 9.278328895568848, "n_input_tokens": 1474, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 1.0}, {"method": "full", "video": "Wz3-b0MeWrQ.mp4", "idx": 361, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.340005397796631, "peak_mem_gb": 10.401102066040039, "n_input_tokens": 4787, "n_video_tokens": 3888, "n_audio_tokens": 695, "retained_av_ratio": 1.0}, {"method": "full", "video": "cr6VlmDjePc.mp4", "idx": 363, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.1628127098083496, "peak_mem_gb": 10.014049530029297, "n_input_tokens": 3623, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 1.0}, {"method": "full", "video": "wuHkSR0XJdo.mp4", "idx": 365, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.960998296737671, "peak_mem_gb": 10.298738479614258, "n_input_tokens": 4479, "n_video_tokens": 3600, "n_audio_tokens": 630, "retained_av_ratio": 1.0}, {"method": "full", "video": "bNu5CZVQUNI.mp4", "idx": 367, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.712392568588257, "peak_mem_gb": 12.150953769683838, "n_input_tokens": 10015, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 1.0}, {"method": "full", "video": "gx9mV8QfXso.mp4", "idx": 369, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8846902847290039, "peak_mem_gb": 9.150652885437012, "n_input_tokens": 1097, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 1.0}, {"method": "full", "video": "YHZ6mWsKDfg.mp4", "idx": 371, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.875776529312134, "peak_mem_gb": 9.898785591125488, "n_input_tokens": 3270, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 1.0}, {"method": "full", "video": "KezGJkHX78E.mp4", "idx": 373, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6101460456848145, "peak_mem_gb": 9.422062873840332, "n_input_tokens": 1901, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 1.0}, {"method": "full", "video": "U1xMaQtB6sE.mp4", "idx": 375, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.717365264892578, "peak_mem_gb": 9.866324424743652, "n_input_tokens": 3166, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 1.0}, {"method": "full", "video": "-YuuVgM_b1Q.mp4", "idx": 377, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.863489866256714, "peak_mem_gb": 9.905532360076904, "n_input_tokens": 3290, "n_video_tokens": 2592, "n_audio_tokens": 455, "retained_av_ratio": 1.0}, {"method": "full", "video": "bczYqpZQMWg.mp4", "idx": 379, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.543237447738647, "peak_mem_gb": 11.584203243255615, "n_input_tokens": 8330, "n_video_tokens": 6912, "n_audio_tokens": 1225, "retained_av_ratio": 1.0}, {"method": "full", "video": "xnx5D0RNCCA.mp4", "idx": 381, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2752807140350342, "peak_mem_gb": 9.26780080795288, "n_input_tokens": 1441, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 1.0}, {"method": "full", "video": "uvGT_4WNYUw.mp4", "idx": 383, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8342576026916504, "peak_mem_gb": 9.507067203521729, "n_input_tokens": 2140, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 1.0}, {"method": "full", "video": "_Y7Q2syNYxY.mp4", "idx": 385, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.759706497192383, "peak_mem_gb": 12.23003339767456, "n_input_tokens": 10252, "n_video_tokens": 8496, "n_audio_tokens": 1480, "retained_av_ratio": 1.0}, {"method": "full", "video": "fv-EzOxlpjo.mp4", "idx": 387, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0814809799194336, "peak_mem_gb": 9.6502046585083, "n_input_tokens": 2479, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 1.0}, {"method": "full", "video": "z9FPvQc9yoI.mp4", "idx": 389, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4395670890808105, "peak_mem_gb": 9.731885433197021, "n_input_tokens": 2744, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 1.0}, {"method": "full", "video": "uhWITF_qwqw.mp4", "idx": 391, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0268759727478027, "peak_mem_gb": 9.137666702270508, "n_input_tokens": 1057, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 1.0}, {"method": "full", "video": "Syz-bdm5oEE.mp4", "idx": 393, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.014413833618164, "peak_mem_gb": 9.628999710083008, "n_input_tokens": 2404, "n_video_tokens": 1872, "n_audio_tokens": 331, "retained_av_ratio": 1.0}, {"method": "full", "video": "aJwO65HE8A4.mp4", "idx": 395, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.865460634231567, "peak_mem_gb": 11.986988544464111, "n_input_tokens": 9525, "n_video_tokens": 7920, "n_audio_tokens": 1380, "retained_av_ratio": 1.0}, {"method": "full", "video": "ZFIh8hf4btQ.mp4", "idx": 397, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.142981767654419, "peak_mem_gb": 9.18726921081543, "n_input_tokens": 1200, "n_video_tokens": 864, "n_audio_tokens": 155, "retained_av_ratio": 1.0}, {"method": "full", "video": "MfjWOF8-npQ.mp4", "idx": 399, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2039895057678223, "peak_mem_gb": 9.252071380615234, "n_input_tokens": 1392, "n_video_tokens": 1008, "n_audio_tokens": 177, "retained_av_ratio": 1.0}, {"method": "full", "video": "hVG1xDJwMbY.mp4", "idx": 401, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.56 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.31862211227417, "n_input_tokens": 12882, "n_video_tokens": 9216, "n_audio_tokens": 3455, "retained_av_ratio": null}, {"method": "full", "video": "AghXXz9hI34.mp4", "idx": 403, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3630855083465576, "peak_mem_gb": 9.348382472991943, "n_input_tokens": 1685, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 1.0}, {"method": "full", "video": "spmQ5-8a6Ao.mp4", "idx": 405, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5702776908874512, "peak_mem_gb": 9.42368745803833, "n_input_tokens": 1906, "n_video_tokens": 1440, "n_audio_tokens": 259, "retained_av_ratio": 1.0}, {"method": "full", "video": "2Gfm1i3DTaw.mp4", "idx": 407, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.885107517242432, "peak_mem_gb": 12.224387645721436, "n_input_tokens": 10233, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 1.0}, {"method": "full", "video": "nKN_c5WAVOw.mp4", "idx": 409, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4398233890533447, "peak_mem_gb": 9.745076179504395, "n_input_tokens": 2787, "n_video_tokens": 2160, "n_audio_tokens": 395, "retained_av_ratio": 1.0}, {"method": "full", "video": "S4oi8EUfLYI.mp4", "idx": 411, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.0082409381866455, "peak_mem_gb": 10.618149280548096, "n_input_tokens": 5440, "n_video_tokens": 4464, "n_audio_tokens": 799, "retained_av_ratio": 1.0}, {"method": "full", "video": "DYfYpkl2OTI.mp4", "idx": 413, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8946750164031982, "peak_mem_gb": 9.543618202209473, "n_input_tokens": 2229, "n_video_tokens": 1728, "n_audio_tokens": 315, "retained_av_ratio": 1.0}, {"method": "full", "video": "51Sg4YQOPhI.mp4", "idx": 415, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.935797929763794, "peak_mem_gb": 9.594668865203857, "n_input_tokens": 2300, "n_video_tokens": 1728, "n_audio_tokens": 326, "retained_av_ratio": 1.0}, {"method": "full", "video": "zVvBiALQtE0.mp4", "idx": 417, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.276977062225342, "peak_mem_gb": 12.081562519073486, "n_input_tokens": 9810, "n_video_tokens": 8064, "n_audio_tokens": 1422, "retained_av_ratio": 1.0}, {"method": "full", "video": "GhuS1QMn9hc.mp4", "idx": 419, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0977320671081543, "peak_mem_gb": 9.212069988250732, "n_input_tokens": 1277, "n_video_tokens": 864, "n_audio_tokens": 173, "retained_av_ratio": 1.0}, {"method": "full", "video": "TUJbjPtM7JE.mp4", "idx": 421, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5206170082092285, "peak_mem_gb": 9.3704195022583, "n_input_tokens": 1750, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 1.0}, {"method": "full", "video": "kq9kiAScGw8.mp4", "idx": 423, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.814958333969116, "peak_mem_gb": 10.23329782485962, "n_input_tokens": 4281, "n_video_tokens": 3456, "n_audio_tokens": 622, "retained_av_ratio": 1.0}, {"method": "full", "video": "NvEO1NCuTR0.mp4", "idx": 425, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6665656566619873, "peak_mem_gb": 10.156251907348633, "n_input_tokens": 4072, "n_video_tokens": 3312, "n_audio_tokens": 595, "retained_av_ratio": 1.0}, {"method": "full", "video": "xhYip8GNPaM.mp4", "idx": 427, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.236933708190918, "peak_mem_gb": 10.017503261566162, "n_input_tokens": 3635, "n_video_tokens": 2880, "n_audio_tokens": 518, "retained_av_ratio": 1.0}, {"method": "full", "video": "u6UOIS1h340.mp4", "idx": 429, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9346654415130615, "peak_mem_gb": 9.08708143234253, "n_input_tokens": 906, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 1.0}, {"method": "full", "video": "hZes-czBp5A.mp4", "idx": 431, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0228145122528076, "peak_mem_gb": 9.622100353240967, "n_input_tokens": 2383, "n_video_tokens": 1872, "n_audio_tokens": 340, "retained_av_ratio": 1.0}, {"method": "full", "video": "bj_1uCcsxrg.mp4", "idx": 433, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.875184774398804, "peak_mem_gb": 12.213167190551758, "n_input_tokens": 10198, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 1.0}, {"method": "full", "video": "aQj9FsOGmFQ.mp4", "idx": 435, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.840240478515625, "peak_mem_gb": 12.212266445159912, "n_input_tokens": 10195, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 1.0}, {"method": "full", "video": "pRcxjOCF3iM.mp4", "idx": 437, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0001342296600342, "peak_mem_gb": 9.169901371002197, "n_input_tokens": 1156, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 1.0}, {"method": "full", "video": "42mYOogHnyE.mp4", "idx": 439, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.29598069190979, "peak_mem_gb": 9.70058298110962, "n_input_tokens": 2640, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 1.0}, {"method": "full", "video": "9yjRUlmmXyk.mp4", "idx": 441, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.923586130142212, "peak_mem_gb": 12.229709148406982, "n_input_tokens": 10251, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 1.0}, {"method": "full", "video": "1C6OKfQaHJ4.mp4", "idx": 443, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.930647611618042, "peak_mem_gb": 9.910704135894775, "n_input_tokens": 3308, "n_video_tokens": 2592, "n_audio_tokens": 467, "retained_av_ratio": 1.0}, {"method": "full", "video": "n4fypasG9N8.mp4", "idx": 445, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0933659076690674, "peak_mem_gb": 9.161457061767578, "n_input_tokens": 1130, "n_video_tokens": 720, "n_audio_tokens": 148, "retained_av_ratio": 1.0}, {"method": "full", "video": "QyR4tUx_eE4.mp4", "idx": 447, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6150376796722412, "peak_mem_gb": 9.373585224151611, "n_input_tokens": 1760, "n_video_tokens": 1296, "n_audio_tokens": 228, "retained_av_ratio": 1.0}, {"method": "full", "video": "Inh0nn0Mehw.mp4", "idx": 449, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2954254150390625, "peak_mem_gb": 9.249218940734863, "n_input_tokens": 1383, "n_video_tokens": 1008, "n_audio_tokens": 178, "retained_av_ratio": 1.0}, {"method": "full", "video": "TqgdJIWKEHU.mp4", "idx": 451, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.213245868682861, "peak_mem_gb": 10.695395469665527, "n_input_tokens": 5679, "n_video_tokens": 4608, "n_audio_tokens": 802, "retained_av_ratio": 1.0}, {"method": "full", "video": "BB4i0vWjSF4.mp4", "idx": 453, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2867987155914307, "peak_mem_gb": 9.66892671585083, "n_input_tokens": 2536, "n_video_tokens": 2002, "n_audio_tokens": 353, "retained_av_ratio": 1.0}, {"method": "full", "video": "2LMNEWSR8K0.mp4", "idx": 455, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6793291568756104, "peak_mem_gb": 9.42595624923706, "n_input_tokens": 1915, "n_video_tokens": 1440, "n_audio_tokens": 276, "retained_av_ratio": 1.0}, {"method": "full", "video": "jfN_xuGVfPg.mp4", "idx": 457, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6049742698669434, "peak_mem_gb": 9.370710372924805, "n_input_tokens": 1751, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 1.0}, {"method": "full", "video": "KMRepMFj6y0.mp4", "idx": 459, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.671702861785889, "peak_mem_gb": 10.848659038543701, "n_input_tokens": 6143, "n_video_tokens": 5040, "n_audio_tokens": 879, "retained_av_ratio": 1.0}, {"method": "full", "video": "iwIjN30bQG8.mp4", "idx": 461, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.549876928329468, "peak_mem_gb": 10.748804569244385, "n_input_tokens": 5844, "n_video_tokens": 4752, "n_audio_tokens": 842, "retained_av_ratio": 1.0}, {"method": "full", "video": "SD3b0eDGylU.mp4", "idx": 463, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.408488512039185, "peak_mem_gb": 9.832918643951416, "n_input_tokens": 3068, "n_video_tokens": 2448, "n_audio_tokens": 442, "retained_av_ratio": 1.0}, {"method": "full", "video": "wskyXFw6C4I.mp4", "idx": 465, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.067488431930542, "peak_mem_gb": 9.14387559890747, "n_input_tokens": 1076, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 1.0}, {"method": "full", "video": "9BTyy18OYe4.mp4", "idx": 467, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.472989559173584, "peak_mem_gb": 9.31815481185913, "n_input_tokens": 1592, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 1.0}, {"method": "full", "video": "-gl0tl4qvzY.mp4", "idx": 469, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0127801895141602, "peak_mem_gb": 9.145458698272705, "n_input_tokens": 1081, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 1.0}, {"method": "full", "video": "_zYW8CObyxM.mp4", "idx": 471, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.911343097686768, "peak_mem_gb": 12.234252452850342, "n_input_tokens": 10265, "n_video_tokens": 8496, "n_audio_tokens": 1500, "retained_av_ratio": 1.0}, {"method": "full", "video": "yEyUrr7d0aI.mp4", "idx": 473, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.657824754714966, "peak_mem_gb": 10.182507038116455, "n_input_tokens": 4117, "n_video_tokens": 3312, "n_audio_tokens": 600, "retained_av_ratio": 1.0}, {"method": "full", "video": "N8S9TYC6mSE.mp4", "idx": 475, "problem_type": "Temporal Reasoning Localization", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.706749439239502, "peak_mem_gb": 9.862450122833252, "n_input_tokens": 3156, "n_video_tokens": 2380, "n_audio_tokens": 436, "retained_av_ratio": 1.0}, {"method": "full", "video": "sHAaDXWyN3c.mp4", "idx": 477, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.005892276763916, "peak_mem_gb": 9.201119899749756, "n_input_tokens": 1243, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 1.0}, {"method": "full", "video": "H3iTVprK7wU.mp4", "idx": 479, "problem_type": "Narrative Comprehension", "answer": "F", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 1.93 GiB. GPU 0 has a total capacity of 14.56 GiB of which 292.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.107272624969482, "n_input_tokens": 11977, "n_video_tokens": 9216, "n_audio_tokens": 2541, "retained_av_ratio": null}, {"method": "full", "video": "O2dajNms-aw.mp4", "idx": 481, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.588111162185669, "peak_mem_gb": 10.188676357269287, "n_input_tokens": 4139, "n_video_tokens": 3312, "n_audio_tokens": 596, "retained_av_ratio": 1.0}, {"method": "full", "video": "w1MSc5_NspY.mp4", "idx": 483, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9229955673217773, "peak_mem_gb": 9.906394481658936, "n_input_tokens": 3293, "n_video_tokens": 2592, "n_audio_tokens": 475, "retained_av_ratio": 1.0}, {"method": "full", "video": "XBRZ_O5FLQ8.mp4", "idx": 485, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.898383617401123, "peak_mem_gb": 9.912715435028076, "n_input_tokens": 3315, "n_video_tokens": 2592, "n_audio_tokens": 470, "retained_av_ratio": 1.0}, {"method": "full", "video": "HuqQmbOK-AQ.mp4", "idx": 487, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9947338104248047, "peak_mem_gb": 9.187613487243652, "n_input_tokens": 1201, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 1.0}, {"method": "full", "video": "jYA4p6xBn2s.mp4", "idx": 489, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.281041622161865, "peak_mem_gb": 10.353510856628418, "n_input_tokens": 4640, "n_video_tokens": 3744, "n_audio_tokens": 661, "retained_av_ratio": 1.0}, {"method": "full", "video": "Zn6Dnm2OAPs.mp4", "idx": 491, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.988696813583374, "peak_mem_gb": 12.186904430389404, "n_input_tokens": 10124, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 1.0}, {"method": "full", "video": "d6zr9dqsvfA.mp4", "idx": 493, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.887552261352539, "peak_mem_gb": 9.89576244354248, "n_input_tokens": 3260, "n_video_tokens": 2592, "n_audio_tokens": 460, "retained_av_ratio": 1.0}, {"method": "full", "video": "UNG7OtwrQoI.mp4", "idx": 495, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.866665363311768, "peak_mem_gb": 11.471067905426025, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1174, "retained_av_ratio": 1.0}, {"method": "full", "video": "3c1sjLnzO1M.mp4", "idx": 497, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.441650152206421, "peak_mem_gb": 10.419672012329102, "n_input_tokens": 4847, "n_video_tokens": 3888, "n_audio_tokens": 696, "retained_av_ratio": 1.0}, {"method": "full", "video": "TIlDW7FLwXk.mp4", "idx": 499, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.024384021759033, "peak_mem_gb": 10.580302715301514, "n_input_tokens": 5332, "n_video_tokens": 4320, "n_audio_tokens": 769, "retained_av_ratio": 1.0}, {"method": "istm_r45", "video": "26TM44a-6a0.mp4", "idx": 0, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.05783486366272, "peak_mem_gb": 9.461158752441406, "n_input_tokens": 3961, "n_video_tokens": 3168, "n_audio_tokens": 575, "retained_av_ratio": 0.4501736574939888}, {"method": "istm_r45", "video": "a_wxf-SGdHU.mp4", "idx": 2, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0262529850006104, "peak_mem_gb": 9.0461745262146, "n_input_tokens": 1232, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 0.4970588235294118}, {"method": "istm_r45", "video": "mecSGL3mrlk.mp4", "idx": 4, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.127078533172607, "peak_mem_gb": 9.779046535491943, "n_input_tokens": 5819, "n_video_tokens": 4760, "n_audio_tokens": 864, "retained_av_ratio": 0.45056899004267426}, {"method": "istm_r45", "video": "Duj0OyTyjNQ.mp4", "idx": 6, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.6485583782196045, "peak_mem_gb": 10.458918571472168, "n_input_tokens": 9792, "n_video_tokens": 8120, "n_audio_tokens": 1458, "retained_av_ratio": 0.45082480684902904}, {"method": "istm_r45", "video": "Q7EqBvNNiT8.mp4", "idx": 8, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.164916038513184, "peak_mem_gb": 10.498050689697266, "n_input_tokens": 10056, "n_video_tokens": 8352, "n_audio_tokens": 1475, "retained_av_ratio": 0.4495776941080696}, {"method": "istm_r45", "video": "87BvhuIeuyc.mp4", "idx": 10, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.5383992195129395, "peak_mem_gb": 9.819988250732422, "n_input_tokens": 6031, "n_video_tokens": 4896, "n_audio_tokens": 863, "retained_av_ratio": 0.4492099322799097}, {"method": "istm_r45", "video": "k9lg5Gd7Q-Y.mp4", "idx": 12, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0764167308807373, "peak_mem_gb": 9.068653583526611, "n_input_tokens": 1439, "n_video_tokens": 1008, "n_audio_tokens": 190, "retained_av_ratio": 0.4532554257095159}, {"method": "istm_r45", "video": "Z2GuiatpQUw.mp4", "idx": 14, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9700736999511719, "peak_mem_gb": 9.049352645874023, "n_input_tokens": 1248, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49167482859941236}, {"method": "istm_r45", "video": "yQvt1MbF-dE.mp4", "idx": 16, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.349283933639526, "peak_mem_gb": 9.960265636444092, "n_input_tokens": 6920, "n_video_tokens": 5616, "n_audio_tokens": 999, "retained_av_ratio": 0.45003779289493573}, {"method": "istm_r45", "video": "ziQsD1EQPOs.mp4", "idx": 18, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0996677875518799, "peak_mem_gb": 9.045242309570312, "n_input_tokens": 1244, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4848780487804878}, {"method": "istm_r45", "video": "rh1PHqq0Q50.mp4", "idx": 20, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7546939849853516, "peak_mem_gb": 8.97169017791748, "n_input_tokens": 728, "n_video_tokens": 432, "n_audio_tokens": 98, "retained_av_ratio": 0.5867924528301887}, {"method": "istm_r45", "video": "nBAE01OGZYI.mp4", "idx": 22, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.649739980697632, "peak_mem_gb": 10.142688274383545, "n_input_tokens": 8211, "n_video_tokens": 6768, "n_audio_tokens": 1201, "retained_av_ratio": 0.44961726690927345}, {"method": "istm_r45", "video": "pAEYLYBhnSk.mp4", "idx": 24, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7216072082519531, "peak_mem_gb": 8.958276748657227, "n_input_tokens": 682, "n_video_tokens": 432, "n_audio_tokens": 77, "retained_av_ratio": 0.5776031434184676}, {"method": "istm_r45", "video": "zrcQYUHNcL8.mp4", "idx": 26, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7044336795806885, "peak_mem_gb": 9.181087017059326, "n_input_tokens": 2219, "n_video_tokens": 1728, "n_audio_tokens": 309, "retained_av_ratio": 0.45017182130584193}, {"method": "istm_r45", "video": "KuDeAIF9rJo.mp4", "idx": 28, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.732136964797974, "peak_mem_gb": 10.556265830993652, "n_input_tokens": 10335, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4486737008184597}, {"method": "istm_r45", "video": "GbJSpPohjDM.mp4", "idx": 30, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4108457565307617, "peak_mem_gb": 9.115773677825928, "n_input_tokens": 1749, "n_video_tokens": 1287, "n_audio_tokens": 232, "retained_av_ratio": 0.4522712310730744}, {"method": "istm_r45", "video": "deDj5V0XxfA.mp4", "idx": 32, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1209425926208496, "peak_mem_gb": 9.07183027267456, "n_input_tokens": 1427, "n_video_tokens": 1008, "n_audio_tokens": 181, "retained_av_ratio": 0.46761984861227923}, {"method": "istm_r45", "video": "9ODSLaN3JFc.mp4", "idx": 34, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.32979154586792, "peak_mem_gb": 9.116836547851562, "n_input_tokens": 1762, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 0.4521795705920625}, {"method": "istm_r45", "video": "ijg1smtVFIQ.mp4", "idx": 36, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.267812490463257, "peak_mem_gb": 9.301447868347168, "n_input_tokens": 2943, "n_video_tokens": 2304, "n_audio_tokens": 420, "retained_av_ratio": 0.4511747430249633}, {"method": "istm_r45", "video": "-abNj3Imno8.mp4", "idx": 38, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8772869110107422, "peak_mem_gb": 9.224681854248047, "n_input_tokens": 2530, "n_video_tokens": 2016, "n_audio_tokens": 356, "retained_av_ratio": 0.44898819561551434}, {"method": "istm_r45", "video": "YYoaOGBfSE4.mp4", "idx": 40, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4521145820617676, "peak_mem_gb": 9.117761135101318, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 0.44937949052906595}, {"method": "istm_r45", "video": "ZQ3iu3wMCPg.mp4", "idx": 42, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0370397567749023, "peak_mem_gb": 9.02387285232544, "n_input_tokens": 1054, "n_video_tokens": 720, "n_audio_tokens": 130, "retained_av_ratio": 0.5305882352941177}, {"method": "istm_r45", "video": "_SrDOLOuTWE.mp4", "idx": 44, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.271307945251465, "peak_mem_gb": 10.524834632873535, "n_input_tokens": 10196, "n_video_tokens": 8496, "n_audio_tokens": 1487, "retained_av_ratio": 0.4489632375037564}, {"method": "istm_r45", "video": "mb5_ZoHnqKM.mp4", "idx": 46, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.611297845840454, "peak_mem_gb": 9.589613914489746, "n_input_tokens": 4666, "n_video_tokens": 3744, "n_audio_tokens": 666, "retained_av_ratio": 0.4492063492063492}, {"method": "istm_r45", "video": "m_IUCFFELcY.mp4", "idx": 48, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4056079387664795, "peak_mem_gb": 9.127691745758057, "n_input_tokens": 1785, "n_video_tokens": 1296, "n_audio_tokens": 247, "retained_av_ratio": 0.46143875567077125}, {"method": "istm_r45", "video": "ESBEIJcZInw.mp4", "idx": 50, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9962248802185059, "peak_mem_gb": 9.057539463043213, "n_input_tokens": 1260, "n_video_tokens": 840, "n_audio_tokens": 164, "retained_av_ratio": 0.49701195219123506}, {"method": "istm_r45", "video": "ltv6vNSstPI.mp4", "idx": 52, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2526662349700928, "peak_mem_gb": 9.304698944091797, "n_input_tokens": 2953, "n_video_tokens": 2304, "n_audio_tokens": 412, "retained_av_ratio": 0.4495581737849779}, {"method": "istm_r45", "video": "R3JcXA8OmEM.mp4", "idx": 54, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9865782260894775, "peak_mem_gb": 9.435484409332275, "n_input_tokens": 3828, "n_video_tokens": 3080, "n_audio_tokens": 559, "retained_av_ratio": 0.45039846111569115}, {"method": "istm_r45", "video": "-H5Yu7-Zb90.mp4", "idx": 56, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6563000679016113, "peak_mem_gb": 9.164394855499268, "n_input_tokens": 2078, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 0.4517333333333333}, {"method": "istm_r45", "video": "TJQ22uyc6XQ.mp4", "idx": 58, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7037622928619385, "peak_mem_gb": 8.965907096862793, "n_input_tokens": 707, "n_video_tokens": 432, "n_audio_tokens": 80, "retained_av_ratio": 0.578125}, {"method": "istm_r45", "video": "9G-6yw9kGIY.mp4", "idx": 60, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.1508073806762695, "peak_mem_gb": 9.9945387840271, "n_input_tokens": 7182, "n_video_tokens": 5904, "n_audio_tokens": 1049, "retained_av_ratio": 0.45332949805839207}, {"method": "istm_r45", "video": "zpeY4mrLlvY.mp4", "idx": 62, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6957929134368896, "peak_mem_gb": 9.172745704650879, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.45154419595314166}, {"method": "istm_r45", "video": "B1U-6d5J7Vw.mp4", "idx": 64, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.3302903175354, "peak_mem_gb": 9.705991268157959, "n_input_tokens": 5310, "n_video_tokens": 4320, "n_audio_tokens": 773, "retained_av_ratio": 0.4496367563322207}, {"method": "istm_r45", "video": "U453o9dA3cs.mp4", "idx": 66, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.917634963989258, "peak_mem_gb": 9.386731624603271, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 0.4504029758214507}, {"method": "istm_r45", "video": "WtjJETMkuiM.mp4", "idx": 68, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.2245185375213623, "peak_mem_gb": 9.429253101348877, "n_input_tokens": 3771, "n_video_tokens": 3024, "n_audio_tokens": 553, "retained_av_ratio": 0.4520547945205479}, {"method": "istm_r45", "video": "LFIUeTL2ClU.mp4", "idx": 70, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.869978904724121, "peak_mem_gb": 9.37516736984253, "n_input_tokens": 3431, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 0.45080445544554454}, {"method": "istm_r45", "video": "31zomD2ZORA.mp4", "idx": 72, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.497821569442749, "peak_mem_gb": 9.13494348526001, "n_input_tokens": 1818, "n_video_tokens": 1296, "n_audio_tokens": 244, "retained_av_ratio": 0.4538961038961039}, {"method": "istm_r45", "video": "Hkzmq5SO0SE.mp4", "idx": 74, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.655274629592896, "peak_mem_gb": 10.54781436920166, "n_input_tokens": 10172, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4498699219531719}, {"method": "istm_r45", "video": "81SdVr4Cqb8.mp4", "idx": 76, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8609740734100342, "peak_mem_gb": 9.018774509429932, "n_input_tokens": 934, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5877318116975749}, {"method": "istm_r45", "video": "kCPt7hqgYis.mp4", "idx": 78, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1684331893920898, "peak_mem_gb": 9.050653457641602, "n_input_tokens": 1253, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4946550048590865}, {"method": "istm_r45", "video": "AbvUzIEW6lw.mp4", "idx": 80, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3704261779785156, "peak_mem_gb": 9.484695434570312, "n_input_tokens": 4026, "n_video_tokens": 3168, "n_audio_tokens": 554, "retained_av_ratio": 0.4492208490059108}, {"method": "istm_r45", "video": "xPTy0rDrzU0.mp4", "idx": 82, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.15559983253479, "peak_mem_gb": 9.446803569793701, "n_input_tokens": 3827, "n_video_tokens": 3024, "n_audio_tokens": 542, "retained_av_ratio": 0.44980370162647226}, {"method": "istm_r45", "video": "GD-P6jsAGWU.mp4", "idx": 84, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.439144611358643, "peak_mem_gb": 10.431095600128174, "n_input_tokens": 9847, "n_video_tokens": 8208, "n_audio_tokens": 1444, "retained_av_ratio": 0.4495441359303771}, {"method": "istm_r45", "video": "aGqCwX-9Oh4.mp4", "idx": 86, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4287419319152832, "peak_mem_gb": 9.076325416564941, "n_input_tokens": 1543, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 0.4515890613451589}, {"method": "istm_r45", "video": "iIDhWvE6jR0.mp4", "idx": 88, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 12.017235279083252, "peak_mem_gb": 11.519179821014404, "n_input_tokens": 11346, "n_video_tokens": 9216, "n_audio_tokens": 1901, "retained_av_ratio": 0.45659800305837905}, {"method": "istm_r45", "video": "b6dkq4sBDyg.mp4", "idx": 90, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5938103199005127, "peak_mem_gb": 9.509133338928223, "n_input_tokens": 4282, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.4492753623188406}, {"method": "istm_r45", "video": "14__F3Bhv4E.mp4", "idx": 92, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9187335968017578, "peak_mem_gb": 9.203473567962646, "n_input_tokens": 2289, "n_video_tokens": 1728, "n_audio_tokens": 316, "retained_av_ratio": 0.4515655577299413}, {"method": "istm_r45", "video": "XxfYKJP7jkE.mp4", "idx": 94, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.125910520553589, "peak_mem_gb": 9.469189167022705, "n_input_tokens": 3984, "n_video_tokens": 3168, "n_audio_tokens": 567, "retained_av_ratio": 0.4495314591700134}, {"method": "istm_r45", "video": "_OCNY0nkjGI.mp4", "idx": 96, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.882375478744507, "peak_mem_gb": 10.20951509475708, "n_input_tokens": 8682, "n_video_tokens": 7200, "n_audio_tokens": 1257, "retained_av_ratio": 0.44850419770604233}, {"method": "istm_r45", "video": "0OCwUDKhb1c.mp4", "idx": 98, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.2962849140167236, "peak_mem_gb": 9.264852046966553, "n_input_tokens": 2742, "n_video_tokens": 2160, "n_audio_tokens": 383, "retained_av_ratio": 0.44907589461266223}, {"method": "istm_r45", "video": "abR9DrA5AyI.mp4", "idx": 100, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0209524631500244, "peak_mem_gb": 9.014890193939209, "n_input_tokens": 1073, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.48791714614499426}, {"method": "istm_r45", "video": "Xkuh_lyCZ0Y.mp4", "idx": 102, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8281564712524414, "peak_mem_gb": 9.376451969146729, "n_input_tokens": 3434, "n_video_tokens": 2736, "n_audio_tokens": 494, "retained_av_ratio": 0.45077399380804956}, {"method": "istm_r45", "video": "uo5ayLrm2xg.mp4", "idx": 104, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.545010805130005, "peak_mem_gb": 10.547049045562744, "n_input_tokens": 10246, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4494696818090855}, {"method": "istm_r45", "video": "SmdRGMI5qsM.mp4", "idx": 106, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.360900640487671, "peak_mem_gb": 9.098493576049805, "n_input_tokens": 1619, "n_video_tokens": 1152, "n_audio_tokens": 218, "retained_av_ratio": 0.45255474452554745}, {"method": "istm_r45", "video": "1kFJgInMGd8.mp4", "idx": 108, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.318230152130127, "peak_mem_gb": 9.09457015991211, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 0.45234604105571846}, {"method": "istm_r45", "video": "QodbVfAEx40.mp4", "idx": 110, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.5786523818969727, "peak_mem_gb": 9.320565223693848, "n_input_tokens": 3088, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 0.45115636865723163}, {"method": "istm_r45", "video": "bSSkJPyZ5vk.mp4", "idx": 112, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.008820295333862, "peak_mem_gb": 10.36132287979126, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1397, "retained_av_ratio": 0.44950091231082967}, {"method": "istm_r45", "video": "-XaFOYkyICI.mp4", "idx": 114, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.147987365722656, "peak_mem_gb": 10.94026803970337, "n_input_tokens": 11168, "n_video_tokens": 9216, "n_audio_tokens": 1670, "retained_av_ratio": 0.4514973360279258}, {"method": "istm_r45", "video": "BVdjSumD5V4.mp4", "idx": 116, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8988978862762451, "peak_mem_gb": 8.998308181762695, "n_input_tokens": 1030, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.4652532391048292}, {"method": "istm_r45", "video": "UMEiemYRr6w.mp4", "idx": 118, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8402411937713623, "peak_mem_gb": 9.197664260864258, "n_input_tokens": 2273, "n_video_tokens": 1728, "n_audio_tokens": 322, "retained_av_ratio": 0.4521951219512195}, {"method": "istm_r45", "video": "_kVH8Wh7tBo.mp4", "idx": 120, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.3661980628967285, "peak_mem_gb": 9.975058555603027, "n_input_tokens": 7140, "n_video_tokens": 5904, "n_audio_tokens": 1041, "retained_av_ratio": 0.4499640028797696}, {"method": "istm_r45", "video": "VTFUr1GXBqY.mp4", "idx": 122, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.3971288204193115, "peak_mem_gb": 9.69353437423706, "n_input_tokens": 5257, "n_video_tokens": 4320, "n_audio_tokens": 755, "retained_av_ratio": 0.4496551724137931}, {"method": "istm_r45", "video": "y4x5PY9JBzw.mp4", "idx": 124, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.881319522857666, "peak_mem_gb": 10.40688943862915, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1432, "retained_av_ratio": 0.4495850622406639}, {"method": "istm_r45", "video": "HT9uHeaOBVY.mp4", "idx": 126, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0487556457519531, "peak_mem_gb": 9.037004470825195, "n_input_tokens": 1208, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "istm_r45", "video": "CjzBW4HOndQ.mp4", "idx": 128, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.01138710975647, "peak_mem_gb": 10.675412654876709, "n_input_tokens": 10748, "n_video_tokens": 8928, "n_audio_tokens": 1555, "retained_av_ratio": 0.4491080797481637}, {"method": "istm_r45", "video": "2Rl9dPOJk2w.mp4", "idx": 130, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.451693058013916, "peak_mem_gb": 9.838339805603027, "n_input_tokens": 6176, "n_video_tokens": 5040, "n_audio_tokens": 878, "retained_av_ratio": 0.4486312943562014}, {"method": "istm_r45", "video": "Cno_MIpe7Yg.mp4", "idx": 132, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.898399591445923, "peak_mem_gb": 10.556572914123535, "n_input_tokens": 10415, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4488709200276107}, {"method": "istm_r45", "video": "ajvnJTQvl_Y.mp4", "idx": 134, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.91743803024292, "peak_mem_gb": 10.535204410552979, "n_input_tokens": 10171, "n_video_tokens": 8496, "n_audio_tokens": 1492, "retained_av_ratio": 0.44923908690428516}, {"method": "istm_r45", "video": "BcREVtMNh4w.mp4", "idx": 136, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0957584381103516, "peak_mem_gb": 9.021704196929932, "n_input_tokens": 1095, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.4820393974507532}, {"method": "istm_r45", "video": "WgGReZb7igo.mp4", "idx": 138, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0854625701904297, "peak_mem_gb": 9.041131973266602, "n_input_tokens": 1217, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.500485908649174}, {"method": "istm_r45", "video": "6wMx9nKgqNE.mp4", "idx": 140, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.227637767791748, "peak_mem_gb": 9.075718402862549, "n_input_tokens": 1430, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 0.47651006711409394}, {"method": "istm_r45", "video": "XslxClszbTU.mp4", "idx": 142, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.2778396606445312, "peak_mem_gb": 9.46877908706665, "n_input_tokens": 4015, "n_video_tokens": 3220, "n_audio_tokens": 582, "retained_av_ratio": 0.45028932140978434}, {"method": "istm_r45", "video": "MJKdCd93wDc.mp4", "idx": 144, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.168853998184204, "peak_mem_gb": 9.279481410980225, "n_input_tokens": 2788, "n_video_tokens": 2160, "n_audio_tokens": 401, "retained_av_ratio": 0.4525575946895744}, {"method": "istm_r45", "video": "FPCuC9ZQlFo.mp4", "idx": 146, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.584519863128662, "peak_mem_gb": 9.879264831542969, "n_input_tokens": 6580, "n_video_tokens": 5472, "n_audio_tokens": 967, "retained_av_ratio": 0.44898276129833825}, {"method": "istm_r45", "video": "fujA0PoO_5w.mp4", "idx": 148, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1870801448822021, "peak_mem_gb": 9.06803274154663, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 0.48255813953488375}, {"method": "istm_r45", "video": "EhwV8BqcpSA.mp4", "idx": 150, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7011587619781494, "peak_mem_gb": 9.150729656219482, "n_input_tokens": 2037, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45008008542445277}, {"method": "istm_r45", "video": "YnjbpidCSN8.mp4", "idx": 152, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8365795612335205, "peak_mem_gb": 9.015429019927979, "n_input_tokens": 927, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5848787446504993}, {"method": "istm_r45", "video": "mJpo1ijaQOs.mp4", "idx": 154, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.413872241973877, "peak_mem_gb": 9.295272827148438, "n_input_tokens": 2923, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 0.45073529411764707}, {"method": "istm_r45", "video": "Zi4RV-TrTus.mp4", "idx": 156, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.42299509048462, "peak_mem_gb": 10.419170379638672, "n_input_tokens": 9908, "n_video_tokens": 8208, "n_audio_tokens": 1438, "retained_av_ratio": 0.44920174165457183}, {"method": "istm_r45", "video": "Mmz1Aba-1P0.mp4", "idx": 158, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.503838777542114, "peak_mem_gb": 9.871943950653076, "n_input_tokens": 6373, "n_video_tokens": 5184, "n_audio_tokens": 902, "retained_av_ratio": 0.44889911271771277}, {"method": "istm_r45", "video": "w9H2ibZd9zk.mp4", "idx": 160, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.463667392730713, "peak_mem_gb": 9.105916976928711, "n_input_tokens": 1732, "n_video_tokens": 1296, "n_audio_tokens": 239, "retained_av_ratio": 0.4501628664495114}, {"method": "istm_r45", "video": "GiN9w2a7wTc.mp4", "idx": 162, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.008185386657715, "peak_mem_gb": 9.78158187866211, "n_input_tokens": 5816, "n_video_tokens": 4752, "n_audio_tokens": 832, "retained_av_ratio": 0.4487822349570201}, {"method": "istm_r45", "video": "a0PA53cVK-I.mp4", "idx": 164, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.930337905883789, "peak_mem_gb": 9.638270378112793, "n_input_tokens": 4724, "n_video_tokens": 3744, "n_audio_tokens": 673, "retained_av_ratio": 0.45075843332578674}, {"method": "istm_r45", "video": "T7lDs4GGNjU.mp4", "idx": 166, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8786797523498535, "peak_mem_gb": 9.384161949157715, "n_input_tokens": 3457, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 0.4504029758214507}, {"method": "istm_r45", "video": "zutS7gJ4Po0.mp4", "idx": 168, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.658457040786743, "peak_mem_gb": 9.501657962799072, "n_input_tokens": 4256, "n_video_tokens": 3456, "n_audio_tokens": 605, "retained_av_ratio": 0.4486579660182221}, {"method": "istm_r45", "video": "_GTQj7_xyME.mp4", "idx": 170, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 13.414403200149536, "peak_mem_gb": 12.45910120010376, "n_input_tokens": 11704, "n_video_tokens": 9216, "n_audio_tokens": 2226, "retained_av_ratio": 0.4650410767348366}, {"method": "istm_r45", "video": "pcnVQwKXsSc.mp4", "idx": 172, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2432961463928223, "peak_mem_gb": 9.074473857879639, "n_input_tokens": 1461, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.45356550580431176}, {"method": "istm_r45", "video": "eE2SsX4F8TM.mp4", "idx": 174, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8248844146728516, "peak_mem_gb": 9.013525009155273, "n_input_tokens": 923, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5820256776034237}, {"method": "istm_r45", "video": "0CCRotntwdA.mp4", "idx": 176, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7197906970977783, "peak_mem_gb": 9.149483680725098, "n_input_tokens": 2034, "n_video_tokens": 1584, "n_audio_tokens": 293, "retained_av_ratio": 0.4507192328183271}, {"method": "istm_r45", "video": "wREFmTKv5DM.mp4", "idx": 178, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.02766752243042, "peak_mem_gb": 10.383824825286865, "n_input_tokens": 9675, "n_video_tokens": 8064, "n_audio_tokens": 1421, "retained_av_ratio": 0.44913020558777017}, {"method": "istm_r45", "video": "h7Mjce0y7SY.mp4", "idx": 180, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1730670928955078, "peak_mem_gb": 9.052568912506104, "n_input_tokens": 1263, "n_video_tokens": 864, "n_audio_tokens": 168, "retained_av_ratio": 0.49224806201550386}, {"method": "istm_r45", "video": "V2CSquBY4pk.mp4", "idx": 182, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.752875566482544, "peak_mem_gb": 9.891140937805176, "n_input_tokens": 6521, "n_video_tokens": 5328, "n_audio_tokens": 928, "retained_av_ratio": 0.4493286445012788}, {"method": "istm_r45", "video": "bHsmSHIoThw.mp4", "idx": 184, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5770854949951172, "peak_mem_gb": 9.132380962371826, "n_input_tokens": 1894, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.4547048509643483}, {"method": "istm_r45", "video": "WLZ8GAD_Mus.mp4", "idx": 186, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.259894609451294, "peak_mem_gb": 9.267780780792236, "n_input_tokens": 2753, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 0.45023510971786834}, {"method": "istm_r45", "video": "UaBfM-tONmI.mp4", "idx": 188, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.608980894088745, "peak_mem_gb": 11.359811782836914, "n_input_tokens": 11265, "n_video_tokens": 9216, "n_audio_tokens": 1840, "retained_av_ratio": 0.4556801736613603}, {"method": "istm_r45", "video": "KF19sEfFHM8.mp4", "idx": 190, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7970588207244873, "peak_mem_gb": 8.97547960281372, "n_input_tokens": 739, "n_video_tokens": 432, "n_audio_tokens": 93, "retained_av_ratio": 0.5847619047619048}, {"method": "istm_r45", "video": "7Np8y8RdAxA.mp4", "idx": 192, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.023179769515991, "peak_mem_gb": 9.198527812957764, "n_input_tokens": 2358, "n_video_tokens": 1872, "n_audio_tokens": 326, "retained_av_ratio": 0.44904458598726116}, {"method": "istm_r45", "video": "x-RjxvYlzrE.mp4", "idx": 194, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7441525459289551, "peak_mem_gb": 8.99390459060669, "n_input_tokens": 795, "n_video_tokens": 432, "n_audio_tokens": 97, "retained_av_ratio": 0.5841209829867675}, {"method": "istm_r45", "video": "FkKh29UxzEc.mp4", "idx": 196, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 12.57371473312378, "peak_mem_gb": 12.122392177581787, "n_input_tokens": 11521, "n_video_tokens": 9216, "n_audio_tokens": 2115, "retained_av_ratio": 0.4628011649457241}, {"method": "istm_r45", "video": "fYYV5CoB5QM.mp4", "idx": 198, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4962329864501953, "peak_mem_gb": 9.131096363067627, "n_input_tokens": 1807, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 0.45003265839320705}, {"method": "istm_r45", "video": "nsbOHGP_uGI.mp4", "idx": 200, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.6800394058227539, "peak_mem_gb": 8.979632377624512, "n_input_tokens": 751, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 0.5921052631578947}, {"method": "istm_r45", "video": "4tyX4O4it5U.mp4", "idx": 202, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.248010873794556, "peak_mem_gb": 10.430200099945068, "n_input_tokens": 9878, "n_video_tokens": 8208, "n_audio_tokens": 1443, "retained_av_ratio": 0.44938348357683144}, {"method": "istm_r45", "video": "-32uRUUd-KU.mp4", "idx": 204, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9951574802398682, "peak_mem_gb": 9.04426097869873, "n_input_tokens": 1139, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "istm_r45", "video": "LQsQCioPS4U.mp4", "idx": 206, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.08040452003479, "peak_mem_gb": 9.046483993530273, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 154, "retained_av_ratio": 0.4931237721021611}, {"method": "istm_r45", "video": "21IQzeQBnGo.mp4", "idx": 208, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.107799530029297, "peak_mem_gb": 9.217224597930908, "n_input_tokens": 2420, "n_video_tokens": 1872, "n_audio_tokens": 334, "retained_av_ratio": 0.4496826835902085}, {"method": "istm_r45", "video": "Q0GG4iuGL3g.mp4", "idx": 210, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0948669910430908, "peak_mem_gb": 9.03349781036377, "n_input_tokens": 1199, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4926829268292683}, {"method": "istm_r45", "video": "PBhCY56XIcI.mp4", "idx": 212, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2564637660980225, "peak_mem_gb": 9.052009105682373, "n_input_tokens": 1391, "n_video_tokens": 1008, "n_audio_tokens": 201, "retained_av_ratio": 0.45492142266335817}, {"method": "istm_r45", "video": "eEV120knBoA.mp4", "idx": 214, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.8618416786193848, "peak_mem_gb": 9.391228675842285, "n_input_tokens": 3478, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 0.4520717377860235}, {"method": "istm_r45", "video": "2lXyTywKtkw.mp4", "idx": 216, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.486955404281616, "peak_mem_gb": 9.670341968536377, "n_input_tokens": 5093, "n_video_tokens": 4176, "n_audio_tokens": 747, "retained_av_ratio": 0.45053828966077597}, {"method": "istm_r45", "video": "olx5oFJbivo.mp4", "idx": 218, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.671331405639648, "peak_mem_gb": 10.455708026885986, "n_input_tokens": 10026, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": 0.44916896094626285}, {"method": "istm_r45", "video": "_uDlP4vW190.mp4", "idx": 220, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.109675407409668, "peak_mem_gb": 10.340779781341553, "n_input_tokens": 9506, "n_video_tokens": 7920, "n_audio_tokens": 1394, "retained_av_ratio": 0.44953832939660726}, {"method": "istm_r45", "video": "ZJovAHQU0o8.mp4", "idx": 222, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2067291736602783, "peak_mem_gb": 9.053826808929443, "n_input_tokens": 1261, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "istm_r45", "video": "YB3bxufnn7s.mp4", "idx": 224, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5254507064819336, "peak_mem_gb": 9.098107814788818, "n_input_tokens": 1701, "n_video_tokens": 1296, "n_audio_tokens": 237, "retained_av_ratio": 0.45401174168297453}, {"method": "istm_r45", "video": "SexJ0OetXZA.mp4", "idx": 226, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7603514194488525, "peak_mem_gb": 8.968206882476807, "n_input_tokens": 723, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 0.5770676691729323}, {"method": "istm_r45", "video": "3l4sUgCflzc.mp4", "idx": 228, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9879047870635986, "peak_mem_gb": 9.007246017456055, "n_input_tokens": 1052, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.4802784222737819}, {"method": "istm_r45", "video": "KwhBtGE-mDc.mp4", "idx": 230, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.872883319854736, "peak_mem_gb": 10.557207107543945, "n_input_tokens": 10305, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4488266614080063}, {"method": "istm_r45", "video": "BojPeImI2QE.mp4", "idx": 232, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.940027713775635, "peak_mem_gb": 10.554561614990234, "n_input_tokens": 10220, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44953486045813745}, {"method": "istm_r45", "video": "4NlfaBAaGrM.mp4", "idx": 234, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.449460744857788, "peak_mem_gb": 9.705988883972168, "n_input_tokens": 5305, "n_video_tokens": 4320, "n_audio_tokens": 760, "retained_av_ratio": 0.44921259842519684}, {"method": "istm_r45", "video": "2Yp6UqQPQNE.mp4", "idx": 236, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.430890321731567, "peak_mem_gb": 10.488563060760498, "n_input_tokens": 10012, "n_video_tokens": 8352, "n_audio_tokens": 1471, "retained_av_ratio": 0.4489463504021175}, {"method": "istm_r45", "video": "qzUdGKGEF6o.mp4", "idx": 238, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4665312767028809, "peak_mem_gb": 9.116217613220215, "n_input_tokens": 1761, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45330739299610895}, {"method": "istm_r45", "video": "WgWdcSg5_to.mp4", "idx": 240, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.838070631027222, "peak_mem_gb": 10.539609432220459, "n_input_tokens": 10225, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": 0.44994994994994997}, {"method": "istm_r45", "video": "rD5uIy9UgxA.mp4", "idx": 242, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3685357570648193, "peak_mem_gb": 9.111191749572754, "n_input_tokens": 1646, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 0.4636627906976744}, {"method": "istm_r45", "video": "y_MMvaZvGIA.mp4", "idx": 244, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.220156669616699, "peak_mem_gb": 9.95290756225586, "n_input_tokens": 6979, "n_video_tokens": 5760, "n_audio_tokens": 1010, "retained_av_ratio": 0.44918759231905464}, {"method": "istm_r45", "video": "HO65gX-k31g.mp4", "idx": 246, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0591611862182617, "peak_mem_gb": 9.065366744995117, "n_input_tokens": 1295, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.4921259842519685}, {"method": "istm_r45", "video": "DGNI8-twVys.mp4", "idx": 248, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7684075832366943, "peak_mem_gb": 9.370350360870361, "n_input_tokens": 3415, "n_video_tokens": 2736, "n_audio_tokens": 497, "retained_av_ratio": 0.4512836374884009}, {"method": "istm_r45", "video": "Igf8n7hy1HI.mp4", "idx": 250, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2996094226837158, "peak_mem_gb": 9.067954063415527, "n_input_tokens": 1408, "n_video_tokens": 1008, "n_audio_tokens": 176, "retained_av_ratio": 0.47128378378378377}, {"method": "istm_r45", "video": "JPqjEZXKoQc.mp4", "idx": 252, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5303809642791748, "peak_mem_gb": 9.139312267303467, "n_input_tokens": 1912, "n_video_tokens": 1440, "n_audio_tokens": 253, "retained_av_ratio": 0.4524512699350266}, {"method": "istm_r45", "video": "IhiQi9X8l8o.mp4", "idx": 254, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.637995958328247, "peak_mem_gb": 10.1538405418396, "n_input_tokens": 8332, "n_video_tokens": 6912, "n_audio_tokens": 1210, "retained_av_ratio": 0.44902733316917015}, {"method": "istm_r45", "video": "vR8fuThRlA4.mp4", "idx": 256, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5881564617156982, "peak_mem_gb": 9.13388204574585, "n_input_tokens": 1900, "n_video_tokens": 1440, "n_audio_tokens": 257, "retained_av_ratio": 0.44961697112551563}, {"method": "istm_r45", "video": "qP6IYn3MrcA.mp4", "idx": 258, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.078290939331055, "peak_mem_gb": 10.102481842041016, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 0.4491210060310535}, {"method": "istm_r45", "video": "WBtbl0O4CZA.mp4", "idx": 260, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.80848741531372, "peak_mem_gb": 10.638450145721436, "n_input_tokens": 10580, "n_video_tokens": 8784, "n_audio_tokens": 1539, "retained_av_ratio": 0.44938486873970745}, {"method": "istm_r45", "video": "HAjJZ1GOXFA.mp4", "idx": 262, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.6451711654663086, "peak_mem_gb": 8.971384525299072, "n_input_tokens": 738, "n_video_tokens": 432, "n_audio_tokens": 101, "retained_av_ratio": 0.5684803001876173}, {"method": "istm_r45", "video": "kzZ7TfwT9Ss.mp4", "idx": 264, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.50520634651184, "peak_mem_gb": 10.42507266998291, "n_input_tokens": 9863, "n_video_tokens": 8208, "n_audio_tokens": 1441, "retained_av_ratio": 0.4486475282412685}, {"method": "istm_r45", "video": "oZlh0hjcXfA.mp4", "idx": 266, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5302155017852783, "peak_mem_gb": 9.313786506652832, "n_input_tokens": 3066, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 0.44892286309937457}, {"method": "istm_r45", "video": "hIWZDSR_uns.mp4", "idx": 268, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.371455192565918, "peak_mem_gb": 9.27883005142212, "n_input_tokens": 2784, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 0.45141065830721006}, {"method": "istm_r45", "video": "91kNCJOoHcY.mp4", "idx": 270, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0736193656921387, "peak_mem_gb": 9.046475887298584, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.4921259842519685}, {"method": "istm_r45", "video": "4HYLaF6fbFY.mp4", "idx": 272, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.780245304107666, "peak_mem_gb": 9.37225341796875, "n_input_tokens": 3330, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 0.45184459680052236}, {"method": "istm_r45", "video": "dtGj9oet70E.mp4", "idx": 274, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.990368127822876, "peak_mem_gb": 10.55712890625, "n_input_tokens": 10285, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4490238611713666}, {"method": "istm_r45", "video": "PwSOZli5_xg.mp4", "idx": 276, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.873104572296143, "peak_mem_gb": 9.768975257873535, "n_input_tokens": 5769, "n_video_tokens": 4752, "n_audio_tokens": 843, "retained_av_ratio": 0.45004468275245757}, {"method": "istm_r45", "video": "MYM3IFheI-E.mp4", "idx": 278, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1390185356140137, "peak_mem_gb": 9.060871124267578, "n_input_tokens": 1403, "n_video_tokens": 1008, "n_audio_tokens": 195, "retained_av_ratio": 0.46550290939318373}, {"method": "istm_r45", "video": "EB9bA1a5TEs.mp4", "idx": 280, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.552936315536499, "peak_mem_gb": 9.121941566467285, "n_input_tokens": 1780, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 0.4540750323415265}, {"method": "istm_r45", "video": "oZLqkTvWXko.mp4", "idx": 282, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.628103256225586, "peak_mem_gb": 9.138713836669922, "n_input_tokens": 1918, "n_video_tokens": 1440, "n_audio_tokens": 266, "retained_av_ratio": 0.45193434935521687}, {"method": "istm_r45", "video": "kS6aRrdqI7M.mp4", "idx": 284, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5188143253326416, "peak_mem_gb": 9.334038734436035, "n_input_tokens": 3122, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 0.44869565217391305}, {"method": "istm_r45", "video": "1P6XRN-amuU.mp4", "idx": 286, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6149985790252686, "peak_mem_gb": 9.125555515289307, "n_input_tokens": 1876, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 0.4532710280373832}, {"method": "istm_r45", "video": "akm-KQiLsvI.mp4", "idx": 288, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.538997173309326, "peak_mem_gb": 9.48619031906128, "n_input_tokens": 4120, "n_video_tokens": 3312, "n_audio_tokens": 584, "retained_av_ratio": 0.449435318275154}, {"method": "istm_r45", "video": "JjuQ2ink05Q.mp4", "idx": 290, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.8301515579223633, "peak_mem_gb": 9.580150127410889, "n_input_tokens": 4632, "n_video_tokens": 3744, "n_audio_tokens": 660, "retained_av_ratio": 0.4493642143505904}, {"method": "istm_r45", "video": "TQjp3EaFCsc.mp4", "idx": 292, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.600325107574463, "peak_mem_gb": 9.854316234588623, "n_input_tokens": 6320, "n_video_tokens": 5184, "n_audio_tokens": 919, "retained_av_ratio": 0.4494510896280518}, {"method": "istm_r45", "video": "pyfj75fxjj8.mp4", "idx": 294, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5491480827331543, "peak_mem_gb": 9.310022830963135, "n_input_tokens": 2972, "n_video_tokens": 2304, "n_audio_tokens": 422, "retained_av_ratio": 0.450476889214967}, {"method": "istm_r45", "video": "05FGA7Febzc.mp4", "idx": 296, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9065423011779785, "peak_mem_gb": 9.008442878723145, "n_input_tokens": 907, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "istm_r45", "video": "8OqmJB8oGtM.mp4", "idx": 298, "problem_type": "Narrative Comprehension", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 5.63 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.485508441925049, "n_input_tokens": 13764, "n_video_tokens": 9216, "n_audio_tokens": 4347, "retained_av_ratio": null}, {"method": "istm_r45", "video": "bajfnk7lwL4.mp4", "idx": 300, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.8141191005706787, "peak_mem_gb": 9.384805679321289, "n_input_tokens": 3462, "n_video_tokens": 2736, "n_audio_tokens": 500, "retained_av_ratio": 0.4511742892459827}, {"method": "istm_r45", "video": "mrYBGL9-DDE.mp4", "idx": 302, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.339607238769531, "peak_mem_gb": 9.83223581314087, "n_input_tokens": 6155, "n_video_tokens": 5040, "n_audio_tokens": 890, "retained_av_ratio": 0.44974704890387857}, {"method": "istm_r45", "video": "odgiRt2sitM.mp4", "idx": 304, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9728958606719971, "peak_mem_gb": 9.029514789581299, "n_input_tokens": 969, "n_video_tokens": 576, "n_audio_tokens": 112, "retained_av_ratio": 0.5784883720930233}, {"method": "istm_r45", "video": "iIZiJR-MHnM.mp4", "idx": 306, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7817842960357666, "peak_mem_gb": 9.173067092895508, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.4520766773162939}, {"method": "istm_r45", "video": "vcsHoRWlD7E.mp4", "idx": 308, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6949129104614258, "peak_mem_gb": 9.188485145568848, "n_input_tokens": 2154, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45061398825413773}, {"method": "istm_r45", "video": "ge-zGSapyeQ.mp4", "idx": 310, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0837249755859375, "peak_mem_gb": 9.056365966796875, "n_input_tokens": 1269, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "istm_r45", "video": "eAOKPLl6AVY.mp4", "idx": 312, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.4922285079956055, "peak_mem_gb": 9.491066932678223, "n_input_tokens": 4134, "n_video_tokens": 3312, "n_audio_tokens": 593, "retained_av_ratio": 0.45096030729833547}, {"method": "istm_r45", "video": "HchGCB08bLg.mp4", "idx": 314, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7875785827636719, "peak_mem_gb": 9.011565685272217, "n_input_tokens": 911, "n_video_tokens": 576, "n_audio_tokens": 111, "retained_av_ratio": 0.5822416302765647}, {"method": "istm_r45", "video": "ECvz4IPwFQ8.mp4", "idx": 316, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.887945413589478, "peak_mem_gb": 10.324250221252441, "n_input_tokens": 9358, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 0.4498686514886165}, {"method": "istm_r45", "video": "Bj7N9gscARc.mp4", "idx": 318, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8746824264526367, "peak_mem_gb": 9.402460098266602, "n_input_tokens": 3515, "n_video_tokens": 2736, "n_audio_tokens": 501, "retained_av_ratio": 0.45103490886623415}, {"method": "istm_r45", "video": "3IIaJnEspcI.mp4", "idx": 320, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.776751518249512, "peak_mem_gb": 9.915806293487549, "n_input_tokens": 6692, "n_video_tokens": 5472, "n_audio_tokens": 974, "retained_av_ratio": 0.4494260006205399}, {"method": "istm_r45", "video": "rYtG6G6tYG0.mp4", "idx": 322, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0077097415924072, "peak_mem_gb": 9.022289752960205, "n_input_tokens": 1050, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 0.5299647473560517}, {"method": "istm_r45", "video": "aneq_Adxc8U.mp4", "idx": 324, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2191166877746582, "peak_mem_gb": 9.058916091918945, "n_input_tokens": 1384, "n_video_tokens": 1008, "n_audio_tokens": 182, "retained_av_ratio": 0.47058823529411764}, {"method": "istm_r45", "video": "4u7eslydITE.mp4", "idx": 326, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.49613356590271, "peak_mem_gb": 10.490667819976807, "n_input_tokens": 10022, "n_video_tokens": 8352, "n_audio_tokens": 1472, "retained_av_ratio": 0.4495114006514658}, {"method": "istm_r45", "video": "5_tgh8PTlxk.mp4", "idx": 328, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.312595844268799, "peak_mem_gb": 9.273305892944336, "n_input_tokens": 2767, "n_video_tokens": 2160, "n_audio_tokens": 391, "retained_av_ratio": 0.45119560956487653}, {"method": "istm_r45", "video": "xyt5IrIgprc.mp4", "idx": 330, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1538641452789307, "peak_mem_gb": 9.24924087524414, "n_input_tokens": 2609, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 0.44970662196144173}, {"method": "istm_r45", "video": "huSqwxMjL-Q.mp4", "idx": 332, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3939805030822754, "peak_mem_gb": 9.118668556213379, "n_input_tokens": 1679, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 0.4563953488372093}, {"method": "istm_r45", "video": "2IvR2LGWvcg.mp4", "idx": 334, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9664742946624756, "peak_mem_gb": 9.39368486404419, "n_input_tokens": 3574, "n_video_tokens": 2880, "n_audio_tokens": 502, "retained_av_ratio": 0.4482554701360142}, {"method": "istm_r45", "video": "4WcVCJr0UwY.mp4", "idx": 336, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8686771392822266, "peak_mem_gb": 9.006868362426758, "n_input_tokens": 903, "n_video_tokens": 576, "n_audio_tokens": 127, "retained_av_ratio": 0.5817923186344239}, {"method": "istm_r45", "video": "t3YK5ezf8u4.mp4", "idx": 338, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.199153423309326, "peak_mem_gb": 9.256307125091553, "n_input_tokens": 2627, "n_video_tokens": 2016, "n_audio_tokens": 358, "retained_av_ratio": 0.44860994102780116}, {"method": "istm_r45", "video": "1pNEF7I8EIk.mp4", "idx": 340, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.201114654541016, "peak_mem_gb": 10.389559268951416, "n_input_tokens": 9747, "n_video_tokens": 8064, "n_audio_tokens": 1405, "retained_av_ratio": 0.44872742633857854}, {"method": "istm_r45", "video": "3D5kuvs3LY8.mp4", "idx": 342, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.458446741104126, "peak_mem_gb": 9.109878063201904, "n_input_tokens": 1741, "n_video_tokens": 1296, "n_audio_tokens": 248, "retained_av_ratio": 0.4540155440414508}, {"method": "istm_r45", "video": "zdvIyL1uFeY.mp4", "idx": 344, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.719876289367676, "peak_mem_gb": 9.773133277893066, "n_input_tokens": 5701, "n_video_tokens": 4608, "n_audio_tokens": 821, "retained_av_ratio": 0.45017498618530116}, {"method": "istm_r45", "video": "yB0eIpT8DBM.mp4", "idx": 346, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8984215259552002, "peak_mem_gb": 9.01064157485962, "n_input_tokens": 910, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 0.5835734870317003}, {"method": "istm_r45", "video": "MbiOwSUz4TY.mp4", "idx": 348, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3613297939300537, "peak_mem_gb": 9.084453582763672, "n_input_tokens": 1571, "n_video_tokens": 1152, "n_audio_tokens": 202, "retained_av_ratio": 0.44977843426883307}, {"method": "istm_r45", "video": "6C33lbRzaMc.mp4", "idx": 350, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.504697561264038, "peak_mem_gb": 9.521168231964111, "n_input_tokens": 4275, "n_video_tokens": 3432, "n_audio_tokens": 602, "retained_av_ratio": 0.44942984630639565}, {"method": "istm_r45", "video": "zBC5SPmMBiI.mp4", "idx": 352, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.721759796142578, "peak_mem_gb": 10.540867328643799, "n_input_tokens": 10279, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 0.4493043739365429}, {"method": "istm_r45", "video": "cI7g9nPVw8s.mp4", "idx": 354, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.842573642730713, "peak_mem_gb": 9.593721866607666, "n_input_tokens": 4674, "n_video_tokens": 3744, "n_audio_tokens": 663, "retained_av_ratio": 0.4499659632402995}, {"method": "istm_r45", "video": "V8ipyMRS5co.mp4", "idx": 356, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.924150466918945, "peak_mem_gb": 10.55671215057373, "n_input_tokens": 10451, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4494625776550636}, {"method": "istm_r45", "video": "aw0_Q6ZOx0o.mp4", "idx": 358, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5915324687957764, "peak_mem_gb": 9.148183345794678, "n_input_tokens": 1944, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 0.4499411071849234}, {"method": "istm_r45", "video": "gJjosUoZiWs.mp4", "idx": 360, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.279296398162842, "peak_mem_gb": 9.95923137664795, "n_input_tokens": 7004, "n_video_tokens": 5760, "n_audio_tokens": 1009, "retained_av_ratio": 0.44881075491209926}, {"method": "istm_r45", "video": "i2fOL059y1A.mp4", "idx": 362, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.472398281097412, "peak_mem_gb": 9.699318885803223, "n_input_tokens": 5276, "n_video_tokens": 4320, "n_audio_tokens": 753, "retained_av_ratio": 0.4496353242657205}, {"method": "istm_r45", "video": "j5wq7pLFlU4.mp4", "idx": 364, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.673261165618896, "peak_mem_gb": 10.335119247436523, "n_input_tokens": 9390, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 0.4498686514886165}, {"method": "istm_r45", "video": "nprAhUMzWag.mp4", "idx": 366, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.908919095993042, "peak_mem_gb": 10.553366661071777, "n_input_tokens": 10183, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44953486045813745}, {"method": "istm_r45", "video": "IiHvzFP6VQk.mp4", "idx": 368, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.70821237564087, "peak_mem_gb": 10.32949686050415, "n_input_tokens": 9385, "n_video_tokens": 7776, "n_audio_tokens": 1376, "retained_av_ratio": 0.4493006993006993}, {"method": "istm_r45", "video": "oT-R-2YcqRQ.mp4", "idx": 370, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.745737314224243, "peak_mem_gb": 10.511322498321533, "n_input_tokens": 10208, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.4491330059136013}, {"method": "istm_r45", "video": "hfS1Io1bQSU.mp4", "idx": 372, "problem_type": "Narrative Comprehension", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.792896270751953, "peak_mem_gb": 10.53748369216919, "n_input_tokens": 10218, "n_video_tokens": 8496, "n_audio_tokens": 1493, "retained_av_ratio": 0.4496946641305436}, {"method": "istm_r45", "video": "rPRGQOics9A.mp4", "idx": 374, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0297234058380127, "peak_mem_gb": 9.05092716217041, "n_input_tokens": 1160, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "istm_r45", "video": "sYxaFHBsxKs.mp4", "idx": 376, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.4602742195129395, "peak_mem_gb": 10.145650863647461, "n_input_tokens": 8302, "n_video_tokens": 6912, "n_audio_tokens": 1205, "retained_av_ratio": 0.44893433534557103}, {"method": "istm_r45", "video": "Ibe_srSbN28.mp4", "idx": 378, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.467052698135376, "peak_mem_gb": 9.88093376159668, "n_input_tokens": 6408, "n_video_tokens": 5184, "n_audio_tokens": 923, "retained_av_ratio": 0.44964794498116917}, {"method": "istm_r45", "video": "gRqQQYJDD5Y.mp4", "idx": 380, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6115193367004395, "peak_mem_gb": 9.503933429718018, "n_input_tokens": 4266, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.4492753623188406}, {"method": "istm_r45", "video": "T2VIyWfYZFI.mp4", "idx": 382, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.7591049671173096, "peak_mem_gb": 9.536531925201416, "n_input_tokens": 4346, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.45074920167035126}, {"method": "istm_r45", "video": "3olxUYqO8cQ.mp4", "idx": 384, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.3616249561309814, "peak_mem_gb": 9.284677982330322, "n_input_tokens": 2799, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.4493496255419787}, {"method": "istm_r45", "video": "XjZaY7vgAzg.mp4", "idx": 386, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8212320804595947, "peak_mem_gb": 9.189539432525635, "n_input_tokens": 2247, "n_video_tokens": 1728, "n_audio_tokens": 320, "retained_av_ratio": 0.4521484375}, {"method": "istm_r45", "video": "EPzdZd7DnsQ.mp4", "idx": 388, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6112747192382812, "peak_mem_gb": 9.327308654785156, "n_input_tokens": 3105, "n_video_tokens": 2448, "n_audio_tokens": 439, "retained_av_ratio": 0.45064080360235537}, {"method": "istm_r45", "video": "xwfHWw5A2HI.mp4", "idx": 390, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.040867805480957, "peak_mem_gb": 10.55726432800293, "n_input_tokens": 10320, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "istm_r45", "video": "otVWxaBSzL0.mp4", "idx": 392, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8090074062347412, "peak_mem_gb": 9.184988021850586, "n_input_tokens": 2231, "n_video_tokens": 1728, "n_audio_tokens": 319, "retained_av_ratio": 0.4528578407425501}, {"method": "istm_r45", "video": "s6C8atMqM1Q.mp4", "idx": 394, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5176944732666016, "peak_mem_gb": 9.520901679992676, "n_input_tokens": 4291, "n_video_tokens": 3456, "n_audio_tokens": 614, "retained_av_ratio": 0.4498771498771499}, {"method": "istm_r45", "video": "xlC9oqCHQ84.mp4", "idx": 396, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4805006980895996, "peak_mem_gb": 9.107530117034912, "n_input_tokens": 1733, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 0.4516971279373368}, {"method": "istm_r45", "video": "wR4Za_pz2ec.mp4", "idx": 398, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5535907745361328, "peak_mem_gb": 9.16325855255127, "n_input_tokens": 1992, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 0.4532710280373832}, {"method": "istm_r45", "video": "Nql8w5AYwAo.mp4", "idx": 400, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9814283847808838, "peak_mem_gb": 9.222397804260254, "n_input_tokens": 2436, "n_video_tokens": 1872, "n_audio_tokens": 347, "retained_av_ratio": 0.45290671473636773}, {"method": "istm_r45", "video": "1t0JN_QdBkA.mp4", "idx": 402, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.565362215042114, "peak_mem_gb": 10.149942398071289, "n_input_tokens": 8319, "n_video_tokens": 6912, "n_audio_tokens": 1209, "retained_av_ratio": 0.4489594877478143}, {"method": "istm_r45", "video": "-dtEpy4XUbQ.mp4", "idx": 404, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9800753593444824, "peak_mem_gb": 9.002117156982422, "n_input_tokens": 1042, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.4652532391048292}, {"method": "istm_r45", "video": "i9LSUR1OSjs.mp4", "idx": 406, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1859462261199951, "peak_mem_gb": 9.08481502532959, "n_input_tokens": 1472, "n_video_tokens": 1008, "n_audio_tokens": 183, "retained_av_ratio": 0.4643157010915197}, {"method": "istm_r45", "video": "F7M7aY5fofo.mp4", "idx": 408, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0291812419891357, "peak_mem_gb": 9.045197010040283, "n_input_tokens": 1126, "n_video_tokens": 720, "n_audio_tokens": 145, "retained_av_ratio": 0.5329479768786127}, {"method": "istm_r45", "video": "eMPOMRBnBpc.mp4", "idx": 410, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.924729824066162, "peak_mem_gb": 9.205080032348633, "n_input_tokens": 2294, "n_video_tokens": 1728, "n_audio_tokens": 318, "retained_av_ratio": 0.4521016617790811}, {"method": "istm_r45", "video": "bSppSU7Njig.mp4", "idx": 412, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.888289451599121, "peak_mem_gb": 10.557605743408203, "n_input_tokens": 10409, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "istm_r45", "video": "72rcNxQbIoI.mp4", "idx": 414, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1001620292663574, "peak_mem_gb": 9.025853633880615, "n_input_tokens": 1108, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.4856156501726122}, {"method": "istm_r45", "video": "jl38NDYzQV0.mp4", "idx": 416, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.235199451446533, "peak_mem_gb": 9.46372652053833, "n_input_tokens": 3966, "n_video_tokens": 3168, "n_audio_tokens": 564, "retained_av_ratio": 0.44935691318327975}, {"method": "istm_r45", "video": "V30jgIuBG94.mp4", "idx": 418, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.611414909362793, "peak_mem_gb": 9.127382278442383, "n_input_tokens": 1881, "n_video_tokens": 1440, "n_audio_tokens": 261, "retained_av_ratio": 0.4503233392122281}, {"method": "istm_r45", "video": "9XbP_d_u_OI.mp4", "idx": 420, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.103671312332153, "peak_mem_gb": 9.950849533081055, "n_input_tokens": 6888, "n_video_tokens": 5616, "n_audio_tokens": 983, "retained_av_ratio": 0.4494620397029853}, {"method": "istm_r45", "video": "FP7u3iJxruw.mp4", "idx": 422, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.064376831054688, "peak_mem_gb": 10.564021587371826, "n_input_tokens": 10430, "n_video_tokens": 8640, "n_audio_tokens": 1505, "retained_av_ratio": 0.4487925086249384}, {"method": "istm_r45", "video": "DWqcf5QvkJI.mp4", "idx": 424, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8767085075378418, "peak_mem_gb": 9.224420547485352, "n_input_tokens": 2353, "n_video_tokens": 1728, "n_audio_tokens": 312, "retained_av_ratio": 0.45098039215686275}, {"method": "istm_r45", "video": "P_cYNsKKpGg.mp4", "idx": 426, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.811401128768921, "peak_mem_gb": 9.19311237335205, "n_input_tokens": 2253, "n_video_tokens": 1728, "n_audio_tokens": 301, "retained_av_ratio": 0.4494825036964022}, {"method": "istm_r45", "video": "J01MxqCqkyE.mp4", "idx": 428, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7891831398010254, "peak_mem_gb": 9.360069751739502, "n_input_tokens": 3381, "n_video_tokens": 2736, "n_audio_tokens": 486, "retained_av_ratio": 0.4500310366232154}, {"method": "istm_r45", "video": "wdESby14IZY.mp4", "idx": 430, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.20640230178833, "peak_mem_gb": 10.112696170806885, "n_input_tokens": 8023, "n_video_tokens": 6624, "n_audio_tokens": 1163, "retained_av_ratio": 0.44921022216514706}, {"method": "istm_r45", "video": "n7vwcWG5ZY0.mp4", "idx": 432, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.791720151901245, "peak_mem_gb": 10.544877529144287, "n_input_tokens": 10229, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.4499149404583208}, {"method": "istm_r45", "video": "18OOnHDat4E.mp4", "idx": 434, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.665565013885498, "peak_mem_gb": 9.486514568328857, "n_input_tokens": 4122, "n_video_tokens": 3312, "n_audio_tokens": 580, "retained_av_ratio": 0.4486125385405961}, {"method": "istm_r45", "video": "w6C29ECeFs8.mp4", "idx": 436, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.025571584701538, "peak_mem_gb": 9.02807331085205, "n_input_tokens": 1088, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "istm_r45", "video": "JB8rAfaUUKY.mp4", "idx": 438, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6882901191711426, "peak_mem_gb": 9.358439922332764, "n_input_tokens": 3286, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 0.45091623036649214}, {"method": "istm_r45", "video": "1YORkMT9JoY.mp4", "idx": 440, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1578834056854248, "peak_mem_gb": 9.056334495544434, "n_input_tokens": 1262, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49951028403525954}, {"method": "istm_r45", "video": "D980E_PsoIs.mp4", "idx": 442, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4946401119232178, "peak_mem_gb": 9.329346179962158, "n_input_tokens": 3026, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 0.45137614678899085}, {"method": "istm_r45", "video": "nGHKrbtPRb4.mp4", "idx": 444, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6840295791625977, "peak_mem_gb": 9.35426378250122, "n_input_tokens": 3273, "n_video_tokens": 2592, "n_audio_tokens": 458, "retained_av_ratio": 0.4498360655737705}, {"method": "istm_r45", "video": "AmWG0oYhDh4.mp4", "idx": 446, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.958664417266846, "peak_mem_gb": 10.363263130187988, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1383, "retained_av_ratio": 0.44931742448672474}, {"method": "istm_r45", "video": "gmKpD8QPvSA.mp4", "idx": 448, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4414539337158203, "peak_mem_gb": 9.112365245819092, "n_input_tokens": 1746, "n_video_tokens": 1296, "n_audio_tokens": 232, "retained_av_ratio": 0.450261780104712}, {"method": "istm_r45", "video": "Wl8UjlAedeQ.mp4", "idx": 450, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7792201042175293, "peak_mem_gb": 9.18752145767212, "n_input_tokens": 2148, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45221569674319273}, {"method": "istm_r45", "video": "dEp4CVa18-s.mp4", "idx": 452, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6103219985961914, "peak_mem_gb": 9.316384315490723, "n_input_tokens": 3072, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 0.44857539958304377}, {"method": "istm_r45", "video": "aWo90tAPcBs.mp4", "idx": 454, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.580785274505615, "peak_mem_gb": 9.879369735717773, "n_input_tokens": 6494, "n_video_tokens": 5328, "n_audio_tokens": 951, "retained_av_ratio": 0.44991240643414554}, {"method": "istm_r45", "video": "XHdk-W9L9RQ.mp4", "idx": 456, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0017645359039307, "peak_mem_gb": 9.00817060470581, "n_input_tokens": 1053, "n_video_tokens": 720, "n_audio_tokens": 135, "retained_av_ratio": 0.4783625730994152}, {"method": "istm_r45", "video": "1SjR4zREgq8.mp4", "idx": 458, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0386056900024414, "peak_mem_gb": 9.041972160339355, "n_input_tokens": 1115, "n_video_tokens": 720, "n_audio_tokens": 132, "retained_av_ratio": 0.5269953051643192}, {"method": "istm_r45", "video": "mVJ32Ex_gqk.mp4", "idx": 460, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.176367998123169, "peak_mem_gb": 9.982984066009521, "n_input_tokens": 7168, "n_video_tokens": 5904, "n_audio_tokens": 1042, "retained_av_ratio": 0.4498992225741434}, {"method": "istm_r45", "video": "RFm_F9hS0Y8.mp4", "idx": 462, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1376864910125732, "peak_mem_gb": 9.044571876525879, "n_input_tokens": 1226, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.49606299212598426}, {"method": "istm_r45", "video": "H3Mg7I8ElNo.mp4", "idx": 464, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0435736179351807, "peak_mem_gb": 9.024820804595947, "n_input_tokens": 1059, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.5276796230859835}, {"method": "istm_r45", "video": "RYQlChSooWM.mp4", "idx": 466, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.095059156417847, "peak_mem_gb": 10.257105350494385, "n_input_tokens": 8887, "n_video_tokens": 7344, "n_audio_tokens": 1289, "retained_av_ratio": 0.4490906984825669}, {"method": "istm_r45", "video": "b0x6z9gDzHQ.mp4", "idx": 468, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.111784219741821, "peak_mem_gb": 10.103671073913574, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 0.4496342871808033}, {"method": "istm_r45", "video": "KwPGlGTOBkU.mp4", "idx": 470, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.420015573501587, "peak_mem_gb": 9.115986347198486, "n_input_tokens": 1668, "n_video_tokens": 1152, "n_audio_tokens": 203, "retained_av_ratio": 0.45018450184501846}, {"method": "istm_r45", "video": "6X1IEKFL33Y.mp4", "idx": 472, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.674858331680298, "peak_mem_gb": 10.223778247833252, "n_input_tokens": 8816, "n_video_tokens": 7344, "n_audio_tokens": 1278, "retained_av_ratio": 0.44873579215959175}, {"method": "istm_r45", "video": "YWhL_jNOzVc.mp4", "idx": 474, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2835586071014404, "peak_mem_gb": 9.08158826828003, "n_input_tokens": 1565, "n_video_tokens": 1152, "n_audio_tokens": 217, "retained_av_ratio": 0.45361577794010227}, {"method": "istm_r45", "video": "OY-uPQ4oLKM.mp4", "idx": 476, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.30946135520935, "peak_mem_gb": 11.4266939163208, "n_input_tokens": 11296, "n_video_tokens": 9216, "n_audio_tokens": 1866, "retained_av_ratio": 0.4552427359682368}, {"method": "istm_r45", "video": "3Y2mE1Oph6A.mp4", "idx": 478, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.869629859924316, "peak_mem_gb": 9.895129680633545, "n_input_tokens": 6620, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 0.44952558718307667}, {"method": "istm_r45", "video": "ZbsjWDzPPx4.mp4", "idx": 480, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0020432472229004, "peak_mem_gb": 9.021700382232666, "n_input_tokens": 1090, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.4872389791183295}, {"method": "istm_r45", "video": "jtFI4H55Clw.mp4", "idx": 482, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.159409999847412, "peak_mem_gb": 9.63200330734253, "n_input_tokens": 4793, "n_video_tokens": 3888, "n_audio_tokens": 693, "retained_av_ratio": 0.449028596376337}, {"method": "istm_r45", "video": "QD1l3iZ2nZs.mp4", "idx": 484, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.313581943511963, "peak_mem_gb": 10.280664920806885, "n_input_tokens": 9145, "n_video_tokens": 7632, "n_audio_tokens": 1337, "retained_av_ratio": 0.4487679785929312}, {"method": "istm_r45", "video": "qPibP0Y6HUU.mp4", "idx": 486, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.839847326278687, "peak_mem_gb": 10.612367630004883, "n_input_tokens": 10494, "n_video_tokens": 8784, "n_audio_tokens": 1527, "retained_av_ratio": 0.44893802734943267}, {"method": "istm_r45", "video": "AOkiHqncwF8.mp4", "idx": 488, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.873991012573242, "peak_mem_gb": 9.546356678009033, "n_input_tokens": 4475, "n_video_tokens": 3600, "n_audio_tokens": 651, "retained_av_ratio": 0.450717478240414}, {"method": "istm_r45", "video": "FPBSzdA1oKU.mp4", "idx": 490, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.210843563079834, "peak_mem_gb": 9.06540060043335, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 180, "retained_av_ratio": 0.46885521885521886}, {"method": "istm_r45", "video": "6RNtg0cLgZk.mp4", "idx": 492, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.618663787841797, "peak_mem_gb": 10.816018104553223, "n_input_tokens": 11041, "n_video_tokens": 9216, "n_audio_tokens": 1616, "retained_av_ratio": 0.4494091580502216}, {"method": "istm_r45", "video": "0avoXj7I3yA.mp4", "idx": 494, "problem_type": "Narrative Comprehension", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.72 GiB. GPU 0 has a total capacity of 14.56 GiB of which 480.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.986373901367188, "n_input_tokens": 12512, "n_video_tokens": 9216, "n_audio_tokens": 3020, "retained_av_ratio": null}, {"method": "istm_r45", "video": "DcyrFhN6ncE.mp4", "idx": 496, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.795781373977661, "peak_mem_gb": 10.257698059082031, "n_input_tokens": 8889, "n_video_tokens": 7344, "n_audio_tokens": 1281, "retained_av_ratio": 0.44857971014492753}, {"method": "istm_r45", "video": "HAh-h5nuzhI.mp4", "idx": 498, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.7610585689544678, "peak_mem_gb": 9.587649822235107, "n_input_tokens": 4655, "n_video_tokens": 3744, "n_audio_tokens": 670, "retained_av_ratio": 0.4506116900770276}, {"method": "istm_r45", "video": "Hre27BhZvM8.mp4", "idx": 1, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.736160755157471, "peak_mem_gb": 10.277089595794678, "n_input_tokens": 8959, "n_video_tokens": 7344, "n_audio_tokens": 1299, "retained_av_ratio": 0.44938100196690967}, {"method": "istm_r45", "video": "f7YdwVUy0ss.mp4", "idx": 3, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0743529796600342, "peak_mem_gb": 9.06082820892334, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 0.4513422818791946}, {"method": "istm_r45", "video": "EgHDY0VpFS0.mp4", "idx": 5, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.7019872665405273, "peak_mem_gb": 9.489328384399414, "n_input_tokens": 4037, "n_video_tokens": 3168, "n_audio_tokens": 565, "retained_av_ratio": 0.4505759442807393}, {"method": "istm_r45", "video": "egcR6-Dq0Tc.mp4", "idx": 7, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.837353944778442, "peak_mem_gb": 10.557397842407227, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "istm_r45", "video": "Bk60US4MacA.mp4", "idx": 9, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.243904113769531, "peak_mem_gb": 10.410010814666748, "n_input_tokens": 9895, "n_video_tokens": 8208, "n_audio_tokens": 1427, "retained_av_ratio": 0.4485729112610275}, {"method": "istm_r45", "video": "ft_pDeYZA6E.mp4", "idx": 11, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3784985542297363, "peak_mem_gb": 9.165887832641602, "n_input_tokens": 2084, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 0.4523682810005322}, {"method": "istm_r45", "video": "TJR8ibF26vw.mp4", "idx": 13, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3804514408111572, "peak_mem_gb": 9.648540019989014, "n_input_tokens": 4872, "n_video_tokens": 3920, "n_audio_tokens": 723, "retained_av_ratio": 0.45143226362265776}, {"method": "istm_r45", "video": "Qps4jZVasg4.mp4", "idx": 15, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1843295097351074, "peak_mem_gb": 9.096837043762207, "n_input_tokens": 1605, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 0.45594713656387664}, {"method": "istm_r45", "video": "lUEjk9CVcEY.mp4", "idx": 17, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.86395263671875, "peak_mem_gb": 9.265458583831787, "n_input_tokens": 2740, "n_video_tokens": 2160, "n_audio_tokens": 385, "retained_av_ratio": 0.4506876227897839}, {"method": "istm_r45", "video": "cdOd3EXqagY.mp4", "idx": 19, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.030500411987305, "peak_mem_gb": 10.39179277420044, "n_input_tokens": 9664, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 0.44978396037517127}, {"method": "istm_r45", "video": "Z6U4PbCGiP0.mp4", "idx": 21, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.891078472137451, "peak_mem_gb": 9.527192115783691, "n_input_tokens": 4316, "n_video_tokens": 3456, "n_audio_tokens": 624, "retained_av_ratio": 0.4504901960784314}, {"method": "istm_r45", "video": "NaDxBMFiiNQ.mp4", "idx": 23, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0769016742706299, "peak_mem_gb": 9.087089538574219, "n_input_tokens": 1570, "n_video_tokens": 1152, "n_audio_tokens": 211, "retained_av_ratio": 0.4600146735143067}, {"method": "istm_r45", "video": "UxlWLoM3H-8.mp4", "idx": 25, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.600395917892456, "peak_mem_gb": 10.49581527709961, "n_input_tokens": 10014, "n_video_tokens": 8352, "n_audio_tokens": 1474, "retained_av_ratio": 0.44982698961937717}, {"method": "istm_r45", "video": "_mYtKUIalEo.mp4", "idx": 27, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2053778171539307, "peak_mem_gb": 9.354033470153809, "n_input_tokens": 3275, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 0.4505386875612145}, {"method": "istm_r45", "video": "P9jqFr3TvSE.mp4", "idx": 29, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.820239305496216, "peak_mem_gb": 10.228647232055664, "n_input_tokens": 8742, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 0.4500472366556448}, {"method": "istm_r45", "video": "X1c3Z8eAhwI.mp4", "idx": 31, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.44168758392334, "peak_mem_gb": 9.87455415725708, "n_input_tokens": 6473, "n_video_tokens": 5328, "n_audio_tokens": 946, "retained_av_ratio": 0.45011157156518966}, {"method": "istm_r45", "video": "O7EgKJ5Km7Y.mp4", "idx": 33, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.874725580215454, "peak_mem_gb": 10.55742073059082, "n_input_tokens": 10361, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4488266614080063}, {"method": "istm_r45", "video": "xlO7zTTgtHg.mp4", "idx": 35, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.117915391921997, "peak_mem_gb": 9.065770149230957, "n_input_tokens": 1428, "n_video_tokens": 1008, "n_audio_tokens": 194, "retained_av_ratio": 0.4550748752079867}, {"method": "istm_r45", "video": "52smiyrFybw.mp4", "idx": 37, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7573399543762207, "peak_mem_gb": 8.999790668487549, "n_input_tokens": 874, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 0.5752212389380531}, {"method": "istm_r45", "video": "_PzDR-Nf-e8.mp4", "idx": 39, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.895740270614624, "peak_mem_gb": 9.062152862548828, "n_input_tokens": 1420, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.45439469320066334}, {"method": "istm_r45", "video": "aiVuez4WsBo.mp4", "idx": 41, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5900835990905762, "peak_mem_gb": 9.18974781036377, "n_input_tokens": 2220, "n_video_tokens": 1680, "n_audio_tokens": 324, "retained_av_ratio": 0.4555888223552894}, {"method": "istm_r45", "video": "3o2KqGljhbQ.mp4", "idx": 43, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.456361532211304, "peak_mem_gb": 10.463076114654541, "n_input_tokens": 10078, "n_video_tokens": 8352, "n_audio_tokens": 1458, "retained_av_ratio": 0.4487257900101937}, {"method": "istm_r45", "video": "TIxlSCV6E_A.mp4", "idx": 45, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8278131484985352, "peak_mem_gb": 9.007154941558838, "n_input_tokens": 902, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 0.5778097982708934}, {"method": "istm_r45", "video": "2dnuTGnFZyo.mp4", "idx": 47, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9900543689727783, "peak_mem_gb": 9.045610904693604, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 174, "retained_av_ratio": 0.5}, {"method": "istm_r45", "video": "9ngLXMGF90I.mp4", "idx": 49, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.238218069076538, "peak_mem_gb": 9.374741077423096, "n_input_tokens": 3426, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 0.45142326732673266}, {"method": "istm_r45", "video": "BPpr-5KW5BQ.mp4", "idx": 51, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.8142523765563965, "peak_mem_gb": 10.532127380371094, "n_input_tokens": 10188, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.4494843296285171}, {"method": "istm_r45", "video": "m_haCHnvhj4.mp4", "idx": 53, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.949629545211792, "peak_mem_gb": 9.040441036224365, "n_input_tokens": 1114, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5327963176064442}, {"method": "istm_r45", "video": "xrOu-TB6osc.mp4", "idx": 55, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8571934700012207, "peak_mem_gb": 9.035278797149658, "n_input_tokens": 989, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "istm_r45", "video": "9IGdrQU77pk.mp4", "idx": 57, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3872671127319336, "peak_mem_gb": 9.122539520263672, "n_input_tokens": 1781, "n_video_tokens": 1296, "n_audio_tokens": 251, "retained_av_ratio": 0.4557207498383969}, {"method": "istm_r45", "video": "wPWguBt2hhs.mp4", "idx": 59, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.939861536026001, "peak_mem_gb": 9.019089221954346, "n_input_tokens": 1044, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.525383707201889}, {"method": "istm_r45", "video": "7moy-Foct8U.mp4", "idx": 61, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.9673736095428467, "peak_mem_gb": 9.775559902191162, "n_input_tokens": 5801, "n_video_tokens": 4752, "n_audio_tokens": 845, "retained_av_ratio": 0.4491691977845274}, {"method": "istm_r45", "video": "GSLBOuwOdRE.mp4", "idx": 63, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.241551399230957, "peak_mem_gb": 11.646289825439453, "n_input_tokens": 11393, "n_video_tokens": 9216, "n_audio_tokens": 1948, "retained_av_ratio": 0.45834826227158726}, {"method": "istm_r45", "video": "LXXmslrnPrg.mp4", "idx": 65, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7924008369445801, "peak_mem_gb": 9.0059814453125, "n_input_tokens": 1051, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "istm_r45", "video": "oPhbjcl7TOg.mp4", "idx": 67, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.199537754058838, "peak_mem_gb": 9.335229396820068, "n_input_tokens": 3132, "n_video_tokens": 2448, "n_audio_tokens": 451, "retained_av_ratio": 0.4525698516729907}, {"method": "istm_r45", "video": "iTdOOLn-ICM.mp4", "idx": 69, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8593449592590332, "peak_mem_gb": 9.250942707061768, "n_input_tokens": 2611, "n_video_tokens": 2016, "n_audio_tokens": 369, "retained_av_ratio": 0.4511530398322851}, {"method": "istm_r45", "video": "CpeBZFxOeqQ.mp4", "idx": 71, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.32649564743042, "peak_mem_gb": 9.128870964050293, "n_input_tokens": 1862, "n_video_tokens": 1400, "n_audio_tokens": 273, "retained_av_ratio": 0.4560669456066946}, {"method": "istm_r45", "video": "B7g9hmHAUzo.mp4", "idx": 73, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.4125871658325195, "peak_mem_gb": 10.008424282073975, "n_input_tokens": 7326, "n_video_tokens": 6048, "n_audio_tokens": 1059, "retained_av_ratio": 0.4496974813564092}, {"method": "istm_r45", "video": "tUucQrJAzsI.mp4", "idx": 75, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0803136825561523, "peak_mem_gb": 9.313063621520996, "n_input_tokens": 3063, "n_video_tokens": 2448, "n_audio_tokens": 443, "retained_av_ratio": 0.4510549982704946}, {"method": "istm_r45", "video": "TA6OBcjG1Oc.mp4", "idx": 77, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.77858304977417, "peak_mem_gb": 10.557226181030273, "n_input_tokens": 10310, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "istm_r45", "video": "CdwdC6E8Cco.mp4", "idx": 79, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0583689212799072, "peak_mem_gb": 9.291515350341797, "n_input_tokens": 2910, "n_video_tokens": 2304, "n_audio_tokens": 409, "retained_av_ratio": 0.44968669369701436}, {"method": "istm_r45", "video": "d-wkkuvKoGM.mp4", "idx": 81, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.033346652984619, "peak_mem_gb": 9.506208419799805, "n_input_tokens": 4268, "n_video_tokens": 3456, "n_audio_tokens": 609, "retained_av_ratio": 0.44969249692496926}, {"method": "istm_r45", "video": "i_mCozcHi9o.mp4", "idx": 83, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.280677080154419, "peak_mem_gb": 9.09642744064331, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 0.45890909090909093}, {"method": "istm_r45", "video": "g_RmJp0IgkY.mp4", "idx": 85, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9726123809814453, "peak_mem_gb": 9.038536071777344, "n_input_tokens": 1122, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "istm_r45", "video": "GOg9dmnHSSw.mp4", "idx": 87, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.161570072174072, "peak_mem_gb": 10.338329315185547, "n_input_tokens": 9500, "n_video_tokens": 7920, "n_audio_tokens": 1393, "retained_av_ratio": 0.44937184580693657}, {"method": "istm_r45", "video": "-DxPwypTfKI.mp4", "idx": 89, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.59171724319458, "peak_mem_gb": 10.468195915222168, "n_input_tokens": 10075, "n_video_tokens": 8352, "n_audio_tokens": 1461, "retained_av_ratio": 0.4489962294914909}, {"method": "istm_r45", "video": "l7QiQSivnAc.mp4", "idx": 91, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.2335076332092285, "peak_mem_gb": 9.355889797210693, "n_input_tokens": 3373, "n_video_tokens": 2736, "n_audio_tokens": 499, "retained_av_ratio": 0.45038639876352393}, {"method": "istm_r45", "video": "T-PGGc1OJcM.mp4", "idx": 93, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8981297016143799, "peak_mem_gb": 9.041763305664062, "n_input_tokens": 1224, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.49513618677042803}, {"method": "istm_r45", "video": "k4o3XkT0jbM.mp4", "idx": 95, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.255526065826416, "peak_mem_gb": 10.1772780418396, "n_input_tokens": 8485, "n_video_tokens": 7056, "n_audio_tokens": 1227, "retained_av_ratio": 0.44923336955209464}, {"method": "istm_r45", "video": "baTfh9Ryxoo.mp4", "idx": 97, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5426366329193115, "peak_mem_gb": 9.676557540893555, "n_input_tokens": 5113, "n_video_tokens": 4176, "n_audio_tokens": 749, "retained_av_ratio": 0.4511675126903553}, {"method": "istm_r45", "video": "CcZr-QjAvTQ.mp4", "idx": 99, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9846351146697998, "peak_mem_gb": 9.052208423614502, "n_input_tokens": 1255, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49363369245837413}, {"method": "istm_r45", "video": "-0J1uWKJKfk.mp4", "idx": 101, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8727023601531982, "peak_mem_gb": 9.050243377685547, "n_input_tokens": 1146, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.5267441860465116}, {"method": "istm_r45", "video": "4WgA39tgBSo.mp4", "idx": 103, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.499781370162964, "peak_mem_gb": 9.400065422058105, "n_input_tokens": 3596, "n_video_tokens": 2880, "n_audio_tokens": 504, "retained_av_ratio": 0.44799054373522457}, {"method": "istm_r45", "video": "QUC2d9FSkcU.mp4", "idx": 105, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.050562620162964, "peak_mem_gb": 9.29761791229248, "n_input_tokens": 2930, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 0.45073529411764707}, {"method": "istm_r45", "video": "uGoKOpZOSUg.mp4", "idx": 107, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 6.04 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.513656616210938, "n_input_tokens": 14039, "n_video_tokens": 9216, "n_audio_tokens": 4501, "retained_av_ratio": null}, {"method": "istm_r45", "video": "MY7puORABAk.mp4", "idx": 109, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0292069911956787, "peak_mem_gb": 9.319489479064941, "n_input_tokens": 3079, "n_video_tokens": 2448, "n_audio_tokens": 436, "retained_av_ratio": 0.4511095700416089}, {"method": "istm_r45", "video": "yl79pxDOjJ0.mp4", "idx": 111, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.955244779586792, "peak_mem_gb": 9.791065692901611, "n_input_tokens": 5843, "n_video_tokens": 4752, "n_audio_tokens": 831, "retained_av_ratio": 0.4493999641769658}, {"method": "istm_r45", "video": "g8JpgtFpciY.mp4", "idx": 113, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.516616106033325, "peak_mem_gb": 10.06151294708252, "n_input_tokens": 7682, "n_video_tokens": 6336, "n_audio_tokens": 1120, "retained_av_ratio": 0.44957081545064376}, {"method": "istm_r45", "video": "1Q3LeFo88xI.mp4", "idx": 115, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.891844749450684, "peak_mem_gb": 10.567143440246582, "n_input_tokens": 10417, "n_video_tokens": 8640, "n_audio_tokens": 1506, "retained_av_ratio": 0.44835403114527894}, {"method": "istm_r45", "video": "HdwFA-Hb0pQ.mp4", "idx": 117, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8976945877075195, "peak_mem_gb": 9.017394065856934, "n_input_tokens": 1086, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.48088064889918886}, {"method": "istm_r45", "video": "GFrTD7uV8mA.mp4", "idx": 119, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8188190460205078, "peak_mem_gb": 9.024575233459473, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.532183908045977}, {"method": "istm_r45", "video": "Dawm3vWn97A.mp4", "idx": 121, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.356771469116211, "peak_mem_gb": 9.152956485748291, "n_input_tokens": 1962, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 0.4536443148688047}, {"method": "istm_r45", "video": "BQc182PZbeU.mp4", "idx": 123, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.340593338012695, "peak_mem_gb": 10.018795490264893, "n_input_tokens": 7363, "n_video_tokens": 6048, "n_audio_tokens": 1069, "retained_av_ratio": 0.44976816074188564}, {"method": "istm_r45", "video": "tjXVd3qBMhw.mp4", "idx": 125, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9614758491516113, "peak_mem_gb": 9.032220363616943, "n_input_tokens": 1195, "n_video_tokens": 864, "n_audio_tokens": 159, "retained_av_ratio": 0.4916911045943304}, {"method": "istm_r45", "video": "YGsvvmBiRHA.mp4", "idx": 127, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8568296432495117, "peak_mem_gb": 9.27749252319336, "n_input_tokens": 2781, "n_video_tokens": 2160, "n_audio_tokens": 386, "retained_av_ratio": 0.4493322859387274}, {"method": "istm_r45", "video": "5yssh2Aqtps.mp4", "idx": 129, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.80031943321228, "peak_mem_gb": 10.512125492095947, "n_input_tokens": 10146, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.4491330059136013}, {"method": "istm_r45", "video": "auFMvAywvbk.mp4", "idx": 131, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8300740718841553, "peak_mem_gb": 9.27235221862793, "n_input_tokens": 2757, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.4505321245565629}, {"method": "istm_r45", "video": "qTX0XvwA0qU.mp4", "idx": 133, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9297866821289062, "peak_mem_gb": 9.0389084815979, "n_input_tokens": 1216, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4946550048590865}, {"method": "istm_r45", "video": "X2ZDXPulIy0.mp4", "idx": 135, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8069319725036621, "peak_mem_gb": 9.01637887954712, "n_input_tokens": 1084, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.46989374262101535}, {"method": "istm_r45", "video": "qHakVESEOec.mp4", "idx": 137, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2134015560150146, "peak_mem_gb": 9.091484546661377, "n_input_tokens": 1587, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 0.45890909090909093}, {"method": "istm_r45", "video": "StG5yDCruTw.mp4", "idx": 139, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.748234987258911, "peak_mem_gb": 9.495290756225586, "n_input_tokens": 4150, "n_video_tokens": 3312, "n_audio_tokens": 586, "retained_av_ratio": 0.4492047203694202}, {"method": "istm_r45", "video": "6kpIdgWGRAg.mp4", "idx": 141, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9092738628387451, "peak_mem_gb": 9.014867305755615, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.49653579676674364}, {"method": "istm_r45", "video": "Z0YlK_pGTMQ.mp4", "idx": 143, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9232559204101562, "peak_mem_gb": 9.01742172241211, "n_input_tokens": 1047, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.5298850574712644}, {"method": "istm_r45", "video": "ZJyYchnUJL4.mp4", "idx": 145, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8705096244812012, "peak_mem_gb": 9.019801139831543, "n_input_tokens": 1093, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "istm_r45", "video": "7uxn4eM979g.mp4", "idx": 147, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8526246547698975, "peak_mem_gb": 9.039445877075195, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 138, "retained_av_ratio": 0.5314685314685315}, {"method": "istm_r45", "video": "GO1FuCX8YHI.mp4", "idx": 149, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.757344007492065, "peak_mem_gb": 10.291738986968994, "n_input_tokens": 9176, "n_video_tokens": 7632, "n_audio_tokens": 1335, "retained_av_ratio": 0.4488680718188915}, {"method": "istm_r45", "video": "tQCM5mtqkeo.mp4", "idx": 151, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.448373079299927, "peak_mem_gb": 10.433043003082275, "n_input_tokens": 9825, "n_video_tokens": 8208, "n_audio_tokens": 1445, "retained_av_ratio": 0.44949756552367137}, {"method": "istm_r45", "video": "OYIX_zxxS9Q.mp4", "idx": 153, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.109370470046997, "peak_mem_gb": 9.54296350479126, "n_input_tokens": 4457, "n_video_tokens": 3600, "n_audio_tokens": 641, "retained_av_ratio": 0.45036547983966047}, {"method": "istm_r45", "video": "GZ_pUlsnTGQ.mp4", "idx": 155, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.919680118560791, "peak_mem_gb": 9.030990600585938, "n_input_tokens": 1196, "n_video_tokens": 864, "n_audio_tokens": 169, "retained_av_ratio": 0.49177153920619554}, {"method": "istm_r45", "video": "VyHYgZamj58.mp4", "idx": 157, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.6512470245361328, "peak_mem_gb": 9.0093994140625, "n_input_tokens": 905, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5871428571428572}, {"method": "istm_r45", "video": "CQHAoYgQLis.mp4", "idx": 159, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.2190940380096436, "peak_mem_gb": 9.568448543548584, "n_input_tokens": 4601, "n_video_tokens": 3744, "n_audio_tokens": 659, "retained_av_ratio": 0.4483306836248013}, {"method": "istm_r45", "video": "ySiKjxAnE4I.mp4", "idx": 161, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.294612884521484, "peak_mem_gb": 10.391505241394043, "n_input_tokens": 9711, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 0.44978396037517127}, {"method": "istm_r45", "video": "9xWVEqxUJGY.mp4", "idx": 163, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9821977615356445, "peak_mem_gb": 9.039540767669678, "n_input_tokens": 1221, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.4912451361867704}, {"method": "istm_r45", "video": "egkxfFQ_C7g.mp4", "idx": 165, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4474232196807861, "peak_mem_gb": 9.153939247131348, "n_input_tokens": 1957, "n_video_tokens": 1440, "n_audio_tokens": 260, "retained_av_ratio": 0.45235294117647057}, {"method": "istm_r45", "video": "U-R7VXme0ps.mp4", "idx": 167, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4777460098266602, "peak_mem_gb": 9.14611530303955, "n_input_tokens": 1939, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.45353594389246055}, {"method": "istm_r45", "video": "OCRdFBuuW2M.mp4", "idx": 169, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.7661900520324707, "peak_mem_gb": 9.755118370056152, "n_input_tokens": 5554, "n_video_tokens": 4464, "n_audio_tokens": 791, "retained_av_ratio": 0.449476688867745}, {"method": "istm_r45", "video": "5yUY-afqZSs.mp4", "idx": 171, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.786356210708618, "peak_mem_gb": 10.532022476196289, "n_input_tokens": 10161, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.4494843296285171}, {"method": "istm_r45", "video": "T0p4FtF99l0.mp4", "idx": 173, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9201476573944092, "peak_mem_gb": 9.039465427398682, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.5341830822711472}, {"method": "istm_r45", "video": "2tVZFAqM7lw.mp4", "idx": 175, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4750442504882812, "peak_mem_gb": 9.38771390914917, "n_input_tokens": 3555, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 0.4498525073746313}, {"method": "istm_r45", "video": "uBXycVLybjY.mp4", "idx": 177, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.29231595993042, "peak_mem_gb": 10.426908493041992, "n_input_tokens": 9806, "n_video_tokens": 8208, "n_audio_tokens": 1442, "retained_av_ratio": 0.44963730569948185}, {"method": "istm_r45", "video": "JiRQff5ADps.mp4", "idx": 179, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.882171392440796, "peak_mem_gb": 9.75663423538208, "n_input_tokens": 5661, "n_video_tokens": 4620, "n_audio_tokens": 845, "retained_av_ratio": 0.45032021957914}, {"method": "istm_r45", "video": "tNn-m_qLsvo.mp4", "idx": 181, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1302928924560547, "peak_mem_gb": 9.094379425048828, "n_input_tokens": 1598, "n_video_tokens": 1152, "n_audio_tokens": 221, "retained_av_ratio": 0.4573925710123816}, {"method": "istm_r45", "video": "0Z3NeEoE4cc.mp4", "idx": 183, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6147003173828125, "peak_mem_gb": 9.443287372589111, "n_input_tokens": 3855, "n_video_tokens": 3080, "n_audio_tokens": 569, "retained_av_ratio": 0.45108248835297343}, {"method": "istm_r45", "video": "K8S_8sdPRDI.mp4", "idx": 185, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9974315166473389, "peak_mem_gb": 9.291672229766846, "n_input_tokens": 2871, "n_video_tokens": 2240, "n_audio_tokens": 414, "retained_av_ratio": 0.45327807083647326}, {"method": "istm_r45", "video": "dxSsSfynCUQ.mp4", "idx": 187, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.459404706954956, "peak_mem_gb": 10.055179119110107, "n_input_tokens": 7661, "n_video_tokens": 6336, "n_audio_tokens": 1104, "retained_av_ratio": 0.4489247311827957}, {"method": "istm_r45", "video": "PW-6MC05h3A.mp4", "idx": 189, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.518832206726074, "peak_mem_gb": 10.26003122329712, "n_input_tokens": 8750, "n_video_tokens": 7128, "n_audio_tokens": 1364, "retained_av_ratio": 0.4525435704192181}, {"method": "istm_r45", "video": "O61cCbK3hMY.mp4", "idx": 191, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.88 GiB. GPU 0 has a total capacity of 14.56 GiB of which 320.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 12.157180309295654, "n_input_tokens": 12527, "n_video_tokens": 9216, "n_audio_tokens": 3107, "retained_av_ratio": null}, {"method": "istm_r45", "video": "ypwjQw-1Ltg.mp4", "idx": 193, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.268332004547119, "peak_mem_gb": 10.055908679962158, "n_input_tokens": 7666, "n_video_tokens": 6336, "n_audio_tokens": 1108, "retained_av_ratio": 0.4482804943578721}, {"method": "istm_r45", "video": "PKoZQwUxvnQ.mp4", "idx": 195, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8944640159606934, "peak_mem_gb": 9.264962673187256, "n_input_tokens": 2735, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.45013795821836816}, {"method": "istm_r45", "video": "yM0pV0G3-gc.mp4", "idx": 197, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.3373823165893555, "peak_mem_gb": 10.055888175964355, "n_input_tokens": 7578, "n_video_tokens": 6192, "n_audio_tokens": 1095, "retained_av_ratio": 0.4488815699190339}, {"method": "istm_r45", "video": "rTcusJ3Nfqc.mp4", "idx": 199, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.497555255889893, "peak_mem_gb": 10.455715656280518, "n_input_tokens": 10028, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": 0.44855715305394106}, {"method": "istm_r45", "video": "mXZjcxVwrLY.mp4", "idx": 201, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9480545520782471, "peak_mem_gb": 9.077675819396973, "n_input_tokens": 1336, "n_video_tokens": 864, "n_audio_tokens": 162, "retained_av_ratio": 0.49415204678362573}, {"method": "istm_r45", "video": "oQVdUilguhc.mp4", "idx": 203, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.3269684314727783, "peak_mem_gb": 9.416994571685791, "n_input_tokens": 3556, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 0.4517625231910946}, {"method": "istm_r45", "video": "E9NW_SJTrco.mp4", "idx": 205, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7289409637451172, "peak_mem_gb": 9.010971069335938, "n_input_tokens": 913, "n_video_tokens": 576, "n_audio_tokens": 120, "retained_av_ratio": 0.5804597701149425}, {"method": "istm_r45", "video": "iYrFMa1TBQQ.mp4", "idx": 207, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.221833229064941, "peak_mem_gb": 10.201101303100586, "n_input_tokens": 8657, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 0.449574870099197}, {"method": "istm_r45", "video": "Sin_6xmILaE.mp4", "idx": 209, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3157830238342285, "peak_mem_gb": 9.118086814880371, "n_input_tokens": 1767, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 0.45039164490861616}, {"method": "istm_r45", "video": "w7K419G-nJM.mp4", "idx": 211, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5766017436981201, "peak_mem_gb": 9.170538425445557, "n_input_tokens": 2102, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 0.45130388504523683}, {"method": "istm_r45", "video": "pOMKJXCtap0.mp4", "idx": 213, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.565109491348267, "peak_mem_gb": 10.51113748550415, "n_input_tokens": 10160, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.44863185326250377}, {"method": "istm_r45", "video": "HWpq9mZ5Sp4.mp4", "idx": 215, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9556446075439453, "peak_mem_gb": 9.04049301147461, "n_input_tokens": 1220, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.49513618677042803}, {"method": "istm_r45", "video": "NOpfz1iawe0.mp4", "idx": 217, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.594879150390625, "peak_mem_gb": 8.977001667022705, "n_input_tokens": 740, "n_video_tokens": 432, "n_audio_tokens": 94, "retained_av_ratio": 0.5931558935361216}, {"method": "istm_r45", "video": "HiujBqqlyvE.mp4", "idx": 219, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7785193920135498, "peak_mem_gb": 9.006949424743652, "n_input_tokens": 1055, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.4827586206896552}, {"method": "istm_r45", "video": "b8RVswCLGnM.mp4", "idx": 221, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.858511924743652, "peak_mem_gb": 10.54798412322998, "n_input_tokens": 10216, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4498699219531719}, {"method": "istm_r45", "video": "UMdQ1VnfT54.mp4", "idx": 223, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.127374887466431, "peak_mem_gb": 10.345779418945312, "n_input_tokens": 9518, "n_video_tokens": 7920, "n_audio_tokens": 1385, "retained_av_ratio": 0.44954325631380976}, {"method": "istm_r45", "video": "xcxsvhrlfc0.mp4", "idx": 225, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4066298007965088, "peak_mem_gb": 9.159489154815674, "n_input_tokens": 2066, "n_video_tokens": 1584, "n_audio_tokens": 298, "retained_av_ratio": 0.4532412327311371}, {"method": "istm_r45", "video": "sAgv2JvK0iA.mp4", "idx": 227, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.657110691070557, "peak_mem_gb": 9.921886920928955, "n_input_tokens": 6703, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 0.4498366775548297}, {"method": "istm_r45", "video": "u45J1YdCWvo.mp4", "idx": 229, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.371724843978882, "peak_mem_gb": 9.386443138122559, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 495, "retained_av_ratio": 0.4503249767873723}, {"method": "istm_r45", "video": "VMltK9d0ewk.mp4", "idx": 231, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.723018646240234, "peak_mem_gb": 10.28231143951416, "n_input_tokens": 9151, "n_video_tokens": 7632, "n_audio_tokens": 1342, "retained_av_ratio": 0.44929797191887677}, {"method": "istm_r45", "video": "bt1qugf_ej8.mp4", "idx": 233, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.402690887451172, "peak_mem_gb": 10.418152332305908, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1437, "retained_av_ratio": 0.44935199585277347}, {"method": "istm_r45", "video": "eVp2MEAkC44.mp4", "idx": 235, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9480865001678467, "peak_mem_gb": 9.546911716461182, "n_input_tokens": 4383, "n_video_tokens": 3456, "n_audio_tokens": 610, "retained_av_ratio": 0.4498278406296114}, {"method": "istm_r45", "video": "3j9Eq0Ky7f8.mp4", "idx": 237, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3932175636291504, "peak_mem_gb": 9.141886711120605, "n_input_tokens": 1928, "n_video_tokens": 1440, "n_audio_tokens": 270, "retained_av_ratio": 0.452046783625731}, {"method": "istm_r45", "video": "yr8b945WDdY.mp4", "idx": 239, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.50519871711731, "peak_mem_gb": 10.499009609222412, "n_input_tokens": 10036, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4494301994301994}, {"method": "istm_r45", "video": "4wJs2hhedj0.mp4", "idx": 241, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.044245719909668, "peak_mem_gb": 9.064030170440674, "n_input_tokens": 1394, "n_video_tokens": 980, "n_audio_tokens": 200, "retained_av_ratio": 0.47203389830508474}, {"method": "istm_r45", "video": "KABpAbGs6bQ.mp4", "idx": 243, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7978203296661377, "peak_mem_gb": 9.262073040008545, "n_input_tokens": 2729, "n_video_tokens": 2160, "n_audio_tokens": 382, "retained_av_ratio": 0.4500393391030684}, {"method": "istm_r45", "video": "Bn2HxjKRi7s.mp4", "idx": 245, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.687938928604126, "peak_mem_gb": 10.094130516052246, "n_input_tokens": 7882, "n_video_tokens": 6480, "n_audio_tokens": 1148, "retained_av_ratio": 0.4491347666491872}, {"method": "istm_r45", "video": "E1FfWv-iIJM.mp4", "idx": 247, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.578052520751953, "peak_mem_gb": 10.273545265197754, "n_input_tokens": 9031, "n_video_tokens": 7488, "n_audio_tokens": 1307, "retained_av_ratio": 0.4491188175099488}, {"method": "istm_r45", "video": "a7YYrZ3mdsY.mp4", "idx": 249, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4810781478881836, "peak_mem_gb": 9.177025318145752, "n_input_tokens": 2206, "n_video_tokens": 1728, "n_audio_tokens": 313, "retained_av_ratio": 0.45222929936305734}, {"method": "istm_r45", "video": "4R3s-co2_lU.mp4", "idx": 251, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.758159637451172, "peak_mem_gb": 10.544929027557373, "n_input_tokens": 10242, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.4494145902131492}, {"method": "istm_r45", "video": "e-3Km4K2G2I.mp4", "idx": 253, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.48016095161438, "peak_mem_gb": 9.680834293365479, "n_input_tokens": 5132, "n_video_tokens": 4176, "n_audio_tokens": 741, "retained_av_ratio": 0.4494610534878991}, {"method": "istm_r45", "video": "Bc3vnp7VHVM.mp4", "idx": 255, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.9811797142028809, "peak_mem_gb": 9.285112857818604, "n_input_tokens": 2891, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 0.45137614678899085}, {"method": "istm_r45", "video": "dNqYbetriJ4.mp4", "idx": 257, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.878512859344482, "peak_mem_gb": 10.582814693450928, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1514, "retained_av_ratio": 0.44888713807366554}, {"method": "istm_r45", "video": "-eUTGNm9ZFY.mp4", "idx": 259, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2398300170898438, "peak_mem_gb": 9.115288257598877, "n_input_tokens": 1754, "n_video_tokens": 1296, "n_audio_tokens": 242, "retained_av_ratio": 0.45513654096228867}, {"method": "istm_r45", "video": "68_42cTzS80.mp4", "idx": 261, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4628956317901611, "peak_mem_gb": 9.18287467956543, "n_input_tokens": 2226, "n_video_tokens": 1728, "n_audio_tokens": 305, "retained_av_ratio": 0.44909001475651744}, {"method": "istm_r45", "video": "OdrWu71KIdI.mp4", "idx": 263, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4498224258422852, "peak_mem_gb": 9.154573440551758, "n_input_tokens": 1964, "n_video_tokens": 1440, "n_audio_tokens": 273, "retained_av_ratio": 0.45475773496789257}, {"method": "istm_r45", "video": "Q4ftOkPtOB0.mp4", "idx": 265, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.756870985031128, "peak_mem_gb": 9.745325088500977, "n_input_tokens": 5536, "n_video_tokens": 4480, "n_audio_tokens": 816, "retained_av_ratio": 0.45128398791540786}, {"method": "istm_r45", "video": "l0Ilew3e9CI.mp4", "idx": 267, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8583853244781494, "peak_mem_gb": 9.052823066711426, "n_input_tokens": 1252, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.49606299212598426}, {"method": "istm_r45", "video": "IgGPXPR1J1A.mp4", "idx": 269, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.056157112121582, "peak_mem_gb": 9.053891658782959, "n_input_tokens": 1394, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 0.45348837209302323}, {"method": "istm_r45", "video": "jNQep4ocOGQ.mp4", "idx": 271, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9654676914215088, "peak_mem_gb": 9.053190231323242, "n_input_tokens": 1388, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.44903117101937656}, {"method": "istm_r45", "video": "qB66a2x8KXE.mp4", "idx": 273, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7267036437988281, "peak_mem_gb": 9.250621795654297, "n_input_tokens": 2608, "n_video_tokens": 2016, "n_audio_tokens": 373, "retained_av_ratio": 0.45290916701548767}, {"method": "istm_r45", "video": "vVlQm8-AHt4.mp4", "idx": 275, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.6928226947784424, "peak_mem_gb": 9.018005847930908, "n_input_tokens": 936, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "istm_r45", "video": "XOUUsvMm1PM.mp4", "idx": 277, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.305926561355591, "peak_mem_gb": 9.381240844726562, "n_input_tokens": 3448, "n_video_tokens": 2736, "n_audio_tokens": 489, "retained_av_ratio": 0.4496124031007752}, {"method": "istm_r45", "video": "_Nqw7iCl4eM.mp4", "idx": 279, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.939633369445801, "peak_mem_gb": 10.350749492645264, "n_input_tokens": 9543, "n_video_tokens": 7920, "n_audio_tokens": 1400, "retained_av_ratio": 0.4494635193133047}, {"method": "istm_r45", "video": "8kMaXa0PYow.mp4", "idx": 281, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1038203239440918, "peak_mem_gb": 9.06571102142334, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.46166807076663857}, {"method": "istm_r45", "video": "A3VEekCcv_s.mp4", "idx": 283, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.9261269569396973, "peak_mem_gb": 9.727665901184082, "n_input_tokens": 5461, "n_video_tokens": 4464, "n_audio_tokens": 796, "retained_av_ratio": 0.45}, {"method": "istm_r45", "video": "Mm6HOkr_HXc.mp4", "idx": 285, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.907314777374268, "peak_mem_gb": 9.95828914642334, "n_input_tokens": 6995, "n_video_tokens": 5760, "n_audio_tokens": 1005, "retained_av_ratio": 0.44907612712490763}, {"method": "istm_r45", "video": "c4x0ezi7rP8.mp4", "idx": 287, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.6496763229370117, "peak_mem_gb": 9.00241994857788, "n_input_tokens": 883, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5877318116975749}, {"method": "istm_r45", "video": "mqFCIRSDSxE.mp4", "idx": 289, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.136138677597046, "peak_mem_gb": 10.35385274887085, "n_input_tokens": 9539, "n_video_tokens": 7920, "n_audio_tokens": 1382, "retained_av_ratio": 0.44990324661363146}, {"method": "istm_r45", "video": "zCwz5h0TPmE.mp4", "idx": 291, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3292889595031738, "peak_mem_gb": 9.13701343536377, "n_input_tokens": 1911, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.45353594389246055}, {"method": "istm_r45", "video": "SbFqCuO11JU.mp4", "idx": 293, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.255706787109375, "peak_mem_gb": 9.372790813446045, "n_input_tokens": 3420, "n_video_tokens": 2736, "n_audio_tokens": 492, "retained_av_ratio": 0.4507434944237918}, {"method": "istm_r45", "video": "nl7-1jkIIgA.mp4", "idx": 295, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.2811648845672607, "peak_mem_gb": 9.647794723510742, "n_input_tokens": 4843, "n_video_tokens": 3888, "n_audio_tokens": 680, "retained_av_ratio": 0.4498686514886165}, {"method": "istm_r45", "video": "gAc9Kp7CHn8.mp4", "idx": 297, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.427565336227417, "peak_mem_gb": 9.176389694213867, "n_input_tokens": 2117, "n_video_tokens": 1584, "n_audio_tokens": 285, "retained_av_ratio": 0.44997324772605674}, {"method": "istm_r45", "video": "0vl66E4x5OI.mp4", "idx": 299, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3353171348571777, "peak_mem_gb": 9.117499351501465, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45330739299610895}, {"method": "istm_r45", "video": "KLeJVuRRk84.mp4", "idx": 301, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9359939098358154, "peak_mem_gb": 9.033431053161621, "n_input_tokens": 1092, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.5290023201856149}, {"method": "istm_r45", "video": "Lzccd-ol1Ks.mp4", "idx": 303, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.842190504074097, "peak_mem_gb": 10.549083232879639, "n_input_tokens": 10228, "n_video_tokens": 8496, "n_audio_tokens": 1499, "retained_av_ratio": 0.4494247123561781}, {"method": "istm_r45", "video": "-7dHm4baH38.mp4", "idx": 305, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3070940971374512, "peak_mem_gb": 9.137694835662842, "n_input_tokens": 1914, "n_video_tokens": 1440, "n_audio_tokens": 279, "retained_av_ratio": 0.45549738219895286}, {"method": "istm_r45", "video": "b53-JgOg1i8.mp4", "idx": 307, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0881378650665283, "peak_mem_gb": 9.075776100158691, "n_input_tokens": 1549, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.4538852578068264}, {"method": "istm_r45", "video": "TQ1qkHdbyzM.mp4", "idx": 309, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.075695514678955, "peak_mem_gb": 9.091599941253662, "n_input_tokens": 1582, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 0.4575018477457502}, {"method": "istm_r45", "video": "nfqtKk9tob0.mp4", "idx": 311, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.753836393356323, "peak_mem_gb": 10.533228397369385, "n_input_tokens": 10202, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.44908380895163713}, {"method": "istm_r45", "video": "Ata-X6fKUas.mp4", "idx": 313, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4117612838745117, "peak_mem_gb": 9.41014289855957, "n_input_tokens": 3625, "n_video_tokens": 2880, "n_audio_tokens": 513, "retained_av_ratio": 0.4500442086648983}, {"method": "istm_r45", "video": "qdbBYgRMOEk.mp4", "idx": 315, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.262632608413696, "peak_mem_gb": 9.804669380187988, "n_input_tokens": 5982, "n_video_tokens": 4896, "n_audio_tokens": 869, "retained_av_ratio": 0.44978317432784043}, {"method": "istm_r45", "video": "mBMrGwB0v9w.mp4", "idx": 317, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2017381191253662, "peak_mem_gb": 9.085142612457275, "n_input_tokens": 1569, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 0.4567448680351906}, {"method": "istm_r45", "video": "RgygVeaQXkU.mp4", "idx": 319, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.438852071762085, "peak_mem_gb": 9.877683162689209, "n_input_tokens": 6479, "n_video_tokens": 5328, "n_audio_tokens": 934, "retained_av_ratio": 0.4498562759501757}, {"method": "istm_r45", "video": "xQUom0axvAg.mp4", "idx": 321, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2953741550445557, "peak_mem_gb": 9.14185905456543, "n_input_tokens": 1925, "n_video_tokens": 1440, "n_audio_tokens": 263, "retained_av_ratio": 0.451556077510276}, {"method": "istm_r45", "video": "7WGoDmVMbU4.mp4", "idx": 323, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.4566123485565186, "peak_mem_gb": 9.651337146759033, "n_input_tokens": 4945, "n_video_tokens": 4032, "n_audio_tokens": 713, "retained_av_ratio": 0.4499473129610116}, {"method": "istm_r45", "video": "AZS9Vz54oWs.mp4", "idx": 325, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.6370537281036377, "peak_mem_gb": 9.691013813018799, "n_input_tokens": 5184, "n_video_tokens": 4200, "n_audio_tokens": 769, "retained_av_ratio": 0.45119742402897967}, {"method": "istm_r45", "video": "ZUXFwwscme0.mp4", "idx": 327, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3699495792388916, "peak_mem_gb": 9.135079383850098, "n_input_tokens": 1909, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 0.4524781341107872}, {"method": "istm_r45", "video": "y6S8EUrrpP4.mp4", "idx": 329, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6163768768310547, "peak_mem_gb": 9.230515003204346, "n_input_tokens": 2458, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 0.4509715318572074}, {"method": "istm_r45", "video": "B0M-_FeUYVE.mp4", "idx": 331, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7794859409332275, "peak_mem_gb": 9.02544641494751, "n_input_tokens": 1109, "n_video_tokens": 715, "n_audio_tokens": 140, "retained_av_ratio": 0.47719298245614034}, {"method": "istm_r45", "video": "DM8l9e9cEXY.mp4", "idx": 333, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.806664228439331, "peak_mem_gb": 9.027705669403076, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 139, "retained_av_ratio": 0.5378346915017462}, {"method": "istm_r45", "video": "jmrtSmuXI9M.mp4", "idx": 335, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1944961547851562, "peak_mem_gb": 9.095500946044922, "n_input_tokens": 1602, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 0.45301027900146845}, {"method": "istm_r45", "video": "DPnvcwAWV-Y.mp4", "idx": 337, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.859803915023804, "peak_mem_gb": 10.539390087127686, "n_input_tokens": 10168, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": 0.44914914914914916}, {"method": "istm_r45", "video": "jBLELOjSYfY.mp4", "idx": 339, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.18344783782959, "peak_mem_gb": 9.319491386413574, "n_input_tokens": 3087, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 0.4508111839834311}, {"method": "istm_r45", "video": "4kslnZ1gICo.mp4", "idx": 341, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.909865140914917, "peak_mem_gb": 9.043635845184326, "n_input_tokens": 1234, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 0.4872549019607843}, {"method": "istm_r45", "video": "toQnAXHr5uo.mp4", "idx": 343, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6244313716888428, "peak_mem_gb": 9.475403308868408, "n_input_tokens": 3996, "n_video_tokens": 3168, "n_audio_tokens": 561, "retained_av_ratio": 0.4497184231697506}, {"method": "istm_r45", "video": "__aZIudBQjw.mp4", "idx": 345, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.771313428878784, "peak_mem_gb": 10.316758155822754, "n_input_tokens": 9254, "n_video_tokens": 7632, "n_audio_tokens": 1343, "retained_av_ratio": 0.4498050139275766}, {"method": "istm_r45", "video": "wRWKim49T6Y.mp4", "idx": 347, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9193689823150635, "peak_mem_gb": 9.009448528289795, "n_input_tokens": 1060, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.48023255813953486}, {"method": "istm_r45", "video": "7lh_sO9EWkE.mp4", "idx": 349, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2129170894622803, "peak_mem_gb": 9.11679744720459, "n_input_tokens": 1758, "n_video_tokens": 1296, "n_audio_tokens": 227, "retained_av_ratio": 0.45042678923177937}, {"method": "istm_r45", "video": "KMngzlwkYCU.mp4", "idx": 351, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4697272777557373, "peak_mem_gb": 9.152009963989258, "n_input_tokens": 2045, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.45101171458998934}, {"method": "istm_r45", "video": "_J6U2xhnBX0.mp4", "idx": 353, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.90934419631958, "peak_mem_gb": 10.614707946777344, "n_input_tokens": 10545, "n_video_tokens": 8784, "n_audio_tokens": 1528, "retained_av_ratio": 0.45044608223429017}, {"method": "istm_r45", "video": "8-QqR2J8qxQ.mp4", "idx": 355, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2136116027832031, "peak_mem_gb": 9.114363670349121, "n_input_tokens": 1759, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45136186770428016}, {"method": "istm_r45", "video": "S0ye2uVX2v0.mp4", "idx": 357, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4832704067230225, "peak_mem_gb": 9.193275928497314, "n_input_tokens": 2256, "n_video_tokens": 1728, "n_audio_tokens": 303, "retained_av_ratio": 0.44953225012309206}, {"method": "istm_r45", "video": "N8q8dFk0Ark.mp4", "idx": 359, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0390496253967285, "peak_mem_gb": 9.081533908843994, "n_input_tokens": 1474, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.45071609098567816}, {"method": "istm_r45", "video": "Wz3-b0MeWrQ.mp4", "idx": 361, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3494651317596436, "peak_mem_gb": 9.629902362823486, "n_input_tokens": 4787, "n_video_tokens": 3888, "n_audio_tokens": 695, "retained_av_ratio": 0.45057822387082697}, {"method": "istm_r45", "video": "cr6VlmDjePc.mp4", "idx": 363, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.465916156768799, "peak_mem_gb": 9.41046667098999, "n_input_tokens": 3623, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 0.4504424778761062}, {"method": "istm_r45", "video": "wuHkSR0XJdo.mp4", "idx": 365, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.0609326362609863, "peak_mem_gb": 9.549880027770996, "n_input_tokens": 4479, "n_video_tokens": 3600, "n_audio_tokens": 630, "retained_av_ratio": 0.4494089834515366}, {"method": "istm_r45", "video": "bNu5CZVQUNI.mp4", "idx": 367, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.528297662734985, "peak_mem_gb": 10.499967098236084, "n_input_tokens": 10015, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4494301994301994}, {"method": "istm_r45", "video": "gx9mV8QfXso.mp4", "idx": 369, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7531161308288574, "peak_mem_gb": 9.022899150848389, "n_input_tokens": 1097, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.4769775678866588}, {"method": "istm_r45", "video": "YHZ6mWsKDfg.mp4", "idx": 371, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.169940948486328, "peak_mem_gb": 9.35315990447998, "n_input_tokens": 3270, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 0.450261780104712}, {"method": "istm_r45", "video": "KezGJkHX78E.mp4", "idx": 373, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3302457332611084, "peak_mem_gb": 9.13476276397705, "n_input_tokens": 1901, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 0.4499411071849234}, {"method": "istm_r45", "video": "U1xMaQtB6sE.mp4", "idx": 375, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.183854341506958, "peak_mem_gb": 9.346469402313232, "n_input_tokens": 3166, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 0.4483478260869565}, {"method": "istm_r45", "video": "-YuuVgM_b1Q.mp4", "idx": 377, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1740384101867676, "peak_mem_gb": 9.36095905303955, "n_input_tokens": 3290, "n_video_tokens": 2592, "n_audio_tokens": 455, "retained_av_ratio": 0.4499507712504102}, {"method": "istm_r45", "video": "bczYqpZQMWg.mp4", "idx": 379, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.171550273895264, "peak_mem_gb": 10.155686378479004, "n_input_tokens": 8330, "n_video_tokens": 6912, "n_audio_tokens": 1225, "retained_av_ratio": 0.45016590881160135}, {"method": "istm_r45", "video": "xnx5D0RNCCA.mp4", "idx": 381, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.087721824645996, "peak_mem_gb": 9.069318294525146, "n_input_tokens": 1441, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.4552238805970149}, {"method": "istm_r45", "video": "uvGT_4WNYUw.mp4", "idx": 383, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.515077829360962, "peak_mem_gb": 9.184839725494385, "n_input_tokens": 2140, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 0.4533333333333333}, {"method": "istm_r45", "video": "_Y7Q2syNYxY.mp4", "idx": 385, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.779638290405273, "peak_mem_gb": 10.51045274734497, "n_input_tokens": 10252, "n_video_tokens": 8496, "n_audio_tokens": 1480, "retained_av_ratio": 0.44917802726543704}, {"method": "istm_r45", "video": "fv-EzOxlpjo.mp4", "idx": 387, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6139857769012451, "peak_mem_gb": 9.23790168762207, "n_input_tokens": 2479, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 0.4518752824220515}, {"method": "istm_r45", "video": "z9FPvQc9yoI.mp4", "idx": 389, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8840184211730957, "peak_mem_gb": 9.266890525817871, "n_input_tokens": 2744, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.448955459203784}, {"method": "istm_r45", "video": "uhWITF_qwqw.mp4", "idx": 391, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9606149196624756, "peak_mem_gb": 9.024500370025635, "n_input_tokens": 1057, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 0.5323149236192715}, {"method": "istm_r45", "video": "Syz-bdm5oEE.mp4", "idx": 393, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6989917755126953, "peak_mem_gb": 9.214430332183838, "n_input_tokens": 2404, "n_video_tokens": 1872, "n_audio_tokens": 331, "retained_av_ratio": 0.4516568315932819}, {"method": "istm_r45", "video": "aJwO65HE8A4.mp4", "idx": 395, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.9697630405426025, "peak_mem_gb": 10.346709728240967, "n_input_tokens": 9525, "n_video_tokens": 7920, "n_audio_tokens": 1380, "retained_av_ratio": 0.44881720430107525}, {"method": "istm_r45", "video": "ZFIh8hf4btQ.mp4", "idx": 397, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8808166980743408, "peak_mem_gb": 9.034743785858154, "n_input_tokens": 1200, "n_video_tokens": 864, "n_audio_tokens": 155, "retained_av_ratio": 0.492639842983317}, {"method": "istm_r45", "video": "MfjWOF8-npQ.mp4", "idx": 399, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9823741912841797, "peak_mem_gb": 9.054769515991211, "n_input_tokens": 1392, "n_video_tokens": 1008, "n_audio_tokens": 177, "retained_av_ratio": 0.4489451476793249}, {"method": "istm_r45", "video": "hVG1xDJwMbY.mp4", "idx": 401, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.56 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.334262371063232, "n_input_tokens": 12882, "n_video_tokens": 9216, "n_audio_tokens": 3455, "retained_av_ratio": null}, {"method": "istm_r45", "video": "AghXXz9hI34.mp4", "idx": 403, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2867300510406494, "peak_mem_gb": 9.1258225440979, "n_input_tokens": 1685, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.4684095860566449}, {"method": "istm_r45", "video": "spmQ5-8a6Ao.mp4", "idx": 405, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2959082126617432, "peak_mem_gb": 9.137362480163574, "n_input_tokens": 1906, "n_video_tokens": 1440, "n_audio_tokens": 259, "retained_av_ratio": 0.45203060623896407}, {"method": "istm_r45", "video": "2Gfm1i3DTaw.mp4", "idx": 407, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.804864406585693, "peak_mem_gb": 10.545945167541504, "n_input_tokens": 10233, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.44961473031121785}, {"method": "istm_r45", "video": "nKN_c5WAVOw.mp4", "idx": 409, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8438169956207275, "peak_mem_gb": 9.278456687927246, "n_input_tokens": 2787, "n_video_tokens": 2160, "n_audio_tokens": 395, "retained_av_ratio": 0.4500978473581213}, {"method": "istm_r45", "video": "S4oi8EUfLYI.mp4", "idx": 411, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.693897008895874, "peak_mem_gb": 9.721322059631348, "n_input_tokens": 5440, "n_video_tokens": 4464, "n_audio_tokens": 799, "retained_av_ratio": 0.4506935208056242}, {"method": "istm_r45", "video": "DYfYpkl2OTI.mp4", "idx": 413, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.471405267715454, "peak_mem_gb": 9.184175968170166, "n_input_tokens": 2229, "n_video_tokens": 1728, "n_audio_tokens": 315, "retained_av_ratio": 0.4522760646108664}, {"method": "istm_r45", "video": "51Sg4YQOPhI.mp4", "idx": 415, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6212353706359863, "peak_mem_gb": 9.206908702850342, "n_input_tokens": 2300, "n_video_tokens": 1728, "n_audio_tokens": 326, "retained_av_ratio": 0.4527750730282376}, {"method": "istm_r45", "video": "zVvBiALQtE0.mp4", "idx": 417, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.2289040088653564, "peak_mem_gb": 10.408946990966797, "n_input_tokens": 9810, "n_video_tokens": 8064, "n_audio_tokens": 1422, "retained_av_ratio": 0.4495045329959941}, {"method": "istm_r45", "video": "GhuS1QMn9hc.mp4", "idx": 419, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9452800750732422, "peak_mem_gb": 9.0611572265625, "n_input_tokens": 1277, "n_video_tokens": 864, "n_audio_tokens": 173, "retained_av_ratio": 0.5072324011571842}, {"method": "istm_r45", "video": "TUJbjPtM7JE.mp4", "idx": 421, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3395535945892334, "peak_mem_gb": 9.111557483673096, "n_input_tokens": 1750, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 0.45278137128072443}, {"method": "istm_r45", "video": "kq9kiAScGw8.mp4", "idx": 423, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.0172948837280273, "peak_mem_gb": 9.508809566497803, "n_input_tokens": 4281, "n_video_tokens": 3456, "n_audio_tokens": 622, "retained_av_ratio": 0.45022069641981366}, {"method": "istm_r45", "video": "NvEO1NCuTR0.mp4", "idx": 425, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.887669086456299, "peak_mem_gb": 9.469536304473877, "n_input_tokens": 4072, "n_video_tokens": 3312, "n_audio_tokens": 595, "retained_av_ratio": 0.44970565651394934}, {"method": "istm_r45", "video": "xhYip8GNPaM.mp4", "idx": 427, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.603219509124756, "peak_mem_gb": 9.412417888641357, "n_input_tokens": 3635, "n_video_tokens": 2880, "n_audio_tokens": 518, "retained_av_ratio": 0.449970570924073}, {"method": "istm_r45", "video": "u6UOIS1h340.mp4", "idx": 429, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8365309238433838, "peak_mem_gb": 9.009947776794434, "n_input_tokens": 906, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 0.5752212389380531}, {"method": "istm_r45", "video": "hZes-czBp5A.mp4", "idx": 431, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.768430233001709, "peak_mem_gb": 9.205981254577637, "n_input_tokens": 2383, "n_video_tokens": 1872, "n_audio_tokens": 340, "retained_av_ratio": 0.451627486437613}, {"method": "istm_r45", "video": "bj_1uCcsxrg.mp4", "idx": 433, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.691504240036011, "peak_mem_gb": 10.541606903076172, "n_input_tokens": 10198, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 0.44950455409868884}, {"method": "istm_r45", "video": "aQj9FsOGmFQ.mp4", "idx": 435, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.693686246871948, "peak_mem_gb": 10.553412437438965, "n_input_tokens": 10195, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44963489046714017}, {"method": "istm_r45", "video": "pRcxjOCF3iM.mp4", "idx": 437, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8526771068572998, "peak_mem_gb": 9.040113925933838, "n_input_tokens": 1156, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "istm_r45", "video": "42mYOogHnyE.mp4", "idx": 439, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.8615937232971191, "peak_mem_gb": 9.260900974273682, "n_input_tokens": 2640, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 0.45222129086336965}, {"method": "istm_r45", "video": "9yjRUlmmXyk.mp4", "idx": 441, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.886963605880737, "peak_mem_gb": 10.547068119049072, "n_input_tokens": 10251, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.44966980188112865}, {"method": "istm_r45", "video": "1C6OKfQaHJ4.mp4", "idx": 443, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.365297317504883, "peak_mem_gb": 9.364535808563232, "n_input_tokens": 3308, "n_video_tokens": 2592, "n_audio_tokens": 467, "retained_av_ratio": 0.44982020268061457}, {"method": "istm_r45", "video": "n4fypasG9N8.mp4", "idx": 445, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9656710624694824, "peak_mem_gb": 9.031550884246826, "n_input_tokens": 1130, "n_video_tokens": 720, "n_audio_tokens": 148, "retained_av_ratio": 0.4827188940092166}, {"method": "istm_r45", "video": "QyR4tUx_eE4.mp4", "idx": 447, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3687827587127686, "peak_mem_gb": 9.116174221038818, "n_input_tokens": 1760, "n_video_tokens": 1296, "n_audio_tokens": 228, "retained_av_ratio": 0.44816272965879267}, {"method": "istm_r45", "video": "Inh0nn0Mehw.mp4", "idx": 449, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0772767066955566, "peak_mem_gb": 9.052551746368408, "n_input_tokens": 1383, "n_video_tokens": 1008, "n_audio_tokens": 178, "retained_av_ratio": 0.45109612141652616}, {"method": "istm_r45", "video": "TqgdJIWKEHU.mp4", "idx": 451, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.9130585193634033, "peak_mem_gb": 9.767869472503662, "n_input_tokens": 5679, "n_video_tokens": 4608, "n_audio_tokens": 802, "retained_av_ratio": 0.44898336414048057}, {"method": "istm_r45", "video": "BB4i0vWjSF4.mp4", "idx": 453, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6868095397949219, "peak_mem_gb": 9.23115348815918, "n_input_tokens": 2536, "n_video_tokens": 2002, "n_audio_tokens": 353, "retained_av_ratio": 0.4496815286624204}, {"method": "istm_r45", "video": "2LMNEWSR8K0.mp4", "idx": 455, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3192789554595947, "peak_mem_gb": 9.137032985687256, "n_input_tokens": 1915, "n_video_tokens": 1440, "n_audio_tokens": 276, "retained_av_ratio": 0.4527972027972028}, {"method": "istm_r45", "video": "jfN_xuGVfPg.mp4", "idx": 457, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2398467063903809, "peak_mem_gb": 9.114030361175537, "n_input_tokens": 1751, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 0.4541314248536109}, {"method": "istm_r45", "video": "KMRepMFj6y0.mp4", "idx": 459, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.4103474617004395, "peak_mem_gb": 9.829770565032959, "n_input_tokens": 6143, "n_video_tokens": 5040, "n_audio_tokens": 879, "retained_av_ratio": 0.4497381314411218}, {"method": "istm_r45", "video": "iwIjN30bQG8.mp4", "idx": 461, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.1753315925598145, "peak_mem_gb": 9.788096904754639, "n_input_tokens": 5844, "n_video_tokens": 4752, "n_audio_tokens": 842, "retained_av_ratio": 0.4490525563103325}, {"method": "istm_r45", "video": "SD3b0eDGylU.mp4", "idx": 463, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.0887908935546875, "peak_mem_gb": 9.313385009765625, "n_input_tokens": 3068, "n_video_tokens": 2448, "n_audio_tokens": 442, "retained_av_ratio": 0.4494809688581315}, {"method": "istm_r45", "video": "wskyXFw6C4I.mp4", "idx": 465, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.912956714630127, "peak_mem_gb": 9.015161037445068, "n_input_tokens": 1076, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.48255813953488375}, {"method": "istm_r45", "video": "9BTyy18OYe4.mp4", "idx": 467, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2360467910766602, "peak_mem_gb": 9.09235668182373, "n_input_tokens": 1592, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.45824255628177196}, {"method": "istm_r45", "video": "-gl0tl4qvzY.mp4", "idx": 469, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8866000175476074, "peak_mem_gb": 9.017022132873535, "n_input_tokens": 1081, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.47703180212014135}, {"method": "istm_r45", "video": "_zYW8CObyxM.mp4", "idx": 471, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.67844033241272, "peak_mem_gb": 10.552626609802246, "n_input_tokens": 10265, "n_video_tokens": 8496, "n_audio_tokens": 1500, "retained_av_ratio": 0.4491796718687475}, {"method": "istm_r45", "video": "yEyUrr7d0aI.mp4", "idx": 473, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.86165714263916, "peak_mem_gb": 9.484463691711426, "n_input_tokens": 4117, "n_video_tokens": 3312, "n_audio_tokens": 600, "retained_av_ratio": 0.450920245398773}, {"method": "istm_r45", "video": "N8S9TYC6mSE.mp4", "idx": 475, "problem_type": "Temporal Reasoning Localization", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1411497592926025, "peak_mem_gb": 9.35508108139038, "n_input_tokens": 3156, "n_video_tokens": 2380, "n_audio_tokens": 436, "retained_av_ratio": 0.45063920454545453}, {"method": "istm_r45", "video": "sHAaDXWyN3c.mp4", "idx": 477, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9800450801849365, "peak_mem_gb": 9.04430627822876, "n_input_tokens": 1243, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4849368318756074}, {"method": "istm_r45", "video": "H3iTVprK7wU.mp4", "idx": 479, "problem_type": "Narrative Comprehension", "answer": "F", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 1.93 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.26 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.124424934387207, "n_input_tokens": 11977, "n_video_tokens": 9216, "n_audio_tokens": 2541, "retained_av_ratio": null}, {"method": "istm_r45", "video": "O2dajNms-aw.mp4", "idx": 481, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.72739315032959, "peak_mem_gb": 9.491124153137207, "n_input_tokens": 4139, "n_video_tokens": 3312, "n_audio_tokens": 596, "retained_av_ratio": 0.4501023541453429}, {"method": "istm_r45", "video": "w1MSc5_NspY.mp4", "idx": 483, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2632579803466797, "peak_mem_gb": 9.35966157913208, "n_input_tokens": 3293, "n_video_tokens": 2592, "n_audio_tokens": 475, "retained_av_ratio": 0.4512552983371373}, {"method": "istm_r45", "video": "XBRZ_O5FLQ8.mp4", "idx": 485, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2041656970977783, "peak_mem_gb": 9.368110656738281, "n_input_tokens": 3315, "n_video_tokens": 2592, "n_audio_tokens": 470, "retained_av_ratio": 0.4516655780535598}, {"method": "istm_r45", "video": "HuqQmbOK-AQ.mp4", "idx": 487, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8146872520446777, "peak_mem_gb": 9.034767627716064, "n_input_tokens": 1201, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4946341463414634}, {"method": "istm_r45", "video": "jYA4p6xBn2s.mp4", "idx": 489, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.304337501525879, "peak_mem_gb": 9.582050800323486, "n_input_tokens": 4640, "n_video_tokens": 3744, "n_audio_tokens": 661, "retained_av_ratio": 0.4490351872871737}, {"method": "istm_r45", "video": "Zn6Dnm2OAPs.mp4", "idx": 491, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.6092209815979, "peak_mem_gb": 10.500386714935303, "n_input_tokens": 10124, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4500407000407}, {"method": "istm_r45", "video": "d6zr9dqsvfA.mp4", "idx": 493, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.163679361343384, "peak_mem_gb": 9.349608421325684, "n_input_tokens": 3260, "n_video_tokens": 2592, "n_audio_tokens": 460, "retained_av_ratio": 0.449213630406291}, {"method": "istm_r45", "video": "UNG7OtwrQoI.mp4", "idx": 495, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.835576772689819, "peak_mem_gb": 10.106907844543457, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1174, "retained_av_ratio": 0.4499871761990254}, {"method": "istm_r45", "video": "3c1sjLnzO1M.mp4", "idx": 497, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3722410202026367, "peak_mem_gb": 9.647185325622559, "n_input_tokens": 4847, "n_video_tokens": 3888, "n_audio_tokens": 696, "retained_av_ratio": 0.450043630017452}, {"method": "istm_r45", "video": "TIlDW7FLwXk.mp4", "idx": 499, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5301902294158936, "peak_mem_gb": 9.713338851928711, "n_input_tokens": 5332, "n_video_tokens": 4320, "n_audio_tokens": 769, "retained_av_ratio": 0.4495971703674592}, {"method": "omnizip_r45", "video": "26TM44a-6a0.mp4", "idx": 0, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.273409843444824, "peak_mem_gb": 9.461158752441406, "n_input_tokens": 3961, "n_video_tokens": 3168, "n_audio_tokens": 575, "retained_av_ratio": 0.4501736574939888}, {"method": "omnizip_r45", "video": "a_wxf-SGdHU.mp4", "idx": 2, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0712683200836182, "peak_mem_gb": 9.0461745262146, "n_input_tokens": 1232, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 0.4970588235294118}, {"method": "omnizip_r45", "video": "mecSGL3mrlk.mp4", "idx": 4, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.090875148773193, "peak_mem_gb": 9.779046535491943, "n_input_tokens": 5819, "n_video_tokens": 4760, "n_audio_tokens": 864, "retained_av_ratio": 0.45056899004267426}, {"method": "omnizip_r45", "video": "Duj0OyTyjNQ.mp4", "idx": 6, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.850189447402954, "peak_mem_gb": 10.458918571472168, "n_input_tokens": 9792, "n_video_tokens": 8120, "n_audio_tokens": 1458, "retained_av_ratio": 0.45082480684902904}, {"method": "omnizip_r45", "video": "Q7EqBvNNiT8.mp4", "idx": 8, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.2200927734375, "peak_mem_gb": 10.498050689697266, "n_input_tokens": 10056, "n_video_tokens": 8352, "n_audio_tokens": 1475, "retained_av_ratio": 0.4495776941080696}, {"method": "omnizip_r45", "video": "87BvhuIeuyc.mp4", "idx": 10, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.47840428352356, "peak_mem_gb": 9.819988250732422, "n_input_tokens": 6031, "n_video_tokens": 4896, "n_audio_tokens": 863, "retained_av_ratio": 0.4492099322799097}, {"method": "omnizip_r45", "video": "k9lg5Gd7Q-Y.mp4", "idx": 12, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1539299488067627, "peak_mem_gb": 9.068653583526611, "n_input_tokens": 1439, "n_video_tokens": 1008, "n_audio_tokens": 190, "retained_av_ratio": 0.4532554257095159}, {"method": "omnizip_r45", "video": "Z2GuiatpQUw.mp4", "idx": 14, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9801478385925293, "peak_mem_gb": 9.049352645874023, "n_input_tokens": 1248, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49167482859941236}, {"method": "omnizip_r45", "video": "yQvt1MbF-dE.mp4", "idx": 16, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.5987327098846436, "peak_mem_gb": 9.960265636444092, "n_input_tokens": 6920, "n_video_tokens": 5616, "n_audio_tokens": 999, "retained_av_ratio": 0.45003779289493573}, {"method": "omnizip_r45", "video": "ziQsD1EQPOs.mp4", "idx": 18, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.110393762588501, "peak_mem_gb": 9.045242309570312, "n_input_tokens": 1244, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4848780487804878}, {"method": "omnizip_r45", "video": "rh1PHqq0Q50.mp4", "idx": 20, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7792050838470459, "peak_mem_gb": 8.97169017791748, "n_input_tokens": 728, "n_video_tokens": 432, "n_audio_tokens": 98, "retained_av_ratio": 0.5867924528301887}, {"method": "omnizip_r45", "video": "nBAE01OGZYI.mp4", "idx": 22, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.8722991943359375, "peak_mem_gb": 10.142688274383545, "n_input_tokens": 8211, "n_video_tokens": 6768, "n_audio_tokens": 1201, "retained_av_ratio": 0.44961726690927345}, {"method": "omnizip_r45", "video": "pAEYLYBhnSk.mp4", "idx": 24, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7896814346313477, "peak_mem_gb": 8.958276748657227, "n_input_tokens": 682, "n_video_tokens": 432, "n_audio_tokens": 77, "retained_av_ratio": 0.5776031434184676}, {"method": "omnizip_r45", "video": "zrcQYUHNcL8.mp4", "idx": 26, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.873978853225708, "peak_mem_gb": 9.181087017059326, "n_input_tokens": 2219, "n_video_tokens": 1728, "n_audio_tokens": 309, "retained_av_ratio": 0.45017182130584193}, {"method": "omnizip_r45", "video": "KuDeAIF9rJo.mp4", "idx": 28, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.238083600997925, "peak_mem_gb": 10.556265830993652, "n_input_tokens": 10335, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4486737008184597}, {"method": "omnizip_r45", "video": "GbJSpPohjDM.mp4", "idx": 30, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.456908941268921, "peak_mem_gb": 9.115773677825928, "n_input_tokens": 1749, "n_video_tokens": 1287, "n_audio_tokens": 232, "retained_av_ratio": 0.4522712310730744}, {"method": "omnizip_r45", "video": "deDj5V0XxfA.mp4", "idx": 32, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2468326091766357, "peak_mem_gb": 9.07183027267456, "n_input_tokens": 1427, "n_video_tokens": 1008, "n_audio_tokens": 181, "retained_av_ratio": 0.46761984861227923}, {"method": "omnizip_r45", "video": "9ODSLaN3JFc.mp4", "idx": 34, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4455270767211914, "peak_mem_gb": 9.116836547851562, "n_input_tokens": 1762, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 0.4521795705920625}, {"method": "omnizip_r45", "video": "ijg1smtVFIQ.mp4", "idx": 36, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3792777061462402, "peak_mem_gb": 9.301447868347168, "n_input_tokens": 2943, "n_video_tokens": 2304, "n_audio_tokens": 420, "retained_av_ratio": 0.4511747430249633}, {"method": "omnizip_r45", "video": "-abNj3Imno8.mp4", "idx": 38, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.00814151763916, "peak_mem_gb": 9.224681854248047, "n_input_tokens": 2530, "n_video_tokens": 2016, "n_audio_tokens": 356, "retained_av_ratio": 0.44898819561551434}, {"method": "omnizip_r45", "video": "YYoaOGBfSE4.mp4", "idx": 40, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.388381004333496, "peak_mem_gb": 9.117761135101318, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 0.44937949052906595}, {"method": "omnizip_r45", "video": "ZQ3iu3wMCPg.mp4", "idx": 42, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0284700393676758, "peak_mem_gb": 9.02387285232544, "n_input_tokens": 1054, "n_video_tokens": 720, "n_audio_tokens": 130, "retained_av_ratio": 0.5305882352941177}, {"method": "omnizip_r45", "video": "_SrDOLOuTWE.mp4", "idx": 44, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 10.414945125579834, "peak_mem_gb": 10.524834632873535, "n_input_tokens": 10196, "n_video_tokens": 8496, "n_audio_tokens": 1487, "retained_av_ratio": 0.4489632375037564}, {"method": "omnizip_r45", "video": "mb5_ZoHnqKM.mp4", "idx": 46, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.7937004566192627, "peak_mem_gb": 9.589613914489746, "n_input_tokens": 4666, "n_video_tokens": 3744, "n_audio_tokens": 666, "retained_av_ratio": 0.4492063492063492}, {"method": "omnizip_r45", "video": "m_IUCFFELcY.mp4", "idx": 48, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4609651565551758, "peak_mem_gb": 9.127691745758057, "n_input_tokens": 1785, "n_video_tokens": 1296, "n_audio_tokens": 247, "retained_av_ratio": 0.46143875567077125}, {"method": "omnizip_r45", "video": "ESBEIJcZInw.mp4", "idx": 50, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9953784942626953, "peak_mem_gb": 9.057539463043213, "n_input_tokens": 1260, "n_video_tokens": 840, "n_audio_tokens": 164, "retained_av_ratio": 0.49701195219123506}, {"method": "omnizip_r45", "video": "ltv6vNSstPI.mp4", "idx": 52, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3055076599121094, "peak_mem_gb": 9.304698944091797, "n_input_tokens": 2953, "n_video_tokens": 2304, "n_audio_tokens": 412, "retained_av_ratio": 0.4495581737849779}, {"method": "omnizip_r45", "video": "R3JcXA8OmEM.mp4", "idx": 54, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.984612464904785, "peak_mem_gb": 9.435484409332275, "n_input_tokens": 3828, "n_video_tokens": 3080, "n_audio_tokens": 559, "retained_av_ratio": 0.45039846111569115}, {"method": "omnizip_r45", "video": "-H5Yu7-Zb90.mp4", "idx": 56, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6654016971588135, "peak_mem_gb": 9.164394855499268, "n_input_tokens": 2078, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 0.4517333333333333}, {"method": "omnizip_r45", "video": "TJQ22uyc6XQ.mp4", "idx": 58, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.6899833679199219, "peak_mem_gb": 8.965907096862793, "n_input_tokens": 707, "n_video_tokens": 432, "n_audio_tokens": 80, "retained_av_ratio": 0.578125}, {"method": "omnizip_r45", "video": "9G-6yw9kGIY.mp4", "idx": 60, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.17938756942749, "peak_mem_gb": 9.9945387840271, "n_input_tokens": 7182, "n_video_tokens": 5904, "n_audio_tokens": 1049, "retained_av_ratio": 0.45332949805839207}, {"method": "omnizip_r45", "video": "zpeY4mrLlvY.mp4", "idx": 62, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6793639659881592, "peak_mem_gb": 9.172745704650879, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.45154419595314166}, {"method": "omnizip_r45", "video": "B1U-6d5J7Vw.mp4", "idx": 64, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.3281309604644775, "peak_mem_gb": 9.705991268157959, "n_input_tokens": 5310, "n_video_tokens": 4320, "n_audio_tokens": 773, "retained_av_ratio": 0.4496367563322207}, {"method": "omnizip_r45", "video": "U453o9dA3cs.mp4", "idx": 66, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8003783226013184, "peak_mem_gb": 9.386731624603271, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 0.4504029758214507}, {"method": "omnizip_r45", "video": "WtjJETMkuiM.mp4", "idx": 68, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.09395694732666, "peak_mem_gb": 9.429253101348877, "n_input_tokens": 3771, "n_video_tokens": 3024, "n_audio_tokens": 553, "retained_av_ratio": 0.4520547945205479}, {"method": "omnizip_r45", "video": "LFIUeTL2ClU.mp4", "idx": 70, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.779750347137451, "peak_mem_gb": 9.37516736984253, "n_input_tokens": 3431, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 0.45080445544554454}, {"method": "omnizip_r45", "video": "31zomD2ZORA.mp4", "idx": 72, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4750151634216309, "peak_mem_gb": 9.13494348526001, "n_input_tokens": 1818, "n_video_tokens": 1296, "n_audio_tokens": 244, "retained_av_ratio": 0.4538961038961039}, {"method": "omnizip_r45", "video": "Hkzmq5SO0SE.mp4", "idx": 74, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.36442232131958, "peak_mem_gb": 10.54781436920166, "n_input_tokens": 10172, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4498699219531719}, {"method": "omnizip_r45", "video": "81SdVr4Cqb8.mp4", "idx": 76, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0512745380401611, "peak_mem_gb": 9.018774509429932, "n_input_tokens": 934, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5877318116975749}, {"method": "omnizip_r45", "video": "kCPt7hqgYis.mp4", "idx": 78, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.204392671585083, "peak_mem_gb": 9.050653457641602, "n_input_tokens": 1253, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4946550048590865}, {"method": "omnizip_r45", "video": "AbvUzIEW6lw.mp4", "idx": 80, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.2857823371887207, "peak_mem_gb": 9.484695434570312, "n_input_tokens": 4026, "n_video_tokens": 3168, "n_audio_tokens": 554, "retained_av_ratio": 0.4492208490059108}, {"method": "omnizip_r45", "video": "xPTy0rDrzU0.mp4", "idx": 82, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.2094955444335938, "peak_mem_gb": 9.446803569793701, "n_input_tokens": 3827, "n_video_tokens": 3024, "n_audio_tokens": 542, "retained_av_ratio": 0.44980370162647226}, {"method": "omnizip_r45", "video": "GD-P6jsAGWU.mp4", "idx": 84, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.072552919387817, "peak_mem_gb": 10.431095600128174, "n_input_tokens": 9847, "n_video_tokens": 8208, "n_audio_tokens": 1444, "retained_av_ratio": 0.4495441359303771}, {"method": "omnizip_r45", "video": "aGqCwX-9Oh4.mp4", "idx": 86, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.448296308517456, "peak_mem_gb": 9.076325416564941, "n_input_tokens": 1543, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 0.4515890613451589}, {"method": "omnizip_r45", "video": "iIDhWvE6jR0.mp4", "idx": 88, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.37249207496643, "peak_mem_gb": 11.519179821014404, "n_input_tokens": 11346, "n_video_tokens": 9216, "n_audio_tokens": 1901, "retained_av_ratio": 0.45659800305837905}, {"method": "omnizip_r45", "video": "b6dkq4sBDyg.mp4", "idx": 90, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.4052789211273193, "peak_mem_gb": 9.509133338928223, "n_input_tokens": 4282, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.4492753623188406}, {"method": "omnizip_r45", "video": "14__F3Bhv4E.mp4", "idx": 92, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7661466598510742, "peak_mem_gb": 9.203473567962646, "n_input_tokens": 2289, "n_video_tokens": 1728, "n_audio_tokens": 316, "retained_av_ratio": 0.4515655577299413}, {"method": "omnizip_r45", "video": "XxfYKJP7jkE.mp4", "idx": 94, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.1404926776885986, "peak_mem_gb": 9.469189167022705, "n_input_tokens": 3984, "n_video_tokens": 3168, "n_audio_tokens": 567, "retained_av_ratio": 0.4495314591700134}, {"method": "omnizip_r45", "video": "_OCNY0nkjGI.mp4", "idx": 96, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.6696038246154785, "peak_mem_gb": 10.20951509475708, "n_input_tokens": 8682, "n_video_tokens": 7200, "n_audio_tokens": 1257, "retained_av_ratio": 0.44850419770604233}, {"method": "omnizip_r45", "video": "0OCwUDKhb1c.mp4", "idx": 98, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.2487010955810547, "peak_mem_gb": 9.264852046966553, "n_input_tokens": 2742, "n_video_tokens": 2160, "n_audio_tokens": 383, "retained_av_ratio": 0.44907589461266223}, {"method": "omnizip_r45", "video": "abR9DrA5AyI.mp4", "idx": 100, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9473872184753418, "peak_mem_gb": 9.014890193939209, "n_input_tokens": 1073, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.48791714614499426}, {"method": "omnizip_r45", "video": "Xkuh_lyCZ0Y.mp4", "idx": 102, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6853718757629395, "peak_mem_gb": 9.376451969146729, "n_input_tokens": 3434, "n_video_tokens": 2736, "n_audio_tokens": 494, "retained_av_ratio": 0.45077399380804956}, {"method": "omnizip_r45", "video": "uo5ayLrm2xg.mp4", "idx": 104, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.742047786712646, "peak_mem_gb": 10.547049045562744, "n_input_tokens": 10246, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4494696818090855}, {"method": "omnizip_r45", "video": "SmdRGMI5qsM.mp4", "idx": 106, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.358367919921875, "peak_mem_gb": 9.098493576049805, "n_input_tokens": 1619, "n_video_tokens": 1152, "n_audio_tokens": 218, "retained_av_ratio": 0.45255474452554745}, {"method": "omnizip_r45", "video": "1kFJgInMGd8.mp4", "idx": 108, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3848271369934082, "peak_mem_gb": 9.09457015991211, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 0.45234604105571846}, {"method": "omnizip_r45", "video": "QodbVfAEx40.mp4", "idx": 110, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.410078763961792, "peak_mem_gb": 9.320565223693848, "n_input_tokens": 3088, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 0.45115636865723163}, {"method": "omnizip_r45", "video": "bSSkJPyZ5vk.mp4", "idx": 112, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.491753816604614, "peak_mem_gb": 10.36132287979126, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1397, "retained_av_ratio": 0.44950091231082967}, {"method": "omnizip_r45", "video": "-XaFOYkyICI.mp4", "idx": 114, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.776928186416626, "peak_mem_gb": 10.94026803970337, "n_input_tokens": 11168, "n_video_tokens": 9216, "n_audio_tokens": 1670, "retained_av_ratio": 0.4514973360279258}, {"method": "omnizip_r45", "video": "BVdjSumD5V4.mp4", "idx": 116, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9223229885101318, "peak_mem_gb": 8.998308181762695, "n_input_tokens": 1030, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.4652532391048292}, {"method": "omnizip_r45", "video": "UMEiemYRr6w.mp4", "idx": 118, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8127870559692383, "peak_mem_gb": 9.197664260864258, "n_input_tokens": 2273, "n_video_tokens": 1728, "n_audio_tokens": 322, "retained_av_ratio": 0.4521951219512195}, {"method": "omnizip_r45", "video": "_kVH8Wh7tBo.mp4", "idx": 120, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.126166820526123, "peak_mem_gb": 9.975058555603027, "n_input_tokens": 7140, "n_video_tokens": 5904, "n_audio_tokens": 1041, "retained_av_ratio": 0.4499640028797696}, {"method": "omnizip_r45", "video": "VTFUr1GXBqY.mp4", "idx": 122, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.278297185897827, "peak_mem_gb": 9.69353437423706, "n_input_tokens": 5257, "n_video_tokens": 4320, "n_audio_tokens": 755, "retained_av_ratio": 0.4496551724137931}, {"method": "omnizip_r45", "video": "y4x5PY9JBzw.mp4", "idx": 124, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.478984117507935, "peak_mem_gb": 10.40688943862915, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1432, "retained_av_ratio": 0.4495850622406639}, {"method": "omnizip_r45", "video": "HT9uHeaOBVY.mp4", "idx": 126, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.03743314743042, "peak_mem_gb": 9.037004470825195, "n_input_tokens": 1208, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "omnizip_r45", "video": "CjzBW4HOndQ.mp4", "idx": 128, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.518490314483643, "peak_mem_gb": 10.675412654876709, "n_input_tokens": 10748, "n_video_tokens": 8928, "n_audio_tokens": 1555, "retained_av_ratio": 0.4491080797481637}, {"method": "omnizip_r45", "video": "2Rl9dPOJk2w.mp4", "idx": 130, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.2736475467681885, "peak_mem_gb": 9.838339805603027, "n_input_tokens": 6176, "n_video_tokens": 5040, "n_audio_tokens": 878, "retained_av_ratio": 0.4486312943562014}, {"method": "omnizip_r45", "video": "Cno_MIpe7Yg.mp4", "idx": 132, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.335066080093384, "peak_mem_gb": 10.556572914123535, "n_input_tokens": 10415, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4488709200276107}, {"method": "omnizip_r45", "video": "ajvnJTQvl_Y.mp4", "idx": 134, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.323050737380981, "peak_mem_gb": 10.535204410552979, "n_input_tokens": 10171, "n_video_tokens": 8496, "n_audio_tokens": 1492, "retained_av_ratio": 0.44923908690428516}, {"method": "omnizip_r45", "video": "BcREVtMNh4w.mp4", "idx": 136, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0529475212097168, "peak_mem_gb": 9.021704196929932, "n_input_tokens": 1095, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.4820393974507532}, {"method": "omnizip_r45", "video": "WgGReZb7igo.mp4", "idx": 138, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1442575454711914, "peak_mem_gb": 9.041131973266602, "n_input_tokens": 1217, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.500485908649174}, {"method": "omnizip_r45", "video": "6wMx9nKgqNE.mp4", "idx": 140, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1732454299926758, "peak_mem_gb": 9.075718402862549, "n_input_tokens": 1430, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 0.47651006711409394}, {"method": "omnizip_r45", "video": "XslxClszbTU.mp4", "idx": 142, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.2762129306793213, "peak_mem_gb": 9.46877908706665, "n_input_tokens": 4015, "n_video_tokens": 3220, "n_audio_tokens": 582, "retained_av_ratio": 0.45028932140978434}, {"method": "omnizip_r45", "video": "MJKdCd93wDc.mp4", "idx": 144, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.170424461364746, "peak_mem_gb": 9.279481410980225, "n_input_tokens": 2788, "n_video_tokens": 2160, "n_audio_tokens": 401, "retained_av_ratio": 0.4525575946895744}, {"method": "omnizip_r45", "video": "FPCuC9ZQlFo.mp4", "idx": 146, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.425995588302612, "peak_mem_gb": 9.879264831542969, "n_input_tokens": 6580, "n_video_tokens": 5472, "n_audio_tokens": 967, "retained_av_ratio": 0.44898276129833825}, {"method": "omnizip_r45", "video": "fujA0PoO_5w.mp4", "idx": 148, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.220973014831543, "peak_mem_gb": 9.06803274154663, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 0.48255813953488375}, {"method": "omnizip_r45", "video": "EhwV8BqcpSA.mp4", "idx": 150, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6592614650726318, "peak_mem_gb": 9.150729656219482, "n_input_tokens": 2037, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45008008542445277}, {"method": "omnizip_r45", "video": "YnjbpidCSN8.mp4", "idx": 152, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.875197172164917, "peak_mem_gb": 9.015429019927979, "n_input_tokens": 927, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5848787446504993}, {"method": "omnizip_r45", "video": "mJpo1ijaQOs.mp4", "idx": 154, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.261111259460449, "peak_mem_gb": 9.295272827148438, "n_input_tokens": 2923, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 0.45073529411764707}, {"method": "omnizip_r45", "video": "Zi4RV-TrTus.mp4", "idx": 156, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.941348791122437, "peak_mem_gb": 10.419170379638672, "n_input_tokens": 9908, "n_video_tokens": 8208, "n_audio_tokens": 1438, "retained_av_ratio": 0.44920174165457183}, {"method": "omnizip_r45", "video": "Mmz1Aba-1P0.mp4", "idx": 158, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.279335975646973, "peak_mem_gb": 9.871943950653076, "n_input_tokens": 6373, "n_video_tokens": 5184, "n_audio_tokens": 902, "retained_av_ratio": 0.44889911271771277}, {"method": "omnizip_r45", "video": "w9H2ibZd9zk.mp4", "idx": 160, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.400275468826294, "peak_mem_gb": 9.105916976928711, "n_input_tokens": 1732, "n_video_tokens": 1296, "n_audio_tokens": 239, "retained_av_ratio": 0.4501628664495114}, {"method": "omnizip_r45", "video": "GiN9w2a7wTc.mp4", "idx": 162, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.747494697570801, "peak_mem_gb": 9.78158187866211, "n_input_tokens": 5816, "n_video_tokens": 4752, "n_audio_tokens": 832, "retained_av_ratio": 0.4487822349570201}, {"method": "omnizip_r45", "video": "a0PA53cVK-I.mp4", "idx": 164, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.7569549083709717, "peak_mem_gb": 9.638270378112793, "n_input_tokens": 4724, "n_video_tokens": 3744, "n_audio_tokens": 673, "retained_av_ratio": 0.45075843332578674}, {"method": "omnizip_r45", "video": "T7lDs4GGNjU.mp4", "idx": 166, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8188726902008057, "peak_mem_gb": 9.384161949157715, "n_input_tokens": 3457, "n_video_tokens": 2736, "n_audio_tokens": 490, "retained_av_ratio": 0.4504029758214507}, {"method": "omnizip_r45", "video": "zutS7gJ4Po0.mp4", "idx": 168, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.36860990524292, "peak_mem_gb": 9.501657962799072, "n_input_tokens": 4256, "n_video_tokens": 3456, "n_audio_tokens": 605, "retained_av_ratio": 0.4486579660182221}, {"method": "omnizip_r45", "video": "_GTQj7_xyME.mp4", "idx": 170, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 12.376801252365112, "peak_mem_gb": 12.45910120010376, "n_input_tokens": 11704, "n_video_tokens": 9216, "n_audio_tokens": 2226, "retained_av_ratio": 0.4650410767348366}, {"method": "omnizip_r45", "video": "pcnVQwKXsSc.mp4", "idx": 172, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2110357284545898, "peak_mem_gb": 9.074473857879639, "n_input_tokens": 1461, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.45356550580431176}, {"method": "omnizip_r45", "video": "eE2SsX4F8TM.mp4", "idx": 174, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.772707462310791, "peak_mem_gb": 9.013525009155273, "n_input_tokens": 923, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5820256776034237}, {"method": "omnizip_r45", "video": "0CCRotntwdA.mp4", "idx": 176, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6418938636779785, "peak_mem_gb": 9.149483680725098, "n_input_tokens": 2034, "n_video_tokens": 1584, "n_audio_tokens": 293, "retained_av_ratio": 0.4507192328183271}, {"method": "omnizip_r45", "video": "wREFmTKv5DM.mp4", "idx": 178, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.615975618362427, "peak_mem_gb": 10.383824825286865, "n_input_tokens": 9675, "n_video_tokens": 8064, "n_audio_tokens": 1421, "retained_av_ratio": 0.44913020558777017}, {"method": "omnizip_r45", "video": "h7Mjce0y7SY.mp4", "idx": 180, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0624666213989258, "peak_mem_gb": 9.052568912506104, "n_input_tokens": 1263, "n_video_tokens": 864, "n_audio_tokens": 168, "retained_av_ratio": 0.49224806201550386}, {"method": "omnizip_r45", "video": "V2CSquBY4pk.mp4", "idx": 182, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.427307605743408, "peak_mem_gb": 9.891140937805176, "n_input_tokens": 6521, "n_video_tokens": 5328, "n_audio_tokens": 928, "retained_av_ratio": 0.4493286445012788}, {"method": "omnizip_r45", "video": "bHsmSHIoThw.mp4", "idx": 184, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5920076370239258, "peak_mem_gb": 9.132380962371826, "n_input_tokens": 1894, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.4547048509643483}, {"method": "omnizip_r45", "video": "WLZ8GAD_Mus.mp4", "idx": 186, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.1735970973968506, "peak_mem_gb": 9.267780780792236, "n_input_tokens": 2753, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 0.45023510971786834}, {"method": "omnizip_r45", "video": "UaBfM-tONmI.mp4", "idx": 188, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.896426439285278, "peak_mem_gb": 11.359811782836914, "n_input_tokens": 11265, "n_video_tokens": 9216, "n_audio_tokens": 1840, "retained_av_ratio": 0.4556801736613603}, {"method": "omnizip_r45", "video": "KF19sEfFHM8.mp4", "idx": 190, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7236907482147217, "peak_mem_gb": 8.97547960281372, "n_input_tokens": 739, "n_video_tokens": 432, "n_audio_tokens": 93, "retained_av_ratio": 0.5847619047619048}, {"method": "omnizip_r45", "video": "7Np8y8RdAxA.mp4", "idx": 192, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8735971450805664, "peak_mem_gb": 9.198527812957764, "n_input_tokens": 2358, "n_video_tokens": 1872, "n_audio_tokens": 326, "retained_av_ratio": 0.44904458598726116}, {"method": "omnizip_r45", "video": "x-RjxvYlzrE.mp4", "idx": 194, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7125699520111084, "peak_mem_gb": 8.99390459060669, "n_input_tokens": 795, "n_video_tokens": 432, "n_audio_tokens": 97, "retained_av_ratio": 0.5841209829867675}, {"method": "omnizip_r45", "video": "FkKh29UxzEc.mp4", "idx": 196, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.912856817245483, "peak_mem_gb": 12.122392177581787, "n_input_tokens": 11521, "n_video_tokens": 9216, "n_audio_tokens": 2115, "retained_av_ratio": 0.4628011649457241}, {"method": "omnizip_r45", "video": "fYYV5CoB5QM.mp4", "idx": 198, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5172665119171143, "peak_mem_gb": 9.131096363067627, "n_input_tokens": 1807, "n_video_tokens": 1296, "n_audio_tokens": 235, "retained_av_ratio": 0.45003265839320705}, {"method": "omnizip_r45", "video": "nsbOHGP_uGI.mp4", "idx": 200, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.7601075172424316, "peak_mem_gb": 8.979632377624512, "n_input_tokens": 751, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 0.5921052631578947}, {"method": "omnizip_r45", "video": "4tyX4O4it5U.mp4", "idx": 202, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.774900197982788, "peak_mem_gb": 10.430200099945068, "n_input_tokens": 9878, "n_video_tokens": 8208, "n_audio_tokens": 1443, "retained_av_ratio": 0.44938348357683144}, {"method": "omnizip_r45", "video": "-32uRUUd-KU.mp4", "idx": 204, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.957655668258667, "peak_mem_gb": 9.04426097869873, "n_input_tokens": 1139, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "omnizip_r45", "video": "LQsQCioPS4U.mp4", "idx": 206, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1085712909698486, "peak_mem_gb": 9.046483993530273, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 154, "retained_av_ratio": 0.4931237721021611}, {"method": "omnizip_r45", "video": "21IQzeQBnGo.mp4", "idx": 208, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.887115716934204, "peak_mem_gb": 9.217224597930908, "n_input_tokens": 2420, "n_video_tokens": 1872, "n_audio_tokens": 334, "retained_av_ratio": 0.4496826835902085}, {"method": "omnizip_r45", "video": "Q0GG4iuGL3g.mp4", "idx": 210, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0414011478424072, "peak_mem_gb": 9.03349781036377, "n_input_tokens": 1199, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4926829268292683}, {"method": "omnizip_r45", "video": "PBhCY56XIcI.mp4", "idx": 212, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1324381828308105, "peak_mem_gb": 9.052009105682373, "n_input_tokens": 1391, "n_video_tokens": 1008, "n_audio_tokens": 201, "retained_av_ratio": 0.45492142266335817}, {"method": "omnizip_r45", "video": "eEV120knBoA.mp4", "idx": 214, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7352957725524902, "peak_mem_gb": 9.391228675842285, "n_input_tokens": 3478, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 0.4520717377860235}, {"method": "omnizip_r45", "video": "2lXyTywKtkw.mp4", "idx": 216, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.155632019042969, "peak_mem_gb": 9.670341968536377, "n_input_tokens": 5093, "n_video_tokens": 4176, "n_audio_tokens": 747, "retained_av_ratio": 0.45053828966077597}, {"method": "omnizip_r45", "video": "olx5oFJbivo.mp4", "idx": 218, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.149531602859497, "peak_mem_gb": 10.455708026885986, "n_input_tokens": 10026, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": 0.44916896094626285}, {"method": "omnizip_r45", "video": "_uDlP4vW190.mp4", "idx": 220, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.597428560256958, "peak_mem_gb": 10.340779781341553, "n_input_tokens": 9506, "n_video_tokens": 7920, "n_audio_tokens": 1394, "retained_av_ratio": 0.44953832939660726}, {"method": "omnizip_r45", "video": "ZJovAHQU0o8.mp4", "idx": 222, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0532333850860596, "peak_mem_gb": 9.053826808929443, "n_input_tokens": 1261, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "omnizip_r45", "video": "YB3bxufnn7s.mp4", "idx": 224, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4464735984802246, "peak_mem_gb": 9.098107814788818, "n_input_tokens": 1701, "n_video_tokens": 1296, "n_audio_tokens": 237, "retained_av_ratio": 0.45401174168297453}, {"method": "omnizip_r45", "video": "SexJ0OetXZA.mp4", "idx": 226, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.6624398231506348, "peak_mem_gb": 8.968206882476807, "n_input_tokens": 723, "n_video_tokens": 432, "n_audio_tokens": 100, "retained_av_ratio": 0.5770676691729323}, {"method": "omnizip_r45", "video": "3l4sUgCflzc.mp4", "idx": 228, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9787628650665283, "peak_mem_gb": 9.007246017456055, "n_input_tokens": 1052, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.4802784222737819}, {"method": "omnizip_r45", "video": "KwhBtGE-mDc.mp4", "idx": 230, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.401922225952148, "peak_mem_gb": 10.557207107543945, "n_input_tokens": 10305, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4488266614080063}, {"method": "omnizip_r45", "video": "BojPeImI2QE.mp4", "idx": 232, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.455384016036987, "peak_mem_gb": 10.554561614990234, "n_input_tokens": 10220, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44953486045813745}, {"method": "omnizip_r45", "video": "4NlfaBAaGrM.mp4", "idx": 234, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.314038991928101, "peak_mem_gb": 9.705988883972168, "n_input_tokens": 5305, "n_video_tokens": 4320, "n_audio_tokens": 760, "retained_av_ratio": 0.44921259842519684}, {"method": "omnizip_r45", "video": "2Yp6UqQPQNE.mp4", "idx": 236, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.203689098358154, "peak_mem_gb": 10.488563060760498, "n_input_tokens": 10012, "n_video_tokens": 8352, "n_audio_tokens": 1471, "retained_av_ratio": 0.4489463504021175}, {"method": "omnizip_r45", "video": "qzUdGKGEF6o.mp4", "idx": 238, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5667073726654053, "peak_mem_gb": 9.116217613220215, "n_input_tokens": 1761, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45330739299610895}, {"method": "omnizip_r45", "video": "WgWdcSg5_to.mp4", "idx": 240, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.589823722839355, "peak_mem_gb": 10.539609432220459, "n_input_tokens": 10225, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": 0.44994994994994997}, {"method": "omnizip_r45", "video": "rD5uIy9UgxA.mp4", "idx": 242, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4601974487304688, "peak_mem_gb": 9.111191749572754, "n_input_tokens": 1646, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 0.4636627906976744}, {"method": "omnizip_r45", "video": "y_MMvaZvGIA.mp4", "idx": 244, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.954668998718262, "peak_mem_gb": 9.95290756225586, "n_input_tokens": 6979, "n_video_tokens": 5760, "n_audio_tokens": 1010, "retained_av_ratio": 0.44918759231905464}, {"method": "omnizip_r45", "video": "HO65gX-k31g.mp4", "idx": 246, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2026245594024658, "peak_mem_gb": 9.065366744995117, "n_input_tokens": 1295, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.4921259842519685}, {"method": "omnizip_r45", "video": "DGNI8-twVys.mp4", "idx": 248, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7773807048797607, "peak_mem_gb": 9.370350360870361, "n_input_tokens": 3415, "n_video_tokens": 2736, "n_audio_tokens": 497, "retained_av_ratio": 0.4512836374884009}, {"method": "omnizip_r45", "video": "Igf8n7hy1HI.mp4", "idx": 250, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1810755729675293, "peak_mem_gb": 9.067954063415527, "n_input_tokens": 1408, "n_video_tokens": 1008, "n_audio_tokens": 176, "retained_av_ratio": 0.47128378378378377}, {"method": "omnizip_r45", "video": "JPqjEZXKoQc.mp4", "idx": 252, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6781392097473145, "peak_mem_gb": 9.139312267303467, "n_input_tokens": 1912, "n_video_tokens": 1440, "n_audio_tokens": 253, "retained_av_ratio": 0.4524512699350266}, {"method": "omnizip_r45", "video": "IhiQi9X8l8o.mp4", "idx": 254, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.407042741775513, "peak_mem_gb": 10.1538405418396, "n_input_tokens": 8332, "n_video_tokens": 6912, "n_audio_tokens": 1210, "retained_av_ratio": 0.44902733316917015}, {"method": "omnizip_r45", "video": "vR8fuThRlA4.mp4", "idx": 256, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5612773895263672, "peak_mem_gb": 9.13388204574585, "n_input_tokens": 1900, "n_video_tokens": 1440, "n_audio_tokens": 257, "retained_av_ratio": 0.44961697112551563}, {"method": "omnizip_r45", "video": "qP6IYn3MrcA.mp4", "idx": 258, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.665640830993652, "peak_mem_gb": 10.102481842041016, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 0.4491210060310535}, {"method": "omnizip_r45", "video": "WBtbl0O4CZA.mp4", "idx": 260, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.546719312667847, "peak_mem_gb": 10.638450145721436, "n_input_tokens": 10580, "n_video_tokens": 8784, "n_audio_tokens": 1539, "retained_av_ratio": 0.44938486873970745}, {"method": "omnizip_r45", "video": "HAjJZ1GOXFA.mp4", "idx": 262, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7719876766204834, "peak_mem_gb": 8.971384525299072, "n_input_tokens": 738, "n_video_tokens": 432, "n_audio_tokens": 101, "retained_av_ratio": 0.5684803001876173}, {"method": "omnizip_r45", "video": "kzZ7TfwT9Ss.mp4", "idx": 264, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.938629627227783, "peak_mem_gb": 10.42507266998291, "n_input_tokens": 9863, "n_video_tokens": 8208, "n_audio_tokens": 1441, "retained_av_ratio": 0.4486475282412685}, {"method": "omnizip_r45", "video": "oZlh0hjcXfA.mp4", "idx": 266, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4450604915618896, "peak_mem_gb": 9.313786506652832, "n_input_tokens": 3066, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 0.44892286309937457}, {"method": "omnizip_r45", "video": "hIWZDSR_uns.mp4", "idx": 268, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3362483978271484, "peak_mem_gb": 9.27883005142212, "n_input_tokens": 2784, "n_video_tokens": 2160, "n_audio_tokens": 392, "retained_av_ratio": 0.45141065830721006}, {"method": "omnizip_r45", "video": "91kNCJOoHcY.mp4", "idx": 270, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1193954944610596, "peak_mem_gb": 9.046475887298584, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.4921259842519685}, {"method": "omnizip_r45", "video": "4HYLaF6fbFY.mp4", "idx": 272, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.670814275741577, "peak_mem_gb": 9.37225341796875, "n_input_tokens": 3330, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 0.45184459680052236}, {"method": "omnizip_r45", "video": "dtGj9oet70E.mp4", "idx": 274, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.291043996810913, "peak_mem_gb": 10.55712890625, "n_input_tokens": 10285, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4490238611713666}, {"method": "omnizip_r45", "video": "PwSOZli5_xg.mp4", "idx": 276, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.676975965499878, "peak_mem_gb": 9.768975257873535, "n_input_tokens": 5769, "n_video_tokens": 4752, "n_audio_tokens": 843, "retained_av_ratio": 0.45004468275245757}, {"method": "omnizip_r45", "video": "MYM3IFheI-E.mp4", "idx": 278, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1244959831237793, "peak_mem_gb": 9.060871124267578, "n_input_tokens": 1403, "n_video_tokens": 1008, "n_audio_tokens": 195, "retained_av_ratio": 0.46550290939318373}, {"method": "omnizip_r45", "video": "EB9bA1a5TEs.mp4", "idx": 280, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4066071510314941, "peak_mem_gb": 9.121941566467285, "n_input_tokens": 1780, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 0.4540750323415265}, {"method": "omnizip_r45", "video": "oZLqkTvWXko.mp4", "idx": 282, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5576024055480957, "peak_mem_gb": 9.138713836669922, "n_input_tokens": 1918, "n_video_tokens": 1440, "n_audio_tokens": 266, "retained_av_ratio": 0.45193434935521687}, {"method": "omnizip_r45", "video": "kS6aRrdqI7M.mp4", "idx": 284, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.547750949859619, "peak_mem_gb": 9.334038734436035, "n_input_tokens": 3122, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 0.44869565217391305}, {"method": "omnizip_r45", "video": "1P6XRN-amuU.mp4", "idx": 286, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6134490966796875, "peak_mem_gb": 9.125555515289307, "n_input_tokens": 1876, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 0.4532710280373832}, {"method": "omnizip_r45", "video": "akm-KQiLsvI.mp4", "idx": 288, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.239523410797119, "peak_mem_gb": 9.48619031906128, "n_input_tokens": 4120, "n_video_tokens": 3312, "n_audio_tokens": 584, "retained_av_ratio": 0.449435318275154}, {"method": "omnizip_r45", "video": "JjuQ2ink05Q.mp4", "idx": 290, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6794772148132324, "peak_mem_gb": 9.580150127410889, "n_input_tokens": 4632, "n_video_tokens": 3744, "n_audio_tokens": 660, "retained_av_ratio": 0.4493642143505904}, {"method": "omnizip_r45", "video": "TQjp3EaFCsc.mp4", "idx": 292, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.24075984954834, "peak_mem_gb": 9.854316234588623, "n_input_tokens": 6320, "n_video_tokens": 5184, "n_audio_tokens": 919, "retained_av_ratio": 0.4494510896280518}, {"method": "omnizip_r45", "video": "pyfj75fxjj8.mp4", "idx": 294, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5434234142303467, "peak_mem_gb": 9.310022830963135, "n_input_tokens": 2972, "n_video_tokens": 2304, "n_audio_tokens": 422, "retained_av_ratio": 0.450476889214967}, {"method": "omnizip_r45", "video": "05FGA7Febzc.mp4", "idx": 296, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9817748069763184, "peak_mem_gb": 9.008442878723145, "n_input_tokens": 907, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "omnizip_r45", "video": "8OqmJB8oGtM.mp4", "idx": 298, "problem_type": "Narrative Comprehension", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 5.63 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.485508441925049, "n_input_tokens": 13764, "n_video_tokens": 9216, "n_audio_tokens": 4347, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "bajfnk7lwL4.mp4", "idx": 300, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.71704363822937, "peak_mem_gb": 9.384805679321289, "n_input_tokens": 3462, "n_video_tokens": 2736, "n_audio_tokens": 500, "retained_av_ratio": 0.4511742892459827}, {"method": "omnizip_r45", "video": "mrYBGL9-DDE.mp4", "idx": 302, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.96196174621582, "peak_mem_gb": 9.83223581314087, "n_input_tokens": 6155, "n_video_tokens": 5040, "n_audio_tokens": 890, "retained_av_ratio": 0.44974704890387857}, {"method": "omnizip_r45", "video": "odgiRt2sitM.mp4", "idx": 304, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.859614372253418, "peak_mem_gb": 9.029514789581299, "n_input_tokens": 969, "n_video_tokens": 576, "n_audio_tokens": 112, "retained_av_ratio": 0.5784883720930233}, {"method": "omnizip_r45", "video": "iIZiJR-MHnM.mp4", "idx": 306, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7120084762573242, "peak_mem_gb": 9.173067092895508, "n_input_tokens": 2106, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.4520766773162939}, {"method": "omnizip_r45", "video": "vcsHoRWlD7E.mp4", "idx": 308, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.748145341873169, "peak_mem_gb": 9.188485145568848, "n_input_tokens": 2154, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45061398825413773}, {"method": "omnizip_r45", "video": "ge-zGSapyeQ.mp4", "idx": 310, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0632884502410889, "peak_mem_gb": 9.056365966796875, "n_input_tokens": 1269, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4965986394557823}, {"method": "omnizip_r45", "video": "eAOKPLl6AVY.mp4", "idx": 312, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3641583919525146, "peak_mem_gb": 9.491066932678223, "n_input_tokens": 4134, "n_video_tokens": 3312, "n_audio_tokens": 593, "retained_av_ratio": 0.45096030729833547}, {"method": "omnizip_r45", "video": "HchGCB08bLg.mp4", "idx": 314, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9141561985015869, "peak_mem_gb": 9.011565685272217, "n_input_tokens": 911, "n_video_tokens": 576, "n_audio_tokens": 111, "retained_av_ratio": 0.5822416302765647}, {"method": "omnizip_r45", "video": "ECvz4IPwFQ8.mp4", "idx": 316, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.327603101730347, "peak_mem_gb": 10.324250221252441, "n_input_tokens": 9358, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 0.4498686514886165}, {"method": "omnizip_r45", "video": "Bj7N9gscARc.mp4", "idx": 318, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.882579803466797, "peak_mem_gb": 9.402460098266602, "n_input_tokens": 3515, "n_video_tokens": 2736, "n_audio_tokens": 501, "retained_av_ratio": 0.45103490886623415}, {"method": "omnizip_r45", "video": "3IIaJnEspcI.mp4", "idx": 320, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.717767000198364, "peak_mem_gb": 9.915806293487549, "n_input_tokens": 6692, "n_video_tokens": 5472, "n_audio_tokens": 974, "retained_av_ratio": 0.4494260006205399}, {"method": "omnizip_r45", "video": "rYtG6G6tYG0.mp4", "idx": 322, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0329313278198242, "peak_mem_gb": 9.022289752960205, "n_input_tokens": 1050, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 0.5299647473560517}, {"method": "omnizip_r45", "video": "aneq_Adxc8U.mp4", "idx": 324, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1179492473602295, "peak_mem_gb": 9.058916091918945, "n_input_tokens": 1384, "n_video_tokens": 1008, "n_audio_tokens": 182, "retained_av_ratio": 0.47058823529411764}, {"method": "omnizip_r45", "video": "4u7eslydITE.mp4", "idx": 326, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.847688436508179, "peak_mem_gb": 10.490667819976807, "n_input_tokens": 10022, "n_video_tokens": 8352, "n_audio_tokens": 1472, "retained_av_ratio": 0.4495114006514658}, {"method": "omnizip_r45", "video": "5_tgh8PTlxk.mp4", "idx": 328, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1556241512298584, "peak_mem_gb": 9.273305892944336, "n_input_tokens": 2767, "n_video_tokens": 2160, "n_audio_tokens": 391, "retained_av_ratio": 0.45119560956487653}, {"method": "omnizip_r45", "video": "xyt5IrIgprc.mp4", "idx": 330, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.106513261795044, "peak_mem_gb": 9.24924087524414, "n_input_tokens": 2609, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 0.44970662196144173}, {"method": "omnizip_r45", "video": "huSqwxMjL-Q.mp4", "idx": 332, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3457694053649902, "peak_mem_gb": 9.118668556213379, "n_input_tokens": 1679, "n_video_tokens": 1152, "n_audio_tokens": 224, "retained_av_ratio": 0.4563953488372093}, {"method": "omnizip_r45", "video": "2IvR2LGWvcg.mp4", "idx": 334, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.9357526302337646, "peak_mem_gb": 9.39368486404419, "n_input_tokens": 3574, "n_video_tokens": 2880, "n_audio_tokens": 502, "retained_av_ratio": 0.4482554701360142}, {"method": "omnizip_r45", "video": "4WcVCJr0UwY.mp4", "idx": 336, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8106119632720947, "peak_mem_gb": 9.006868362426758, "n_input_tokens": 903, "n_video_tokens": 576, "n_audio_tokens": 127, "retained_av_ratio": 0.5817923186344239}, {"method": "omnizip_r45", "video": "t3YK5ezf8u4.mp4", "idx": 338, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.0587010383605957, "peak_mem_gb": 9.256307125091553, "n_input_tokens": 2627, "n_video_tokens": 2016, "n_audio_tokens": 358, "retained_av_ratio": 0.44860994102780116}, {"method": "omnizip_r45", "video": "1pNEF7I8EIk.mp4", "idx": 340, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.847228050231934, "peak_mem_gb": 10.389559268951416, "n_input_tokens": 9747, "n_video_tokens": 8064, "n_audio_tokens": 1405, "retained_av_ratio": 0.44872742633857854}, {"method": "omnizip_r45", "video": "3D5kuvs3LY8.mp4", "idx": 342, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3807482719421387, "peak_mem_gb": 9.109878063201904, "n_input_tokens": 1741, "n_video_tokens": 1296, "n_audio_tokens": 248, "retained_av_ratio": 0.4540155440414508}, {"method": "omnizip_r45", "video": "zdvIyL1uFeY.mp4", "idx": 344, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.464069366455078, "peak_mem_gb": 9.773133277893066, "n_input_tokens": 5701, "n_video_tokens": 4608, "n_audio_tokens": 821, "retained_av_ratio": 0.45017498618530116}, {"method": "omnizip_r45", "video": "yB0eIpT8DBM.mp4", "idx": 346, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8702106475830078, "peak_mem_gb": 9.01064157485962, "n_input_tokens": 910, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 0.5835734870317003}, {"method": "omnizip_r45", "video": "MbiOwSUz4TY.mp4", "idx": 348, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2217442989349365, "peak_mem_gb": 9.084453582763672, "n_input_tokens": 1571, "n_video_tokens": 1152, "n_audio_tokens": 202, "retained_av_ratio": 0.44977843426883307}, {"method": "omnizip_r45", "video": "6C33lbRzaMc.mp4", "idx": 350, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5034382343292236, "peak_mem_gb": 9.521168231964111, "n_input_tokens": 4275, "n_video_tokens": 3432, "n_audio_tokens": 602, "retained_av_ratio": 0.44942984630639565}, {"method": "omnizip_r45", "video": "zBC5SPmMBiI.mp4", "idx": 352, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.269805431365967, "peak_mem_gb": 10.540867328643799, "n_input_tokens": 10279, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 0.4493043739365429}, {"method": "omnizip_r45", "video": "cI7g9nPVw8s.mp4", "idx": 354, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.7249979972839355, "peak_mem_gb": 9.593721866607666, "n_input_tokens": 4674, "n_video_tokens": 3744, "n_audio_tokens": 663, "retained_av_ratio": 0.4499659632402995}, {"method": "omnizip_r45", "video": "V8ipyMRS5co.mp4", "idx": 356, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.711008310317993, "peak_mem_gb": 10.55671215057373, "n_input_tokens": 10451, "n_video_tokens": 8640, "n_audio_tokens": 1501, "retained_av_ratio": 0.4494625776550636}, {"method": "omnizip_r45", "video": "aw0_Q6ZOx0o.mp4", "idx": 358, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5593159198760986, "peak_mem_gb": 9.148183345794678, "n_input_tokens": 1944, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 0.4499411071849234}, {"method": "omnizip_r45", "video": "gJjosUoZiWs.mp4", "idx": 360, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.884181499481201, "peak_mem_gb": 9.95923137664795, "n_input_tokens": 7004, "n_video_tokens": 5760, "n_audio_tokens": 1009, "retained_av_ratio": 0.44881075491209926}, {"method": "omnizip_r45", "video": "i2fOL059y1A.mp4", "idx": 362, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.246623516082764, "peak_mem_gb": 9.699318885803223, "n_input_tokens": 5276, "n_video_tokens": 4320, "n_audio_tokens": 753, "retained_av_ratio": 0.4496353242657205}, {"method": "omnizip_r45", "video": "j5wq7pLFlU4.mp4", "idx": 364, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.029168367385864, "peak_mem_gb": 10.335119247436523, "n_input_tokens": 9390, "n_video_tokens": 7776, "n_audio_tokens": 1360, "retained_av_ratio": 0.4498686514886165}, {"method": "omnizip_r45", "video": "nprAhUMzWag.mp4", "idx": 366, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.312214136123657, "peak_mem_gb": 10.553366661071777, "n_input_tokens": 10183, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44953486045813745}, {"method": "omnizip_r45", "video": "IiHvzFP6VQk.mp4", "idx": 368, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.462630271911621, "peak_mem_gb": 10.32949686050415, "n_input_tokens": 9385, "n_video_tokens": 7776, "n_audio_tokens": 1376, "retained_av_ratio": 0.4493006993006993}, {"method": "omnizip_r45", "video": "oT-R-2YcqRQ.mp4", "idx": 370, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.358123064041138, "peak_mem_gb": 10.511322498321533, "n_input_tokens": 10208, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.4491330059136013}, {"method": "omnizip_r45", "video": "hfS1Io1bQSU.mp4", "idx": 372, "problem_type": "Narrative Comprehension", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.256853818893433, "peak_mem_gb": 10.53748369216919, "n_input_tokens": 10218, "n_video_tokens": 8496, "n_audio_tokens": 1493, "retained_av_ratio": 0.4496946641305436}, {"method": "omnizip_r45", "video": "rPRGQOics9A.mp4", "idx": 374, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.028864860534668, "peak_mem_gb": 9.05092716217041, "n_input_tokens": 1160, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "omnizip_r45", "video": "sYxaFHBsxKs.mp4", "idx": 376, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.0906877517700195, "peak_mem_gb": 10.145650863647461, "n_input_tokens": 8302, "n_video_tokens": 6912, "n_audio_tokens": 1205, "retained_av_ratio": 0.44893433534557103}, {"method": "omnizip_r45", "video": "Ibe_srSbN28.mp4", "idx": 378, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.0896031856536865, "peak_mem_gb": 9.88093376159668, "n_input_tokens": 6408, "n_video_tokens": 5184, "n_audio_tokens": 923, "retained_av_ratio": 0.44964794498116917}, {"method": "omnizip_r45", "video": "gRqQQYJDD5Y.mp4", "idx": 380, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.527829170227051, "peak_mem_gb": 9.503933429718018, "n_input_tokens": 4266, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.4492753623188406}, {"method": "omnizip_r45", "video": "T2VIyWfYZFI.mp4", "idx": 382, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.3921406269073486, "peak_mem_gb": 9.536531925201416, "n_input_tokens": 4346, "n_video_tokens": 3456, "n_audio_tokens": 615, "retained_av_ratio": 0.45074920167035126}, {"method": "omnizip_r45", "video": "3olxUYqO8cQ.mp4", "idx": 384, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.133574962615967, "peak_mem_gb": 9.284677982330322, "n_input_tokens": 2799, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.4493496255419787}, {"method": "omnizip_r45", "video": "XjZaY7vgAzg.mp4", "idx": 386, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.805140495300293, "peak_mem_gb": 9.189539432525635, "n_input_tokens": 2247, "n_video_tokens": 1728, "n_audio_tokens": 320, "retained_av_ratio": 0.4521484375}, {"method": "omnizip_r45", "video": "EPzdZd7DnsQ.mp4", "idx": 388, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.528695821762085, "peak_mem_gb": 9.327308654785156, "n_input_tokens": 3105, "n_video_tokens": 2448, "n_audio_tokens": 439, "retained_av_ratio": 0.45064080360235537}, {"method": "omnizip_r45", "video": "xwfHWw5A2HI.mp4", "idx": 390, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.542213201522827, "peak_mem_gb": 10.55726432800293, "n_input_tokens": 10320, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "omnizip_r45", "video": "otVWxaBSzL0.mp4", "idx": 392, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7902791500091553, "peak_mem_gb": 9.184988021850586, "n_input_tokens": 2231, "n_video_tokens": 1728, "n_audio_tokens": 319, "retained_av_ratio": 0.4528578407425501}, {"method": "omnizip_r45", "video": "s6C8atMqM1Q.mp4", "idx": 394, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.4170608520507812, "peak_mem_gb": 9.520901679992676, "n_input_tokens": 4291, "n_video_tokens": 3456, "n_audio_tokens": 614, "retained_av_ratio": 0.4498771498771499}, {"method": "omnizip_r45", "video": "xlC9oqCHQ84.mp4", "idx": 396, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4568378925323486, "peak_mem_gb": 9.107530117034912, "n_input_tokens": 1733, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 0.4516971279373368}, {"method": "omnizip_r45", "video": "wR4Za_pz2ec.mp4", "idx": 398, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.588036298751831, "peak_mem_gb": 9.16325855255127, "n_input_tokens": 1992, "n_video_tokens": 1440, "n_audio_tokens": 272, "retained_av_ratio": 0.4532710280373832}, {"method": "omnizip_r45", "video": "Nql8w5AYwAo.mp4", "idx": 400, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9721190929412842, "peak_mem_gb": 9.222397804260254, "n_input_tokens": 2436, "n_video_tokens": 1872, "n_audio_tokens": 347, "retained_av_ratio": 0.45290671473636773}, {"method": "omnizip_r45", "video": "1t0JN_QdBkA.mp4", "idx": 402, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.142086505889893, "peak_mem_gb": 10.149942398071289, "n_input_tokens": 8319, "n_video_tokens": 6912, "n_audio_tokens": 1209, "retained_av_ratio": 0.4489594877478143}, {"method": "omnizip_r45", "video": "-dtEpy4XUbQ.mp4", "idx": 404, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0035145282745361, "peak_mem_gb": 9.002117156982422, "n_input_tokens": 1042, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.4652532391048292}, {"method": "omnizip_r45", "video": "i9LSUR1OSjs.mp4", "idx": 406, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2538681030273438, "peak_mem_gb": 9.08481502532959, "n_input_tokens": 1472, "n_video_tokens": 1008, "n_audio_tokens": 183, "retained_av_ratio": 0.4643157010915197}, {"method": "omnizip_r45", "video": "F7M7aY5fofo.mp4", "idx": 408, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9699351787567139, "peak_mem_gb": 9.045197010040283, "n_input_tokens": 1126, "n_video_tokens": 720, "n_audio_tokens": 145, "retained_av_ratio": 0.5329479768786127}, {"method": "omnizip_r45", "video": "eMPOMRBnBpc.mp4", "idx": 410, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7956879138946533, "peak_mem_gb": 9.205080032348633, "n_input_tokens": 2294, "n_video_tokens": 1728, "n_audio_tokens": 318, "retained_av_ratio": 0.4521016617790811}, {"method": "omnizip_r45", "video": "bSppSU7Njig.mp4", "idx": 412, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.369787216186523, "peak_mem_gb": 10.557605743408203, "n_input_tokens": 10409, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "omnizip_r45", "video": "72rcNxQbIoI.mp4", "idx": 414, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0226960182189941, "peak_mem_gb": 9.025853633880615, "n_input_tokens": 1108, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.4856156501726122}, {"method": "omnizip_r45", "video": "jl38NDYzQV0.mp4", "idx": 416, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.1686203479766846, "peak_mem_gb": 9.46372652053833, "n_input_tokens": 3966, "n_video_tokens": 3168, "n_audio_tokens": 564, "retained_av_ratio": 0.44935691318327975}, {"method": "omnizip_r45", "video": "V30jgIuBG94.mp4", "idx": 418, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6232755184173584, "peak_mem_gb": 9.127382278442383, "n_input_tokens": 1881, "n_video_tokens": 1440, "n_audio_tokens": 261, "retained_av_ratio": 0.4503233392122281}, {"method": "omnizip_r45", "video": "9XbP_d_u_OI.mp4", "idx": 420, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.800222873687744, "peak_mem_gb": 9.950849533081055, "n_input_tokens": 6888, "n_video_tokens": 5616, "n_audio_tokens": 983, "retained_av_ratio": 0.4494620397029853}, {"method": "omnizip_r45", "video": "FP7u3iJxruw.mp4", "idx": 422, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.663774013519287, "peak_mem_gb": 10.564021587371826, "n_input_tokens": 10430, "n_video_tokens": 8640, "n_audio_tokens": 1505, "retained_av_ratio": 0.4487925086249384}, {"method": "omnizip_r45", "video": "DWqcf5QvkJI.mp4", "idx": 424, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7827081680297852, "peak_mem_gb": 9.224420547485352, "n_input_tokens": 2353, "n_video_tokens": 1728, "n_audio_tokens": 312, "retained_av_ratio": 0.45098039215686275}, {"method": "omnizip_r45", "video": "P_cYNsKKpGg.mp4", "idx": 426, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.794020652770996, "peak_mem_gb": 9.19311237335205, "n_input_tokens": 2253, "n_video_tokens": 1728, "n_audio_tokens": 301, "retained_av_ratio": 0.4494825036964022}, {"method": "omnizip_r45", "video": "J01MxqCqkyE.mp4", "idx": 428, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6419641971588135, "peak_mem_gb": 9.360069751739502, "n_input_tokens": 3381, "n_video_tokens": 2736, "n_audio_tokens": 486, "retained_av_ratio": 0.4500310366232154}, {"method": "omnizip_r45", "video": "wdESby14IZY.mp4", "idx": 430, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.832838773727417, "peak_mem_gb": 10.112696170806885, "n_input_tokens": 8023, "n_video_tokens": 6624, "n_audio_tokens": 1163, "retained_av_ratio": 0.44921022216514706}, {"method": "omnizip_r45", "video": "n7vwcWG5ZY0.mp4", "idx": 432, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.216995477676392, "peak_mem_gb": 10.544877529144287, "n_input_tokens": 10229, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.4499149404583208}, {"method": "omnizip_r45", "video": "18OOnHDat4E.mp4", "idx": 434, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.469623327255249, "peak_mem_gb": 9.486514568328857, "n_input_tokens": 4122, "n_video_tokens": 3312, "n_audio_tokens": 580, "retained_av_ratio": 0.4486125385405961}, {"method": "omnizip_r45", "video": "w6C29ECeFs8.mp4", "idx": 436, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0941152572631836, "peak_mem_gb": 9.02807331085205, "n_input_tokens": 1088, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "omnizip_r45", "video": "JB8rAfaUUKY.mp4", "idx": 438, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6750919818878174, "peak_mem_gb": 9.358439922332764, "n_input_tokens": 3286, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 0.45091623036649214}, {"method": "omnizip_r45", "video": "1YORkMT9JoY.mp4", "idx": 440, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1264371871948242, "peak_mem_gb": 9.056334495544434, "n_input_tokens": 1262, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49951028403525954}, {"method": "omnizip_r45", "video": "D980E_PsoIs.mp4", "idx": 442, "problem_type": "Temporal Reasoning Localization", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.3538477420806885, "peak_mem_gb": 9.329346179962158, "n_input_tokens": 3026, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 0.45137614678899085}, {"method": "omnizip_r45", "video": "nGHKrbtPRb4.mp4", "idx": 444, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.660468578338623, "peak_mem_gb": 9.35426378250122, "n_input_tokens": 3273, "n_video_tokens": 2592, "n_audio_tokens": 458, "retained_av_ratio": 0.4498360655737705}, {"method": "omnizip_r45", "video": "AmWG0oYhDh4.mp4", "idx": 446, "problem_type": "Temporal Reasoning Localization", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.687062501907349, "peak_mem_gb": 10.363263130187988, "n_input_tokens": 9573, "n_video_tokens": 7920, "n_audio_tokens": 1383, "retained_av_ratio": 0.44931742448672474}, {"method": "omnizip_r45", "video": "gmKpD8QPvSA.mp4", "idx": 448, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5174951553344727, "peak_mem_gb": 9.112365245819092, "n_input_tokens": 1746, "n_video_tokens": 1296, "n_audio_tokens": 232, "retained_av_ratio": 0.450261780104712}, {"method": "omnizip_r45", "video": "Wl8UjlAedeQ.mp4", "idx": 450, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7820489406585693, "peak_mem_gb": 9.18752145767212, "n_input_tokens": 2148, "n_video_tokens": 1584, "n_audio_tokens": 289, "retained_av_ratio": 0.45221569674319273}, {"method": "omnizip_r45", "video": "dEp4CVa18-s.mp4", "idx": 452, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.515272378921509, "peak_mem_gb": 9.316384315490723, "n_input_tokens": 3072, "n_video_tokens": 2448, "n_audio_tokens": 430, "retained_av_ratio": 0.44857539958304377}, {"method": "omnizip_r45", "video": "aWo90tAPcBs.mp4", "idx": 454, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.518739461898804, "peak_mem_gb": 9.879369735717773, "n_input_tokens": 6494, "n_video_tokens": 5328, "n_audio_tokens": 951, "retained_av_ratio": 0.44991240643414554}, {"method": "omnizip_r45", "video": "XHdk-W9L9RQ.mp4", "idx": 456, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0330982208251953, "peak_mem_gb": 9.00817060470581, "n_input_tokens": 1053, "n_video_tokens": 720, "n_audio_tokens": 135, "retained_av_ratio": 0.4783625730994152}, {"method": "omnizip_r45", "video": "1SjR4zREgq8.mp4", "idx": 458, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0686275959014893, "peak_mem_gb": 9.041972160339355, "n_input_tokens": 1115, "n_video_tokens": 720, "n_audio_tokens": 132, "retained_av_ratio": 0.5269953051643192}, {"method": "omnizip_r45", "video": "mVJ32Ex_gqk.mp4", "idx": 460, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.973301649093628, "peak_mem_gb": 9.982984066009521, "n_input_tokens": 7168, "n_video_tokens": 5904, "n_audio_tokens": 1042, "retained_av_ratio": 0.4498992225741434}, {"method": "omnizip_r45", "video": "RFm_F9hS0Y8.mp4", "idx": 462, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9914951324462891, "peak_mem_gb": 9.044571876525879, "n_input_tokens": 1226, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.49606299212598426}, {"method": "omnizip_r45", "video": "H3Mg7I8ElNo.mp4", "idx": 464, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9618434906005859, "peak_mem_gb": 9.024820804595947, "n_input_tokens": 1059, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.5276796230859835}, {"method": "omnizip_r45", "video": "RYQlChSooWM.mp4", "idx": 466, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.904811143875122, "peak_mem_gb": 10.257105350494385, "n_input_tokens": 8887, "n_video_tokens": 7344, "n_audio_tokens": 1289, "retained_av_ratio": 0.4490906984825669}, {"method": "omnizip_r45", "video": "b0x6z9gDzHQ.mp4", "idx": 468, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.679638385772705, "peak_mem_gb": 10.103671073913574, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1169, "retained_av_ratio": 0.4496342871808033}, {"method": "omnizip_r45", "video": "KwPGlGTOBkU.mp4", "idx": 470, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3485729694366455, "peak_mem_gb": 9.115986347198486, "n_input_tokens": 1668, "n_video_tokens": 1152, "n_audio_tokens": 203, "retained_av_ratio": 0.45018450184501846}, {"method": "omnizip_r45", "video": "6X1IEKFL33Y.mp4", "idx": 472, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.295295476913452, "peak_mem_gb": 10.223778247833252, "n_input_tokens": 8816, "n_video_tokens": 7344, "n_audio_tokens": 1278, "retained_av_ratio": 0.44873579215959175}, {"method": "omnizip_r45", "video": "YWhL_jNOzVc.mp4", "idx": 474, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.295100450515747, "peak_mem_gb": 9.08158826828003, "n_input_tokens": 1565, "n_video_tokens": 1152, "n_audio_tokens": 217, "retained_av_ratio": 0.45361577794010227}, {"method": "omnizip_r45", "video": "OY-uPQ4oLKM.mp4", "idx": 476, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 10.198216676712036, "peak_mem_gb": 11.4266939163208, "n_input_tokens": 11296, "n_video_tokens": 9216, "n_audio_tokens": 1866, "retained_av_ratio": 0.4552427359682368}, {"method": "omnizip_r45", "video": "3Y2mE1Oph6A.mp4", "idx": 478, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.29018235206604, "peak_mem_gb": 9.895129680633545, "n_input_tokens": 6620, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 0.44952558718307667}, {"method": "omnizip_r45", "video": "ZbsjWDzPPx4.mp4", "idx": 480, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9965815544128418, "peak_mem_gb": 9.021700382232666, "n_input_tokens": 1090, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.4872389791183295}, {"method": "omnizip_r45", "video": "jtFI4H55Clw.mp4", "idx": 482, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6104795932769775, "peak_mem_gb": 9.63200330734253, "n_input_tokens": 4793, "n_video_tokens": 3888, "n_audio_tokens": 693, "retained_av_ratio": 0.449028596376337}, {"method": "omnizip_r45", "video": "QD1l3iZ2nZs.mp4", "idx": 484, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.506958484649658, "peak_mem_gb": 10.280664920806885, "n_input_tokens": 9145, "n_video_tokens": 7632, "n_audio_tokens": 1337, "retained_av_ratio": 0.4487679785929312}, {"method": "omnizip_r45", "video": "qPibP0Y6HUU.mp4", "idx": 486, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.716902017593384, "peak_mem_gb": 10.612367630004883, "n_input_tokens": 10494, "n_video_tokens": 8784, "n_audio_tokens": 1527, "retained_av_ratio": 0.44893802734943267}, {"method": "omnizip_r45", "video": "AOkiHqncwF8.mp4", "idx": 488, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.431154251098633, "peak_mem_gb": 9.546356678009033, "n_input_tokens": 4475, "n_video_tokens": 3600, "n_audio_tokens": 651, "retained_av_ratio": 0.450717478240414}, {"method": "omnizip_r45", "video": "FPBSzdA1oKU.mp4", "idx": 490, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1633155345916748, "peak_mem_gb": 9.06540060043335, "n_input_tokens": 1405, "n_video_tokens": 1008, "n_audio_tokens": 180, "retained_av_ratio": 0.46885521885521886}, {"method": "omnizip_r45", "video": "6RNtg0cLgZk.mp4", "idx": 492, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.376153945922852, "peak_mem_gb": 10.816018104553223, "n_input_tokens": 11041, "n_video_tokens": 9216, "n_audio_tokens": 1616, "retained_av_ratio": 0.4494091580502216}, {"method": "omnizip_r45", "video": "0avoXj7I3yA.mp4", "idx": 494, "problem_type": "Narrative Comprehension", "answer": "B", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.72 GiB. GPU 0 has a total capacity of 14.56 GiB of which 480.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.986373901367188, "n_input_tokens": 12512, "n_video_tokens": 9216, "n_audio_tokens": 3020, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "DcyrFhN6ncE.mp4", "idx": 496, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.1145548820495605, "peak_mem_gb": 10.257698059082031, "n_input_tokens": 8889, "n_video_tokens": 7344, "n_audio_tokens": 1281, "retained_av_ratio": 0.44857971014492753}, {"method": "omnizip_r45", "video": "HAh-h5nuzhI.mp4", "idx": 498, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.4217214584350586, "peak_mem_gb": 9.587649822235107, "n_input_tokens": 4655, "n_video_tokens": 3744, "n_audio_tokens": 670, "retained_av_ratio": 0.4506116900770276}, {"method": "omnizip_r45", "video": "Hre27BhZvM8.mp4", "idx": 1, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.223827600479126, "peak_mem_gb": 10.277089595794678, "n_input_tokens": 8959, "n_video_tokens": 7344, "n_audio_tokens": 1299, "retained_av_ratio": 0.44938100196690967}, {"method": "omnizip_r45", "video": "f7YdwVUy0ss.mp4", "idx": 3, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.100036859512329, "peak_mem_gb": 9.06082820892334, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 184, "retained_av_ratio": 0.4513422818791946}, {"method": "omnizip_r45", "video": "EgHDY0VpFS0.mp4", "idx": 5, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8523683547973633, "peak_mem_gb": 9.489328384399414, "n_input_tokens": 4037, "n_video_tokens": 3168, "n_audio_tokens": 565, "retained_av_ratio": 0.4505759442807393}, {"method": "omnizip_r45", "video": "egcR6-Dq0Tc.mp4", "idx": 7, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.315863609313965, "peak_mem_gb": 10.557397842407227, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "omnizip_r45", "video": "Bk60US4MacA.mp4", "idx": 9, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.813795566558838, "peak_mem_gb": 10.410010814666748, "n_input_tokens": 9895, "n_video_tokens": 8208, "n_audio_tokens": 1427, "retained_av_ratio": 0.4485729112610275}, {"method": "omnizip_r45", "video": "ft_pDeYZA6E.mp4", "idx": 11, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4782288074493408, "peak_mem_gb": 9.165887832641602, "n_input_tokens": 2084, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 0.4523682810005322}, {"method": "omnizip_r45", "video": "TJR8ibF26vw.mp4", "idx": 13, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6707611083984375, "peak_mem_gb": 9.648540019989014, "n_input_tokens": 4872, "n_video_tokens": 3920, "n_audio_tokens": 723, "retained_av_ratio": 0.45143226362265776}, {"method": "omnizip_r45", "video": "Qps4jZVasg4.mp4", "idx": 15, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.306117057800293, "peak_mem_gb": 9.096837043762207, "n_input_tokens": 1605, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 0.45594713656387664}, {"method": "omnizip_r45", "video": "lUEjk9CVcEY.mp4", "idx": 17, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.033836603164673, "peak_mem_gb": 9.265458583831787, "n_input_tokens": 2740, "n_video_tokens": 2160, "n_audio_tokens": 385, "retained_av_ratio": 0.4506876227897839}, {"method": "omnizip_r45", "video": "cdOd3EXqagY.mp4", "idx": 19, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.900641202926636, "peak_mem_gb": 10.39179277420044, "n_input_tokens": 9664, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 0.44978396037517127}, {"method": "omnizip_r45", "video": "Z6U4PbCGiP0.mp4", "idx": 21, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.3607518672943115, "peak_mem_gb": 9.527192115783691, "n_input_tokens": 4316, "n_video_tokens": 3456, "n_audio_tokens": 624, "retained_av_ratio": 0.4504901960784314}, {"method": "omnizip_r45", "video": "NaDxBMFiiNQ.mp4", "idx": 23, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1849088668823242, "peak_mem_gb": 9.087089538574219, "n_input_tokens": 1570, "n_video_tokens": 1152, "n_audio_tokens": 211, "retained_av_ratio": 0.4600146735143067}, {"method": "omnizip_r45", "video": "UxlWLoM3H-8.mp4", "idx": 25, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.643619537353516, "peak_mem_gb": 10.49581527709961, "n_input_tokens": 10014, "n_video_tokens": 8352, "n_audio_tokens": 1474, "retained_av_ratio": 0.44982698961937717}, {"method": "omnizip_r45", "video": "_mYtKUIalEo.mp4", "idx": 27, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4829607009887695, "peak_mem_gb": 9.354033470153809, "n_input_tokens": 3275, "n_video_tokens": 2592, "n_audio_tokens": 471, "retained_av_ratio": 0.4505386875612145}, {"method": "omnizip_r45", "video": "P9jqFr3TvSE.mp4", "idx": 29, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 16.54264736175537, "peak_mem_gb": 10.228647232055664, "n_input_tokens": 8742, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 0.4500472366556448}, {"method": "omnizip_r45", "video": "X1c3Z8eAhwI.mp4", "idx": 31, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.794737815856934, "peak_mem_gb": 9.87455415725708, "n_input_tokens": 6473, "n_video_tokens": 5328, "n_audio_tokens": 946, "retained_av_ratio": 0.45011157156518966}, {"method": "omnizip_r45", "video": "O7EgKJ5Km7Y.mp4", "idx": 33, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.464510917663574, "peak_mem_gb": 10.55742073059082, "n_input_tokens": 10361, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.4488266614080063}, {"method": "omnizip_r45", "video": "xlO7zTTgtHg.mp4", "idx": 35, "problem_type": "Affective Intent Classification", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2368946075439453, "peak_mem_gb": 9.065770149230957, "n_input_tokens": 1428, "n_video_tokens": 1008, "n_audio_tokens": 194, "retained_av_ratio": 0.4550748752079867}, {"method": "omnizip_r45", "video": "52smiyrFybw.mp4", "idx": 37, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8652553558349609, "peak_mem_gb": 8.999790668487549, "n_input_tokens": 874, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 0.5752212389380531}, {"method": "omnizip_r45", "video": "_PzDR-Nf-e8.mp4", "idx": 39, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0739049911499023, "peak_mem_gb": 9.062152862548828, "n_input_tokens": 1420, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.45439469320066334}, {"method": "omnizip_r45", "video": "aiVuez4WsBo.mp4", "idx": 41, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.730553150177002, "peak_mem_gb": 9.18974781036377, "n_input_tokens": 2220, "n_video_tokens": 1680, "n_audio_tokens": 324, "retained_av_ratio": 0.4555888223552894}, {"method": "omnizip_r45", "video": "3o2KqGljhbQ.mp4", "idx": 43, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.347785711288452, "peak_mem_gb": 10.463076114654541, "n_input_tokens": 10078, "n_video_tokens": 8352, "n_audio_tokens": 1458, "retained_av_ratio": 0.4487257900101937}, {"method": "omnizip_r45", "video": "TIxlSCV6E_A.mp4", "idx": 45, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9035866260528564, "peak_mem_gb": 9.007154941558838, "n_input_tokens": 902, "n_video_tokens": 576, "n_audio_tokens": 118, "retained_av_ratio": 0.5778097982708934}, {"method": "omnizip_r45", "video": "2dnuTGnFZyo.mp4", "idx": 47, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0938875675201416, "peak_mem_gb": 9.045610904693604, "n_input_tokens": 1236, "n_video_tokens": 864, "n_audio_tokens": 174, "retained_av_ratio": 0.5}, {"method": "omnizip_r45", "video": "9ngLXMGF90I.mp4", "idx": 49, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.660673141479492, "peak_mem_gb": 9.374741077423096, "n_input_tokens": 3426, "n_video_tokens": 2736, "n_audio_tokens": 496, "retained_av_ratio": 0.45142326732673266}, {"method": "omnizip_r45", "video": "BPpr-5KW5BQ.mp4", "idx": 51, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.59167194366455, "peak_mem_gb": 10.532127380371094, "n_input_tokens": 10188, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.4494843296285171}, {"method": "omnizip_r45", "video": "m_haCHnvhj4.mp4", "idx": 53, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0059638023376465, "peak_mem_gb": 9.040441036224365, "n_input_tokens": 1114, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5327963176064442}, {"method": "omnizip_r45", "video": "xrOu-TB6osc.mp4", "idx": 55, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8981068134307861, "peak_mem_gb": 9.035278797149658, "n_input_tokens": 989, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "omnizip_r45", "video": "9IGdrQU77pk.mp4", "idx": 57, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4740099906921387, "peak_mem_gb": 9.122539520263672, "n_input_tokens": 1781, "n_video_tokens": 1296, "n_audio_tokens": 251, "retained_av_ratio": 0.4557207498383969}, {"method": "omnizip_r45", "video": "wPWguBt2hhs.mp4", "idx": 59, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9559054374694824, "peak_mem_gb": 9.019089221954346, "n_input_tokens": 1044, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.525383707201889}, {"method": "omnizip_r45", "video": "7moy-Foct8U.mp4", "idx": 61, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.773390531539917, "peak_mem_gb": 9.775559902191162, "n_input_tokens": 5801, "n_video_tokens": 4752, "n_audio_tokens": 845, "retained_av_ratio": 0.4491691977845274}, {"method": "omnizip_r45", "video": "GSLBOuwOdRE.mp4", "idx": 63, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 11.065389156341553, "peak_mem_gb": 11.646289825439453, "n_input_tokens": 11393, "n_video_tokens": 9216, "n_audio_tokens": 1948, "retained_av_ratio": 0.45834826227158726}, {"method": "omnizip_r45", "video": "LXXmslrnPrg.mp4", "idx": 65, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.8472869396209717, "peak_mem_gb": 9.0059814453125, "n_input_tokens": 1051, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "omnizip_r45", "video": "oPhbjcl7TOg.mp4", "idx": 67, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6334598064422607, "peak_mem_gb": 9.335229396820068, "n_input_tokens": 3132, "n_video_tokens": 2448, "n_audio_tokens": 451, "retained_av_ratio": 0.4525698516729907}, {"method": "omnizip_r45", "video": "iTdOOLn-ICM.mp4", "idx": 69, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.0920774936676025, "peak_mem_gb": 9.250942707061768, "n_input_tokens": 2611, "n_video_tokens": 2016, "n_audio_tokens": 369, "retained_av_ratio": 0.4511530398322851}, {"method": "omnizip_r45", "video": "CpeBZFxOeqQ.mp4", "idx": 71, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.542809247970581, "peak_mem_gb": 9.128870964050293, "n_input_tokens": 1862, "n_video_tokens": 1400, "n_audio_tokens": 273, "retained_av_ratio": 0.4560669456066946}, {"method": "omnizip_r45", "video": "B7g9hmHAUzo.mp4", "idx": 73, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.186798810958862, "peak_mem_gb": 10.008424282073975, "n_input_tokens": 7326, "n_video_tokens": 6048, "n_audio_tokens": 1059, "retained_av_ratio": 0.4496974813564092}, {"method": "omnizip_r45", "video": "tUucQrJAzsI.mp4", "idx": 75, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "E", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5346150398254395, "peak_mem_gb": 9.313063621520996, "n_input_tokens": 3063, "n_video_tokens": 2448, "n_audio_tokens": 443, "retained_av_ratio": 0.4510549982704946}, {"method": "omnizip_r45", "video": "TA6OBcjG1Oc.mp4", "idx": 77, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.73151969909668, "peak_mem_gb": 10.557226181030273, "n_input_tokens": 10310, "n_video_tokens": 8640, "n_audio_tokens": 1502, "retained_av_ratio": 0.448629461644646}, {"method": "omnizip_r45", "video": "CdwdC6E8Cco.mp4", "idx": 79, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5369293689727783, "peak_mem_gb": 9.291515350341797, "n_input_tokens": 2910, "n_video_tokens": 2304, "n_audio_tokens": 409, "retained_av_ratio": 0.44968669369701436}, {"method": "omnizip_r45", "video": "d-wkkuvKoGM.mp4", "idx": 81, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5754010677337646, "peak_mem_gb": 9.506208419799805, "n_input_tokens": 4268, "n_video_tokens": 3456, "n_audio_tokens": 609, "retained_av_ratio": 0.44969249692496926}, {"method": "omnizip_r45", "video": "i_mCozcHi9o.mp4", "idx": 83, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4153780937194824, "peak_mem_gb": 9.09642744064331, "n_input_tokens": 1604, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 0.45890909090909093}, {"method": "omnizip_r45", "video": "g_RmJp0IgkY.mp4", "idx": 85, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0575318336486816, "peak_mem_gb": 9.038536071777344, "n_input_tokens": 1122, "n_video_tokens": 720, "n_audio_tokens": 149, "retained_av_ratio": 0.5166858457997698}, {"method": "omnizip_r45", "video": "GOg9dmnHSSw.mp4", "idx": 87, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.562599658966064, "peak_mem_gb": 10.338329315185547, "n_input_tokens": 9500, "n_video_tokens": 7920, "n_audio_tokens": 1393, "retained_av_ratio": 0.44937184580693657}, {"method": "omnizip_r45", "video": "-DxPwypTfKI.mp4", "idx": 89, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.305899858474731, "peak_mem_gb": 10.468195915222168, "n_input_tokens": 10075, "n_video_tokens": 8352, "n_audio_tokens": 1461, "retained_av_ratio": 0.4489962294914909}, {"method": "omnizip_r45", "video": "l7QiQSivnAc.mp4", "idx": 91, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.746664047241211, "peak_mem_gb": 9.355889797210693, "n_input_tokens": 3373, "n_video_tokens": 2736, "n_audio_tokens": 499, "retained_av_ratio": 0.45038639876352393}, {"method": "omnizip_r45", "video": "T-PGGc1OJcM.mp4", "idx": 93, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1089189052581787, "peak_mem_gb": 9.041763305664062, "n_input_tokens": 1224, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.49513618677042803}, {"method": "omnizip_r45", "video": "k4o3XkT0jbM.mp4", "idx": 95, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.54811692237854, "peak_mem_gb": 10.1772780418396, "n_input_tokens": 8485, "n_video_tokens": 7056, "n_audio_tokens": 1227, "retained_av_ratio": 0.44923336955209464}, {"method": "omnizip_r45", "video": "baTfh9Ryxoo.mp4", "idx": 97, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.326330184936523, "peak_mem_gb": 9.676557540893555, "n_input_tokens": 5113, "n_video_tokens": 4176, "n_audio_tokens": 749, "retained_av_ratio": 0.4511675126903553}, {"method": "omnizip_r45", "video": "CcZr-QjAvTQ.mp4", "idx": 99, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0776307582855225, "peak_mem_gb": 9.052208423614502, "n_input_tokens": 1255, "n_video_tokens": 864, "n_audio_tokens": 157, "retained_av_ratio": 0.49363369245837413}, {"method": "omnizip_r45", "video": "-0J1uWKJKfk.mp4", "idx": 101, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9085984230041504, "peak_mem_gb": 9.050243377685547, "n_input_tokens": 1146, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.5267441860465116}, {"method": "omnizip_r45", "video": "4WgA39tgBSo.mp4", "idx": 103, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.922589063644409, "peak_mem_gb": 9.400065422058105, "n_input_tokens": 3596, "n_video_tokens": 2880, "n_audio_tokens": 504, "retained_av_ratio": 0.44799054373522457}, {"method": "omnizip_r45", "video": "QUC2d9FSkcU.mp4", "idx": 105, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.269340753555298, "peak_mem_gb": 9.29761791229248, "n_input_tokens": 2930, "n_video_tokens": 2304, "n_audio_tokens": 416, "retained_av_ratio": 0.45073529411764707}, {"method": "omnizip_r45", "video": "uGoKOpZOSUg.mp4", "idx": 107, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 6.04 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.513656616210938, "n_input_tokens": 14039, "n_video_tokens": 9216, "n_audio_tokens": 4501, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "MY7puORABAk.mp4", "idx": 109, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4788589477539062, "peak_mem_gb": 9.319489479064941, "n_input_tokens": 3079, "n_video_tokens": 2448, "n_audio_tokens": 436, "retained_av_ratio": 0.4511095700416089}, {"method": "omnizip_r45", "video": "yl79pxDOjJ0.mp4", "idx": 111, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.7705395221710205, "peak_mem_gb": 9.791065692901611, "n_input_tokens": 5843, "n_video_tokens": 4752, "n_audio_tokens": 831, "retained_av_ratio": 0.4493999641769658}, {"method": "omnizip_r45", "video": "g8JpgtFpciY.mp4", "idx": 113, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.423975944519043, "peak_mem_gb": 10.06151294708252, "n_input_tokens": 7682, "n_video_tokens": 6336, "n_audio_tokens": 1120, "retained_av_ratio": 0.44957081545064376}, {"method": "omnizip_r45", "video": "1Q3LeFo88xI.mp4", "idx": 115, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.536560535430908, "peak_mem_gb": 10.567143440246582, "n_input_tokens": 10417, "n_video_tokens": 8640, "n_audio_tokens": 1506, "retained_av_ratio": 0.44835403114527894}, {"method": "omnizip_r45", "video": "HdwFA-Hb0pQ.mp4", "idx": 117, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.857365608215332, "peak_mem_gb": 9.017394065856934, "n_input_tokens": 1086, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.48088064889918886}, {"method": "omnizip_r45", "video": "GFrTD7uV8mA.mp4", "idx": 119, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9170303344726562, "peak_mem_gb": 9.024575233459473, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.532183908045977}, {"method": "omnizip_r45", "video": "Dawm3vWn97A.mp4", "idx": 121, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6744494438171387, "peak_mem_gb": 9.152956485748291, "n_input_tokens": 1962, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 0.4536443148688047}, {"method": "omnizip_r45", "video": "BQc182PZbeU.mp4", "idx": 123, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 6.326647996902466, "peak_mem_gb": 10.018795490264893, "n_input_tokens": 7363, "n_video_tokens": 6048, "n_audio_tokens": 1069, "retained_av_ratio": 0.44976816074188564}, {"method": "omnizip_r45", "video": "tjXVd3qBMhw.mp4", "idx": 125, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.125685691833496, "peak_mem_gb": 9.032220363616943, "n_input_tokens": 1195, "n_video_tokens": 864, "n_audio_tokens": 159, "retained_av_ratio": 0.4916911045943304}, {"method": "omnizip_r45", "video": "YGsvvmBiRHA.mp4", "idx": 127, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3359668254852295, "peak_mem_gb": 9.27749252319336, "n_input_tokens": 2781, "n_video_tokens": 2160, "n_audio_tokens": 386, "retained_av_ratio": 0.4493322859387274}, {"method": "omnizip_r45", "video": "5yssh2Aqtps.mp4", "idx": 129, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.44648790359497, "peak_mem_gb": 10.512125492095947, "n_input_tokens": 10146, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.4491330059136013}, {"method": "omnizip_r45", "video": "auFMvAywvbk.mp4", "idx": 131, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.138209819793701, "peak_mem_gb": 9.27235221862793, "n_input_tokens": 2757, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.4505321245565629}, {"method": "omnizip_r45", "video": "qTX0XvwA0qU.mp4", "idx": 133, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0070264339447021, "peak_mem_gb": 9.0389084815979, "n_input_tokens": 1216, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4946550048590865}, {"method": "omnizip_r45", "video": "X2ZDXPulIy0.mp4", "idx": 135, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8392536640167236, "peak_mem_gb": 9.01637887954712, "n_input_tokens": 1084, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.46989374262101535}, {"method": "omnizip_r45", "video": "qHakVESEOec.mp4", "idx": 137, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2807679176330566, "peak_mem_gb": 9.091484546661377, "n_input_tokens": 1587, "n_video_tokens": 1152, "n_audio_tokens": 223, "retained_av_ratio": 0.45890909090909093}, {"method": "omnizip_r45", "video": "StG5yDCruTw.mp4", "idx": 139, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.4195878505706787, "peak_mem_gb": 9.495290756225586, "n_input_tokens": 4150, "n_video_tokens": 3312, "n_audio_tokens": 586, "retained_av_ratio": 0.4492047203694202}, {"method": "omnizip_r45", "video": "6kpIdgWGRAg.mp4", "idx": 141, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9758410453796387, "peak_mem_gb": 9.014867305755615, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.49653579676674364}, {"method": "omnizip_r45", "video": "Z0YlK_pGTMQ.mp4", "idx": 143, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9996640682220459, "peak_mem_gb": 9.01742172241211, "n_input_tokens": 1047, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.5298850574712644}, {"method": "omnizip_r45", "video": "ZJyYchnUJL4.mp4", "idx": 145, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9953057765960693, "peak_mem_gb": 9.019801139831543, "n_input_tokens": 1093, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "omnizip_r45", "video": "7uxn4eM979g.mp4", "idx": 147, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9472157955169678, "peak_mem_gb": 9.039445877075195, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 138, "retained_av_ratio": 0.5314685314685315}, {"method": "omnizip_r45", "video": "GO1FuCX8YHI.mp4", "idx": 149, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.29244065284729, "peak_mem_gb": 10.291738986968994, "n_input_tokens": 9176, "n_video_tokens": 7632, "n_audio_tokens": 1335, "retained_av_ratio": 0.4488680718188915}, {"method": "omnizip_r45", "video": "tQCM5mtqkeo.mp4", "idx": 151, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.055258512496948, "peak_mem_gb": 10.433043003082275, "n_input_tokens": 9825, "n_video_tokens": 8208, "n_audio_tokens": 1445, "retained_av_ratio": 0.44949756552367137}, {"method": "omnizip_r45", "video": "OYIX_zxxS9Q.mp4", "idx": 153, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.746265172958374, "peak_mem_gb": 9.54296350479126, "n_input_tokens": 4457, "n_video_tokens": 3600, "n_audio_tokens": 641, "retained_av_ratio": 0.45036547983966047}, {"method": "omnizip_r45", "video": "GZ_pUlsnTGQ.mp4", "idx": 155, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2144389152526855, "peak_mem_gb": 9.030990600585938, "n_input_tokens": 1196, "n_video_tokens": 864, "n_audio_tokens": 169, "retained_av_ratio": 0.49177153920619554}, {"method": "omnizip_r45", "video": "VyHYgZamj58.mp4", "idx": 157, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.7966020107269287, "peak_mem_gb": 9.0093994140625, "n_input_tokens": 905, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5871428571428572}, {"method": "omnizip_r45", "video": "CQHAoYgQLis.mp4", "idx": 159, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.7407584190368652, "peak_mem_gb": 9.568448543548584, "n_input_tokens": 4601, "n_video_tokens": 3744, "n_audio_tokens": 659, "retained_av_ratio": 0.4483306836248013}, {"method": "omnizip_r45", "video": "ySiKjxAnE4I.mp4", "idx": 161, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.00650930404663, "peak_mem_gb": 10.391505241394043, "n_input_tokens": 9711, "n_video_tokens": 8064, "n_audio_tokens": 1425, "retained_av_ratio": 0.44978396037517127}, {"method": "omnizip_r45", "video": "9xWVEqxUJGY.mp4", "idx": 163, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1533005237579346, "peak_mem_gb": 9.039540767669678, "n_input_tokens": 1221, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.4912451361867704}, {"method": "omnizip_r45", "video": "egkxfFQ_C7g.mp4", "idx": 165, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.7099428176879883, "peak_mem_gb": 9.153939247131348, "n_input_tokens": 1957, "n_video_tokens": 1440, "n_audio_tokens": 260, "retained_av_ratio": 0.45235294117647057}, {"method": "omnizip_r45", "video": "U-R7VXme0ps.mp4", "idx": 167, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5629065036773682, "peak_mem_gb": 9.14611530303955, "n_input_tokens": 1939, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.45353594389246055}, {"method": "omnizip_r45", "video": "OCRdFBuuW2M.mp4", "idx": 169, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.642545938491821, "peak_mem_gb": 9.755118370056152, "n_input_tokens": 5554, "n_video_tokens": 4464, "n_audio_tokens": 791, "retained_av_ratio": 0.449476688867745}, {"method": "omnizip_r45", "video": "5yUY-afqZSs.mp4", "idx": 171, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.62177324295044, "peak_mem_gb": 10.532022476196289, "n_input_tokens": 10161, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.4494843296285171}, {"method": "omnizip_r45", "video": "T0p4FtF99l0.mp4", "idx": 173, "problem_type": "Creative Innovation Analysis", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9664044380187988, "peak_mem_gb": 9.039465427398682, "n_input_tokens": 1107, "n_video_tokens": 720, "n_audio_tokens": 143, "retained_av_ratio": 0.5341830822711472}, {"method": "omnizip_r45", "video": "2tVZFAqM7lw.mp4", "idx": 175, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.8022444248199463, "peak_mem_gb": 9.38771390914917, "n_input_tokens": 3555, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 0.4498525073746313}, {"method": "omnizip_r45", "video": "uBXycVLybjY.mp4", "idx": 177, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.97194504737854, "peak_mem_gb": 10.426908493041992, "n_input_tokens": 9806, "n_video_tokens": 8208, "n_audio_tokens": 1442, "retained_av_ratio": 0.44963730569948185}, {"method": "omnizip_r45", "video": "JiRQff5ADps.mp4", "idx": 179, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.925326108932495, "peak_mem_gb": 9.75663423538208, "n_input_tokens": 5661, "n_video_tokens": 4620, "n_audio_tokens": 845, "retained_av_ratio": 0.45032021957914}, {"method": "omnizip_r45", "video": "tNn-m_qLsvo.mp4", "idx": 181, "problem_type": "Temporal Reasoning Localization", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.3051402568817139, "peak_mem_gb": 9.094379425048828, "n_input_tokens": 1598, "n_video_tokens": 1152, "n_audio_tokens": 221, "retained_av_ratio": 0.4573925710123816}, {"method": "omnizip_r45", "video": "0Z3NeEoE4cc.mp4", "idx": 183, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.137367010116577, "peak_mem_gb": 9.443287372589111, "n_input_tokens": 3855, "n_video_tokens": 3080, "n_audio_tokens": 569, "retained_av_ratio": 0.45108248835297343}, {"method": "omnizip_r45", "video": "K8S_8sdPRDI.mp4", "idx": 185, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.3393096923828125, "peak_mem_gb": 9.291672229766846, "n_input_tokens": 2871, "n_video_tokens": 2240, "n_audio_tokens": 414, "retained_av_ratio": 0.45327807083647326}, {"method": "omnizip_r45", "video": "dxSsSfynCUQ.mp4", "idx": 187, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.621243715286255, "peak_mem_gb": 10.055179119110107, "n_input_tokens": 7661, "n_video_tokens": 6336, "n_audio_tokens": 1104, "retained_av_ratio": 0.4489247311827957}, {"method": "omnizip_r45", "video": "PW-6MC05h3A.mp4", "idx": 189, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.981442213058472, "peak_mem_gb": 10.26003122329712, "n_input_tokens": 8750, "n_video_tokens": 7128, "n_audio_tokens": 1364, "retained_av_ratio": 0.4525435704192181}, {"method": "omnizip_r45", "video": "O61cCbK3hMY.mp4", "idx": 191, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 2.88 GiB. GPU 0 has a total capacity of 14.56 GiB of which 320.81 MiB is free. Including non-PyTorch mem", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 12.157180309295654, "n_input_tokens": 12527, "n_video_tokens": 9216, "n_audio_tokens": 3107, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "ypwjQw-1Ltg.mp4", "idx": 193, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.192783355712891, "peak_mem_gb": 10.055908679962158, "n_input_tokens": 7666, "n_video_tokens": 6336, "n_audio_tokens": 1108, "retained_av_ratio": 0.4482804943578721}, {"method": "omnizip_r45", "video": "PKoZQwUxvnQ.mp4", "idx": 195, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.2005152702331543, "peak_mem_gb": 9.264962673187256, "n_input_tokens": 2735, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.45013795821836816}, {"method": "omnizip_r45", "video": "yM0pV0G3-gc.mp4", "idx": 197, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.322643280029297, "peak_mem_gb": 10.055888175964355, "n_input_tokens": 7578, "n_video_tokens": 6192, "n_audio_tokens": 1095, "retained_av_ratio": 0.4488815699190339}, {"method": "omnizip_r45", "video": "rTcusJ3Nfqc.mp4", "idx": 199, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.99370551109314, "peak_mem_gb": 10.455715656280518, "n_input_tokens": 10028, "n_video_tokens": 8352, "n_audio_tokens": 1455, "retained_av_ratio": 0.44855715305394106}, {"method": "omnizip_r45", "video": "mXZjcxVwrLY.mp4", "idx": 201, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0704929828643799, "peak_mem_gb": 9.077675819396973, "n_input_tokens": 1336, "n_video_tokens": 864, "n_audio_tokens": 162, "retained_av_ratio": 0.49415204678362573}, {"method": "omnizip_r45", "video": "oQVdUilguhc.mp4", "idx": 203, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6437273025512695, "peak_mem_gb": 9.416994571685791, "n_input_tokens": 3556, "n_video_tokens": 2736, "n_audio_tokens": 498, "retained_av_ratio": 0.4517625231910946}, {"method": "omnizip_r45", "video": "E9NW_SJTrco.mp4", "idx": 205, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8380856513977051, "peak_mem_gb": 9.010971069335938, "n_input_tokens": 913, "n_video_tokens": 576, "n_audio_tokens": 120, "retained_av_ratio": 0.5804597701149425}, {"method": "omnizip_r45", "video": "iYrFMa1TBQQ.mp4", "idx": 207, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.336568593978882, "peak_mem_gb": 10.201101303100586, "n_input_tokens": 8657, "n_video_tokens": 7200, "n_audio_tokens": 1268, "retained_av_ratio": 0.449574870099197}, {"method": "omnizip_r45", "video": "Sin_6xmILaE.mp4", "idx": 209, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.420881748199463, "peak_mem_gb": 9.118086814880371, "n_input_tokens": 1767, "n_video_tokens": 1296, "n_audio_tokens": 236, "retained_av_ratio": 0.45039164490861616}, {"method": "omnizip_r45", "video": "w7K419G-nJM.mp4", "idx": 211, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6823809146881104, "peak_mem_gb": 9.170538425445557, "n_input_tokens": 2102, "n_video_tokens": 1584, "n_audio_tokens": 295, "retained_av_ratio": 0.45130388504523683}, {"method": "omnizip_r45", "video": "pOMKJXCtap0.mp4", "idx": 213, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.909597158432007, "peak_mem_gb": 10.51113748550415, "n_input_tokens": 10160, "n_video_tokens": 8496, "n_audio_tokens": 1481, "retained_av_ratio": 0.44863185326250377}, {"method": "omnizip_r45", "video": "HWpq9mZ5Sp4.mp4", "idx": 215, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1160202026367188, "peak_mem_gb": 9.04049301147461, "n_input_tokens": 1220, "n_video_tokens": 864, "n_audio_tokens": 164, "retained_av_ratio": 0.49513618677042803}, {"method": "omnizip_r45", "video": "NOpfz1iawe0.mp4", "idx": 217, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.806018590927124, "peak_mem_gb": 8.977001667022705, "n_input_tokens": 740, "n_video_tokens": 432, "n_audio_tokens": 94, "retained_av_ratio": 0.5931558935361216}, {"method": "omnizip_r45", "video": "HiujBqqlyvE.mp4", "idx": 219, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9839613437652588, "peak_mem_gb": 9.006949424743652, "n_input_tokens": 1055, "n_video_tokens": 720, "n_audio_tokens": 150, "retained_av_ratio": 0.4827586206896552}, {"method": "omnizip_r45", "video": "b8RVswCLGnM.mp4", "idx": 221, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.473613023757935, "peak_mem_gb": 10.54798412322998, "n_input_tokens": 10216, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.4498699219531719}, {"method": "omnizip_r45", "video": "UMdQ1VnfT54.mp4", "idx": 223, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.603150844573975, "peak_mem_gb": 10.345779418945312, "n_input_tokens": 9518, "n_video_tokens": 7920, "n_audio_tokens": 1385, "retained_av_ratio": 0.44954325631380976}, {"method": "omnizip_r45", "video": "xcxsvhrlfc0.mp4", "idx": 225, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7497382164001465, "peak_mem_gb": 9.159489154815674, "n_input_tokens": 2066, "n_video_tokens": 1584, "n_audio_tokens": 298, "retained_av_ratio": 0.4532412327311371}, {"method": "omnizip_r45", "video": "sAgv2JvK0iA.mp4", "idx": 227, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.756588935852051, "peak_mem_gb": 9.921886920928955, "n_input_tokens": 6703, "n_video_tokens": 5472, "n_audio_tokens": 957, "retained_av_ratio": 0.4498366775548297}, {"method": "omnizip_r45", "video": "u45J1YdCWvo.mp4", "idx": 229, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7708404064178467, "peak_mem_gb": 9.386443138122559, "n_input_tokens": 3465, "n_video_tokens": 2736, "n_audio_tokens": 495, "retained_av_ratio": 0.4503249767873723}, {"method": "omnizip_r45", "video": "VMltK9d0ewk.mp4", "idx": 231, "problem_type": "Creator Intent Taxonomy", "answer": "E", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.401801109313965, "peak_mem_gb": 10.28231143951416, "n_input_tokens": 9151, "n_video_tokens": 7632, "n_audio_tokens": 1342, "retained_av_ratio": 0.44929797191887677}, {"method": "omnizip_r45", "video": "bt1qugf_ej8.mp4", "idx": 233, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.875181913375854, "peak_mem_gb": 10.418152332305908, "n_input_tokens": 9864, "n_video_tokens": 8208, "n_audio_tokens": 1437, "retained_av_ratio": 0.44935199585277347}, {"method": "omnizip_r45", "video": "eVp2MEAkC44.mp4", "idx": 235, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5892868041992188, "peak_mem_gb": 9.546911716461182, "n_input_tokens": 4383, "n_video_tokens": 3456, "n_audio_tokens": 610, "retained_av_ratio": 0.4498278406296114}, {"method": "omnizip_r45", "video": "3j9Eq0Ky7f8.mp4", "idx": 237, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5121524333953857, "peak_mem_gb": 9.141886711120605, "n_input_tokens": 1928, "n_video_tokens": 1440, "n_audio_tokens": 270, "retained_av_ratio": 0.452046783625731}, {"method": "omnizip_r45", "video": "yr8b945WDdY.mp4", "idx": 239, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.272574663162231, "peak_mem_gb": 10.499009609222412, "n_input_tokens": 10036, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4494301994301994}, {"method": "omnizip_r45", "video": "4wJs2hhedj0.mp4", "idx": 241, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2258248329162598, "peak_mem_gb": 9.064030170440674, "n_input_tokens": 1394, "n_video_tokens": 980, "n_audio_tokens": 200, "retained_av_ratio": 0.47203389830508474}, {"method": "omnizip_r45", "video": "KABpAbGs6bQ.mp4", "idx": 243, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.2742865085601807, "peak_mem_gb": 9.262073040008545, "n_input_tokens": 2729, "n_video_tokens": 2160, "n_audio_tokens": 382, "retained_av_ratio": 0.4500393391030684}, {"method": "omnizip_r45", "video": "Bn2HxjKRi7s.mp4", "idx": 245, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 6.739444255828857, "peak_mem_gb": 10.094130516052246, "n_input_tokens": 7882, "n_video_tokens": 6480, "n_audio_tokens": 1148, "retained_av_ratio": 0.4491347666491872}, {"method": "omnizip_r45", "video": "E1FfWv-iIJM.mp4", "idx": 247, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.947092533111572, "peak_mem_gb": 10.273545265197754, "n_input_tokens": 9031, "n_video_tokens": 7488, "n_audio_tokens": 1307, "retained_av_ratio": 0.4491188175099488}, {"method": "omnizip_r45", "video": "a7YYrZ3mdsY.mp4", "idx": 249, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7710356712341309, "peak_mem_gb": 9.177025318145752, "n_input_tokens": 2206, "n_video_tokens": 1728, "n_audio_tokens": 313, "retained_av_ratio": 0.45222929936305734}, {"method": "omnizip_r45", "video": "4R3s-co2_lU.mp4", "idx": 251, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.296194553375244, "peak_mem_gb": 10.544929027557373, "n_input_tokens": 10242, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.4494145902131492}, {"method": "omnizip_r45", "video": "e-3Km4K2G2I.mp4", "idx": 253, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.156999111175537, "peak_mem_gb": 9.680834293365479, "n_input_tokens": 5132, "n_video_tokens": 4176, "n_audio_tokens": 741, "retained_av_ratio": 0.4494610534878991}, {"method": "omnizip_r45", "video": "Bc3vnp7VHVM.mp4", "idx": 255, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.4029147624969482, "peak_mem_gb": 9.285112857818604, "n_input_tokens": 2891, "n_video_tokens": 2304, "n_audio_tokens": 421, "retained_av_ratio": 0.45137614678899085}, {"method": "omnizip_r45", "video": "dNqYbetriJ4.mp4", "idx": 257, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.272922992706299, "peak_mem_gb": 10.582814693450928, "n_input_tokens": 10355, "n_video_tokens": 8640, "n_audio_tokens": 1514, "retained_av_ratio": 0.44888713807366554}, {"method": "omnizip_r45", "video": "-eUTGNm9ZFY.mp4", "idx": 259, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4289774894714355, "peak_mem_gb": 9.115288257598877, "n_input_tokens": 1754, "n_video_tokens": 1296, "n_audio_tokens": 242, "retained_av_ratio": 0.45513654096228867}, {"method": "omnizip_r45", "video": "68_42cTzS80.mp4", "idx": 261, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.867678165435791, "peak_mem_gb": 9.18287467956543, "n_input_tokens": 2226, "n_video_tokens": 1728, "n_audio_tokens": 305, "retained_av_ratio": 0.44909001475651744}, {"method": "omnizip_r45", "video": "OdrWu71KIdI.mp4", "idx": 263, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5442886352539062, "peak_mem_gb": 9.154573440551758, "n_input_tokens": 1964, "n_video_tokens": 1440, "n_audio_tokens": 273, "retained_av_ratio": 0.45475773496789257}, {"method": "omnizip_r45", "video": "Q4ftOkPtOB0.mp4", "idx": 265, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.756898880004883, "peak_mem_gb": 9.745325088500977, "n_input_tokens": 5536, "n_video_tokens": 4480, "n_audio_tokens": 816, "retained_av_ratio": 0.45128398791540786}, {"method": "omnizip_r45", "video": "l0Ilew3e9CI.mp4", "idx": 267, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0779731273651123, "peak_mem_gb": 9.052823066711426, "n_input_tokens": 1252, "n_video_tokens": 864, "n_audio_tokens": 152, "retained_av_ratio": 0.49606299212598426}, {"method": "omnizip_r45", "video": "IgGPXPR1J1A.mp4", "idx": 269, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.128648281097412, "peak_mem_gb": 9.053891658782959, "n_input_tokens": 1394, "n_video_tokens": 1008, "n_audio_tokens": 196, "retained_av_ratio": 0.45348837209302323}, {"method": "omnizip_r45", "video": "jNQep4ocOGQ.mp4", "idx": 271, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.170947790145874, "peak_mem_gb": 9.053190231323242, "n_input_tokens": 1388, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.44903117101937656}, {"method": "omnizip_r45", "video": "qB66a2x8KXE.mp4", "idx": 273, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1610159873962402, "peak_mem_gb": 9.250621795654297, "n_input_tokens": 2608, "n_video_tokens": 2016, "n_audio_tokens": 373, "retained_av_ratio": 0.45290916701548767}, {"method": "omnizip_r45", "video": "vVlQm8-AHt4.mp4", "idx": 275, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8501746654510498, "peak_mem_gb": 9.018005847930908, "n_input_tokens": 936, "n_video_tokens": 576, "n_audio_tokens": 124, "retained_av_ratio": 0.5814285714285714}, {"method": "omnizip_r45", "video": "XOUUsvMm1PM.mp4", "idx": 277, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.7743985652923584, "peak_mem_gb": 9.381240844726562, "n_input_tokens": 3448, "n_video_tokens": 2736, "n_audio_tokens": 489, "retained_av_ratio": 0.4496124031007752}, {"method": "omnizip_r45", "video": "_Nqw7iCl4eM.mp4", "idx": 279, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.434006214141846, "peak_mem_gb": 10.350749492645264, "n_input_tokens": 9543, "n_video_tokens": 7920, "n_audio_tokens": 1400, "retained_av_ratio": 0.4494635193133047}, {"method": "omnizip_r45", "video": "8kMaXa0PYow.mp4", "idx": 281, "problem_type": "Humor & Meme Deconstruction", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2442989349365234, "peak_mem_gb": 9.06571102142334, "n_input_tokens": 1412, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.46166807076663857}, {"method": "omnizip_r45", "video": "A3VEekCcv_s.mp4", "idx": 283, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.654634714126587, "peak_mem_gb": 9.727665901184082, "n_input_tokens": 5461, "n_video_tokens": 4464, "n_audio_tokens": 796, "retained_av_ratio": 0.45}, {"method": "omnizip_r45", "video": "Mm6HOkr_HXc.mp4", "idx": 285, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.826455354690552, "peak_mem_gb": 9.95828914642334, "n_input_tokens": 6995, "n_video_tokens": 5760, "n_audio_tokens": 1005, "retained_av_ratio": 0.44907612712490763}, {"method": "omnizip_r45", "video": "c4x0ezi7rP8.mp4", "idx": 287, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.810427188873291, "peak_mem_gb": 9.00241994857788, "n_input_tokens": 883, "n_video_tokens": 576, "n_audio_tokens": 125, "retained_av_ratio": 0.5877318116975749}, {"method": "omnizip_r45", "video": "mqFCIRSDSxE.mp4", "idx": 289, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.662669658660889, "peak_mem_gb": 10.35385274887085, "n_input_tokens": 9539, "n_video_tokens": 7920, "n_audio_tokens": 1382, "retained_av_ratio": 0.44990324661363146}, {"method": "omnizip_r45", "video": "zCwz5h0TPmE.mp4", "idx": 291, "problem_type": "Humor & Meme Deconstruction", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5710484981536865, "peak_mem_gb": 9.13701343536377, "n_input_tokens": 1911, "n_video_tokens": 1440, "n_audio_tokens": 271, "retained_av_ratio": 0.45353594389246055}, {"method": "omnizip_r45", "video": "SbFqCuO11JU.mp4", "idx": 293, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.6980135440826416, "peak_mem_gb": 9.372790813446045, "n_input_tokens": 3420, "n_video_tokens": 2736, "n_audio_tokens": 492, "retained_av_ratio": 0.4507434944237918}, {"method": "omnizip_r45", "video": "nl7-1jkIIgA.mp4", "idx": 295, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.056987285614014, "peak_mem_gb": 9.647794723510742, "n_input_tokens": 4843, "n_video_tokens": 3888, "n_audio_tokens": 680, "retained_av_ratio": 0.4498686514886165}, {"method": "omnizip_r45", "video": "gAc9Kp7CHn8.mp4", "idx": 297, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.7661545276641846, "peak_mem_gb": 9.176389694213867, "n_input_tokens": 2117, "n_video_tokens": 1584, "n_audio_tokens": 285, "retained_av_ratio": 0.44997324772605674}, {"method": "omnizip_r45", "video": "0vl66E4x5OI.mp4", "idx": 299, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5059871673583984, "peak_mem_gb": 9.117499351501465, "n_input_tokens": 1766, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45330739299610895}, {"method": "omnizip_r45", "video": "KLeJVuRRk84.mp4", "idx": 301, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9779765605926514, "peak_mem_gb": 9.033431053161621, "n_input_tokens": 1092, "n_video_tokens": 720, "n_audio_tokens": 142, "retained_av_ratio": 0.5290023201856149}, {"method": "omnizip_r45", "video": "Lzccd-ol1Ks.mp4", "idx": 303, "problem_type": "Affective Intent Classification", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.739398717880249, "peak_mem_gb": 10.549083232879639, "n_input_tokens": 10228, "n_video_tokens": 8496, "n_audio_tokens": 1499, "retained_av_ratio": 0.4494247123561781}, {"method": "omnizip_r45", "video": "-7dHm4baH38.mp4", "idx": 305, "problem_type": "Humor & Meme Deconstruction", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5210788249969482, "peak_mem_gb": 9.137694835662842, "n_input_tokens": 1914, "n_video_tokens": 1440, "n_audio_tokens": 279, "retained_av_ratio": 0.45549738219895286}, {"method": "omnizip_r45", "video": "b53-JgOg1i8.mp4", "idx": 307, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2400414943695068, "peak_mem_gb": 9.075776100158691, "n_input_tokens": 1549, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.4538852578068264}, {"method": "omnizip_r45", "video": "TQ1qkHdbyzM.mp4", "idx": 309, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2823224067687988, "peak_mem_gb": 9.091599941253662, "n_input_tokens": 1582, "n_video_tokens": 1152, "n_audio_tokens": 201, "retained_av_ratio": 0.4575018477457502}, {"method": "omnizip_r45", "video": "nfqtKk9tob0.mp4", "idx": 311, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.630110502243042, "peak_mem_gb": 10.533228397369385, "n_input_tokens": 10202, "n_video_tokens": 8496, "n_audio_tokens": 1491, "retained_av_ratio": 0.44908380895163713}, {"method": "omnizip_r45", "video": "Ata-X6fKUas.mp4", "idx": 313, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.9099135398864746, "peak_mem_gb": 9.41014289855957, "n_input_tokens": 3625, "n_video_tokens": 2880, "n_audio_tokens": 513, "retained_av_ratio": 0.4500442086648983}, {"method": "omnizip_r45", "video": "qdbBYgRMOEk.mp4", "idx": 315, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.18129563331604, "peak_mem_gb": 9.804669380187988, "n_input_tokens": 5982, "n_video_tokens": 4896, "n_audio_tokens": 869, "retained_av_ratio": 0.44978317432784043}, {"method": "omnizip_r45", "video": "mBMrGwB0v9w.mp4", "idx": 317, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2898938655853271, "peak_mem_gb": 9.085142612457275, "n_input_tokens": 1569, "n_video_tokens": 1152, "n_audio_tokens": 212, "retained_av_ratio": 0.4567448680351906}, {"method": "omnizip_r45", "video": "RgygVeaQXkU.mp4", "idx": 319, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.507701635360718, "peak_mem_gb": 9.877683162689209, "n_input_tokens": 6479, "n_video_tokens": 5328, "n_audio_tokens": 934, "retained_av_ratio": 0.4498562759501757}, {"method": "omnizip_r45", "video": "xQUom0axvAg.mp4", "idx": 321, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.5886521339416504, "peak_mem_gb": 9.14185905456543, "n_input_tokens": 1925, "n_video_tokens": 1440, "n_audio_tokens": 263, "retained_av_ratio": 0.451556077510276}, {"method": "omnizip_r45", "video": "7WGoDmVMbU4.mp4", "idx": 323, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.164819955825806, "peak_mem_gb": 9.651337146759033, "n_input_tokens": 4945, "n_video_tokens": 4032, "n_audio_tokens": 713, "retained_av_ratio": 0.4499473129610116}, {"method": "omnizip_r45", "video": "AZS9Vz54oWs.mp4", "idx": 325, "problem_type": "Affective Intent Classification", "answer": "D", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 4.3482935428619385, "peak_mem_gb": 9.691013813018799, "n_input_tokens": 5184, "n_video_tokens": 4200, "n_audio_tokens": 769, "retained_av_ratio": 0.45119742402897967}, {"method": "omnizip_r45", "video": "ZUXFwwscme0.mp4", "idx": 327, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6748664379119873, "peak_mem_gb": 9.135079383850098, "n_input_tokens": 1909, "n_video_tokens": 1440, "n_audio_tokens": 275, "retained_av_ratio": 0.4524781341107872}, {"method": "omnizip_r45", "video": "y6S8EUrrpP4.mp4", "idx": 329, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.079314947128296, "peak_mem_gb": 9.230515003204346, "n_input_tokens": 2458, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 0.4509715318572074}, {"method": "omnizip_r45", "video": "B0M-_FeUYVE.mp4", "idx": 331, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0413882732391357, "peak_mem_gb": 9.02544641494751, "n_input_tokens": 1109, "n_video_tokens": 715, "n_audio_tokens": 140, "retained_av_ratio": 0.47719298245614034}, {"method": "omnizip_r45", "video": "DM8l9e9cEXY.mp4", "idx": 333, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9189660549163818, "peak_mem_gb": 9.027705669403076, "n_input_tokens": 1066, "n_video_tokens": 720, "n_audio_tokens": 139, "retained_av_ratio": 0.5378346915017462}, {"method": "omnizip_r45", "video": "jmrtSmuXI9M.mp4", "idx": 335, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.26828932762146, "peak_mem_gb": 9.095500946044922, "n_input_tokens": 1602, "n_video_tokens": 1152, "n_audio_tokens": 210, "retained_av_ratio": 0.45301027900146845}, {"method": "omnizip_r45", "video": "DPnvcwAWV-Y.mp4", "idx": 337, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.30994725227356, "peak_mem_gb": 10.539390087127686, "n_input_tokens": 10168, "n_video_tokens": 8496, "n_audio_tokens": 1494, "retained_av_ratio": 0.44914914914914916}, {"method": "omnizip_r45", "video": "jBLELOjSYfY.mp4", "idx": 339, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.451942205429077, "peak_mem_gb": 9.319491386413574, "n_input_tokens": 3087, "n_video_tokens": 2448, "n_audio_tokens": 449, "retained_av_ratio": 0.4508111839834311}, {"method": "omnizip_r45", "video": "4kslnZ1gICo.mp4", "idx": 341, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.039193868637085, "peak_mem_gb": 9.043635845184326, "n_input_tokens": 1234, "n_video_tokens": 864, "n_audio_tokens": 156, "retained_av_ratio": 0.4872549019607843}, {"method": "omnizip_r45", "video": "toQnAXHr5uo.mp4", "idx": 343, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.3095948696136475, "peak_mem_gb": 9.475403308868408, "n_input_tokens": 3996, "n_video_tokens": 3168, "n_audio_tokens": 561, "retained_av_ratio": 0.4497184231697506}, {"method": "omnizip_r45", "video": "__aZIudBQjw.mp4", "idx": 345, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 8.35782527923584, "peak_mem_gb": 10.316758155822754, "n_input_tokens": 9254, "n_video_tokens": 7632, "n_audio_tokens": 1343, "retained_av_ratio": 0.4498050139275766}, {"method": "omnizip_r45", "video": "wRWKim49T6Y.mp4", "idx": 347, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9418880939483643, "peak_mem_gb": 9.009448528289795, "n_input_tokens": 1060, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.48023255813953486}, {"method": "omnizip_r45", "video": "7lh_sO9EWkE.mp4", "idx": 349, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5602242946624756, "peak_mem_gb": 9.11679744720459, "n_input_tokens": 1758, "n_video_tokens": 1296, "n_audio_tokens": 227, "retained_av_ratio": 0.45042678923177937}, {"method": "omnizip_r45", "video": "KMngzlwkYCU.mp4", "idx": 351, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6544177532196045, "peak_mem_gb": 9.152009963989258, "n_input_tokens": 2045, "n_video_tokens": 1584, "n_audio_tokens": 294, "retained_av_ratio": 0.45101171458998934}, {"method": "omnizip_r45", "video": "_J6U2xhnBX0.mp4", "idx": 353, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.67907452583313, "peak_mem_gb": 10.614707946777344, "n_input_tokens": 10545, "n_video_tokens": 8784, "n_audio_tokens": 1528, "retained_av_ratio": 0.45044608223429017}, {"method": "omnizip_r45", "video": "8-QqR2J8qxQ.mp4", "idx": 355, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.4574611186981201, "peak_mem_gb": 9.114363670349121, "n_input_tokens": 1759, "n_video_tokens": 1296, "n_audio_tokens": 246, "retained_av_ratio": 0.45136186770428016}, {"method": "omnizip_r45", "video": "S0ye2uVX2v0.mp4", "idx": 357, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.781641960144043, "peak_mem_gb": 9.193275928497314, "n_input_tokens": 2256, "n_video_tokens": 1728, "n_audio_tokens": 303, "retained_av_ratio": 0.44953225012309206}, {"method": "omnizip_r45", "video": "N8q8dFk0Ark.mp4", "idx": 359, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.1865167617797852, "peak_mem_gb": 9.081533908843994, "n_input_tokens": 1474, "n_video_tokens": 1008, "n_audio_tokens": 179, "retained_av_ratio": 0.45071609098567816}, {"method": "omnizip_r45", "video": "Wz3-b0MeWrQ.mp4", "idx": 361, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.018997430801392, "peak_mem_gb": 9.629902362823486, "n_input_tokens": 4787, "n_video_tokens": 3888, "n_audio_tokens": 695, "retained_av_ratio": 0.45057822387082697}, {"method": "omnizip_r45", "video": "cr6VlmDjePc.mp4", "idx": 363, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.832601547241211, "peak_mem_gb": 9.41046667098999, "n_input_tokens": 3623, "n_video_tokens": 2880, "n_audio_tokens": 510, "retained_av_ratio": 0.4504424778761062}, {"method": "omnizip_r45", "video": "wuHkSR0XJdo.mp4", "idx": 365, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.6935312747955322, "peak_mem_gb": 9.549880027770996, "n_input_tokens": 4479, "n_video_tokens": 3600, "n_audio_tokens": 630, "retained_av_ratio": 0.4494089834515366}, {"method": "omnizip_r45", "video": "bNu5CZVQUNI.mp4", "idx": 367, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.448297023773193, "peak_mem_gb": 10.499967098236084, "n_input_tokens": 10015, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4494301994301994}, {"method": "omnizip_r45", "video": "gx9mV8QfXso.mp4", "idx": 369, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0552394390106201, "peak_mem_gb": 9.022899150848389, "n_input_tokens": 1097, "n_video_tokens": 720, "n_audio_tokens": 127, "retained_av_ratio": 0.4769775678866588}, {"method": "omnizip_r45", "video": "YHZ6mWsKDfg.mp4", "idx": 371, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.5881850719451904, "peak_mem_gb": 9.35315990447998, "n_input_tokens": 3270, "n_video_tokens": 2592, "n_audio_tokens": 464, "retained_av_ratio": 0.450261780104712}, {"method": "omnizip_r45", "video": "KezGJkHX78E.mp4", "idx": 373, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5103297233581543, "peak_mem_gb": 9.13476276397705, "n_input_tokens": 1901, "n_video_tokens": 1440, "n_audio_tokens": 258, "retained_av_ratio": 0.4499411071849234}, {"method": "omnizip_r45", "video": "U1xMaQtB6sE.mp4", "idx": 375, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.535144805908203, "peak_mem_gb": 9.346469402313232, "n_input_tokens": 3166, "n_video_tokens": 2448, "n_audio_tokens": 427, "retained_av_ratio": 0.4483478260869565}, {"method": "omnizip_r45", "video": "-YuuVgM_b1Q.mp4", "idx": 377, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.564992904663086, "peak_mem_gb": 9.36095905303955, "n_input_tokens": 3290, "n_video_tokens": 2592, "n_audio_tokens": 455, "retained_av_ratio": 0.4499507712504102}, {"method": "omnizip_r45", "video": "bczYqpZQMWg.mp4", "idx": 379, "problem_type": "Narrative Comprehension", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 7.4236249923706055, "peak_mem_gb": 10.155686378479004, "n_input_tokens": 8330, "n_video_tokens": 6912, "n_audio_tokens": 1225, "retained_av_ratio": 0.45016590881160135}, {"method": "omnizip_r45", "video": "xnx5D0RNCCA.mp4", "idx": 381, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.184859275817871, "peak_mem_gb": 9.069318294525146, "n_input_tokens": 1441, "n_video_tokens": 1008, "n_audio_tokens": 198, "retained_av_ratio": 0.4552238805970149}, {"method": "omnizip_r45", "video": "uvGT_4WNYUw.mp4", "idx": 383, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6680607795715332, "peak_mem_gb": 9.184839725494385, "n_input_tokens": 2140, "n_video_tokens": 1584, "n_audio_tokens": 291, "retained_av_ratio": 0.4533333333333333}, {"method": "omnizip_r45", "video": "_Y7Q2syNYxY.mp4", "idx": 385, "problem_type": "Temporal Reasoning Localization", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.237180233001709, "peak_mem_gb": 10.51045274734497, "n_input_tokens": 10252, "n_video_tokens": 8496, "n_audio_tokens": 1480, "retained_av_ratio": 0.44917802726543704}, {"method": "omnizip_r45", "video": "fv-EzOxlpjo.mp4", "idx": 387, "problem_type": "Affective Intent Classification", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.01627779006958, "peak_mem_gb": 9.23790168762207, "n_input_tokens": 2479, "n_video_tokens": 1872, "n_audio_tokens": 341, "retained_av_ratio": 0.4518752824220515}, {"method": "omnizip_r45", "video": "z9FPvQc9yoI.mp4", "idx": 389, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.276942014694214, "peak_mem_gb": 9.266890525817871, "n_input_tokens": 2744, "n_video_tokens": 2160, "n_audio_tokens": 377, "retained_av_ratio": 0.448955459203784}, {"method": "omnizip_r45", "video": "uhWITF_qwqw.mp4", "idx": 391, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0703659057617188, "peak_mem_gb": 9.024500370025635, "n_input_tokens": 1057, "n_video_tokens": 720, "n_audio_tokens": 131, "retained_av_ratio": 0.5323149236192715}, {"method": "omnizip_r45", "video": "Syz-bdm5oEE.mp4", "idx": 393, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8789565563201904, "peak_mem_gb": 9.214430332183838, "n_input_tokens": 2404, "n_video_tokens": 1872, "n_audio_tokens": 331, "retained_av_ratio": 0.4516568315932819}, {"method": "omnizip_r45", "video": "aJwO65HE8A4.mp4", "idx": 395, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 8.604759216308594, "peak_mem_gb": 10.346709728240967, "n_input_tokens": 9525, "n_video_tokens": 7920, "n_audio_tokens": 1380, "retained_av_ratio": 0.44881720430107525}, {"method": "omnizip_r45", "video": "ZFIh8hf4btQ.mp4", "idx": 397, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "F", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.1297023296356201, "peak_mem_gb": 9.034743785858154, "n_input_tokens": 1200, "n_video_tokens": 864, "n_audio_tokens": 155, "retained_av_ratio": 0.492639842983317}, {"method": "omnizip_r45", "video": "MfjWOF8-npQ.mp4", "idx": 399, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.2589964866638184, "peak_mem_gb": 9.054769515991211, "n_input_tokens": 1392, "n_video_tokens": 1008, "n_audio_tokens": 177, "retained_av_ratio": 0.4489451476793249}, {"method": "omnizip_r45", "video": "hVG1xDJwMbY.mp4", "idx": 401, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 3.56 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.19 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 9.334262371063232, "n_input_tokens": 12882, "n_video_tokens": 9216, "n_audio_tokens": 3455, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "AghXXz9hI34.mp4", "idx": 403, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.4212052822113037, "peak_mem_gb": 9.1258225440979, "n_input_tokens": 1685, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.4684095860566449}, {"method": "omnizip_r45", "video": "spmQ5-8a6Ao.mp4", "idx": 405, "problem_type": "Creator Intent Taxonomy", "answer": "A", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.6267330646514893, "peak_mem_gb": 9.137362480163574, "n_input_tokens": 1906, "n_video_tokens": 1440, "n_audio_tokens": 259, "retained_av_ratio": 0.45203060623896407}, {"method": "omnizip_r45", "video": "2Gfm1i3DTaw.mp4", "idx": 407, "problem_type": "Creative Innovation Analysis", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.719947576522827, "peak_mem_gb": 10.545945167541504, "n_input_tokens": 10233, "n_video_tokens": 8496, "n_audio_tokens": 1497, "retained_av_ratio": 0.44961473031121785}, {"method": "omnizip_r45", "video": "nKN_c5WAVOw.mp4", "idx": 409, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.4031360149383545, "peak_mem_gb": 9.278456687927246, "n_input_tokens": 2787, "n_video_tokens": 2160, "n_audio_tokens": 395, "retained_av_ratio": 0.4500978473581213}, {"method": "omnizip_r45", "video": "S4oi8EUfLYI.mp4", "idx": 411, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.505157470703125, "peak_mem_gb": 9.721322059631348, "n_input_tokens": 5440, "n_video_tokens": 4464, "n_audio_tokens": 799, "retained_av_ratio": 0.4506935208056242}, {"method": "omnizip_r45", "video": "DYfYpkl2OTI.mp4", "idx": 413, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8019237518310547, "peak_mem_gb": 9.184175968170166, "n_input_tokens": 2229, "n_video_tokens": 1728, "n_audio_tokens": 315, "retained_av_ratio": 0.4522760646108664}, {"method": "omnizip_r45", "video": "51Sg4YQOPhI.mp4", "idx": 415, "problem_type": "Temporal Reasoning Localization", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.8196101188659668, "peak_mem_gb": 9.206908702850342, "n_input_tokens": 2300, "n_video_tokens": 1728, "n_audio_tokens": 326, "retained_av_ratio": 0.4527750730282376}, {"method": "omnizip_r45", "video": "zVvBiALQtE0.mp4", "idx": 417, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.433343410491943, "peak_mem_gb": 10.408946990966797, "n_input_tokens": 9810, "n_video_tokens": 8064, "n_audio_tokens": 1422, "retained_av_ratio": 0.4495045329959941}, {"method": "omnizip_r45", "video": "GhuS1QMn9hc.mp4", "idx": 419, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2520923614501953, "peak_mem_gb": 9.0611572265625, "n_input_tokens": 1277, "n_video_tokens": 864, "n_audio_tokens": 173, "retained_av_ratio": 0.5072324011571842}, {"method": "omnizip_r45", "video": "TUJbjPtM7JE.mp4", "idx": 421, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5430822372436523, "peak_mem_gb": 9.111557483673096, "n_input_tokens": 1750, "n_video_tokens": 1296, "n_audio_tokens": 250, "retained_av_ratio": 0.45278137128072443}, {"method": "omnizip_r45", "video": "kq9kiAScGw8.mp4", "idx": 423, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.581538677215576, "peak_mem_gb": 9.508809566497803, "n_input_tokens": 4281, "n_video_tokens": 3456, "n_audio_tokens": 622, "retained_av_ratio": 0.45022069641981366}, {"method": "omnizip_r45", "video": "NvEO1NCuTR0.mp4", "idx": 425, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.547142505645752, "peak_mem_gb": 9.469536304473877, "n_input_tokens": 4072, "n_video_tokens": 3312, "n_audio_tokens": 595, "retained_av_ratio": 0.44970565651394934}, {"method": "omnizip_r45", "video": "xhYip8GNPaM.mp4", "idx": 427, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "D", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.1082546710968018, "peak_mem_gb": 9.412417888641357, "n_input_tokens": 3635, "n_video_tokens": 2880, "n_audio_tokens": 518, "retained_av_ratio": 0.449970570924073}, {"method": "omnizip_r45", "video": "u6UOIS1h340.mp4", "idx": 429, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.8368439674377441, "peak_mem_gb": 9.009947776794434, "n_input_tokens": 906, "n_video_tokens": 576, "n_audio_tokens": 102, "retained_av_ratio": 0.5752212389380531}, {"method": "omnizip_r45", "video": "hZes-czBp5A.mp4", "idx": 431, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.9628922939300537, "peak_mem_gb": 9.205981254577637, "n_input_tokens": 2383, "n_video_tokens": 1872, "n_audio_tokens": 340, "retained_av_ratio": 0.451627486437613}, {"method": "omnizip_r45", "video": "bj_1uCcsxrg.mp4", "idx": 433, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.897549867630005, "peak_mem_gb": 10.541606903076172, "n_input_tokens": 10198, "n_video_tokens": 8496, "n_audio_tokens": 1495, "retained_av_ratio": 0.44950455409868884}, {"method": "omnizip_r45", "video": "aQj9FsOGmFQ.mp4", "idx": 435, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.59591007232666, "peak_mem_gb": 10.553412437438965, "n_input_tokens": 10195, "n_video_tokens": 8496, "n_audio_tokens": 1501, "retained_av_ratio": 0.44963489046714017}, {"method": "omnizip_r45", "video": "pRcxjOCF3iM.mp4", "idx": 437, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.0852415561676025, "peak_mem_gb": 9.040113925933838, "n_input_tokens": 1156, "n_video_tokens": 720, "n_audio_tokens": 146, "retained_av_ratio": 0.4815242494226328}, {"method": "omnizip_r45", "video": "42mYOogHnyE.mp4", "idx": 439, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.1136045455932617, "peak_mem_gb": 9.260900974273682, "n_input_tokens": 2640, "n_video_tokens": 2016, "n_audio_tokens": 370, "retained_av_ratio": 0.45222129086336965}, {"method": "omnizip_r45", "video": "9yjRUlmmXyk.mp4", "idx": 441, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 9.722120761871338, "peak_mem_gb": 10.547068119049072, "n_input_tokens": 10251, "n_video_tokens": 8496, "n_audio_tokens": 1498, "retained_av_ratio": 0.44966980188112865}, {"method": "omnizip_r45", "video": "1C6OKfQaHJ4.mp4", "idx": 443, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.851914644241333, "peak_mem_gb": 9.364535808563232, "n_input_tokens": 3308, "n_video_tokens": 2592, "n_audio_tokens": 467, "retained_av_ratio": 0.44982020268061457}, {"method": "omnizip_r45", "video": "n4fypasG9N8.mp4", "idx": 445, "problem_type": "Narrative Comprehension", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.0678489208221436, "peak_mem_gb": 9.031550884246826, "n_input_tokens": 1130, "n_video_tokens": 720, "n_audio_tokens": 148, "retained_av_ratio": 0.4827188940092166}, {"method": "omnizip_r45", "video": "QyR4tUx_eE4.mp4", "idx": 447, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.6307237148284912, "peak_mem_gb": 9.116174221038818, "n_input_tokens": 1760, "n_video_tokens": 1296, "n_audio_tokens": 228, "retained_av_ratio": 0.44816272965879267}, {"method": "omnizip_r45", "video": "Inh0nn0Mehw.mp4", "idx": 449, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.2306087017059326, "peak_mem_gb": 9.052551746368408, "n_input_tokens": 1383, "n_video_tokens": 1008, "n_audio_tokens": 178, "retained_av_ratio": 0.45109612141652616}, {"method": "omnizip_r45", "video": "TqgdJIWKEHU.mp4", "idx": 451, "problem_type": "Creative Innovation Analysis", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.869392395019531, "peak_mem_gb": 9.767869472503662, "n_input_tokens": 5679, "n_video_tokens": 4608, "n_audio_tokens": 802, "retained_av_ratio": 0.44898336414048057}, {"method": "omnizip_r45", "video": "BB4i0vWjSF4.mp4", "idx": 453, "problem_type": "Narrative Comprehension", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.022923231124878, "peak_mem_gb": 9.23115348815918, "n_input_tokens": 2536, "n_video_tokens": 2002, "n_audio_tokens": 353, "retained_av_ratio": 0.4496815286624204}, {"method": "omnizip_r45", "video": "2LMNEWSR8K0.mp4", "idx": 455, "problem_type": "Affective Intent Classification", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5519418716430664, "peak_mem_gb": 9.137032985687256, "n_input_tokens": 1915, "n_video_tokens": 1440, "n_audio_tokens": 276, "retained_av_ratio": 0.4527972027972028}, {"method": "omnizip_r45", "video": "jfN_xuGVfPg.mp4", "idx": 457, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.5380094051361084, "peak_mem_gb": 9.114030361175537, "n_input_tokens": 1751, "n_video_tokens": 1296, "n_audio_tokens": 241, "retained_av_ratio": 0.4541314248536109}, {"method": "omnizip_r45", "video": "KMRepMFj6y0.mp4", "idx": 459, "problem_type": "Creator Intent Taxonomy", "answer": "F", "pred": "E", "correct": false, "error": null, "has_audio": true, "gen_seconds": 5.193942070007324, "peak_mem_gb": 9.829770565032959, "n_input_tokens": 6143, "n_video_tokens": 5040, "n_audio_tokens": 879, "retained_av_ratio": 0.4497381314411218}, {"method": "omnizip_r45", "video": "iwIjN30bQG8.mp4", "idx": 461, "problem_type": "Humor & Meme Deconstruction", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 5.049201726913452, "peak_mem_gb": 9.788096904754639, "n_input_tokens": 5844, "n_video_tokens": 4752, "n_audio_tokens": 842, "retained_av_ratio": 0.4490525563103325}, {"method": "omnizip_r45", "video": "SD3b0eDGylU.mp4", "idx": 463, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "B", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.471749782562256, "peak_mem_gb": 9.313385009765625, "n_input_tokens": 3068, "n_video_tokens": 2448, "n_audio_tokens": 442, "retained_av_ratio": 0.4494809688581315}, {"method": "omnizip_r45", "video": "wskyXFw6C4I.mp4", "idx": 465, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9562766551971436, "peak_mem_gb": 9.015161037445068, "n_input_tokens": 1076, "n_video_tokens": 720, "n_audio_tokens": 140, "retained_av_ratio": 0.48255813953488375}, {"method": "omnizip_r45", "video": "9BTyy18OYe4.mp4", "idx": 467, "problem_type": "Humor & Meme Deconstruction", "answer": "A", "pred": "D", "correct": false, "error": null, "has_audio": true, "gen_seconds": 1.3971097469329834, "peak_mem_gb": 9.09235668182373, "n_input_tokens": 1592, "n_video_tokens": 1152, "n_audio_tokens": 225, "retained_av_ratio": 0.45824255628177196}, {"method": "omnizip_r45", "video": "-gl0tl4qvzY.mp4", "idx": 469, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 0.9713630676269531, "peak_mem_gb": 9.017022132873535, "n_input_tokens": 1081, "n_video_tokens": 720, "n_audio_tokens": 129, "retained_av_ratio": 0.47703180212014135}, {"method": "omnizip_r45", "video": "_zYW8CObyxM.mp4", "idx": 471, "problem_type": "Creator Intent Taxonomy", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.624833106994629, "peak_mem_gb": 10.552626609802246, "n_input_tokens": 10265, "n_video_tokens": 8496, "n_audio_tokens": 1500, "retained_av_ratio": 0.4491796718687475}, {"method": "omnizip_r45", "video": "yEyUrr7d0aI.mp4", "idx": 473, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.5591557025909424, "peak_mem_gb": 9.484463691711426, "n_input_tokens": 4117, "n_video_tokens": 3312, "n_audio_tokens": 600, "retained_av_ratio": 0.450920245398773}, {"method": "omnizip_r45", "video": "N8S9TYC6mSE.mp4", "idx": 475, "problem_type": "Temporal Reasoning Localization", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.597853899002075, "peak_mem_gb": 9.35508108139038, "n_input_tokens": 3156, "n_video_tokens": 2380, "n_audio_tokens": 436, "retained_av_ratio": 0.45063920454545453}, {"method": "omnizip_r45", "video": "sHAaDXWyN3c.mp4", "idx": 477, "problem_type": "Narrative Comprehension", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 1.009619951248169, "peak_mem_gb": 9.04430627822876, "n_input_tokens": 1243, "n_video_tokens": 864, "n_audio_tokens": 165, "retained_av_ratio": 0.4849368318756074}, {"method": "omnizip_r45", "video": "H3iTVprK7wU.mp4", "idx": 479, "problem_type": "Narrative Comprehension", "answer": "F", "pred": null, "correct": false, "error": "OutOfMemoryError: CUDA out of memory. Tried to allocate 1.93 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.26 GiB is free. Including non-PyTorch memor", "has_audio": true, "gen_seconds": null, "peak_mem_gb": 11.124424934387207, "n_input_tokens": 11977, "n_video_tokens": 9216, "n_audio_tokens": 2541, "retained_av_ratio": null}, {"method": "omnizip_r45", "video": "O2dajNms-aw.mp4", "idx": 481, "problem_type": "Creative Innovation Analysis", "answer": "A", "pred": "A", "correct": true, "error": null, "has_audio": true, "gen_seconds": 3.259314775466919, "peak_mem_gb": 9.491124153137207, "n_input_tokens": 4139, "n_video_tokens": 3312, "n_audio_tokens": 596, "retained_av_ratio": 0.4501023541453429}, {"method": "omnizip_r45", "video": "w1MSc5_NspY.mp4", "idx": 483, "problem_type": "Creator Intent Taxonomy", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5734238624572754, "peak_mem_gb": 9.35966157913208, "n_input_tokens": 3293, "n_video_tokens": 2592, "n_audio_tokens": 475, "retained_av_ratio": 0.4512552983371373}, {"method": "omnizip_r45", "video": "XBRZ_O5FLQ8.mp4", "idx": 485, "problem_type": "Creative Innovation Analysis", "answer": "B", "pred": "B", "correct": true, "error": null, "has_audio": true, "gen_seconds": 2.5976955890655518, "peak_mem_gb": 9.368110656738281, "n_input_tokens": 3315, "n_video_tokens": 2592, "n_audio_tokens": 470, "retained_av_ratio": 0.4516655780535598}, {"method": "omnizip_r45", "video": "HuqQmbOK-AQ.mp4", "idx": 487, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 0.9791262149810791, "peak_mem_gb": 9.034767627716064, "n_input_tokens": 1201, "n_video_tokens": 864, "n_audio_tokens": 161, "retained_av_ratio": 0.4946341463414634}, {"method": "omnizip_r45", "video": "jYA4p6xBn2s.mp4", "idx": 489, "problem_type": "Humor & Meme Deconstruction", "answer": "E", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 3.738391876220703, "peak_mem_gb": 9.582050800323486, "n_input_tokens": 4640, "n_video_tokens": 3744, "n_audio_tokens": 661, "retained_av_ratio": 0.4490351872871737}, {"method": "omnizip_r45", "video": "Zn6Dnm2OAPs.mp4", "idx": 491, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 9.616131782531738, "peak_mem_gb": 10.500386714935303, "n_input_tokens": 10124, "n_video_tokens": 8352, "n_audio_tokens": 1476, "retained_av_ratio": 0.4500407000407}, {"method": "omnizip_r45", "video": "d6zr9dqsvfA.mp4", "idx": 493, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "C", "correct": false, "error": null, "has_audio": true, "gen_seconds": 2.6742844581604004, "peak_mem_gb": 9.349608421325684, "n_input_tokens": 3260, "n_video_tokens": 2592, "n_audio_tokens": 460, "retained_av_ratio": 0.449213630406291}, {"method": "omnizip_r45", "video": "UNG7OtwrQoI.mp4", "idx": 495, "problem_type": "Creator Intent Taxonomy", "answer": "D", "pred": "A", "correct": false, "error": null, "has_audio": true, "gen_seconds": 7.080990791320801, "peak_mem_gb": 10.106907844543457, "n_input_tokens": 7996, "n_video_tokens": 6624, "n_audio_tokens": 1174, "retained_av_ratio": 0.4499871761990254}, {"method": "omnizip_r45", "video": "3c1sjLnzO1M.mp4", "idx": 497, "problem_type": "Narrative Comprehension", "answer": "C", "pred": "C", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.061153411865234, "peak_mem_gb": 9.647185325622559, "n_input_tokens": 4847, "n_video_tokens": 3888, "n_audio_tokens": 696, "retained_av_ratio": 0.450043630017452}, {"method": "omnizip_r45", "video": "TIlDW7FLwXk.mp4", "idx": 499, "problem_type": "Affective Intent Classification", "answer": "F", "pred": "F", "correct": true, "error": null, "has_audio": true, "gen_seconds": 4.320363283157349, "peak_mem_gb": 9.713338851928711, "n_input_tokens": 5332, "n_video_tokens": 4320, "n_audio_tokens": 769, "retained_av_ratio": 0.4495971703674592}]

## 2. GPU

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i} {p.name} {p.total_memory / 2**30:.1f} GB")
assert n_gpu >= 1, "Không có GPU: Settings → Accelerator → GPU T4 x2"
NUM_SHARDS = min(n_gpu, 2)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

## 3. Dữ liệu (chỉ các video cần chạy) + model

In [ ]:
import time
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("dùng HF_TOKEN từ Kaggle Secrets")
except Exception:
    print("không có secret HF_TOKEN — tải ẩn danh, dễ bị 429")
os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import hf_hub_download, snapshot_download

def with_retry(fn, tries=6):
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except Exception as ex:
            if attempt == tries:
                raise
            print(f"lần {attempt} lỗi: {type(ex).__name__}: {str(ex)[:150]} → chờ {60 * attempt}s"); time.sleep(60 * attempt)

REPO = "TencentARC/ShortVid-Bench"
data_json = with_retry(lambda: hf_hub_download(REPO, "ShortVid-Bench-1k.json", repo_type="dataset", local_dir=DATA_DIR))
with_retry(lambda: snapshot_download(REPO, repo_type="dataset", local_dir=DATA_DIR, max_workers=4,
                                     allow_patterns=[f"videos/{v}" for v in RUN_VIDEOS]))
missing = [v for v in RUN_VIDEOS if not os.path.exists(f"{DATA_DIR}/videos/{v}")]
assert not missing, f"thiếu {len(missing)} video — chạy lại cell này"
with_retry(lambda: snapshot_download(MODEL_ID, max_workers=4))
print(len(RUN_VIDEOS), "video đã có")

## 4. Worker

In [ ]:
%%writefile /kaggle/working/svb_eval.py
"""ShortVid-Bench worker: full-token Qwen2.5-Omni vs OmniZip, one process per GPU.

Run by the Kaggle notebook (one subprocess per GPU), or directly:
    python svb_eval.py --config run_config.json --shard 0 --num-shards 2

Every finished sample is appended to <results_dir>/<run_tag>__<method>__shard<k>.jsonl, so a
restarted run skips what is already there. Helpers below the worker (load_records, summarize)
have no torch/transformers import and are reused by the notebook's summary cell.
"""
import argparse
import glob
import json
import math
import os
import random
import sys
import time
import traceback

OPTION_LETTERS = ["A", "B", "C", "D", "E", "F"]


def extract_answer(resp_text):
    """Same rule as eval/eval_shortvideobench.py, so scores stay comparable with the repo script."""
    resp_text = (resp_text or "").strip()
    for opt in OPTION_LETTERS:
        if resp_text.upper().startswith(opt):
            return opt
    return resp_text[0].upper() if resp_text else None


def qid(r):
    """Question id: WorldSense has several questions per video; ShortVid-Bench one, keyed by the video."""
    return r.get("qid") or r["video"]


def result_path(results_dir, run_tag, method, shard):
    return os.path.join(results_dir, f"{run_tag}__{method}__shard{shard}.jsonl")


def done_videos(result_roots, run_tag, method):
    """Question ids (qid) already scored for (run_tag, method) in this session or in restored inputs."""
    done = set()
    for root in result_roots:
        for p in glob.glob(os.path.join(root, "**", f"{run_tag}__{method}__shard*.jsonl"), recursive=True):
            with open(p, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            r = json.loads(line)
                        except json.JSONDecodeError:
                            continue  # a line cut off by a killed session; the sample is simply redone
                        if not r.get("error"):  # failed samples are retried on the next start
                            done.add(r.get("qid", r["video"]))
    return done


def select_samples(samples, mini_test_num, shard, num_shards, videos=None):
    if videos:  # explicit list (e.g. re-running failed videos); idx stays the sample's index in the 1k set
        want = set(videos)
        picked = [(i, s) for i, s in enumerate(samples) if s["video"] in want]
    else:
        picked = list(enumerate(samples[:mini_test_num] if mini_test_num else samples))
    return [p for j, p in enumerate(picked) if j % num_shards == shard]


# ----------------------------------------------------------------------------- T4 workarounds

def segment_sdpa(q, k, v, cu_seqlens):
    """Attention restricted to each [cu[i-1], cu[i]) segment; q, k, v: (S, H, D) -> (S, H, D).

    Same result as SDPA with the dense block-diagonal (1, S, S) mask the vision encoder builds,
    without that mask: on T4 (no FlashAttention) the masked call falls back to a kernel that
    materialises H x S x S scores and runs out of memory at ~32k patches (128 frames). Segments of
    equal length are batched into one call.
    """
    import torch
    import torch.nn.functional as F
    out = torch.empty_like(q)
    cu = cu_seqlens.tolist() if hasattr(cu_seqlens, "tolist") else list(cu_seqlens)
    by_len = {}
    for a, b in zip(cu[:-1], cu[1:]):
        if b > a:
            by_len.setdefault(b - a, []).append(a)
    for length, starts in by_len.items():
        idx = torch.tensor(starts, device=q.device)[:, None] + torch.arange(length, device=q.device)  # (n, L)
        o = F.scaled_dot_product_attention(q[idx].transpose(1, 2), k[idx].transpose(1, 2), v[idx].transpose(1, 2))
        out[idx] = o.transpose(1, 2)
    return out


def patch_vision_sdpa(module):
    """Swap Qwen2_5OmniVisionSdpaAttention.forward in `module` for the segment-wise version (same math)."""
    rope = module.apply_rotary_pos_emb_vision

    def forward(self, hidden_states, cu_seqlens, rotary_pos_emb=None):
        seq_length = hidden_states.shape[0]
        q = self.q(hidden_states).reshape(seq_length, self.num_heads, -1)
        k = self.k(hidden_states).reshape(seq_length, self.num_heads, -1)
        v = self.v(hidden_states).reshape(seq_length, self.num_heads, -1)
        q = rope(q.unsqueeze(0), rotary_pos_emb).squeeze(0)
        k = rope(k.unsqueeze(0), rotary_pos_emb).squeeze(0)
        attn_output = segment_sdpa(q, k, v, cu_seqlens).reshape(seq_length, -1)
        return self.proj(attn_output)

    module.Qwen2_5OmniVisionSdpaAttention.forward = forward


def patch_audio_sdpa(module):
    """Same segment-wise attention for the audio encoder, whose sdpa path also builds a dense (1, S, S) mask
    and ran out of memory on long audio (~4k audio tokens). The OmniZip copy also returns its audio scores,
    computed exactly as before (whole sequence, unmasked); the transformers copy returns only the output."""
    import inspect
    cls = module.Qwen2_5OmniAudioSdpaAttention
    returns_scores = "return_logits" in inspect.signature(cls.forward).parameters

    def forward(self, hidden_states, cu_seqlens=None, return_logits=False, **kwargs):
        seq_length = hidden_states.shape[0]
        q = self.q_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        k = self.k_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        v = self.v_proj(hidden_states).reshape(seq_length, self.num_heads, -1)
        out = self.out_proj(segment_sdpa(q, k, v, cu_seqlens).reshape(seq_length, self.embed_dim))
        if not returns_scores:
            return out
        if not return_logits:
            return out, None, None
        q, k = q.transpose(0, 1), k.transpose(0, 1)  # [H, T, D], as the original sdpa path
        return out, module._audio_token_importance(q, k), k

    cls.forward = forward


def make_last_token_head(head):
    """lm_head that only projects the last position. Prefill otherwise builds logits for the whole sequence
    (11k tokens x 152k vocab x fp16 = 3.2 GB, the OOM of long videos in `full`); greedy generate() reads only
    logits[:, -1], so answers are unchanged."""
    import torch

    class LastTokenHead(torch.nn.Module):
        def __init__(self, head):
            super().__init__()
            self.head = head

        def forward(self, hidden_states):
            return self.head(hidden_states[:, -1:, :])

    return LastTokenHead(head)


class StageTimer:
    """CUDA-synchronised wall time and peak memory per stage of one sample (vision, audio, omnizip, LLM...).

    Hooks synchronise before and after each stage, so stages do not overlap; the peak memory of the gaps
    between stages is folded into `peak_gb` before each reset."""

    def __init__(self, torch):
        self.torch = torch
        self.reset()

    def reset(self):
        self.t, self.peak, self.n, self.peak_gb, self.llm_tokens, self.ttft_at = {}, {}, {}, 0.0, None, None

    def begin(self):
        tc = self.torch.cuda
        tc.synchronize()
        self.peak_gb = max(self.peak_gb, tc.max_memory_allocated() / 2**30)
        tc.reset_peak_memory_stats()
        return time.perf_counter()

    def end(self, name, t0):
        tc = self.torch.cuda
        tc.synchronize()
        p = tc.max_memory_allocated() / 2**30
        self.t[name] = self.t.get(name, 0.0) + time.perf_counter() - t0
        self.peak[name] = max(self.peak.get(name, 0.0), p)
        self.n[name] = self.n.get(name, 0) + 1
        self.peak_gb = max(self.peak_gb, p)

    def hook(self, module, name, llm=False):
        state = {}

        def pre(m, args, kwargs):
            if llm and self.llm_tokens is None:  # first decoder call of the sample = prefill
                x = kwargs.get("inputs_embeds")
                self.llm_tokens = int(x.shape[1]) if x is not None else None
            state["t0"] = self.begin()

        def post(m, args, kwargs, out):
            stage = ("llm_prefill" if "llm_prefill" not in self.t else "llm_decode") if llm else name
            self.end(stage, state["t0"])
            if name == "lm_head" and self.ttft_at is None:
                self.ttft_at = time.perf_counter()

        module.register_forward_pre_hook(pre, with_kwargs=True)
        module.register_forward_hook(post, with_kwargs=True)

    def record(self):
        r = {f"t_{k}": v for k, v in self.t.items()}
        r.update({f"peak_{k}_gb": v for k, v in self.peak.items()})
        r["n_decode_steps"] = self.n.get("llm_decode", 0)
        r["n_llm_tokens"] = self.llm_tokens
        return r


def read_video_pyav(ele):
    """PyAV reader with qwen-omni-utils' sampling; stands in for its torchvision fallback,
    whose torchvision.io.read_video no longer exists in the torchvision Kaggle ships."""
    import av
    import numpy as np
    import torch
    from qwen_omni_utils.v2_5 import vision_process as vp
    path = ele["video"]
    with av.open(path) as c:
        s = c.streams.video[0]
        fps = float(s.average_rate or s.guessed_rate or 30)
        total = sum(1 for p in c.demux(s) if p.size > 0)
    nframes = vp.smart_nframes(ele, total_frames=total, video_fps=fps)
    idx = torch.linspace(0, total - 1, nframes).round().long().tolist()
    want, got = set(idx), {}
    with av.open(path) as c:
        for i, f in enumerate(c.decode(video=0)):
            if i in want:
                got[i] = f.to_ndarray(format="rgb24")
            if i >= idx[-1]:
                break
    assert got, f"PyAV decoded no frames from {path}"
    have = sorted(got)
    frames = [got[j] if j in got else got[min(have, key=lambda h: abs(h - j))] for j in idx]
    video = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2)  # TCHW, like the other readers
    return video, nframes / max(total, 1e-6) * fps


def rss_gb():
    """Resident memory of this process (Linux); nan elsewhere."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
    except OSError:
        pass
    return float("nan")


def trim_memory():
    # Decoding 128 full-HD frames per sample leaves freed GB in glibc arenas; hand them back to the OS
    import gc
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass


def mmr_select(scores, feats, k, lam):
    """Greedy MMR in every frame at once: scores [T, P], feats [T, P, D] -> kept indices [T, k].

    i* = argmax_{i not in S} lam * s_i - (1 - lam) * max_{j in S} cos(x_i, x_j), with s min-max normalised per frame
    (scores and cosines on one scale). lam = 1 is the plain per-frame top-k of the score.
    """
    import torch
    s = scores.float()
    lo, hi = s.amin(1, keepdim=True), s.amax(1, keepdim=True)
    s = (s - lo) / (hi - lo).clamp_min(1e-6)
    x = torch.nn.functional.normalize(feats.float(), dim=-1)
    sim = x @ x.transpose(1, 2)                      # [T, P, P]
    T = s.shape[0]
    rows = torch.arange(T, device=s.device)
    maxsim = torch.full_like(s, -1.0)               # empty S: same penalty for every token, so step 1 = argmax s
    picked = torch.zeros_like(s, dtype=torch.bool)
    out = torch.empty(T, k, dtype=torch.long, device=s.device)
    for step in range(k):
        gain = (lam * s - (1 - lam) * maxsim).masked_fill(picked, float("-inf"))
        i = gain.argmax(1)
        out[:, step] = i
        picked[rows, i] = True
        maxsim = torch.maximum(maxsim, sim[rows, :, i])
    return out


def av_align_select(video_feature, audio_feature, num_tokens_per_frame, merging_ratio, aligner, mmr_lambda):
    """omnizip_core.omnizip_av_align with its per-frame top-k replaced by MMR over the same AV_Aligner scores."""
    import torch
    P = num_tokens_per_frame
    T = video_feature.shape[0] // P
    mask = torch.zeros(video_feature.shape[0], dtype=torch.bool, device=video_feature.device)
    k = int(P * (1.0 - merging_ratio))
    if T == 0 or k >= P:
        mask[: T * P] = True
        return mask
    frames = video_feature[: T * P].reshape(T, P, -1)
    aud = audio_feature.to(frames.dtype).unsqueeze(0).expand(T, -1, -1)
    with torch.no_grad():
        scores = aligner.token_scores(aud, frames)  # [T, P]
        keep = mmr_select(scores, frames, k, mmr_lambda)
    mask[(keep + torch.arange(T, device=mask.device)[:, None] * P).reshape(-1)] = True
    return mask


def save_viz(results_dir, method, video, frames, grid_thw, video_keep, n_show=4):
    """4 frames + the kept-video-token mask (t, h/2, w/2), for the notebook's overlay plot."""
    import numpy as np
    t, h, w = (int(x) for x in grid_thw.tolist())
    keep = (np.ones(t * (h // 2) * (w // 2), bool) if video_keep is None else video_keep).reshape(t, h // 2, w // 2)
    tp = np.linspace(0, t - 1, n_show).round().astype(int)  # temporal patch index; it covers frames 2tp, 2tp+1
    imgs = frames[[min(2 * i, frames.shape[0] - 1) for i in tp]].float().clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
    os.makedirs(os.path.join(results_dir, "viz"), exist_ok=True)
    np.savez_compressed(os.path.join(results_dir, "viz", f"{method}__{video}.npz"),
                        frames=imgs, tp=tp, keep=keep[tp], grid=np.array([t, h, w]))


EXIT_HIGH_RSS = 75  # worker stopped itself between samples; the notebook restarts it and resume skips done samples


# ----------------------------------------------------------------------------- worker

def run_worker(cfg, shard, num_shards, methods):
    os.environ.setdefault("FORCE_QWENVL_VIDEO_READER", cfg["video_reader"])
    sys.path.insert(0, cfg["code_root"])

    import numpy as np
    import torch
    from transformers import Qwen2_5OmniProcessor
    import transformers.models.qwen2_5_omni.modeling_qwen2_5_omni as hf_modeling
    import omnizip.modeling_qwen2_5_omni as oz_modeling
    from qwen_omni_utils import process_mm_info
    from qwen_omni_utils.v2_5 import vision_process
    from qwen_omni_utils.v2_5.audio_process import _check_if_video_has_audio

    # Both methods get the same vision-encoder fix, so the comparison stays like for like
    for m in (hf_modeling, oz_modeling):
        patch_vision_sdpa(m)
        if cfg.get("fix_audio_sdpa"):
            patch_audio_sdpa(m)
    vision_process.VIDEO_READER_BACKENDS["torchvision"] = read_video_pyav

    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    torch.cuda.manual_seed_all(cfg["seed"])

    with open(cfg["data_json"], encoding="utf-8") as f:
        samples = json.load(f)
    todo_all = select_samples(samples, cfg["mini_test_num"], shard, num_shards, cfg.get("videos"))
    viz_videos = set(cfg.get("viz_videos") or [])
    os.makedirs(cfg["results_dir"], exist_ok=True)
    processor = Qwen2_5OmniProcessor.from_pretrained(cfg["model_id"])
    dtype = getattr(torch, cfg["dtype"])

    for method in methods:
        mcfg = cfg["methods"][method]
        done = done_videos(cfg["result_roots"], cfg["run_tag"], method)
        todo = [(i, s) for i, s in todo_all if qid(s) not in done]
        print(f"[shard{shard}] {method}: {len(todo_all) - len(todo)} done, {len(todo)} to go", flush=True)
        if not todo:
            continue

        if mcfg["wrapper"] == "omnizip":
            from omnizip.modeling_qwen2_5_omni import Qwen2_5OmniForConditionalGeneration
        else:
            from transformers import Qwen2_5OmniForConditionalGeneration
        model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
            cfg["model_id"], torch_dtype=dtype, device_map="cuda:0", attn_implementation=cfg["attn_implementation"],
        )
        model.disable_talker()  # text answers only; frees the talker/token2wav weights
        model.eval()
        model.thinker.omnizip_config = (
            {k: mcfg[k] for k in ("rho_audio", "rho_video", "g", "contextual_ratio")}
            if mcfg["wrapper"] == "omnizip" else None
        )

        if cfg.get("fix_lm_head"):
            model.thinker.lm_head = make_last_token_head(model.thinker.lm_head)
        timer = StageTimer(torch) if cfg.get("profile") else None
        if timer:
            th = model.thinker
            for mod, name in ((th.visual, "vision"), (th.audio_tower, "audio"), (th.lm_head, "lm_head")):
                timer.hook(mod, name)
            timer.hook(th.model, "llm", llm=True)

        tcfg = model.config.thinker_config
        audio_id = getattr(tcfg, "audio_token_index", None) or getattr(tcfg, "audio_token_id")
        video_id = getattr(tcfg, "video_token_index", None) or getattr(tcfg, "video_token_id")

        # Record what OmniZip actually kept; the thinker looks the function up on every call
        last = {}
        if mcfg["wrapper"] == "omnizip":
            import omnizip.omnizip_core as oc
            orig = getattr(oc, "_orig_omnizip", oc.omnizip)
            oc._orig_omnizip = orig

            def recorded(*args, **kw):
                t_oz = timer.begin() if timer else None
                embeds, mask = orig(*args, **kw)
                if timer:
                    timer.end("omnizip", t_oz)
                ids = kw["input_ids"].reshape(-1)
                last.update(
                    kept_audio=int(mask[ids == kw["audio_token_id"]].sum()),
                    kept_video=int(mask[ids == kw["video_token_id"]].sum()),
                    n_audio_scores=int(kw["attn_logits"].numel()),
                )
                if last.get("viz"):
                    last["video_keep"] = mask[ids == kw["video_token_id"]].cpu().numpy()
                return embeds, mask
            oc.omnizip = recorded

            orig_align = getattr(oc, "_orig_av_align", oc.omnizip_av_align)
            oc._orig_av_align = orig_align

            def istm_only(video_feature, audio_feature, num_tokens_per_frame, merging_ratio, aligner):
                # Paper's ISTC, as in upstream KD-TAO/OmniZip: same groups and ratios, AV_Aligner bypassed
                last["n_istm_groups"] = last.get("n_istm_groups", 0) + 1
                return oc.omnizip_istm(video_feature, num_tokens_per_frame, merging_ratio)

            def av_align_mmr(video_feature, audio_feature, num_tokens_per_frame, merging_ratio, aligner):
                last["n_mmr_groups"] = last.get("n_mmr_groups", 0) + 1
                return av_align_select(video_feature, audio_feature, num_tokens_per_frame, merging_ratio, aligner,
                                       mmr_lambda=mcfg.get("mmr_lambda", 0.5))
            oc.omnizip_av_align = {"istm": istm_only, "av_aligner_mmr": av_align_mmr}.get(
                mcfg.get("video_compressor"), orig_align)

        out_path = result_path(cfg["results_dir"], cfg["run_tag"], method, shard)
        with open(out_path, "a", encoding="utf-8") as out:
            for n_done, (idx, s) in enumerate(todo, 1):
                last.clear()
                last["viz"] = s["video"] in viz_videos
                if timer:
                    timer.reset()
                rec = {"run_tag": cfg["run_tag"], "method": method, "idx": idx, "qid": qid(s), "video": s["video"],
                       "problem_type": s["problem_type"], "answer": s["answer"], "error": None}
                video_path = os.path.join(cfg["video_dir"], s["video"])
                prompt = (f"{s['question']}\nOptions:\n" + "\n".join(s["candidates"]) +
                          "\nAnswer with the option's letter from the given choices directly.")
                conversation = [
                    {"role": "system", "content": [{"type": "text", "text": cfg["system_prompt"]}]},
                    {"role": "user", "content": [
                        {"type": "video", "video": video_path, "fps": cfg["fps"],
                         "max_frames": cfg["max_frames"], "max_pixels": cfg["video_max_pixels"]},
                        {"type": "text", "text": prompt},
                    ]},
                ]
                try:
                    t0 = time.time()
                    use_audio = _check_if_video_has_audio(video_path)
                    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
                    audios, images, videos = process_mm_info(conversation, use_audio_in_video=use_audio)
                    t_read = time.time()
                    inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt",
                                       padding=True, use_audio_in_video=use_audio)
                    t_proc = time.time()
                    inputs = inputs.to(model.device).to(model.dtype)
                    torch.cuda.synchronize()
                    rec.update(t_read_decode=t_read - t0, t_processor=t_proc - t_read, t_to_gpu=time.time() - t_proc)
                    ids = inputs["input_ids"]
                    n_audio = int((ids == audio_id).sum()); n_video = int((ids == video_id).sum())
                    # recorded before generate() so a failed sample still says what it was
                    rec.update(has_audio=use_audio, n_frames=int(videos[0].shape[0]), n_input_tokens=int(ids.shape[1]),
                               n_audio_tokens=n_audio, n_video_tokens=n_video)
                    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                    t1 = time.time()
                    g0 = time.perf_counter()  # same clock as StageTimer.ttft_at
                    with torch.no_grad():
                        out_ids = model.generate(
                            **inputs, use_audio_in_video=use_audio, return_audio=False,
                            thinker_do_sample=False, thinker_max_new_tokens=cfg["max_new_tokens"],
                            eos_token_id=processor.tokenizer.eos_token_id, pad_token_id=processor.tokenizer.pad_token_id,
                        )
                    torch.cuda.synchronize()
                    t2 = time.time()
                    resp = processor.batch_decode(out_ids[:, ids.shape[1]:], skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False)[0].strip()
                    pred = extract_answer(resp)
                    kept_a = last.get("kept_audio", n_audio); kept_v = last.get("kept_video", n_video)
                    rec.update(
                        pred=pred, correct=pred == s["answer"], response=resp, kept_audio=kept_a, kept_video=kept_v,
                        retained_av_ratio=(kept_a + kept_v) / max(n_audio + n_video, 1),
                        n_audio_scores=last.get("n_audio_scores"), n_istm_groups=last.get("n_istm_groups"),
                        n_mmr_groups=last.get("n_mmr_groups"),
                        n_new_tokens=int(out_ids.shape[1] - ids.shape[1]),
                        prep_seconds=t1 - t0, gen_seconds=t2 - t1,
                        peak_mem_gb=max(torch.cuda.max_memory_allocated() / 2**30, timer.peak_gb if timer else 0),
                    )
                    if timer:
                        rec.update(timer.record(), ttft_seconds=timer.ttft_at - g0 if timer.ttft_at else None)
                    if last["viz"]:
                        save_viz(cfg["results_dir"], method, s["video"], videos[0], inputs["video_grid_thw"][0],
                                 last.get("video_keep"))
                except Exception as e:  # one bad video must not stop the shard; it is scored as wrong
                    rec.update(pred=None, correct=False, response=None,
                               error=f"{type(e).__name__}: {e}".splitlines()[0][:500],
                               traceback=traceback.format_exc()[-4000:],
                               peak_mem_gb=max(torch.cuda.max_memory_allocated() / 2**30, timer.peak_gb if timer else 0))
                    if timer:
                        rec.update(timer.record())  # stages reached before the failure
                    traceback.print_exc()
                    torch.cuda.empty_cache()
                inputs = out_ids = audios = images = videos = None
                trim_memory()
                rec["rss_gb"] = rss_gb()
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out.flush(); os.fsync(out.fileno())
                if n_done % 10 == 0 or n_done == len(todo):
                    print(f"[shard{shard}] {method}: {n_done}/{len(todo)} rss={rec['rss_gb']:.1f}GB", flush=True)
                if rec["rss_gb"] > cfg.get("max_rss_gb", float("inf")):
                    print(f"[shard{shard}] rss {rec['rss_gb']:.1f}GB > {cfg['max_rss_gb']}GB, exiting for a restart", flush=True)
                    sys.exit(EXIT_HIGH_RSS)

        # Free this model before the next method loads one: `th`/`mod` (profiling hooks), the last `ids`, and the cached
        # AV_Aligner (it shares layer-0 modules) would otherwise keep ~7 GB of it on the GPU
        model = th = mod = ids = None
        if mcfg["wrapper"] == "omnizip":
            oc._AV_ALIGNERS.clear()
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[shard{shard}] {method}: done, GPU still allocated {torch.cuda.memory_allocated() / 2**30:.2f} GB",
              flush=True)


# ----------------------------------------------------------------------------- summary helpers

def load_records(result_roots, run_tag):
    """All records of run_tag, one per (method, qid): first non-error record wins."""
    best = {}
    for root in result_roots:
        for p in sorted(glob.glob(os.path.join(root, "**", f"{run_tag}__*__shard*.jsonl"), recursive=True)):
            with open(p, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    key = (r["method"], qid(r))
                    if key not in best or (best[key]["error"] and not r["error"]):
                        best[key] = r
    return list(best.values())


def mcnemar_exact_p(b, c):
    """Two-sided exact McNemar p-value from the discordant counts b, c."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def paired_compare(recs, base, other, n_boot=2000, seed=0):
    """Accuracy difference other - base on the questions both methods scored, with McNemar p and bootstrap CI."""
    a = {qid(r): bool(r["correct"]) for r in recs if r["method"] == base}
    o = {qid(r): bool(r["correct"]) for r in recs if r["method"] == other}
    vids = sorted(a.keys() & o.keys())
    if not vids:
        return None
    x = [a[v] for v in vids]; y = [o[v] for v in vids]
    b = sum(1 for p, q in zip(x, y) if p and not q)   # base right, other wrong
    c = sum(1 for p, q in zip(x, y) if q and not p)   # other right, base wrong
    diff = (sum(y) - sum(x)) / len(vids)
    rng = random.Random(seed)
    n = len(vids)
    boots = sorted(sum(y[j] - x[j] for j in (rng.randrange(n) for _ in range(n))) / n for _ in range(n_boot))
    return {"base": base, "other": other, "n_paired": n, "acc_base": sum(x) / n, "acc_other": sum(y) / n,
            "diff": diff, "ci95_low": boots[int(0.025 * n_boot)], "ci95_high": boots[int(0.975 * n_boot) - 1],
            "base_only_correct": b, "other_only_correct": c, "mcnemar_p": mcnemar_exact_p(b, c),
            "answer_agreement": sum(1 for v in vids if a[v] == o[v]) / n}


def summarize(recs):
    """Per-method accuracy and efficiency rows (plain dicts, so the notebook can build DataFrames)."""
    rows, by_type = [], []
    for m in sorted({r["method"] for r in recs}):
        rs = [r for r in recs if r["method"] == m]
        ok = [r for r in rs if not r["error"]]
        mean = lambda k: sum(r[k] for r in ok) / len(ok) if ok else float("nan")
        gen = sorted(r["gen_seconds"] for r in ok)
        rows.append({"method": m, "n": len(rs), "n_errors": len(rs) - len(ok),
                     "accuracy": sum(r["correct"] for r in rs) / len(rs),
                     "accuracy_no_errors": sum(r["correct"] for r in ok) / len(ok) if ok else float("nan"),
                     "retained_av_ratio": mean("retained_av_ratio"), "n_input_tokens": mean("n_input_tokens"),
                     "gen_seconds_mean": mean("gen_seconds"), "gen_seconds_median": gen[len(gen) // 2] if gen else float("nan"),
                     "peak_mem_gb_mean": mean("peak_mem_gb"), "no_audio_videos": sum(1 for r in ok if not r["has_audio"])})
        for t in sorted({r["problem_type"] for r in rs}):
            tr = [r for r in rs if r["problem_type"] == t]
            by_type.append({"method": m, "problem_type": t, "n": len(tr), "accuracy": sum(r["correct"] for r in tr) / len(tr)})
    return rows, by_type


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--num-shards", type=int, default=1)
    ap.add_argument("--methods", nargs="*", default=None)
    a = ap.parse_args()
    with open(a.config, encoding="utf-8") as f:
        cfg = json.load(f)
    run_worker(cfg, a.shard, a.num_shards, a.methods or list(cfg["methods"]))


In [ ]:
def write_config(path):
    cfg = {"model_id": MODEL_ID, "methods": METHODS, "run_tag": RUN_TAG, "results_dir": RESULTS_DIR,
           "result_roots": [RESULTS_DIR], "code_root": CODE_ROOT, "data_json": data_json,
           "video_dir": f"{DATA_DIR}/videos", "mini_test_num": None, "videos": RUN_VIDEOS, "viz_videos": VIZ_VIDEOS,
           "fix_lm_head": FIX_LM_HEAD, "fix_audio_sdpa": FIX_AUDIO_SDPA, "profile": PROFILE,
           "max_frames": MAX_FRAMES, "fps": FPS, "video_max_pixels": VIDEO_MAX_PIXELS, "max_new_tokens": MAX_NEW_TOKENS,
           "dtype": DTYPE, "attn_implementation": ATTN_IMPLEMENTATION, "seed": SEED, "video_reader": "decord",
           "max_rss_gb": round(RAM_GB * 0.75 / NUM_SHARDS, 1),
           "system_prompt": ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
                             "perceiving auditory and visual inputs, as well as generating text and speech. Please "
                             "analyze the video carefully and select the most appropriate answer from the given options.")}
    json.dump(cfg, open(path, "w"), indent=1)
    return path

def launch(config_path, shard, log_path, gpus):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", MALLOC_ARENA_MAX="2", PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, f"{WORK}/svb_eval.py", "--config", config_path,
                             "--shard", str(shard), "--num-shards", str(NUM_SHARDS)],
                            stdout=open(log_path, "a"), stderr=subprocess.STDOUT, env=env)

def tail(path, n=40):
    return "".join(open(path, encoding="utf-8", errors="replace").readlines()[-n:])

RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
sys.path.insert(0, WORK)
import importlib, svb_eval; importlib.reload(svb_eval)

## 5. Chạy (mỗi GPU một shard; tự khởi động lại khi worker thoát vì RAM)

In [ ]:
SNAPSHOT_ZIP = f"{WORK}/svb_results_fix_rerun.zip"

def snapshot_and_zip():
    shutil.copytree(LOG_DIR, f"{RESULTS_DIR}/logs_fix_rerun", dirs_exist_ok=True)
    shutil.make_archive(SNAPSHOT_ZIP[:-4] + ".tmp", "zip", root_dir=WORK, base_dir="results")
    os.replace(SNAPSHOT_ZIP[:-4] + ".tmp.zip", SNAPSHOT_ZIP)

subprocess.run(["pkill", "-f", "svb_eval.py"], check=False); time.sleep(3)
cfg_path = write_config(f"{WORK}/run_config.json")
procs = {k: launch(cfg_path, k, f"{LOG_DIR}/shard{k}.log", str(k)) for k in range(NUM_SHARDS)}
restarts = {k: 0 for k in procs}
t0 = time.time()
while procs:
    time.sleep(30)
    recs = svb_eval.load_records([RESULTS_DIR], RUN_TAG)
    print(f"[{(time.time() - t0) / 60:.1f} min]", {m: sum(r["method"] == m for r in recs) for m in METHODS},
          f"/ {len(RUN_VIDEOS)}", flush=True)
    for k, p in list(procs.items()):
        rc = p.poll()
        if rc is None:
            continue
        del procs[k]
        print(f"shard{k}: exit {rc}")
        if rc in (svb_eval.EXIT_HIGH_RSS, -9) and restarts[k] < 10:
            restarts[k] += 1; procs[k] = launch(cfg_path, k, f"{LOG_DIR}/shard{k}.log", str(k))
        elif rc != 0:
            print(tail(f"{LOG_DIR}/shard{k}.log"))
snapshot_and_zip()
print(f"xong sau {(time.time() - t0) / 60:.1f} phút")

## 6. Video lỗi lần trước: giờ thế nào?

Mỗi ô: trạng thái lần trước → lần này (`OOM` = lỗi, chữ cái = đáp án, ✓/✗ = đúng/sai).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200)
prev = pd.DataFrame(json.load(open(PREV_JSON, encoding="utf-8")))
new = pd.DataFrame(svb_eval.load_records([RESULTS_DIR], RUN_TAG))
new["set"] = np.where(new["video"].isin(FAILED_VIDEOS), "lỗi lần trước", "đối chứng")
pv = prev.set_index(["method", "video"]); nv = new.set_index(["method", "video"])

def cell(r):
    if r is None:
        return "–"
    return "OOM" if pd.notna(r["error"]) else f"{r['pred']}{'✓' if r['correct'] else '✗'}"

rows = []
for v in FAILED_VIDEOS:
    row = {"video": v, "n_input_tokens": int(np.nan_to_num(pv.loc[("full", v), "n_input_tokens"])),
           "n_audio_tokens": int(np.nan_to_num(pv.loc[("full", v), "n_audio_tokens"]))}
    for m in METHODS:
        a = pv.loc[(m, v)] if (m, v) in pv.index else None
        b = nv.loc[(m, v)] if (m, v) in nv.index else None
        row[m] = f"{cell(a)} → {cell(b)}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("video"))

for m in METHODS:
    was = [v for v in FAILED_VIDEOS if (m, v) in pv.index and pd.notna(pv.loc[(m, v), "error"])]
    now = nv.loc[[(m, v) for v in was if (m, v) in nv.index]] if was else nv.iloc[:0]
    print(f"{m:12s}: lỗi lần trước {len(was):2d} → chạy được {int(now['error'].isna().sum()):2d}, đúng {int(now['correct'].sum()):2d}")
err = new[new["error"].notna()]
if len(err):
    print("\nVẪN LỖI:")
    for _, r in err.iterrows():
        print(f"--- {r['method']} {r['video']}: {r['error']}\n{(r.get('traceback') or '').strip().splitlines()[-1:]}")

## 7. Kiểm tra bản sửa không đổi đáp án (nhóm đối chứng)

Video đối chứng đã chạy được lần trước. Sửa `lm_head` không đổi phép tính đáp án; sửa audio SDPA cùng toán học nhưng
sai số fp16 khác đi một chút, nên có thể lệch vài câu.

In [ ]:
ctl = new[new["video"].isin(CONTROL_VIDEOS) & new["error"].isna()].merge(
    prev[["method", "video", "pred", "correct", "gen_seconds", "peak_mem_gb"]], on=["method", "video"], suffixes=("", "_prev"))
agree = ctl.groupby("method").apply(lambda d: pd.Series({
    "n": len(d), "cùng đáp án": (d["pred"] == d["pred_prev"]).mean(),
    "acc trước": d["correct_prev"].mean(), "acc sau": d["correct"].mean(),
    "gen_s trước (median)": d["gen_seconds_prev"].median(), "gen_s sau (median)": d["gen_seconds"].median(),
    "VRAM đỉnh trước (median)": d["peak_mem_gb_prev"].median(), "VRAM đỉnh sau (median)": d["peak_mem_gb"].median()}))
display(agree.round(3))
diff = ctl[ctl["pred"] != ctl["pred_prev"]]
if len(diff):
    display(diff[["method", "video", "pred_prev", "pred", "answer"]])

## 8. Thời gian từng giai đoạn

`t_other` = phần còn lại của `generate()` ngoài các giai đoạn GPU đã đo (ghép embedding, tính vị trí M-RoPE, overhead
của generate). Ba giai đoạn đầu (đọc video, processor, chép lên GPU) chạy trên CPU, trước `generate()`.

In [ ]:
CPU = ["t_read_decode", "t_processor", "t_to_gpu"]
GPU = ["t_vision", "t_audio", "t_omnizip", "t_llm_prefill", "t_lm_head", "t_llm_decode"]
ok = new[new["error"].isna()].copy()
for c in CPU + GPU:
    ok[c] = ok[c].fillna(0.0) if c in ok else 0.0
ok["t_other"] = (ok["gen_seconds"] - ok[GPU].sum(axis=1)).clip(lower=0)
STAGES = CPU + GPU + ["t_other"]
colors = plt.cm.tab10(np.linspace(0, 1, 10))
mlist = list(METHODS)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=False)
for ax, (name, d) in zip(axes, ok.groupby("set")):
    mean = d.groupby("method")[STAGES].mean().reindex(mlist)
    bottom = np.zeros(len(mean))
    for c, col in zip(STAGES, colors):
        ax.bar(mean.index, mean[c], bottom=bottom, label=c[2:], color=col); bottom += mean[c].values
    for x, tot in enumerate(bottom):
        ax.text(x, tot, f"{tot:.1f}s", ha="center", va="bottom")
    ax.set_title(f"{name}: thời gian trung bình mỗi mẫu (n={d['video'].nunique()})"); ax.set_ylabel("giây")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

display(ok.groupby(["set", "method"])[STAGES + ["ttft_seconds", "gen_seconds"]].median().round(3))

In [ ]:
# Từng video lỗi lần trước, từng method
d = ok[ok["set"] == "lỗi lần trước"]
fig, axes = plt.subplots(1, len(mlist), figsize=(6 * len(mlist), 0.35 * len(FAILED_VIDEOS) + 1.5), sharey=True, sharex=True)
for ax, m in zip(np.atleast_1d(axes), mlist):
    dm = d[d["method"] == m].set_index("video").reindex(FAILED_VIDEOS)
    left = np.zeros(len(dm))
    for c, col in zip(STAGES, colors):
        vals = dm[c].fillna(0).values
        ax.barh(range(len(dm)), vals, left=left, color=col, label=c[2:]); left += vals
    ax.set_yticks(range(len(dm))); ax.set_yticklabels([v[:11] for v in dm.index], fontsize=8); ax.invert_yaxis()
    ax.set_title(m); ax.set_xlabel("giây")
np.atleast_1d(axes)[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.suptitle("Video lỗi lần trước: thời gian theo giai đoạn (ô trống = vẫn lỗi)"); plt.tight_layout(); plt.show()

## 9. Thời gian theo số token, và tốc độ so với `full` (ghép theo từng video)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for m, col in zip(mlist, ["tab:gray", "tab:blue", "tab:orange"]):
    dm = ok[ok["method"] == m]
    axes[0].scatter(dm["n_llm_tokens"], dm["t_llm_prefill"], s=18, label=m, color=col)
    axes[1].scatter(dm["n_video_tokens"], dm["t_vision"], s=18, label=m, color=col)
    axes[2].scatter(dm["n_audio_tokens"], dm["t_audio"], s=18, label=m, color=col)
for ax, (x, y) in zip(axes, [("số token vào LLM", "LLM prefill (s)"), ("số video token", "vision encoder (s)"),
                              ("số audio token", "audio encoder (s)")]):
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

base = ok[ok["method"] == BASELINE].set_index("video")
rows = []
for m in mlist:
    if m == BASELINE:
        continue
    dm = ok[ok["method"] == m].set_index("video")
    v = dm.index.intersection(base.index)
    r = {"method": m, "n_video": len(v), "token vào LLM / full": (dm.loc[v, "n_llm_tokens"] / base.loc[v, "n_llm_tokens"]).median()}
    for c in ["t_llm_prefill", "t_llm_decode", "ttft_seconds", "gen_seconds"]:
        r[f"{c} / full"] = (dm.loc[v, c] / base.loc[v, c]).median()
    r["chi phí OmniZip (s, median)"] = dm.loc[v, "t_omnizip"].median()
    r["VRAM đỉnh − full (GB, median)"] = (dm.loc[v, "peak_mem_gb"] - base.loc[v, "peak_mem_gb"]).median()
    rows.append(r)
display(pd.DataFrame(rows).set_index("method").round(3))

## 10. VRAM đỉnh

In [ ]:
PEAKS = [c for c in ["peak_vision_gb", "peak_audio_gb", "peak_omnizip_gb", "peak_llm_prefill_gb", "peak_lm_head_gb",
                     "peak_llm_decode_gb"] if c in ok]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
med = ok.groupby("method")[PEAKS].median().reindex(mlist)
med.columns = [c[5:-3] for c in PEAKS]
med.T.plot.bar(ax=axes[0]); axes[0].set_ylabel("GB (median)"); axes[0].set_title("VRAM đỉnh theo giai đoạn")
axes[0].axhline(GPU_GB, color="red", ls="--", lw=1, label="VRAM T4")
for m, col in zip(mlist, ["tab:gray", "tab:blue", "tab:orange"]):
    dm = new[new["method"] == m]
    e = dm["error"].notna()
    axes[1].scatter(dm.loc[~e, "n_input_tokens"], dm.loc[~e, "peak_mem_gb"], s=18, color=col, label=m)
    axes[1].scatter(dm.loc[e, "n_input_tokens"], dm.loc[e, "peak_mem_gb"], s=50, color=col, marker="x",
                    label=f"{m} (vẫn lỗi)" if e.any() else None)
axes[1].axhline(GPU_GB, color="red", ls="--", lw=1); axes[1].set_xlabel("số token đầu vào (trước nén)")
axes[1].set_ylabel("VRAM đỉnh cả mẫu (GB)"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 11. Token video được giữ (vùng tối = bị bỏ)

In [ ]:
for v in VIZ_VIDEOS:
    files = {m: f"{RESULTS_DIR}/viz/{m}__{v}.npz" for m in mlist}
    files = {m: f for m, f in files.items() if os.path.exists(f)}
    if not files:
        print("không có ảnh cho", v); continue
    fig, axes = plt.subplots(len(files), 4, figsize=(16, 3.2 * len(files)), squeeze=False)
    for i, (m, f) in enumerate(files.items()):
        z = np.load(f)
        for j in range(4):
            img = z["frames"][j].astype(float) / 255
            mask = np.kron(z["keep"][j], np.ones((28, 28)))[: img.shape[0], : img.shape[1]]
            axes[i, j].imshow(img * (0.25 + 0.75 * mask[..., None]))
            axes[i, j].set_title(f"{m} · t={int(z['tp'][j])} · giữ {z['keep'][j].mean():.0%}", fontsize=9)
            axes[i, j].axis("off")
    plt.suptitle(v); plt.tight_layout(); plt.show()

## 12. Accuracy 500 câu sau khi thay kết quả các video chạy lại

Bản ghi của các video chạy lại (lỗi + đối chứng) thay cho bản ghi cũ; 500 − số video chạy lại giữ kết quả cũ. Accuracy
lúc này không còn thiên vị do OOM của `full`. Thời gian trong bảng trộn lần cũ và lần mới, chỉ đọc cột accuracy.

In [ ]:
import itertools
cols = [c for c in prev.columns]
key_new = set(zip(new["method"], new["video"]))
merged = pd.concat([prev[[k not in key_new for k in zip(prev["method"], prev["video"])]],
                    new[[c for c in cols if c in new]]], ignore_index=True)
mrecs = merged.replace({np.nan: None}).to_dict("records")
for r in mrecs:
    r["correct"] = bool(r["correct"])
rows, by_type = svb_eval.summarize([r for r in mrecs if r["error"] or r["gen_seconds"] is not None])
display(pd.DataFrame(rows).set_index("method")[["n", "n_errors", "accuracy", "accuracy_no_errors", "retained_av_ratio"]].round(4))
paired = pd.DataFrame([svb_eval.paired_compare(mrecs, a, b) for a, b in itertools.combinations(mlist, 2)])
display(paired.round(4))

fig, ax = plt.subplots(figsize=(7, 3.2))
for i, r in paired.iterrows():
    ax.errorbar(r["diff"] * 100, i, xerr=[[(r["diff"] - r["ci95_low"]) * 100], [(r["ci95_high"] - r["diff"]) * 100]],
                fmt="o", capsize=4)
ax.axvline(0, color="gray", lw=1); ax.set_yticks(range(len(paired)))
ax.set_yticklabels([f"{a} → {b}" for a, b in zip(paired["base"], paired["other"])])
ax.set_xlabel("chênh accuracy (điểm %), KTC 95% bootstrap"); ax.set_title("So ghép cặp, 500 câu")
plt.tight_layout(); plt.show()

## 13. Lưu

In [ ]:
new.to_csv(f"{RESULTS_DIR}/{RUN_TAG}__records.csv", index=False)
ok[["method", "video", "set"] + STAGES + ["ttft_seconds", "gen_seconds", "n_llm_tokens"] + PEAKS].to_csv(
    f"{RESULTS_DIR}/{RUN_TAG}__stages.csv", index=False)
merged.to_csv(f"{RESULTS_DIR}/{RUN_TAG}__merged500_records.csv", index=False)
paired.to_csv(f"{RESULTS_DIR}/{RUN_TAG}__merged500_paired.csv", index=False)
snapshot_and_zip()
print("tải về:", SNAPSHOT_ZIP)